# TFM — Agentes de IA generativa y productivización

## Sistema explicable de detección preventiva de indicadores de riesgo en salud mental adulta mediante aprendizaje automático multisalida y agentes de inteligencia artificial generativa a partir del dataset NSDUH 2024

**Autor:** Carlos Ayllón Motto  
**Máster:** Big Data, Data Science e Inteligencia Artificial  
**Dataset:** National Survey on Drug Use and Health 2024  
**Fuente:** SAMHSA  
**Población de análisis:** personas adultas de 18 años o más

---

Este notebook desarrolla la fase final de agentes de inteligencia artificial generativa y productivización a partir de la solución predictiva y explicable cerrada en los notebooks anteriores. Los modelos, variables, preprocesamientos, pesos del ensemble, umbrales y resultados de evaluación se consideran definitivamente fijados y se utilizarán exclusivamente en modo de inferencia.

La solución integrará una entrada estructurada validada, el acceso controlado al entorno predictivo `TFM_ML`, las probabilidades y clasificaciones de las cuatro variables objetivo y, cuando se solicite, sus explicaciones SHAP. Sobre esta salida se construirá una capa de agentes orientada a organizar, interpretar y comunicar los resultados mediante herramientas deterministas, un flujo controlado con LangGraph, recuperación documental controlada y generación de lenguaje natural.

El objetivo final es disponer de un prototipo funcional de extremo a extremo capaz de recibir nuevos registros compatibles con el esquema definido, devolver predicciones explicables y generar un informe preventivo, prudente y trazable. La capa generativa no recalculará resultados predictivos, no modificará umbrales o clasificaciones y no presentará las salidas como diagnósticos clínicos.

## Esquema general del sistema

El Notebook 05 completa la solución predictiva cerrada mediante una capa independiente de agentes y productivización. El flujo general mantiene separadas la inferencia, la explicabilidad, la recuperación documental y la generación de lenguaje natural:

<div style="max-width:900px;margin:20px auto;font-family:Arial,sans-serif;text-align:center;">

<div style="background:#176b70;color:white;padding:12px 18px;border-radius:10px;font-weight:bold;">
NUEVO REGISTRO NSDUH · 813 VARIABLES
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#dcf7f4;border:2px solid #81d8d0;padding:10px;border-radius:10px;">
<strong>Validación Pydantic</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#dcf7f4;border:2px solid #81d8d0;padding:10px;border-radius:10px;">
<strong>Flujo LangGraph</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="display:flex;gap:15px;justify-content:center;align-items:stretch;">
  <div style="flex:2;background:#f4fcfb;border:2px solid #2b9f99;padding:12px;border-radius:10px;">
    <strong>Tool predictiva local</strong><br>
    <span style="font-size:13px;">Mecanismo principal</span>
  </div>
  <div style="flex:1;background:#f7f7f7;border:1px dashed #52656a;padding:12px;border-radius:10px;">
    <strong>MCP</strong><br>
    <span style="font-size:13px;">Alternativa interoperable validada</span>
  </div>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#f4fcfb;border:2px solid #2b9f99;padding:10px;border-radius:10px;">
<strong>Bridge JSON</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#0e4f55;color:white;padding:16px;border-radius:12px;">
<strong>TFM_ML · SOLO LECTURA</strong><br>
<span style="font-size:13px;">ML / DL finales · SHAP opcional</span>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#dcf7f4;padding:10px;border-radius:10px;">
<strong>4 predicciones</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="display:flex;gap:15px;justify-content:center;">
  <div style="flex:1;background:white;border:1px solid #81d8d0;padding:10px;border-radius:10px;">
  SHAP solicitado
  </div>
  <div style="flex:1;background:white;border:1px solid #81d8d0;padding:10px;border-radius:10px;">
  Sin SHAP
  </div>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#f4fcfb;border:2px solid #81d8d0;padding:10px;border-radius:10px;">
<strong>RAG controlado</strong><br>
<span style="font-size:13px;">NSDUH 2024 Codebook + limitaciones</span>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#dcf7f4;padding:10px;border-radius:10px;">
<strong>Contexto estructurado</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#f4fcfb;border:2px solid #2b9f99;padding:10px;border-radius:10px;">
<strong>Mistral</strong><br>
<span style="font-size:13px;">Resumen · interpretación · traducciones</span>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#dcf7f4;padding:10px;border-radius:10px;">
<strong>SalidaGenerativa</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#f4fcfb;border:2px solid #2b9f99;padding:10px;border-radius:10px;">
<strong>Reconstrucción determinista</strong><br>
InformePreventivo
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#f4fcfb;border:2px solid #176b70;padding:10px;border-radius:10px;">
<strong>Guardrails deterministas</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#dcf7f4;padding:10px;border-radius:10px;">
<strong>Flask + Plotly</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#176b70;color:white;padding:13px 18px;border-radius:10px;font-weight:bold;">
INFORME PREVENTIVO FINAL
</div>

</div>

# 0. Preparación y configuración

Esta sección establece la configuración común utilizada por el Notebook 05 antes de iniciar la recuperación de la transferencia predictiva y la ejecución de los distintos componentes del sistema.

Se comprueba el entorno `TFM_Agentes`, se fijan rutas, semillas y parámetros comunes, y se centralizan los diccionarios de presentación. También se declaran de forma previa la configuración del modelo generativo, el prompt de sistema y las plantillas HTML empleadas posteriormente por la aplicación Flask.

Finalmente, se definen las funciones auxiliares reutilizadas a lo largo del notebook. Esta organización permite separar las decisiones de configuración y presentación de la lógica de ejecución posterior, manteniendo una única fuente para cada elemento común.

## 0.1. Importación de librerías, versiones y dispositivo

In [1]:
# =============================================================================
# IMPORTACIÓN DE LIBRERÍAS, VERSIONES Y DISPOSITIVO
# =============================================================================
print('\nIMPORTACIÓN DE LIBRERÍAS', flush=True)

import hmac
import importlib.util
import json
import os
import platform
import re
import subprocess
import sys
import threading

from datetime import datetime
from html import escape
from importlib.metadata import version
from io import BytesIO
from pathlib import Path
from time import perf_counter, sleep
from typing import Any, Literal, TypedDict
from zipfile import ZipFile
from IPython.display import HTML, display

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import requests
import torch

print('Base científica y sistema... OK', flush=True)

from dotenv import load_dotenv
from flask import Flask, jsonify, redirect, render_template_string, request, session, url_for

from werkzeug.serving import make_server

from pypdf import PdfReader
from pydantic import BaseModel, Field, ValidationError

print('Aplicación y contratos... OK', flush=True)

from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mistralai import ChatMistralAI
from langgraph.graph import END, START, StateGraph
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.settings import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print('Agentes, MCP y RAG... OK', flush=True)

# -----------------------------------------------------------------------------
# VERSIONES
# -----------------------------------------------------------------------------
versiones_esperadas = {
    'Python': '3.11.15',
    'NumPy': '2.4.6',
    'pandas': '3.0.5',
    'requests': '2.34.2',
    'Pydantic': '2.13.4',
    'python-dotenv': '1.2.3',
    'Flask': '3.1.3',
    'Plotly': '6.9.0',
    'pypdf': '6.14.2',
    'ipykernel': '7.3.0',
    'PyTorch': '2.13.0',
    'sentence-transformers': '5.7.0',
    'transformers': '5.15.1',
    'huggingface-hub': '1.28.0',
    'LangChain': '1.3.16',
    'langchain-core': '1.6.0',
    'LangGraph': '1.2.11',
    'langchain-mistralai': '1.1.6',
    'MCP': '1.29.1',
    'langchain-mcp-adapters': '0.3.2',
    'llama-index-core': '0.14.23',
    'llama-index-embeddings-huggingface': '0.7.0'
}

paquetes_version = {
    'NumPy': 'numpy',
    'pandas': 'pandas',
    'requests': 'requests',
    'Pydantic': 'pydantic',
    'python-dotenv': 'python-dotenv',
    'Flask': 'flask',
    'Plotly': 'plotly',
    'pypdf': 'pypdf',
    'ipykernel': 'ipykernel',
    'PyTorch': 'torch',
    'sentence-transformers': 'sentence-transformers',
    'transformers': 'transformers',
    'huggingface-hub': 'huggingface-hub',
    'LangChain': 'langchain',
    'langchain-core': 'langchain-core',
    'LangGraph': 'langgraph',
    'langchain-mistralai': 'langchain-mistralai',
    'MCP': 'mcp',
    'langchain-mcp-adapters': 'langchain-mcp-adapters',
    'llama-index-core': 'llama-index-core',
    'llama-index-embeddings-huggingface': 'llama-index-embeddings-huggingface'
}

versiones_detectadas = {
    'Python': platform.python_version(),
    **{nombre: version(paquete) for nombre, paquete in paquetes_version.items()}
}

tabla_versiones = pd.DataFrame({
    'biblioteca': list(versiones_esperadas),
    'esperada': list(versiones_esperadas.values()),
    'detectada': [versiones_detectadas[nombre] for nombre in versiones_esperadas]
})

tabla_versiones['estado'] = np.where(
    tabla_versiones['esperada'].eq(tabla_versiones['detectada']), 'OK', 'REVISAR'
)

print('\nVERSIONES DEL ENTORNO')
display(tabla_versiones)

# -----------------------------------------------------------------------------
# ENTORNO Y DISPOSITIVO
# -----------------------------------------------------------------------------
nombre_entorno = Path(sys.prefix).name
entorno_inicio = os.environ.get('CONDA_DEFAULT_ENV', 'No disponible')
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
nombre_dispositivo = torch.cuda.get_device_name(0) if dispositivo == 'cuda' else 'CPU'

tensor_prueba = torch.tensor([1.0, 2.0], device=dispositivo) * 2
operacion_torch = np.allclose(tensor_prueba.cpu().numpy(), [2.0, 4.0])

print(f'\nEntorno del kernel: {nombre_entorno}')
print(f'Entorno de inicio de Jupyter: {entorno_inicio}')
print(f'Ejecutable: {sys.executable}')
print(f'Dispositivo seleccionado: {dispositivo}')
print(f'Dispositivo disponible: {nombre_dispositivo}')

# -----------------------------------------------------------------------------
# AISLAMIENTO RESPECTO A TFM_ML
# -----------------------------------------------------------------------------
dependencias_aisladas = {
    'TensorFlow': importlib.util.find_spec('tensorflow') is None,
    'XGBoost': importlib.util.find_spec('xgboost') is None,
    'SHAP': importlib.util.find_spec('shap') is None
}

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_entorno = pd.DataFrame({
    'comprobacion': [
        'El kernel pertenece al entorno TFM_Agentes',
        'Las versiones fijadas coinciden',
        'PyTorch ejecuta una operación correctamente',
        'TensorFlow no está instalado en TFM_Agentes',
        'XGBoost no está instalado en TFM_Agentes',
        'SHAP no está instalado en TFM_Agentes'
    ],
    'resultado': [
        nombre_entorno == 'TFM_Agentes',
        tabla_versiones['estado'].eq('OK').all(),
        operacion_torch,
        dependencias_aisladas['TensorFlow'],
        dependencias_aisladas['XGBoost'],
        dependencias_aisladas['SHAP']
    ]
})

print('\nCOMPROBACIONES DEL ENTORNO')
display(comprobaciones_entorno)

if not comprobaciones_entorno['resultado'].all():
    raise RuntimeError('La validación inicial del entorno no es correcta.')


IMPORTACIÓN DE LIBRERÍAS


Base científica y sistema... OK


Aplicación y contratos... OK


/home/cam/miniconda3/envs/TFM_Agentes/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Agentes, MCP y RAG... OK



VERSIONES DEL ENTORNO


,biblioteca,esperada,detectada,estado
0,Python,3.11.15,3.11.15,OK
1,NumPy,2.4.6,2.4.6,OK
2,pandas,3.0.5,3.0.5,OK
3,requests,2.34.2,2.34.2,OK
4,Pydantic,2.13.4,2.13.4,OK
5,python-dotenv,1.2.3,1.2.3,OK
6,Flask,3.1.3,3.1.3,OK
7,Plotly,6.9.0,6.9.0,OK
8,pypdf,6.14.2,6.14.2,OK
9,ipykernel,7.3.0,7.3.0,OK



Entorno del kernel: TFM_Agentes
Entorno de inicio de Jupyter: base
Ejecutable: /home/cam/miniconda3/envs/TFM_Agentes/bin/python
Dispositivo seleccionado: cuda
Dispositivo disponible: NVIDIA RTX 1000 Ada Generation Laptop GPU

COMPROBACIONES DEL ENTORNO


,comprobacion,resultado
0,El kernel pertenece al entorno TFM_Agentes,True
1,Las versiones fijadas coinciden,True
2,PyTorch ejecuta una operación correctamente,True
3,TensorFlow no está instalado en TFM_Agentes,True
4,XGBoost no está instalado en TFM_Agentes,True
5,SHAP no está instalado en TFM_Agentes,True


### Resultados

El notebook se ejecuta con el kernel `Python (TFM_Agentes)` desde el entorno previsto, mientras JupyterLab permanece iniciado desde `base`. Todas las versiones comprobadas coinciden con las fijadas durante la validación del entorno, incluyendo MCP y su adaptador para LangChain.

PyTorch detecta correctamente la GPU NVIDIA RTX 1000 Ada Generation Laptop GPU y selecciona `cuda` como dispositivo de ejecución. La selección se realiza de forma automática, manteniendo `cpu` como alternativa cuando CUDA no esté disponible.

También se confirma el aislamiento respecto al entorno predictivo: TensorFlow, XGBoost y SHAP no están instalados en `TFM_Agentes`, por lo que la separación respecto a `TFM_ML` se mantiene correctamente.

## 0.2. Configuración general y rutas

Se mantiene la semilla `12345` utilizada durante el proyecto y se centralizan las rutas necesarias para recuperar la transferencia cerrada en el Notebook 04, comprobar el entorno de agentes y almacenar las salidas propias de esta última fase.

El Notebook 05 no cargará directamente los modelos de Machine Learning, Deep Learning ni SHAP. Estos permanecerán en el entorno congelado `TFM_ML` y se consumirán posteriormente mediante una interfaz controlada. En esta primera configuración únicamente se comprueba la disponibilidad de los contratos y archivos transferidos.

Las credenciales externas se recuperan desde `.env` sin mostrar sus valores. Se mantienen las comprobaciones de seguridad necesarias para confirmar que el archivo continúa excluido de Git y no está versionado.

El smoke test integral de `TFM_Agentes`, ya validado antes de iniciar el Notebook 05, no se repite en cada reinicio del kernel debido a que comprueba capacidades de mayor coste como embeddings, RAG, LangGraph y la comunicación entre entornos. Su archivo se mantiene disponible y la prueba completa se ejecutará nuevamente durante el cierre definitivo del entorno.

In [2]:
# =============================================================================
# CONFIGURACIÓN GENERAL Y RUTAS
# =============================================================================
print('\nCONFIGURACIÓN GENERAL Y RUTAS', flush=True)

semilla = 12345
np.random.seed(semilla)
torch.manual_seed(semilla)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(semilla)

encoding_csv = 'utf-8-sig'
fecha_ejecucion = datetime.now().astimezone().isoformat(timespec='seconds')

modelo_embeddings = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
dimension_embeddings = 384
dispositivo_embeddings = 'cpu'

ruta_proyecto = Path.home() / 'BD' / 'TFM_NSDUH'
ruta_entornos = ruta_proyecto / 'environments'
ruta_tests = ruta_proyecto / 'tests'
ruta_transferencia = ruta_proyecto / 'results' / 'tables' / '04_evaluacion_explicabilidad'

ruta_referencias_nsduh = ruta_proyecto / 'references' / 'NSDUH_2024'
ruta_zip_documentacion_nsduh = ruta_referencias_nsduh / '2024-nsduh-archivos.zip'
ruta_codebook_nsduh = ruta_referencias_nsduh / 'nsduh-2024-ds0001-info-codebook_v1.pdf'

ruta_tablas = ruta_proyecto / 'results' / 'tables' / '05_agentes_productivizacion'
ruta_figuras = ruta_proyecto / 'results' / 'figures' / '05_agentes_productivizacion'
ruta_informes = ruta_proyecto / 'results' / 'agent_reports'
ruta_logs = ruta_proyecto / 'logs'

ruta_productivizacion = ruta_proyecto / 'src' / 'productivizacion'
ruta_servicio_predictivo = ruta_productivizacion / 'servicio_predictivo.py'
ruta_servidor_mcp = ruta_productivizacion / 'servidor_mcp_predictivo.py'
ruta_base_premodelado = (
    ruta_proyecto / 'data' / 'interim' / 'NSDUH_2024_adultos_premodelado.parquet'
)

ruta_productivizacion.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# ENTORNO
# -----------------------------------------------------------------------------
ruta_environment = ruta_entornos / 'TFM_Agentes_environment.yml'
ruta_reproduccion = ruta_entornos / 'TFM_Agentes_REPRODUCCION.md'
ruta_smoke_test = ruta_tests / 'smoke_test_agents_environment.py'
ruta_env = ruta_proyecto / '.env'

archivos_entorno = {
    'Configuración del entorno': ruta_environment,
    'Reproducción del entorno': ruta_reproduccion,
    'Smoke test': ruta_smoke_test,
    'Archivo de credenciales': ruta_env
}

# -----------------------------------------------------------------------------
# TRANSFERENCIA DEL NOTEBOOK 04
# -----------------------------------------------------------------------------
archivos_transferencia = {
    'Configuración de productivización':
        ruta_transferencia / 'configuracion_productivizacion_v1.joblib',
    'Referencia SHAP': ruta_transferencia / 'background_shap_productivizacion_v1.joblib',
    'Selección final por objetivo': ruta_transferencia / '53_seleccion_final_por_objetivo.csv',
    'Predicciones finales de TEST': ruta_transferencia / '54_predicciones_finales_test.csv',
    'Variables principales SHAP': ruta_transferencia / '69_top_importancia_global_shap.csv',
    'Coincidencias de variables SHAP': ruta_transferencia / '73_coincidencias_variables_shap.csv',
    'Esquema de entrada': ruta_transferencia / '75_esquema_entrada_productivizacion.csv',
    'Esquema de salida': ruta_transferencia / '76_esquema_salida_productivizacion.csv',
    'Limitaciones': ruta_transferencia / '77_limitaciones_modelo_final.csv',
    'Archivos transferidos': ruta_transferencia / '78_archivos_transferencia.csv',
    'Comprobaciones de transferencia': ruta_transferencia / '79_comprobaciones_transferencia.csv',
    'Comprobaciones de cierre': ruta_transferencia / '80_comprobaciones_cierre_notebook.csv'
}

# -----------------------------------------------------------------------------
# SALIDAS
# -----------------------------------------------------------------------------
for ruta in [ruta_tablas, ruta_figuras, ruta_informes, ruta_logs]:
    ruta.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# CREDENCIALES Y SEGURIDAD
# -----------------------------------------------------------------------------
load_dotenv(ruta_env)

credenciales_disponibles = pd.DataFrame({
    'servicio': ['Mistral', 'Hugging Face'],
    'configurada': [bool(os.getenv('MISTRAL_API_KEY')), bool(os.getenv('HF_TOKEN'))]
})

permisos_env = oct(ruta_env.stat().st_mode & 0o777) if ruta_env.is_file() else 'No disponible'

env_ignorado_git = subprocess.run(
    ['git', 'check-ignore', '-q', '.env'], cwd=ruta_proyecto
).returncode == 0

env_no_versionado = subprocess.run(
    ['git', 'ls-files', '--', '.env'], cwd=ruta_proyecto, capture_output=True, text=True
).stdout.strip() == ''

# -----------------------------------------------------------------------------
# COMPROBACIÓN DE ARCHIVOS
# -----------------------------------------------------------------------------
tabla_archivos_entorno = pd.DataFrame([
    {
        'archivo': nombre,
        'ruta': str(ruta.relative_to(ruta_proyecto)),
        'disponible': ruta.is_file()
    }
    for nombre, ruta in archivos_entorno.items()
])

tabla_archivos_transferencia = pd.DataFrame([
    {
        'archivo': nombre,
        'ruta': str(ruta.relative_to(ruta_proyecto)),
        'disponible': ruta.is_file()
    }
    for nombre, ruta in archivos_transferencia.items()
])

configuracion_general = pd.DataFrame({
    'configuracion': [
        'Semilla',
        'Codificación CSV',
        'Fecha de ejecución',
        'Dispositivo PyTorch',
        'Dispositivo embeddings RAG',
        'Modelo de embeddings',
        'Dimensión embeddings',
        'Permisos .env'
    ],
    'valor': [
        semilla,
        encoding_csv,
        fecha_ejecucion,
        dispositivo,
        dispositivo_embeddings,
        modelo_embeddings,
        dimension_embeddings,
        permisos_env
    ]
})

print('\nARCHIVOS DEL ENTORNO')
display(tabla_archivos_entorno)

print('\nARCHIVOS DE TRANSFERENCIA')
display(tabla_archivos_transferencia)

print('\nCREDENCIALES CONFIGURADAS')
display(credenciales_disponibles)

print('\nCONFIGURACIÓN GENERAL')
display(configuracion_general)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_configuracion = pd.DataFrame({
    'comprobacion': [
        'Los archivos del entorno están disponibles',
        'Los archivos transferidos desde el Notebook 04 están disponibles',
        'Las credenciales necesarias están configuradas',
        'El archivo .env mantiene permisos 600',
        'El archivo .env está ignorado por Git',
        'El archivo .env no está versionado'
    ],
    'resultado': [
        tabla_archivos_entorno['disponible'].all(),
        tabla_archivos_transferencia['disponible'].all(),
        credenciales_disponibles['configurada'].all(),
        permisos_env == '0o600',
        env_ignorado_git,
        env_no_versionado
    ]
})

print('\nCOMPROBACIONES DE LA CONFIGURACIÓN')
display(comprobaciones_configuracion)

if not comprobaciones_configuracion['resultado'].all():
    raise RuntimeError('La configuración inicial del Notebook 05 no es correcta.')

print('\nConfiguración inicial validada correctamente.')


CONFIGURACIÓN GENERAL Y RUTAS



ARCHIVOS DEL ENTORNO


,archivo,ruta,disponible
0,Configuración del entorno,environments/TFM_Agentes_environment.yml,True
1,Reproducción del entorno,environments/TFM_Agentes_REPRODUCCION.md,True
2,Smoke test,tests/smoke_test_agents_environment.py,True
3,Archivo de credenciales,.env,True



ARCHIVOS DE TRANSFERENCIA


,archivo,ruta,disponible
0,Configuración de productivización,results/tables/04_evaluacion_explicabilidad/co...,True
1,Referencia SHAP,results/tables/04_evaluacion_explicabilidad/ba...,True
2,Selección final por objetivo,results/tables/04_evaluacion_explicabilidad/53...,True
3,Predicciones finales de TEST,results/tables/04_evaluacion_explicabilidad/54...,True
4,Variables principales SHAP,results/tables/04_evaluacion_explicabilidad/69...,True
5,Coincidencias de variables SHAP,results/tables/04_evaluacion_explicabilidad/73...,True
6,Esquema de entrada,results/tables/04_evaluacion_explicabilidad/75...,True
7,Esquema de salida,results/tables/04_evaluacion_explicabilidad/76...,True
8,Limitaciones,results/tables/04_evaluacion_explicabilidad/77...,True
9,Archivos transferidos,results/tables/04_evaluacion_explicabilidad/78...,True


...


CONFIGURACIÓN GENERAL


,configuracion,valor
0,Semilla,12345
1,Codificación CSV,utf-8-sig
2,Fecha de ejecución,2026-08-31T21:07:00+02:00
3,Dispositivo PyTorch,cuda
4,Dispositivo embeddings RAG,cpu
5,Modelo de embeddings,sentence-transformers/paraphrase-multilingual-...
6,Dimensión embeddings,384
7,Permisos .env,0o600



COMPROBACIONES DE LA CONFIGURACIÓN


,comprobacion,resultado
0,Los archivos del entorno están disponibles,True
1,Los archivos transferidos desde el Notebook 04...,True
2,Las credenciales necesarias están configuradas,True
3,El archivo .env mantiene permisos 600,True
4,El archivo .env está ignorado por Git,True
5,El archivo .env no está versionado,True



Configuración inicial validada correctamente.


### Resultados

La configuración inicial queda validada. Están disponibles los archivos de reproducción y comprobación del entorno y los doce archivos de transferencia requeridos desde el Notebook 04.

Las credenciales de Mistral y Hugging Face se recuperan correctamente desde `.env` sin mostrar sus valores. El archivo mantiene permisos `600`, permanece ignorado por Git y no se encuentra versionado.

Las carpetas de salida del Notebook 05 quedan disponibles y se centralizan las rutas necesarias para la transferencia, el servicio predictivo, MCP, la documentación NSDUH y las salidas propias de agentes y productivización.

El smoke test integral de `TFM_Agentes`, previamente validado durante la preparación del entorno, permanece disponible y se reserva para el cierre definitivo del entorno.

## 0.3. Diccionarios y etiquetas

Se definen las etiquetas y estructuras comunes necesarias para mantener una presentación homogénea durante la productivización. Se conservan los nombres de las familias Machine Learning y Deep Learning utilizados en el Notebook 04 y se incorporan las denominaciones de los dos mecanismos de acceso mediante tools que se contrastarán posteriormente: tool local y MCP.

También se centraliza el renombrado visible de los campos relacionados con predicción, explicabilidad, recuperación documental, agentes y trazabilidad. Este diccionario afecta únicamente a la presentación de tablas y resultados y no modifica las claves contractuales utilizadas internamente.

Las variables objetivo, sus descripciones, los modelos seleccionados, los umbrales y los esquemas definitivos de entrada y salida no se declaran manualmente en este apartado. Estos elementos se recuperarán en la sección siguiente desde los archivos transferidos por el Notebook 04, evitando duplicar información ya cerrada.

In [3]:
# =============================================================================
# DICCIONARIOS Y ETIQUETAS
# =============================================================================
print('\nDICCIONARIOS Y ETIQUETAS')

# -----------------------------------------------------------------------------
# FAMILIAS DE MODELOS
# -----------------------------------------------------------------------------
nombres_familias = {
    'ML': 'Machine Learning',
    'DL': 'Deep Learning'
}

orden_familias = ['ML', 'DL']

# -----------------------------------------------------------------------------
# MECANISMOS DE TOOLS
# -----------------------------------------------------------------------------
nombres_mecanismos_tools = {
    'local': 'Tool local',
    'mcp': 'MCP'
}

orden_mecanismos_tools = ['local', 'mcp']

# -----------------------------------------------------------------------------
# BANCO DE FRASES DEL INFORME
# -----------------------------------------------------------------------------
frases_informe = {
    'tipos_factor': {
        'positiva': 'Factor de riesgo para la predicción',
        'negativa': 'Factor protector para la predicción'
    },
    'nota_factores': (
        'La clasificación como factor de riesgo o factor protector se refiere únicamente '
        'a la dirección de la contribución SHAP sobre la predicción del modelo y no implica '
        'una relación causal ni una valoración clínica.'
    ),
    'orientacion_preventiva': [
        'Considerar una revisión profesional del conjunto de indicadores '
        'cuando el contexto lo justifique.',
        'Favorecer el acceso a recursos de apoyo y seguimiento preventivo.',
        'Mantener la interpretación de los resultados como apoyo complementario '
        'y no como diagnóstico.',
        'Priorizar una valoración profesional si los indicadores coinciden con señales '
        'de especial preocupación.',
        'Contrastar esta información con el contexto individual y con otras fuentes pertinentes '
        'antes de adoptar decisiones.'
    ],
    'limitaciones_obligatorias': [
        'El sistema tiene finalidad preventiva y no diagnóstica.',
        'Las probabilidades estimadas no equivalen a riesgo clínico individual.',
        'Las contribuciones SHAP describen la dirección de la contribución del modelo '
        'y no establecen causalidad.'
    ],
    'advertencia_uso': (
        'Este informe es un prototipo académico de apoyo preventivo. No realiza diagnósticos '
        'ni sustituye la valoración de profesionales cualificados.'
    )
}

# -----------------------------------------------------------------------------
# RENOMBRADO COMÚN DE TABLAS
# -----------------------------------------------------------------------------
renombrado_comun = {
    # Identificación y presentación general
    'target': 'Variable objetivo',
    'variable_objetivo': 'Variable objetivo',
    'descripcion': 'Descripción',
    'familia': 'Familia',
    'modelo': 'Modelo',
    'variable': 'Variable',
    'etiqueta_variable': 'Descripción variable',
    'modulo': 'Módulo',
    'posicion': 'Posición',
    'estado': 'Estado',
    'elemento': 'Elemento',
    'valor': 'Valor',
    'registros': 'Registros',
    'variables': 'Variables',
    'ruta': 'Ruta',
    'disponible': 'Disponible',
    'campos': 'Campos',
    'campos_obligatorios': 'Campos obligatorios',
    'validacion': 'Validación',
    # Predicción y explicabilidad
    'probabilidad': 'Probabilidad',
    'umbral': 'Umbral',
    'clasificacion': 'Clasificación',
    'variables_principales': 'Variables principales',
    'importancia_shap': 'Importancia SHAP',
    'diferencia_reconstruccion': 'Diferencia de reconstrucción',
    'advertencias': 'Advertencias',
    # Tools y flujo
    'mecanismo': 'Mecanismo',
    'tool': 'Tool',
    'nodo': 'Nodo',
    # Recuperación documental
    'consulta': 'Consulta',
    'fuente': 'Fuente',
    'documento': 'Documento',
    'fragmento': 'Fragmento',
    'similitud': 'Similitud',
    'documentos_recuperados': 'Documentos recuperados',
    # Generación y trazabilidad
    'modelo_generativo': 'Modelo generativo',
    'version_prompt': 'Versión del prompt',
    'fecha_ejecucion': 'Fecha de ejecución',
    'tiempo_s': 'Tiempo (s)',
    'guardrails_superados': 'Guardrails superados',
    'valor_observado': 'Valor observado',
    'valor_shap': 'Valor SHAP',
    'respuesta_observada': 'Respuesta observada',
    'nota_variable': 'Aclaración de la variable',
    # Comprobaciones
    'servicio': 'Servicio',
    'configurada': 'Configurada',
    'comprobacion': 'Comprobación',
    'resultado': 'Resultado',
    'funcion': 'Función',
    'finalidad': 'Finalidad',
    # Informe
    'tipo_factor': 'Tipo de factor',
    'descripcion_objetivo': 'Indicador',
    'variable_origen': 'Variable original',
    'descripcion_original': 'Descripción original',
    'posicion_factor': 'Posición',
    # Productivización y cierre
    'archivo': 'Archivo',
    'archivos': 'Archivos',
    'metodos': 'Métodos',
    'solicitud': 'Solicitud',
    'codigo_http': 'Código HTTP',
    'ejemplo': 'Ejemplo',
    'perfil_clasificacion': 'Perfil de clasificación',
    'valores_ausentes': 'Valores ausentes',
    'componente': 'Componente',
    'criterio': 'Criterio',
    'factores': 'Factores',
    'paso': 'Paso',
    'entrada': 'Entrada',
    'salida': 'Salida',
    'bloque': 'Bloque',
    'comprobaciones': 'Comprobaciones',
    'superadas': 'Superadas',
    'tipo': 'Tipo',
    'tamano_kb': 'Tamaño (KB)',
    'evidencia': 'Evidencia',
    'destino': 'Destino',
    # Comparación arquitectónica e informes
    'tool_local': 'Tool local',
    'mcp': 'MCP',
    'lectura': 'Lectura',
    'id_informe': 'Informe',
    'fecha': 'Fecha',
    'archivo_origen': 'Archivo de entrada',
    'indicadores_superados': 'Indicadores que superan el umbral',
}

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
campos_visibles_necesarios = {
    'variable_objetivo', 'descripcion', 'probabilidad', 'umbral', 'clasificacion', 'familia',
    'modelo', 'advertencias'
}

comprobaciones_diccionarios = pd.DataFrame({
    'comprobacion': [
        'Las dos familias de modelos están definidas',
        'El orden de familias coincide con sus etiquetas',
        'Los dos mecanismos de tools están definidos',
        'El orden de mecanismos coincide con sus etiquetas',
        'El renombrado contiene los campos predictivos principales'
    ],
    'resultado': [
        set(nombres_familias) == {'ML', 'DL'},
        set(orden_familias) == set(nombres_familias),
        set(nombres_mecanismos_tools) == {'local', 'mcp'},
        set(orden_mecanismos_tools) == set(nombres_mecanismos_tools),
        campos_visibles_necesarios.issubset(renombrado_comun)
    ]
})

resumen_diccionarios = pd.DataFrame({
    'elemento': [
        'Familias de modelos',
        'Mecanismos de tools',
        'Campos de renombrado común'
    ],
    'valor': [
        len(nombres_familias),
        len(nombres_mecanismos_tools),
        len(renombrado_comun)
    ]
})

tabla_familias = pd.DataFrame({
    'familia': list(nombres_familias),
    'descripcion': list(nombres_familias.values())
}).rename(columns=renombrado_comun)

tabla_mecanismos_tools = pd.DataFrame({
    'mecanismo': list(nombres_mecanismos_tools),
    'descripcion': list(nombres_mecanismos_tools.values())
}).rename(columns=renombrado_comun)

tabla_resumen = resumen_diccionarios.rename(columns=renombrado_comun)

display(tabla_resumen)

print('\nFAMILIAS DE MODELOS')
display(tabla_familias)

print('\nMECANISMOS DE TOOLS')
display(tabla_mecanismos_tools)

tabla_comprobaciones = comprobaciones_diccionarios.rename(columns=renombrado_comun)
print('\nCOMPROBACIONES DE LOS DICCIONARIOS')
display(tabla_comprobaciones)

if not comprobaciones_diccionarios['resultado'].all():
    raise RuntimeError('Los diccionarios y etiquetas no son correctos.')

print('\nDiccionarios y etiquetas preparados correctamente.')


DICCIONARIOS Y ETIQUETAS


,Elemento,Valor
0,Familias de modelos,2
1,Mecanismos de tools,2
2,Campos de renombrado común,83



FAMILIAS DE MODELOS


,Familia,Descripción
0,ML,Machine Learning
1,DL,Deep Learning



MECANISMOS DE TOOLS


,Mecanismo,Descripción
0,local,Tool local
1,mcp,MCP



COMPROBACIONES DE LOS DICCIONARIOS


,Comprobación,Resultado
0,Las dos familias de modelos están definidas,True
1,El orden de familias coincide con sus etiquetas,True
2,Los dos mecanismos de tools están definidos,True
3,El orden de mecanismos coincide con sus etiquetas,True
4,El renombrado contiene los campos predictivos ...,True



Diccionarios y etiquetas preparados correctamente.


### Resultados

Se definen las dos familias predictivas utilizadas en el proyecto —Machine Learning y Deep Learning— y los dos mecanismos de acceso mediante tools que se compararán posteriormente: tool local y MCP.

El diccionario común centraliza 83 campos de presentación para predicción, explicabilidad, recuperación documental, agentes, trazabilidad y comprobaciones, manteniendo separadas las claves técnicas internas de los nombres visibles.

Las cinco comprobaciones realizadas resultan correctas. No se duplican en esta sección variables objetivo, modelos ni umbrales, que se recuperan posteriormente desde la transferencia cerrada del Notebook 04.

## 0.4. Configuración del modelo generativo y prompt

La capa generativa utiliza Mistral exclusivamente para redactar el resumen, la interpretación y las descripciones comprensibles de las variables autorizadas. Las predicciones, los umbrales, las clasificaciones, la selección de factores SHAP, las fuentes, las limitaciones y las advertencias permanecen fuera del control del modelo generativo.

En este bloque se centralizan el modelo utilizado, la temperatura, la versión del prompt y las instrucciones de sistema. El prompt restringe explícitamente la generación a la información validada por la aplicación y establece un lenguaje preventivo, prudente y no diagnóstico.

Esta separación mantiene la configuración generativa definida antes de la ejecución del flujo y permite que las secciones posteriores se centren únicamente en preparar el contexto, realizar la llamada estructurada y validar el resultado.

In [4]:
# =============================================================================
# CONFIGURACIÓN DEL MODELO GENERATIVO Y PROMPT
# =============================================================================
modelo_mistral = 'mistral-small-latest'
version_prompt_informe = 'v5'
temperatura_mistral = 0.0

prompt_sistema_informe = """
Eres la capa de redacción de un sistema académico de apoyo preventivo en salud mental.

Utiliza exclusivamente el contexto validado proporcionado por la aplicación.

Debes generar únicamente:
- un resumen;
- una interpretación;
- la traducción al español de las variables incluidas en variables_traducir.

REGLAS OBLIGATORIAS:
1. No realices diagnósticos ni afirmes que una persona padece una condición clínica.
2. No atribuyas causalidad a ninguna variable ni a las contribuciones SHAP.
3. No incluyas probabilidades, umbrales, porcentajes ni valores numéricos SHAP.
4. En interpretacion menciona exactamente las cuatro descripciones de las variables objetivo.
5. Indica si cada indicador supera o no su umbral validado.
6. Debes informar sobre los cuatro indicadores aunque alguno no supere su umbral.
7. Traduce exactamente las variables incluidas en variables_traducir.
8. Conserva literalmente variable_origen en cada traducción estructurada.
9. No añadas ni elimines variables.
10. descripcion debe contener únicamente una denominación clara y natural en español.
11. No incluyas códigos técnicos de variables dentro de descripcion.
12. No interpretes las variables traducidas como causas clínicas.
13. No añadas información clínica, epidemiológica o documental no incluida en el contexto.

Redacta resumen e interpretacion en español claro, prudente y comprensible.

REGLAS DE REDACCIÓN DEL RESUMEN Y LA INTERPRETACIÓN:

1. Describe los resultados exclusivamente en términos de indicadores y umbrales validados.

2. Cuando un indicador tenga clasificación 1, indica que el indicador
   "supera su umbral validado".

3. Cuando un indicador tenga clasificación 0, indica que el indicador
   "no supera su umbral validado".

4. Un indicador que supera su umbral puede describirse como una señal preventiva
   que justifica una revisión adicional, pero nunca como la confirmación de una
   condición clínica.

5. No interpretes las probabilidades como probabilidades de enfermedad,
   riesgo clínico individual ni probabilidad de que el fenómeno exista.

6. Mantén siempre un lenguaje preventivo, prudente, comprensible y no diagnóstico.

7. Distingue claramente entre el indicador evaluado y la existencia real del
   fenómeno al que hace referencia.

8. Evita expresiones técnicas innecesarias para el usuario, como
   "variable objetivo", "descripción de la variable objetivo" o formulaciones
   equivalentes.

9. Cuando ningún indicador supere su umbral, limita la conclusión a indicar que
   ninguno supera su umbral validado en esta evaluación. No concluyas que los
   fenómenos evaluados están ausentes.

10. Cuando uno o varios indicadores superen su umbral, identifica cuáles son y
    señala la conveniencia de una revisión preventiva adicional, sin convertir
    el resultado en diagnóstico ni recomendación clínica.

FORMULACIONES QUE NO DEBEN UTILIZARSE:

- "Presencia de episodio depresivo mayor".
- "Presencia de ideación suicida".
- "Presencia de planificación suicida".
- "Presencia de intento suicida".
- "Ausencia de episodio depresivo mayor".
- "No existe ideación suicida".
- "Riesgo asociado al intento suicida".
- "Probabilidad de padecer".
- "El paciente presenta".
- "El usuario padece".
- "Se diagnostica".
- "Las descripciones de las variables objetivo que superan el umbral".
- Cualquier formulación equivalente que convierta una salida predictiva
  en una afirmación clínica.

FORMULACIONES PREFERIDAS:

- "El indicador de episodio depresivo mayor supera su umbral validado."
- "El indicador de ideación suicida no supera su umbral validado."
- "El indicador de intento suicida supera su umbral validado. Este resultado
  señala la conveniencia de una revisión preventiva adicional."
- "Ninguno de los cuatro indicadores supera su umbral validado en esta evaluación."
- "Los cuatro indicadores evaluados superan sus respectivos umbrales validados."
""".strip()

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_prompt = pd.DataFrame({
    'elemento': [
        'Modelo generativo',
        'Versión del prompt',
        'Temperatura',
        'Prompt de sistema definido'
    ],
    'valor': [
        modelo_mistral,
        version_prompt_informe,
        temperatura_mistral,
        bool(prompt_sistema_informe)
    ]
})

tabla_prompt = comprobaciones_prompt.rename(columns=renombrado_comun)

print('\nCONFIGURACIÓN GENERATIVA')
display(tabla_prompt)


CONFIGURACIÓN GENERATIVA


,Elemento,Valor
0,Modelo generativo,mistral-small-latest
1,Versión del prompt,v5
2,Temperatura,0.0
3,Prompt de sistema definido,True


## 0.5. Plantillas de la aplicación

La capa de presentación de la aplicación de demostración utiliza plantillas HTML definidas de forma centralizada y separadas de la lógica de ejecución de Flask.

Las plantillas recogen la identificación del usuario autorizado, el panel principal, la presentación del informe preventivo y la consulta del historial de informes. Su definición previa permite mantener las rutas Flask centradas en la validación de entradas, ejecución del flujo, recuperación de resultados y navegación.

La interfaz presenta únicamente información ya validada por la aplicación y no modifica predicciones, umbrales, factores SHAP ni contenido estructurado del informe.

In [5]:
# =============================================================================
# PLANTILLAS DE LA APLICACIÓN
# =============================================================================
# -----------------------------------------------------------------------------
# INTERFAZ PRINCIPAL
# -----------------------------------------------------------------------------
plantilla_interfaz = """
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Sistema preventivo NSDUH 2024</title>
<style>
:root{
 --turquesa:#81d8d0;--medio:#5ccec5;--intenso:#2b9f99;
 --oscuro:#176b70;--profundo:#0e4f55;--claro:#dcf7f4;
 --fondo:#f4fcfb;--gris:#52656a;--blanco:#fff
}
*{box-sizing:border-box}
body{margin:0;background:var(--fondo);font-family:Arial,sans-serif;color:#24383c}
main{max-width:1180px;margin:auto;padding:26px 22px 50px}
nav{display:flex;gap:10px;justify-content:flex-end;margin-bottom:14px;flex-wrap:wrap}
nav a,nav button{border:0;border-radius:9px;padding:9px 13px;background:white;
 color:var(--oscuro);font-weight:700;text-decoration:none;cursor:pointer;
 box-shadow:0 2px 9px #176b7015}
header{background:linear-gradient(135deg,var(--oscuro),var(--turquesa));
 color:white;padding:28px 32px;border-radius:18px;box-shadow:0 10px 28px #176b7030}
header h1{margin:0 0 7px;font-size:30px}
header p{margin:0;opacity:.94}
h2{color:var(--profundo);margin-top:32px}
h3,h4{color:var(--oscuro)}
.aviso{margin:18px 0;padding:14px 18px;border-left:5px solid var(--turquesa);
 background:white;border-radius:10px;box-shadow:0 3px 12px #176b7012}
.grid{display:grid;grid-template-columns:repeat(2,1fr);gap:16px}
.tarjeta,.bloque{background:white;border:1px solid var(--claro);
 border-radius:15px;padding:20px;box-shadow:0 5px 16px #176b7012}
.tarjeta h3{margin:0 0 14px;min-height:42px}
.valor{font-size:32px;font-weight:700;color:var(--intenso)}
.detalle{color:var(--gris);font-size:14px;margin-top:5px}
.estado{display:inline-block;margin-top:12px;padding:6px 11px;
 border-radius:20px;font-size:13px;font-weight:700}
.positivo{background:var(--claro);color:var(--oscuro)}
.negativo{background:#edf5f5;color:#52656a}
.factor{padding:9px 12px;margin:7px 0;border-radius:8px;background:#f9fdfd}
.riesgo{border-left:4px solid var(--oscuro)}
.protector{border-left:4px solid var(--turquesa)}
.bloque{margin-top:15px}
.explicacion{margin-top:12px;padding:4px 2px}
.explicacion-grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}
.explicacion h4{margin-bottom:8px}
.valor-observado{display:block;color:var(--gris);font-size:12px;margin-top:4px}
details{background:white;border:1px solid var(--claro);border-radius:12px;
 padding:13px 16px;margin-top:10px}
summary{cursor:pointer;color:var(--oscuro);font-weight:700}
.nota{color:var(--gris);font-size:13px;font-style:italic}
ul{line-height:1.6}
footer{text-align:center;color:#64748b;font-size:12px;margin-top:35px}
@media(max-width:750px){
 .grid,.explicacion-grid{grid-template-columns:1fr}
 header h1{font-size:24px}
}
@media print{nav{display:none}body{background:white}}
</style>
</head>

<body><main>

<nav>
<a href="/acceso">Panel</a>
<a href="/informes">Informes generados</a>
<button onclick="window.print()">Imprimir / guardar PDF</button>
</nav>

<header>
<h1>Sistema preventivo NSDUH 2024</h1>
<p>Machine Learning · Deep Learning · SHAP · RAG · Inteligencia Artificial Generativa</p>
</header>

<div class="aviso"><strong>Uso preventivo:</strong> {{ advertencia }}</div>

<h2>Resultados predictivos</h2>
<div class="grid">
{% for p in predicciones %}
<div class="tarjeta">
<h3>{{ p.descripcion }}</h3>
<div class="valor">{{ "%.3f"|format(p.probabilidad) }}</div>
<div class="detalle">Umbral validado: {{ "%.3f"|format(p.umbral) }}</div>
<span class="estado {{ 'positivo' if p.clasificacion == 1 else 'negativo' }}">
{{ 'SUPERA EL UMBRAL' if p.clasificacion == 1 else 'NO SUPERA EL UMBRAL' }}
</span>
<div class="detalle">{{ p.modelo }} · {{ p.familia }}</div>
</div>
{% endfor %}
</div>

<div class="bloque">
<h2>Probabilidad y umbral</h2>
{{ grafico|safe }}
</div>

<h2>Factores principales del informe</h2>
{% for objetivo, lista in factores.items() %}
<div class="bloque">
<h3>{{ objetivo }}</h3>
{% for factor in lista %}
<div class="factor {{ 'riesgo' if 'riesgo' in factor.tipo_factor|lower else 'protector' }}">
<strong>{{ factor.tipo_factor }}</strong><br>
{{ factor.descripcion }}

<span class="valor-observado">
<strong>Respuesta registrada:</strong> {{ factor.respuesta_observada }}
</span>

{% if factor.nota_variable %}
<span class="valor-observado">
<strong>Aclaración:</strong> {{ factor.nota_variable }}
</span>
{% endif %}

</div>
{% endfor %}
</div>
{% endfor %}

<p class="nota">{{ nota_factores }}</p>

<h2>Explicabilidad ampliada</h2>
<p class="nota">
Se muestran hasta tres contribuciones que aumentan la predicción y hasta dos
que la reducen entre las variables SHAP principales disponibles.
</p>

{% for objetivo, grupos in detalle.items() %}
<details>
<summary>{{ objetivo }}</summary>
<div class="explicacion-grid">

<div class="explicacion">
<h4>Factores que aumentan la predicción</h4>
{% for factor in grupos['riesgo'] %}
<div class="factor riesgo">
{{ factor.descripcion }}

<span class="valor-observado">
<strong>Respuesta registrada:</strong> {{ factor.respuesta_observada }}
</span>

{% if factor.nota_variable %}
<span class="valor-observado">
<strong>Aclaración:</strong> {{ factor.nota_variable }}
</span>
{% endif %}
</div>

{% else %}
<p class="nota">No se identifican contribuciones positivas adicionales.</p>
{% endfor %}
</div>

<div class="explicacion">
<h4>Factores que reducen la predicción</h4>
{% for factor in grupos['protector'] %}
<div class="factor protector">
{{ factor.descripcion }}

<span class="valor-observado">
<strong>Respuesta registrada:</strong> {{ factor.respuesta_observada }}
</span>

{% if factor.nota_variable %}
<span class="valor-observado">
<strong>Aclaración:</strong> {{ factor.nota_variable }}
</span>
{% endif %}
</div>

{% else %}
<p class="nota">No se identifican contribuciones negativas adicionales.</p>
{% endfor %}
</div>

</div>
</details>
{% endfor %}

<div class="bloque">
<h2>Resumen</h2>
<p>{{ informe.resumen }}</p>
</div>

<div class="bloque">
<h2>Interpretación</h2>
<p>{{ informe.interpretacion }}</p>
</div>

<div class="bloque">
<h2>Orientación preventiva</h2>
<ul>{% for valor in informe.orientacion_preventiva %}<li>{{ valor }}</li>{% endfor %}</ul>
</div>

<div class="bloque">
<h2>Limitaciones</h2>
<ul>{% for valor in informe.limitaciones %}<li>{{ valor }}</li>{% endfor %}</ul>
</div>

<div class="bloque">
<h2>Fuentes</h2>
<ul>{% for valor in informe.fuentes %}<li>{{ valor }}</li>{% endfor %}</ul>
</div>

<footer>CAM · TFM · NSDUH </footer>
</main></body>
</html>
"""

# -----------------------------------------------------------------------------
# INTERFAZ DE ACCESO
# -----------------------------------------------------------------------------
plantilla_acceso = """
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Acceso · Sistema preventivo NSDUH 2024</title>
<style>
:root{
 --turquesa:#81d8d0;--medio:#5ccec5;--intenso:#2b9f99;
 --oscuro:#176b70;--profundo:#0e4f55;--claro:#dcf7f4;
 --fondo:#f4fcfb;--gris:#52656a;--blanco:#fff
}
*{box-sizing:border-box}
body{margin:0;background:linear-gradient(135deg,#f4fcfb,#dcf7f4);
 font-family:Arial,sans-serif;color:#24383c;min-height:100vh}
main{max-width:560px;margin:auto;padding:55px 22px}
.cabecera{text-align:center;margin-bottom:22px}
.logo{width:62px;height:62px;margin:auto;border-radius:18px;
 background:linear-gradient(135deg,var(--oscuro),var(--turquesa));color:white;
 display:flex;align-items:center;justify-content:center;
 font-size:28px;font-weight:700;box-shadow:0 8px 22px #176b7030}
h1{color:var(--profundo);font-size:25px;margin:16px 0 5px}
.subtitulo{color:var(--gris);font-size:14px}
.tarjeta{background:white;border:1px solid var(--claro);border-radius:20px;
 padding:28px;box-shadow:0 14px 38px #176b7020}
h2{margin:0 0 5px;color:var(--profundo);font-size:20px}
.descripcion{color:var(--gris);font-size:14px;margin:0 0 24px}
label{display:block;color:var(--oscuro);font-size:14px;
 font-weight:700;margin:16px 0 7px}
input{width:100%;padding:12px 13px;border:1px solid var(--turquesa);
 border-radius:9px;background:#fbffff;font-size:14px}
input:focus{outline:2px solid var(--turquesa);border-color:var(--intenso)}
.archivo{padding:11px;background:var(--fondo)}
.info{background:var(--fondo);border-left:4px solid var(--turquesa);
 padding:11px 13px;border-radius:7px;margin:18px 0;
 color:var(--gris);font-size:13px}
button,.boton{width:100%;display:block;border:0;border-radius:10px;
 padding:13px;margin-top:10px;text-align:center;text-decoration:none;
 background:linear-gradient(135deg,var(--oscuro),var(--medio));
 color:white;font-size:15px;font-weight:700;cursor:pointer}
button:hover,.boton:hover{
 background:linear-gradient(135deg,var(--profundo),var(--intenso))
}
.boton-secundario{background:white;color:var(--oscuro);
 border:1px solid var(--turquesa)}
.boton-salir{background:#edf5f5;color:var(--gris);
 border:1px solid #d8e5e5}
.error{background:#fef2f2;color:#991b1b;padding:10px 12px;
 border-radius:8px;margin-bottom:16px;font-size:13px}
.pie{text-align:center;color:var(--gris);font-size:12px;
 margin-top:19px;line-height:1.5}
.nota{color:var(--gris);font-size:13px;font-style:italic}
.oculto{display:none}
.procesando{background:white;border:1px solid var(--claro);border-radius:20px;
 padding:32px;text-align:center;box-shadow:0 14px 38px #176b7020}
.spinner{width:52px;height:52px;margin:5px auto 22px;border-radius:50%;
 border:6px solid var(--claro);border-top-color:var(--intenso);
 animation:giro 1s linear infinite}
.pasos{text-align:left;line-height:2;margin:22px auto;
 max-width:330px;color:var(--gris)}
@keyframes giro{to{transform:rotate(360deg)}}
</style>
</head>

<body>
<main>
<div class="cabecera">
<div class="logo">IA</div>
<h1>Sistema preventivo NSDUH 2024</h1>
<div class="subtitulo">ML · DL · SHAP · RAG · IA generativa</div>
</div>

{% if not autorizado %}

<div class="tarjeta">
<h2>Acceso autorizado</h2>
<p class="descripcion">
Introduzca su credencial para acceder al prototipo.
</p>

<div class="info">
<strong>Demostración académica</strong><br>
Credencial del usuario autorizado: <strong>CAM</strong>
</div>

{% if error %}
<div class="error">{{ error }}</div>
{% endif %}

<form method="post">
<label>Credencial de acceso</label>
<input type="password" name="api_key"
 placeholder="Introduzca su credencial" required autocomplete="off">

<button type="submit">ACCEDER</button>
</form>
</div>

{% else %}

<div class="tarjeta">
<h2>Panel de usuario autorizado</h2>
<p class="descripcion">
Seleccione la operación que desea realizar.
</p>

<div class="info">
<strong>Sesión autorizada</strong><br>
Prototipo académico NSDUH 2024.
</div>

{% if error %}
<div class="error">{{ error }}</div>
{% endif %}

<form id="form-informe" method="post" enctype="multipart/form-data"
 onsubmit="mostrarProcesando()">

<label>Generar nuevo informe</label>
<input class="archivo" type="file" name="archivo" accept=".csv" required>

<div class="info">
<strong>Formato esperado</strong><br>
Archivo CSV · un único registro · 813 variables de entrada.
</div>

<button type="submit">GENERAR INFORME PREVENTIVO</button>
</form>

<a class="boton boton-secundario" href="/informes">
VER INFORMES GENERADOS
</a>

<form method="post"
 onsubmit="return confirm('¿Desea finalizar la demostración y cerrar el servidor?')">
<input type="hidden" name="accion" value="finalizar">
<button class="boton-salir" type="submit">
SALIR Y CERRAR DEMOSTRACIÓN
</button>
</form>
</div>

{% endif %}

<div id="procesando" class="procesando oculto">
<div class="spinner"></div>
<h2>Generando informe preventivo</h2>
<p>El sistema está procesando el registro seleccionado.</p>

<div class="pasos">
<div>· Validación del registro</div>
<div>· Predicción de los cuatro indicadores</div>
<div>· Explicabilidad de las predicciones</div>
<div>· Recuperación documental</div>
<div>· Generación y validación del informe</div>
</div>

<p class="nota">
El proceso puede requerir unos minutos. No cierre esta ventana.
</p>
</div>

<script>
function mostrarProcesando(){
 const tarjeta=document.querySelector('.tarjeta');
 if(tarjeta){tarjeta.style.display='none';}
 document.getElementById('procesando').classList.remove('oculto');
 window.scrollTo({top:0,behavior:'smooth'});
}
</script>

<div class="pie">
CAM · TFM · No realiza diagnósticos.<br>
Acceso restringido a usuarios autorizados.
</div>
</main>
</body>
</html>
"""

# -----------------------------------------------------------------------------
# INTERFAZ DE HISTORIAL
# -----------------------------------------------------------------------------
plantilla_historial = """
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Informes generados</title>
<style>
:root{--turquesa:#81d8d0;--intenso:#2b9f99;--oscuro:#176b70;
 --profundo:#0e4f55;--claro:#dcf7f4;--fondo:#f4fcfb;--gris:#52656a}
*{box-sizing:border-box}
body{margin:0;background:var(--fondo);font-family:Arial,sans-serif;color:#24383c}
main{max-width:900px;margin:auto;padding:35px 22px}
header{background:linear-gradient(135deg,var(--oscuro),var(--turquesa));
 color:white;padding:25px 30px;border-radius:17px}
a{text-decoration:none}
.nuevo{display:inline-block;margin:20px 0;padding:11px 16px;border-radius:9px;
 background:var(--intenso);color:white;font-weight:700}
.informe{background:white;border:1px solid var(--claro);border-radius:14px;
 padding:18px 20px;margin:12px 0;box-shadow:0 4px 14px #176b7012;
 display:flex;justify-content:space-between;gap:15px;align-items:center}
.informe h3{margin:0 0 6px;color:var(--profundo)}
.meta{font-size:13px;color:var(--gris);line-height:1.5}
.ver{padding:9px 13px;border-radius:8px;background:var(--turquesa);
 color:var(--profundo);font-weight:700;white-space:nowrap}
</style>
</head>
<body><main>

<header>
<h1>Informes generados</h1>
<p>Historial de informes preventivos validados.</p>
</header>

<a class="nuevo" href="/acceso">← Volver al panel</a>

{% for item in informes %}
<div class="informe">
<div>
<h3>Informe {{ item.id_informe }}</h3>
<div class="meta">
{{ item.fecha }}<br>
Archivo: {{ item.archivo_origen or 'Entrada mediante API' }}<br>
Indicadores que superan el umbral: {{ item.indicadores_superados }} de 4
</div>
</div>
<a class="ver" href="/informe/{{ item.id_informe }}">VER INFORME</a>
</div>
{% else %}
<p>No hay informes almacenados todavía.</p>
{% endfor %}

</main></body>
</html>
"""

# -----------------------------------------------------------------------------
# INTERFAZ DE SALIDA
# -----------------------------------------------------------------------------
plantilla_salida = """
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<title>Demostración finalizada</title>
<style>
body{margin:0;background:#f4fcfb;font-family:Arial,sans-serif;
color:#24383c;text-align:center;padding:80px 20px}

.tarjeta{max-width:520px;margin:auto;background:white;padding:35px;
border-radius:18px;box-shadow:0 10px 30px #176b7020}
h1{color:#0e4f55}
p{color:#52656a}
</style>
</head>
<body>
<div class="tarjeta">
<h1>Demostración finalizada</h1>
<p>La sesión se ha cerrado y el servidor local se está deteniendo.</p>
<p>Puede cerrar esta pestaña.</p>
</div>
</body>
</html>
"""

## 0.6. Funciones auxiliares

Se centralizan las funciones reutilizables del notebook para guardar resultados, trabajar con estructuras JSON, comprobar contratos y mantener homogénea la comunicación con el servicio predictivo.

El guardado sigue el patrón de los notebooks anteriores: las tablas se almacenan como CSV con la codificación común del proyecto y las funciones devuelven la ruta generada. Se incluyen además funciones equivalentes para JSON e informes de texto.

La validación predictiva mantiene funciones deterministas para comprobar el registro de entrada y las salidas estructuradas, junto con la función común que comunica `TFM_Agentes` con el servicio de solo lectura alojado en `TFM_ML`.

La capa RAG incorpora funciones para localizar fragmentos del codebook, identificar cada documento y combinar coincidencia exacta mediante metadatos con recuperación semántica.

Para facilitar la interpretación de los resultados se incluyen funciones de visualización completa y de preparación textual determinista. La lógica reutilizable del flujo LangGraph centraliza además el registro de trazabilidad, los nodos y las decisiones de ruta.

Finalmente se incorporan las funciones comunes de la capa generativa: preparación de un contexto restringido para Mistral, generación estructurada mediante el contrato `SalidaGenerativa`, reconstrucción determinista de `InformePreventivo`, representación textual del informe y evaluación de guardrails deterministas.

El estado, la construcción del grafo, los prompts, las ejecuciones reales y las comprobaciones permanecen visibles en sus correspondientes secciones para mantener trazable el funcionamiento del sistema.

In [6]:
# =============================================================================
# FUNCIONES AUXILIARES
# =============================================================================
# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
def guardar_csv(df, nombre, **kwargs):
    """
    Guarda un DataFrame en la carpeta de tablas del Notebook 05.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError('El objeto debe ser un DataFrame.')

    if not str(nombre).lower().endswith('.csv'):
        nombre = f'{nombre}.csv'

    kwargs.setdefault('index', False)
    kwargs.setdefault('encoding', encoding_csv)

    ruta = ruta_tablas / nombre
    df.to_csv(ruta, **kwargs)

    return ruta


def convertir_json(valor):
    """
    Convierte tipos auxiliares habituales a formatos compatibles con JSON.
    """
    if isinstance(valor, Path):
        salida = str(valor)
    elif isinstance(valor, (datetime, pd.Timestamp)):
        salida = valor.isoformat()
    elif isinstance(valor, np.ndarray):
        salida = valor.tolist()
    elif isinstance(valor, np.generic):
        salida = valor.item()
    else:
        raise TypeError(f'Tipo no serializable en JSON: {type(valor).__name__}')

    return salida


def guardar_json(datos, nombre, ruta_salida=None):
    """
    Guarda una estructura o modelo Pydantic como JSON válido.
    """
    if isinstance(datos, BaseModel):
        datos = datos.model_dump(mode='json')

    if not isinstance(datos, (dict, list)):
        raise TypeError('El objeto debe ser un diccionario, una lista o un modelo Pydantic.')

    if not str(nombre).lower().endswith('.json'):
        nombre = f'{nombre}.json'

    ruta_salida = ruta_tablas if ruta_salida is None else Path(ruta_salida)
    ruta_salida.mkdir(parents=True, exist_ok=True)

    ruta = ruta_salida / nombre

    with ruta.open('w', encoding='utf-8') as archivo:
        json.dump(
            datos, archivo, ensure_ascii=False, indent=2, default=convertir_json, allow_nan=False
        )

    return ruta


def guardar_texto(texto, nombre, ruta_salida=None):
    """
    Guarda un informe o contenido textual utilizando UTF-8.
    """
    if not isinstance(texto, str):
        raise TypeError('El contenido debe ser texto.')

    ruta_salida = ruta_informes if ruta_salida is None else Path(ruta_salida)
    ruta_salida.mkdir(parents=True, exist_ok=True)

    ruta = ruta_salida / nombre
    ruta.write_text(texto, encoding='utf-8')

    return ruta


# -----------------------------------------------------------------------------
# PRESENTACIÓN Y FLUJO
# -----------------------------------------------------------------------------
def mostrar_tabla_completa(tabla):
    """
    Muestra una tabla completa sin modificar globalmente las opciones de pandas.
    """
    with pd.option_context(
        'display.max_rows', None, 'display.max_columns', None,
        'display.max_colwidth', None, 'display.width', None
    ):
        display(tabla)


def resumir_resultado_predictivo(resultado):
    """
    Convierte una salida predictiva estructurada en texto determinista.
    """
    familia = nombres_familias.get(resultado['familia'], resultado['familia'])
    decision = 'supera' if resultado['clasificacion'] == 1 else 'no supera'

    salida = (
        f"{resultado['descripcion']}: la probabilidad estimada es "
        f"{resultado['probabilidad']:.3f} y el umbral validado es {resultado['umbral']:.3f}; "
        f"la probabilidad {decision} el umbral, por lo que la clasificación estructurada es "
        f"{resultado['clasificacion']}. La predicción procede de {resultado['modelo']} ({familia})."
    )

    return salida


def resumir_explicacion_shap(explicacion, n_variables=3):
    """
    Resume de forma determinista las principales contribuciones SHAP locales.
    """
    principales = explicacion['variables_principales'][:n_variables]
    detalles = []

    for variable in principales:
        etiqueta = variable.get('etiqueta_variable') or variable.get('variable', 'Variable')
        signo = variable['signo_contribucion'].lower()
        tipo_factor = frases_informe['tipos_factor'].get(signo, 'Factor explicativo')

        detalles.append(f'{etiqueta} — {tipo_factor}')

    resumen = (
        f"Para {explicacion['descripcion']}, los principales factores asociados a la "
        f"predicción son: {'; '.join(detalles)}."
    )

    return resumen


def construir_vista_previa_flujo(predicciones, explicaciones, documentacion):
    """
    Construye la vista textual determinista previa a la generación.
    """
    partes = [resumir_resultado_predictivo(resultado) for resultado in predicciones]
    partes += [resumir_explicacion_shap(explicacion) for explicacion in explicaciones]

    if documentacion:
        fuentes = ', '.join(sorted({documento['fuente'] for documento in documentacion}))
        partes.append(
            f"Se han recuperado {len(documentacion)} documentos controlados procedentes de "
            f"{fuentes}. Esta información se entregará como contexto a la capa generativa."
        )

    vista_previa = ' '.join(partes)

    return vista_previa


def mostrar_html_aislado(contenido_html, altura=700):
    iframe = (
        '<iframe '
        f'srcdoc="{escape(contenido_html, quote=True)}" '
        'style="width:100%;'
        f'height:{altura}px;'
        'border:1px solid #d0d7de;'
        'border-radius:8px;'
        'background:white;">'
        '</iframe>'
    )

    visionado = HTML(iframe)
    display(visionado)


# -----------------------------------------------------------------------------
# COMPROBACIÓN DE CONTRATOS
# -----------------------------------------------------------------------------
def comprobar_columnas(df, columnas, nombre='DataFrame'):
    """
    Comprueba que un DataFrame contiene todas las columnas requeridas.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f'{nombre} debe ser un DataFrame.')

    faltan = [columna for columna in columnas if columna not in df.columns]

    if faltan:
        raise KeyError(f'Faltan columnas en {nombre}: {faltan}')

    resultado = True

    return resultado


def comprobar_claves(diccionario, claves, nombre='diccionario'):
    """
    Comprueba que un diccionario contiene todas las claves requeridas.
    """
    if not isinstance(diccionario, dict):
        raise TypeError(f'{nombre} debe ser un diccionario.')

    faltan = [clave for clave in claves if clave not in diccionario]

    if faltan:
        raise KeyError(f'Faltan claves en {nombre}: {faltan}')

    resultado = True

    return resultado


def ruta_relativa(ruta):
    """
    Devuelve una ruta relativa al proyecto cuando sea posible.
    """
    ruta = Path(ruta)

    try:
        salida = str(ruta.relative_to(ruta_proyecto))
    except ValueError:
        salida = str(ruta)

    return salida


# -----------------------------------------------------------------------------
# VALIDACIÓN DE ENTRADA Y SALIDA
# -----------------------------------------------------------------------------
def validar_registro_entrada(registro, variables_esperadas):
    """
    Comprueba la estructura general de un registro de entrada.
    """
    if not isinstance(registro, dict):
        raise TypeError('El registro de entrada debe ser un diccionario.')

    variables_recibidas = set(registro)
    variables_esperadas_set = set(variables_esperadas)

    variables_ausentes = [
        variable for variable in variables_esperadas
        if variable not in variables_recibidas
    ]
    variables_desconocidas = sorted(variables_recibidas - variables_esperadas_set)

    if variables_ausentes:
        raise ValueError(f'Faltan {len(variables_ausentes)} variables de entrada.')

    if variables_desconocidas:
        raise ValueError(f'Existen {len(variables_desconocidas)} variables desconocidas.')

    tipos_validos = (str, int, float, bool, type(None), np.integer, np.floating, np.bool_)

    valores_no_validos = [
        variable for variable, valor in registro.items()
        if not isinstance(valor, tipos_validos)
    ]

    if valores_no_validos:
        raise TypeError(f'Existen valores con tipos no admitidos: {valores_no_validos[:10]}')

    valores_no_finitos = [
        variable for variable, valor in registro.items()
        if isinstance(valor, (float, np.floating)) and not np.isfinite(valor)
    ]

    if valores_no_finitos:
        raise ValueError(f'Existen valores numéricos no finitos: {valores_no_finitos[:10]}')

    return registro


def validar_resultado_predictivo(resultado, configuracion):
    """
    Comprueba la coherencia de una salida con la configuración cerrada.
    """
    if not isinstance(resultado, dict):
        raise TypeError('El resultado predictivo debe ser un diccionario.')

    claves_necesarias = {
        'variable_objetivo',
        'descripcion',
        'probabilidad',
        'umbral',
        'clasificacion',
        'familia',
        'modelo',
        'advertencias'
    }

    _ = comprobar_claves(resultado, claves_necesarias, 'resultado_predictivo')

    target = resultado['variable_objetivo']

    if target not in configuracion['targets']:
        raise ValueError(f'Variable objetivo desconocida: {target}')

    if resultado['descripcion'] != configuracion['titulos_targets'][target]:
        raise ValueError('La descripción no coincide con la configuración cerrada.')

    if resultado['familia'] != configuracion['familias_finales'][target]:
        raise ValueError('La familia no coincide con la selección definitiva.')

    if resultado['modelo'] != configuracion['modelos_finales'][target]:
        raise ValueError('El modelo no coincide con la selección definitiva.')

    if not np.isclose(resultado['umbral'], configuracion['umbrales'][target]):
        raise ValueError('El umbral no coincide con la configuración cerrada.')

    clasificacion_esperada = int(resultado['probabilidad'] >= resultado['umbral'])

    if resultado['clasificacion'] != clasificacion_esperada:
        raise ValueError('La clasificación no es coherente con la probabilidad y el umbral.')

    return resultado


def ejecutar_bridge_predictivo(solicitud, ruta_conda, ruta_servicio, timeout=180):
    """
    Ejecuta una solicitud JSON en el servicio predictivo de TFM_ML.
    """
    proceso = subprocess.run(
        [
            str(ruta_conda), 'run', '--no-capture-output', '-n', 'TFM_ML',
            'python', str(ruta_servicio)
        ],
        input=json.dumps(solicitud, ensure_ascii=False, allow_nan=False, default=convertir_json),
        capture_output=True, text=True, check=False, timeout=timeout
    )

    if proceso.returncode != 0:
        raise RuntimeError(f'No se ha podido ejecutar el servicio predictivo:\n {proceso.stderr}')

    lineas = [linea.strip() for linea in proceso.stdout.splitlines() if linea.strip()]

    if not lineas:
        raise RuntimeError('El servicio predictivo no ha devuelto ninguna respuesta.')

    respuesta = json.loads(lineas[-1])

    return respuesta


# -----------------------------------------------------------------------------
# RAG Y RECUPERACIÓN DOCUMENTAL
# -----------------------------------------------------------------------------
def localizar_fragmentos_codebook(lector, patrones):
    """
    Localiza los fragmentos relevantes y conserva el texto del codebook.
    """
    mejores_fragmentos = {}
    textos_paginas = []
    n_paginas = len(lector.pages)

    for numero_pagina, pagina in enumerate(lector.pages, start=1):
        texto_original = pagina.extract_text() or ''
        textos_paginas.append(texto_original)

        texto = ' '.join(texto_original.split())

        for variable, patron in patrones.items():
            coincidencia = patron.search(texto)
            if coincidencia is None:
                continue

            puntuacion = (
                5 * int(f'{variable} Len' in texto) + 2 * int('Freq Pct' in texto)
                + min(len(patron.findall(texto)), 3)
            )

            if (
                variable not in mejores_fragmentos or
                puntuacion > mejores_fragmentos[variable]['puntuacion']
            ):
                inicio = max(0, coincidencia.start() - 500)
                fin = min(len(texto), coincidencia.start() + 1800)

                mejores_fragmentos[variable] = {
                    'pagina': numero_pagina,
                    'fragmento': texto[inicio:fin],
                    'puntuacion': puntuacion
                }

        if numero_pagina % 100 == 0 or numero_pagina == n_paginas:
            print(f'      Páginas revisadas: {numero_pagina}/{n_paginas}', flush=True)

    texto_codebook = '\n'.join(textos_paginas)

    return mejores_fragmentos, texto_codebook


def clave_documento_rag(metadata):
    """
    Genera una clave única a partir de los metadatos documentales.
    """
    campos = ['fuente', 'documento', 'tipo', 'variable', 'pagina', 'registro']
    clave = tuple(metadata.get(campo) for campo in campos)

    return clave


def preparar_resultado_rag(documento, metodo, similitud=None):
    """
    Prepara una recuperación documental con sus metadatos de trazabilidad.
    """
    metadata = documento.metadata

    resultado = {
        'metodo': metodo,
        'fuente': metadata.get('fuente'),
        'tipo': metadata.get('tipo'),
        'variable': metadata.get('variable'),
        'pagina': metadata.get('pagina'),
        'registro': metadata.get('registro'),
        'similitud': similitud,
        'fragmento': documento.get_content()
    }

    return resultado


def recuperar_documentacion(consulta, top_k=3):
    """
    Recupera documentación mediante coincidencia exacta y similitud semántica.
    """
    if not isinstance(consulta, str) or not consulta.strip():
        raise ValueError('La consulta documental debe contener texto.')

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError('top_k debe ser un entero positivo.')

    consulta = consulta.strip()
    variables_consulta = [
        variable for variable, patron in patrones_variables.items() if patron.search(consulta)
    ]

    documentos_exactos = [
        documento for documento in documentos_rag
        if documento.metadata.get('variable') in variables_consulta
    ]
    salida = [
        preparar_resultado_rag(documento, 'Coincidencia exacta') for documento in documentos_exactos
    ]
    claves = {
        clave_documento_rag(documento.metadata) for documento in documentos_exactos
    }

    recuperados = indice_rag.as_retriever(similarity_top_k=max(top_k * 2, 6)).retrieve(consulta)

    for nodo in recuperados:
        clave = clave_documento_rag(nodo.metadata)
        if clave in claves:
            continue

        salida.append(preparar_resultado_rag(nodo, 'Semántica', float(nodo.score or 0.0)))
        claves.add(clave)

        if len(salida) >= top_k:
            break

    resultado = salida[:top_k]

    return resultado


# -----------------------------------------------------------------------------
# FLUJO LANGGRAPH
# -----------------------------------------------------------------------------
def registrar_paso_flujo(trazabilidad, nodo, entrada, salida, tiempo_s):
    """
    Añade un paso a la trazabilidad del flujo.
    """
    registro = {
        'paso': len(trazabilidad) + 1, 'nodo': nodo,
        'entrada': entrada, 'salida': salida,
        'tiempo_s': round(tiempo_s, 3)
    }

    trazabilidad_actualizada = trazabilidad + [registro]

    return trazabilidad_actualizada


def nodo_validar_entrada(estado):
    """
    Valida el registro antes de ejecutar la capacidad predictiva.
    """
    print('[1/5] Validando el registro de entrada...', flush=True)
    inicio = perf_counter()

    entrada = EntradaPredictiva(
        registro=estado['registro'],
        solicitar_explicacion=estado.get('solicitar_explicacion', False)
    )

    _ = validar_registro_entrada(entrada.registro, vars_predictoras_modelado_dl)

    trazabilidad = registrar_paso_flujo(
        estado.get('trazabilidad', []), 'validar_entrada',
        f'Registro recibido con {len(estado["registro"])} variables',
        f'Entrada válida con {len(entrada.registro)} variables', perf_counter() - inicio
    )

    salida = {'registro': entrada.registro, 'trazabilidad': trazabilidad}

    return salida


def nodo_predecir_indicadores(estado):
    """
    Ejecuta la tool predictiva local seleccionada.
    """
    print('[2/5] Ejecutando la tool predictiva local...', flush=True)
    inicio = perf_counter()

    respuesta = tool_predictiva_local.invoke({'registro': estado['registro']})

    if respuesta.get('estado') != 'OK' or len(respuesta.get('resultados', [])) != len(targets):
        raise RuntimeError('La tool predictiva no ha devuelto las cuatro salidas esperadas.')

    predicciones = respuesta['resultados']
    trazabilidad = registrar_paso_flujo(
        estado['trazabilidad'], 'predecir_indicadores', 'Entrada validada con 813 variables',
        f'{len(predicciones)} predicciones estructuradas obtenidas', perf_counter() - inicio
    )

    salida = {
        'predicciones': predicciones,
        'trazabilidad': trazabilidad
    }

    return salida


def nodo_explicar_resultados(estado):
    """
    Recupera mediante el bridge las explicaciones SHAP solicitadas.
    """
    objetivos = estado.get('objetivos_explicacion', [])

    if not objetivos or not set(objetivos).issubset(targets):
        raise ValueError('Las variables objetivo solicitadas para SHAP no son correctas.')

    print(f'[3/5] Generando {len(objetivos)} explicaciones SHAP...', flush=True)
    inicio = perf_counter()
    explicaciones = []

    for target in objetivos:
        print(f'      {target} — {titulos_targets[target]}', flush=True)

        respuesta = ejecutar_bridge_predictivo(
            {
                'operacion': 'explain',
                'registro': estado['registro'],
                'variable_objetivo': target
            },
            ruta_conda, ruta_servicio_predictivo
        )

        if respuesta.get('estado') != 'OK':
            raise RuntimeError(respuesta.get('mensaje', f'Error al explicar {target}.'))

        explicaciones.append(respuesta['explicacion'])

    variables_explicacion = list(dict.fromkeys(
        variable['variable'] for explicacion in explicaciones
        for variable in explicacion['variables_principales'][:n_variables_contexto_shap]
    ))

    trazabilidad = registrar_paso_flujo(
        estado['trazabilidad'], 'explicar_resultados',
        f'{len(objetivos)} variables objetivo solicitadas',
        f'{len(explicaciones)} explicaciones y {len(variables_explicacion)} variables principales',
        perf_counter() - inicio
    )

    salida = {
        'explicaciones': explicaciones,
        'variables_explicacion': variables_explicacion,
        'trazabilidad': trazabilidad
    }

    return salida


def nodo_recuperar_documentacion(estado):
    """
    Recupera mediante la tool RAG el contexto documental necesario.
    """
    print('[4/5] Recuperando contexto documental...', flush=True)
    inicio = perf_counter()

    consultas = [f'¿Cómo define NSDUH la variable {target}?' for target in targets]
    consultas += [
        f'¿Cómo define NSDUH la variable {variable}?'
        for variable in estado.get('variables_explicacion', [])
    ]
    consultas += [
        '¿Los valores SHAP permiten afirmar causalidad?',
        '¿Puede utilizarse el sistema con finalidad diagnóstica?',
        '¿La probabilidad estimada equivale a riesgo clínico?'
    ]
    consultas = list(dict.fromkeys(consultas))

    contexto = []

    for consulta in consultas:
        recuperados = tool_rag.invoke({'consulta': consulta})

        if not recuperados:
            raise RuntimeError(f'No se ha recuperado documentación para: {consulta}')

        contexto.append({'consulta': consulta, **recuperados[0]})

    trazabilidad = registrar_paso_flujo(
        estado['trazabilidad'], 'recuperar_documentacion',
        f'{len(consultas)} consultas documentales',
        f'{len(contexto)} documentos principales recuperados',
        perf_counter() - inicio
    )

    salida = {
        'consultas_documentales': consultas,
        'contexto_documental': contexto,
        'trazabilidad': trazabilidad
    }

    return salida


def nodo_preparar_contexto(estado):
    """
    Consolida la información que utilizará posteriormente la capa generativa.
    """
    print('[5/5] Preparando el contexto para generación...', flush=True)
    inicio = perf_counter()

    predicciones = estado['predicciones']
    explicaciones = estado.get('explicaciones', [])
    documentacion = estado.get('contexto_documental', [])

    vista_previa = construir_vista_previa_flujo(predicciones, explicaciones, documentacion)

    contexto_generacion = {
        'predicciones': predicciones,
        'explicaciones': explicaciones,
        'documentacion': documentacion,
        'advertencias': predicciones[0].get('advertencias', []),
        'vista_previa': vista_previa
    }

    trazabilidad = registrar_paso_flujo(
        estado['trazabilidad'], 'preparar_contexto',
        f'{len(predicciones)} predicciones, {len(explicaciones)} explicaciones y '
        f'{len(documentacion)} documentos',
        'Contexto estructurado preparado para la capa generativa',
        perf_counter() - inicio
    )

    salida = {
        'contexto_generacion': contexto_generacion,
        'vista_previa_informe': vista_previa,
        'trazabilidad': trazabilidad
    }

    return salida


def decidir_despues_prediccion(estado):
    """
    Selecciona la ruta posterior a la predicción.
    """
    if estado.get('solicitar_explicacion') and estado.get('objetivos_explicacion'):
        ruta = 'explicar'
    elif estado.get('solicitar_documentacion'):
        ruta = 'documentar'
    else:
        ruta = 'preparar'

    return ruta


def decidir_despues_explicacion(estado):
    """
    Selecciona la ruta posterior a la explicación SHAP.
    """
    salida = 'documentar' if estado.get('solicitar_documentacion') else 'preparar'

    return salida


# -----------------------------------------------------------------------------
# GENERACIÓN Y GUARDRAILS
# -----------------------------------------------------------------------------
def limpiar_descripcion_factor(texto, variable=None):
    """
    Limpia prefijos técnicos de una descripción destinada a presentación.
    """
    salida = re.sub(
        r'^\s*variable_origen\s*[:\-–]\s*', '', str(texto).strip(), flags=re.IGNORECASE
    )

    if variable:
        salida = re.sub(
            rf'^\s*{re.escape(variable)}(?:\s*[:\-–]\s*|\s+)', '', salida, flags=re.IGNORECASE
        )

    salida = re.sub(r'\s*\((?:recode|recoded)\)\s*', '', salida, flags=re.IGNORECASE).strip()

    # -----------------------------------------------------------------------------
    # DESCRIPCIONES CONTROLADAS
    # -----------------------------------------------------------------------------
    descripciones_controladas = {
        'CAMHPROB': ('Percepción de haber tenido alguna vez un problema de salud mental'),
        'CAMHPROB2': ('Percepción de haber tenido alguna vez un problema de salud mental'),
        'MOVSINPYR2': ('Número de veces que se mudó en los últimos 12 meses')
    }

    if variable in descripciones_controladas:
        salida = descripciones_controladas[variable]

    if salida:
        salida = salida[0].upper() + salida[1:]

    return salida


def localizar_bloque_variable_codebook(variable, descripcion_original):
    """
    Localiza el bloque correspondiente a una variable en el codebook oficial.
    """
    patron_variable = (
        re.escape(variable) if re.search(r'\d$', variable)
        else rf'{re.escape(variable)}(?:\s*[12])?'
    )

    patron_cabecera = re.compile(r'(?im)^\s*[A-Z][A-Z0-9_]*(?:\s*[12])?\s+Len\s*:')

    coincidencia = re.search(rf'(?im)^\s*{patron_variable}\s+Len\s*:', texto_codebook_completo)

    if coincidencia is not None:
        siguiente = patron_cabecera.search(texto_codebook_completo, coincidencia.end())

        fin = (
            siguiente.start() if siguiente is not None
            else min(len(texto_codebook_completo), coincidencia.end() + 5000)
        )

        return texto_codebook_completo[coincidencia.start():fin]

    descripcion = ' '.join(str(descripcion_original).split()).strip()

    if descripcion:
        patron_descripcion = r'\s+'.join(re.escape(parte) for parte in descripcion.split())

        coincidencia = re.search(patron_descripcion, texto_codebook_completo, re.IGNORECASE)

        if coincidencia is not None:
            inicio_busqueda = max(0, coincidencia.start() - 1000)

            anteriores = list(
                patron_cabecera.finditer(
                    texto_codebook_completo, inicio_busqueda, coincidencia.start() + 1
                )
            )

            if anteriores:
                cabecera = anteriores[-1]
                siguiente = patron_cabecera.search(texto_codebook_completo, cabecera.end())

                fin = (
                    siguiente.start() if siguiente is not None
                    else min(len(texto_codebook_completo), cabecera.end() + 5000)
                )

                return texto_codebook_completo[cabecera.start():fin]

    return ''


def traducir_respuesta_codebook(etiqueta):
    """
    Limpia y traduce una respuesta del codebook sin incorporar Freq/Pct.
    """
    salida = ' '.join(str(etiqueta).split()).strip()

    salida = re.sub(r'\.{2,}\s*\d+\s+\d+(?:\.\d+)?\s*$', '', salida)

    salida = re.sub(r'\s{2,}\d+\s+\d+(?:\.\d+)?\s*$', '', salida)

    salida = re.sub(r'\s*\((?=[^)]*=)[^)]*\)\s*$', '', salida).strip(' .')

    clave = salida.upper()

    if clave.startswith('LEGIT SKIP MALES/FEMALES NOT AGED 12-50'):
        return ('No aplicable: hombre o mujer fuera del intervalo de edad correspondiente')

    if clave.startswith('LEGITIMATE SKIP'):
        return 'No aplicable según el flujo del cuestionario'

    if clave.startswith('BAD DATA'):
        return 'Dato no válido según las reglas del cuestionario'

    if clave.startswith('BLANK (NO ANSWER)'):
        return 'Sin respuesta'

    if clave.startswith("DON'T KNOW"):
        return 'No sabe'

    if clave.startswith('REFUSED'):
        return 'Prefiere no responder'

    traducciones = {
        'YES': 'Sí',
        'NO': 'No',
        'NO/NO ISSUE': 'No / no se identificó el problema',
        'NO DIFFICULTY': 'Sin dificultad',
        'SOME DIFFICULTY': 'Alguna dificultad',
        'A LOT OF DIFFICULTY OR CANNOT DO AT ALL':
            'Mucha dificultad o no puede hacerlo',
        'EXCELLENT': 'Excelente',
        'VERY GOOD': 'Muy buena',
        'GOOD': 'Buena',
        'FAIR': 'Regular',
        'POOR': 'Mala',
        'FAIR/POOR': 'Regular o mala',
        '0 TIMES': 'Ninguna vez',
        'ONE TIME': 'Una vez',
        'TWO TIMES': 'Dos veces',
        'THREE OR MORE TIMES': 'Tres o más veces',
        'STRONGLY DISAGREE': 'Totalmente en desacuerdo',
        'DISAGREE': 'En desacuerdo',
        'AGREE': 'De acuerdo',
        'STRONGLY AGREE': 'Totalmente de acuerdo',
        'NOT PREGNANT': 'No embarazada',
        'PREGNANT': 'Embarazada',
        'PROBATION': 'Libertad vigilada',
        'EVER USED': 'Sí, alguna vez',
        'SCHOOL NOT IN SESSION': 'Fuera del periodo lectivo',
    }

    if clave in traducciones:
        return traducciones[clave]

    coincidencia_dias = re.fullmatch(r'(\d+)\s*-\s*(\d+)\s+DAYS', salida, flags=re.IGNORECASE)

    if coincidencia_dias:
        return (f'{coincidencia_dias.group(1)}–{coincidencia_dias.group(2)} días')

    coincidencia_sexo_edad = re.fullmatch(
        r'(Males|Females)\s+Aged\s+(\d+)-(\d+)', salida, flags=re.IGNORECASE
    )

    if coincidencia_sexo_edad:
        sexo, edad_minima, edad_maxima = coincidencia_sexo_edad.groups()
        sexo = 'Hombre' if sexo.lower() == 'males' else 'Mujer'

        return (f'{sexo} de {edad_minima} a {edad_maxima} años')

    coincidencia_ingresos = re.fullmatch(
        r'\$(\d{1,3}(?:,\d{3})*)\s*-\s*\$(\d{1,3}(?:,\d{3})*)', salida
    )

    if coincidencia_ingresos:
        minimo, maximo = coincidencia_ingresos.groups()
        minimo = minimo.replace(',', '.')
        maximo = maximo.replace(',', '.')

        return f'Entre {minimo} y {maximo} dólares'

    return salida


def formatear_valor_numerico_codebook(valor, descripcion_original):
    """
    Añade la unidad cuando el codebook define un valor numérico directo.
    """
    valor = int(valor) if float(valor).is_integer() else float(valor)
    descripcion = descripcion_original.upper()

    if 'AGE' in descripcion:
        return f'{valor} años'

    if '# OF TIMES' in descripcion or 'NUMBER OF TIMES' in descripcion:
        return f'{valor} veces'

    if 'DAY' in descripcion:
        return f'{valor} días'

    return str(valor)


def obtener_respuesta_observada(variable, descripcion_original, valor_observado):
    """
    Obtiene el significado del valor registrado utilizando el codebook.
    """
    if valor_observado is None:
        return 'Sin respuesta disponible'

    try:
        valor = float(valor_observado)
        valor = int(valor) if valor.is_integer() else valor
    except (TypeError, ValueError):
        valor = valor_observado

    bloque = localizar_bloque_variable_codebook(variable, descripcion_original)

    if bloque:
        coincidencia = re.search(rf'(?m)^\s*{re.escape(str(valor))}\s*=\s*([^\r\n]+)', bloque)

        if coincidencia is not None:
            return traducir_respuesta_codebook(coincidencia.group(1))

        rango = re.search(r'(?im)^\s*RANGE\s*=\s*(-?\d+)\s*-\s*(-?\d+)', bloque)

        if (
            rango is not None and isinstance(valor, (int, float))
            and int(rango.group(1)) <= valor <= int(rango.group(2))
        ):
            return formatear_valor_numerico_codebook(valor, descripcion_original)

    if isinstance(valor, (int, float)):
        return formatear_valor_numerico_codebook(valor, descripcion_original)

    return str(valor)


def obtener_aclaracion_variable(variable, descripcion_original):
    """
    Explica de forma comprensible las variables derivadas cuando procede.
    """
    descripcion = descripcion_original.upper()

    if variable == 'CAMHPROB2':
        return (
            'NSDUH obtiene esta variable recodificando en Sí/No la respuesta '
            'original sobre si la persona considera haber tenido alguna vez '
            'un problema de salud mental; no corresponde a una pregunta '
            'independiente.'
        )

    if variable == 'RCVYMHPRB':
        return (
            'NSDUH obtiene esta variable combinando la respuesta sobre haber '
            'tenido un problema de salud mental con la respuesta sobre '
            'recuperación; no corresponde a una pregunta independiente.'
        )

    if variable == 'EDUSCKCOM':
        return (
            'NSDUH obtiene esta variable combinando las respuestas relativas '
            'a las ausencias escolares según las reglas establecidas en el '
            'codebook; no corresponde a una pregunta independiente.'
        )

    if variable == 'SEXRACE':
        return (
            'NSDUH combina la información de sexo y raza/etnia en una única '
            'categoría derivada.'
        )

    if 'COMBINED' in descripcion:
        return (
            'NSDUH obtiene esta variable combinando varias respuestas originales según las reglas '
            'de su documentación oficial; no corresponde a una pregunta independiente.'
        )

    if 'RECODED' in descripcion or re.match(r'^\s*RC\s*-', descripcion):
        return (
            'NSDUH obtiene esta variable recodificando una respuesta original según las reglas '
            'de su documentación oficial; no corresponde a una pregunta independiente.'
        )

    return None


def seleccionar_factores_ampliados(explicacion, n_riesgo=3, n_proteccion=2):
    """
    Selecciona contribuciones SHAP positivas y negativas para presentación.
    """
    limites = {'positiva': n_riesgo, 'negativa': n_proteccion}
    factores = []

    for signo, limite in limites.items():
        seleccion = [
            variable for variable in explicacion['variables_principales']
            if variable['signo_contribucion'].lower() == signo
        ][:limite]

        for posicion, variable in enumerate(seleccion, start=1):
            factores.append({
                'variable_objetivo': explicacion['variable_objetivo'],
                'descripcion_objetivo': explicacion['descripcion'],
                'variable_origen': variable['variable'],
                'descripcion_original': (
                    variable.get('etiqueta_variable')
                    or variable.get('descripcion')
                    or variable['variable']
                ),
                'tipo_factor': frases_informe['tipos_factor'][signo],
                'posicion_factor': posicion,
                'valor_observado': variable.get('valor_observado'),
                'valor_shap': variable['valor_shap']
            })

    return factores


def construir_detalle_explicabilidad(factores_ampliados, traducciones):
    """
    Construye la vista ampliada sin modificar las explicaciones SHAP.
    """
    traducciones_por_variable = {
        traduccion.variable_origen: limpiar_descripcion_factor(
            traduccion.descripcion, traduccion.variable_origen
        )
        for traduccion in traducciones
    }

    variables_necesarias = {factor['variable_origen'] for factor in factores_ampliados}

    if not variables_necesarias.issubset(traducciones_por_variable):
        raise ValueError('Faltan traducciones para variables de la explicación ampliada.')

    detalle = []

    for factor in factores_ampliados:
        detalle.append({
            **factor,
            'descripcion': traducciones_por_variable[factor['variable_origen']],
            'respuesta_observada': obtener_respuesta_observada(
                factor['variable_origen'], factor['descripcion_original'], factor['valor_observado']
            ),
            'nota_variable': obtener_aclaracion_variable(
                factor['variable_origen'], factor['descripcion_original']
            )
        })

    return detalle


def preparar_contexto_generativo(estado, n_variables_shap=3, max_caracteres_fragmento=900):
    """
    Prepara la información validada que puede recibir el modelo generativo.
    """
    predicciones = [
        {
            'variable_objetivo': resultado['variable_objetivo'],
            'descripcion': resultado['descripcion'],
            'decision': (
                'supera el umbral validado'
                if resultado['clasificacion'] == 1 else 'no supera el umbral validado'
            ),
            'familia': resultado['familia'],
            'modelo': resultado['modelo']
        }
        for resultado in estado['predicciones']
    ]

    explicaciones = []
    factores_permitidos = []
    factores_ampliados = []

    for explicacion in estado.get('explicaciones', []):
        variables_validas = [
            variable for variable in explicacion['variables_principales']
            if variable['signo_contribucion'].lower() in frases_informe['tipos_factor']
        ]

        if len(variables_validas) < n_variables_shap:
            raise ValueError(
                f"No existen {n_variables_shap} contribuciones SHAP válidas "
                f"para {explicacion['variable_objetivo']}."
            )

        factores = []

        for posicion, variable in enumerate(variables_validas[:n_variables_shap], start=1):
            signo = variable['signo_contribucion'].lower()

            factor = {
                'variable_objetivo': explicacion['variable_objetivo'],
                'descripcion_objetivo': explicacion['descripcion'],
                'variable_origen': variable['variable'],
                'descripcion_original': (
                    variable.get('etiqueta_variable') or variable.get('descripcion') or
                    variable['variable']
                ),
                'tipo_factor': frases_informe['tipos_factor'][signo],
                'posicion_factor': posicion,
                'valor_observado': variable.get('valor_observado')
            }

            factores.append(factor)
            factores_permitidos.append(factor)

        factores_ampliados.extend(seleccionar_factores_ampliados(explicacion))

        prediccion_objetivo = next(
            resultado for resultado in predicciones
            if resultado['variable_objetivo'] == explicacion['variable_objetivo']
        )

        explicaciones.append({
            'variable_objetivo': explicacion['variable_objetivo'],
            'descripcion': explicacion['descripcion'],
            'decision': prediccion_objetivo['decision'],
            'factores': factores
        })

    variables_traducir = {}

    for factor in factores_permitidos + factores_ampliados:
        variables_traducir.setdefault(
            factor['variable_origen'],
            {
                'variable_origen': factor['variable_origen'],
                'descripcion_original': factor['descripcion_original']
            }
        )

    documentacion = [
        {
            'consulta': documento['consulta'],
            'fuente': documento['fuente'],
            'tipo': documento['tipo'],
            'variable': documento.get('variable'),
            'pagina': documento.get('pagina'),
            'registro': documento.get('registro'),
            'fragmento': documento['fragmento'][:max_caracteres_fragmento]
        }
        for documento in estado.get('contexto_documental', [])
    ]

    contexto = {
        'predicciones': predicciones,
        'explicaciones': explicaciones,
        'factores_permitidos': factores_permitidos,
        'factores_ampliados': factores_ampliados,
        'variables_traducir': list(variables_traducir.values()),
        'documentacion': documentacion,
        'advertencias': estado['contexto_generacion'].get('advertencias', []),
        'descripciones_objetivos': list(titulos_targets.values()),
        'orientaciones_permitidas': frases_informe['orientacion_preventiva'],
        'limitaciones_obligatorias': frases_informe['limitaciones_obligatorias'],
        'advertencia_obligatoria': frases_informe['advertencia_uso'],
        'nota_factores': frases_informe['nota_factores'],
        'fuentes_permitidas': sorted({documento['fuente'] for documento in documentacion})
    }

    return contexto


def formatear_informe_preventivo(informe):
    """
    Convierte el contrato del informe en una salida textual legible.
    """
    datos = informe.model_dump(mode='json') if isinstance(informe, BaseModel) else informe

    objetivos = list(dict.fromkeys(
        factor['descripcion_objetivo'] for factor in datos['factores_relevantes']
    ))

    bloques_factores = []

    for objetivo in objetivos:
        factores = [
            factor for factor in datos['factores_relevantes']
            if factor['descripcion_objetivo'] == objetivo
        ]
        factores = sorted(factores, key=lambda factor: factor['posicion_factor'])

        riesgo = [
            factor['descripcion'] for factor in factores
            if factor['tipo_factor'] == 'Factor de riesgo para la predicción'
        ]
        protectores = [
            factor['descripcion'] for factor in factores
            if factor['tipo_factor'] == 'Factor protector para la predicción'
        ]

        bloque = [objetivo]

        if riesgo:
            bloque.append('Factores de riesgo para la predicción:')
            bloque.extend(f'- {factor}' for factor in riesgo)

        if protectores:
            bloque.append('Factores protectores para la predicción:')
            bloque.extend(f'- {factor}' for factor in protectores)

        bloques_factores.append('\n'.join(bloque))

    texto = (
        f"RESUMEN\n{datos['resumen']}\n\n"
        f"INTERPRETACIÓN\n{datos['interpretacion']}\n\n"
        f"FACTORES RELEVANTES POR INDICADOR\n\n"
        + '\n\n'.join(bloques_factores)
        + f"\n\nNOTA SOBRE LOS FACTORES\n{frases_informe['nota_factores']}"
        + "\n\nORIENTACIÓN PREVENTIVA\n"
        + '\n'.join(f"- {valor}" for valor in datos['orientacion_preventiva'])
        + "\n\nLIMITACIONES\n"
        + '\n'.join(f"- {valor}" for valor in datos['limitaciones'])
        + "\n\nFUENTES\n"
        + '\n'.join(f"- {valor}" for valor in datos['fuentes'])
        + f"\n\nADVERTENCIA DE USO\n{datos['advertencia_uso']}"
    )

    return texto


def generar_informe_mistral(modelo_estructurado, contexto, prompt_sistema):
    """
    Genera mediante Mistral únicamente el contenido permitido.
    """
    claves_modelo = [
        'predicciones',
        'variables_traducir',
        'documentacion',
        'advertencias',
        'descripciones_objetivos'
    ]

    contexto_modelo = {clave: contexto[clave] for clave in claves_modelo}

    contexto_json = json.dumps(
        contexto_modelo, ensure_ascii=False, default=convertir_json, allow_nan=False
    )

    mensaje_usuario = (
        'Genera el contenido solicitado utilizando exclusivamente el contexto '
        'validado que se proporciona a continuación.\n\n' + contexto_json
    )

    inicio = perf_counter()

    respuesta = modelo_estructurado.invoke([
        ('system', prompt_sistema),
        ('human', mensaje_usuario)
    ])

    tiempo_s = perf_counter() - inicio

    if (respuesta.get('parsing_error') is not None or respuesta.get('parsed') is None):
        raise RuntimeError(
            f'La respuesta de Mistral no cumple el contrato: {respuesta["parsing_error"]}'
        )

    salida = respuesta['parsed']

    if not isinstance(salida, SalidaGenerativa):
        salida = SalidaGenerativa.model_validate(salida)

    raw = respuesta.get('raw')
    uso_tokens = getattr(raw, 'usage_metadata', {}) or {}

    resultado = {
        'salida': salida,
        'tiempo_s': tiempo_s,
        'uso_tokens': uso_tokens
    }

    return resultado


def construir_factores_informe(factores_permitidos, traducciones):
    """
    Construye determinísticamente los factores finales a partir de SHAP
    y de las traducciones generadas.
    """
    traducciones_por_variable = {
        traduccion.variable_origen: limpiar_descripcion_factor(
            traduccion.descripcion, traduccion.variable_origen
        )
        for traduccion in traducciones
    }

    variables_esperadas = {factor['variable_origen'] for factor in factores_permitidos}

    if not variables_esperadas.issubset(traducciones_por_variable):
        raise ValueError('Las traducciones generadas no cubren los factores del informe.')

    factores = []

    for factor in factores_permitidos:
        respuesta_observada = obtener_respuesta_observada(
            factor['variable_origen'], factor['descripcion_original'], factor['valor_observado']
        )

        nota_variable = obtener_aclaracion_variable(
            factor['variable_origen'], factor['descripcion_original']
        )

        factores.append(FactorInforme(
            variable_objetivo=factor['variable_objetivo'],
            descripcion_objetivo=factor['descripcion_objetivo'],
            tipo_factor=factor['tipo_factor'],
            variable_origen=factor['variable_origen'],
            descripcion=traducciones_por_variable[factor['variable_origen']],
            respuesta_observada=respuesta_observada,
            nota_variable=nota_variable,
            posicion_factor=factor['posicion_factor']
        ))

    return factores


def construir_informe_preventivo(salida_generativa, contexto):
    """
    Construye el informe final combinando generación y contenido determinista.
    """
    factores = construir_factores_informe(
        contexto['factores_permitidos'], salida_generativa.traducciones_factores
    )

    informe = InformePreventivo(
        resumen=salida_generativa.resumen,
        interpretacion=salida_generativa.interpretacion,
        factores_relevantes=factores,
        orientacion_preventiva=contexto['orientaciones_permitidas'][:3],
        limitaciones=contexto['limitaciones_obligatorias'],
        fuentes=contexto['fuentes_permitidas'],
        advertencia_uso=contexto['advertencia_obligatoria']
    )

    return informe


def evaluar_guardrails_informe(
    informe, descripciones_objetivos, factores_permitidos, fuentes_permitidas, predicciones, frases
):
    """
    Evalúa reglas deterministas sobre el informe generado.
    """
    datos = (
        informe.model_dump(mode='json') if isinstance(informe, BaseModel) else informe
    )

    factores = datos['factores_relevantes']

    factores_esperados = {
        (factor['variable_objetivo'], factor['variable_origen']): factor
        for factor in factores_permitidos
    }

    factores_generados = {
        (factor['variable_objetivo'], factor['variable_origen']): factor
        for factor in factores
    }

    factores_exactos = (
        len(factores) == len(factores_permitidos) and set(factores_generados) ==
        set(factores_esperados)
    )

    tipos_correctos = factores_exactos and all(
        factor['tipo_factor'] ==
        factores_esperados[(factor['variable_objetivo'], factor['variable_origen'])]['tipo_factor']
        for factor in factores
    )

    factores_por_objetivo = pd.Series([
        factor['variable_objetivo'] for factor in factores
    ]).value_counts()

    tres_factores_por_objetivo = (
        set(factores_por_objetivo.index) == set(targets) and factores_por_objetivo.eq(3).all()
    )

    objetivos_no_superados = {
        resultado['variable_objetivo'] for resultado in predicciones
        if resultado['decision'] == 'no supera el umbral validado'
    }

    objetivos_no_superados_con_factores = (
        objetivos_no_superados.issubset(set(factores_por_objetivo.index))
    )

    traducciones_correctas = factores_exactos and all(
        factor['descripcion'].strip() and factor['variable_origen'].casefold() not in
        factor['descripcion'].casefold() for factor in factores
    )

    texto_objetivos = (f"{datos['resumen']} {datos['interpretacion']}").lower()

    texto_factores = ' '.join(factor['descripcion'] for factor in factores)

    texto_narrativo = ' '.join([
        datos['resumen'],
        datos['interpretacion'],
        texto_factores,
        ' '.join(datos['orientacion_preventiva']),
        ' '.join(datos['limitaciones']),
        datos['advertencia_uso']
    ]).lower()

    patrones_causales = [
        r'\bcausa\b',
        r'\bcausan\b',
        r'\bprovoca\b',
        r'\bprovocan\b',
        r'\bdemuestra que\b',
        r'\bdetermina\b',
        r'\bdeterminan\b'
    ]

    patrones_diagnosticos = [
        r'\bpadece\b',
        r'\bdiagnóstico confirmado\b',
        r'\bconfirma un diagnóstico\b',
        r'\bdiagnosticado\b',
        r'\bdiagnosticada\b'
    ]

    patrones_prescriptivos = [
        r'\bdosis\b',
        r'\bprescrib',
        r'\bmedicamento\b',
        r'\bfármaco\b'
    ]

    comprobaciones = pd.DataFrame({
        'comprobacion': [
            'El informe mantiene los siete campos contractuales',
            'Las cuatro variables objetivo aparecen en la interpretación',
            'Se generan exactamente tres factores por variable objetivo',
            'Los factores proceden exclusivamente de SHAP',
            'El tipo de factor coincide con la dirección SHAP',
            'Las descripciones visibles se presentan traducidas',
            'Las fuentes coinciden con las fuentes documentales permitidas',
            'La orientación utiliza únicamente el banco de frases',
            'Las limitaciones obligatorias se mantienen exactamente',
            'La advertencia de uso coincide con el banco de frases',
            'No se formulan relaciones causales',
            'No se formulan afirmaciones diagnósticas',
            'La narración no introduce cifras predictivas',
            'La orientación no contiene indicaciones farmacológicas',
            'Los indicadores que no superan el umbral también conservan sus factores'
        ],
        'resultado': [
            set(datos) == set(InformePreventivo.model_fields),
            all(descripcion.lower() in texto_objetivos for descripcion in descripciones_objetivos),
            tres_factores_por_objetivo,
            factores_exactos,
            tipos_correctos,
            traducciones_correctas,
            set(datos['fuentes']) == set(fuentes_permitidas),
            len(datos['orientacion_preventiva']) == 3 and
            set(datos['orientacion_preventiva']).issubset(frases['orientacion_preventiva']),
            set(datos['limitaciones']) == set(frases['limitaciones_obligatorias']),
            datos['advertencia_uso'] == frases['advertencia_uso'],
            not any(re.search(patron, texto_narrativo) for patron in patrones_causales),
            not any(re.search(patron, texto_narrativo) for patron in patrones_diagnosticos),
            re.search(r'\b\d+[.,]\d+\b|%', texto_narrativo) is None,
            not any(re.search(patron, texto_narrativo) for patron in patrones_prescriptivos),
            objetivos_no_superados_con_factores
        ]
    })

    return comprobaciones


# -----------------------------------------------------------------------------
# APLICACIÓN, ACCESO Y VISUALIZACIÓN
# -----------------------------------------------------------------------------
def preparar_tabla_factores_informe(informe):
    """
    Convierte los factores del informe en una tabla trazable.
    """
    tabla = pd.DataFrame([
        factor.model_dump(mode='json') if isinstance(factor, BaseModel) else factor
        for factor in informe.factores_relevantes
    ])

    return tabla


def preparar_respuesta_aplicacion(
    estado, informe, detalle_explicabilidad, modelo_generativo, version_prompt
):
    """
    Consolida la salida determinista y generativa de la aplicación.
    """
    datos_informe = informe.model_dump(mode='json') if isinstance(informe, BaseModel) else informe

    respuesta = {
        'estado': 'OK',
        'fecha_ejecucion': datetime.now().isoformat(timespec='seconds'),
        'version_predictiva': configuracion_productivizacion['version'],
        'modelo_generativo': modelo_generativo,
        'version_prompt': version_prompt,
        'predicciones': estado['predicciones'],
        'factores': datos_informe['factores_relevantes'],
        'detalle_explicabilidad': detalle_explicabilidad,
        'informe': datos_informe,
        'trazabilidad': estado.get('trazabilidad', [])
    }

    return respuesta


def crear_figura_predicciones(predicciones):
    """
    Representa probabilidades y umbrales de los cuatro indicadores.
    """
    datos = pd.DataFrame(predicciones)[
        ['descripcion', 'probabilidad', 'umbral', 'clasificacion']
    ].melt(
        id_vars=['descripcion', 'clasificacion'], value_vars=['probabilidad', 'umbral'],
        var_name='medida', value_name='valor'
    )

    figura = px.bar(
        datos, x='valor', y='descripcion', color='medida', orientation='h', barmode='group',
        color_discrete_map={'probabilidad': '#2B9F99', 'umbral': '#81D8D0'},
        labels={'descripcion': 'Indicador', 'valor': 'Valor', 'medida': ''},
        title='Probabilidad estimada y umbral validado',
        category_orders={'descripcion': [
            'Intento suicida', 'Planificación suicida', 'Ideación suicida',
            'Episodio depresivo mayor'
        ], 'medida': ['probabilidad', 'umbral']}
    )

    figura.update_traces(texttemplate='%{x:.1%}', textposition='outside', cliponaxis=False)

    figura.update_layout(
        height=500, bargap=0.25, plot_bgcolor='white', paper_bgcolor='white',
        font={'color': '#0E4F55', 'size': 13}, title_font={'size': 20, 'color': '#176B70'},
        legend={'orientation': 'h', 'y': 1.12, 'x': 0, 'title': None},
        margin={'l': 190, 'r': 65, 't': 85, 'b': 55}
    )

    figura.update_xaxes(
        range=[0, 1], tickformat='.0%', dtick=0.1, gridcolor='#DCF7F4', zeroline=False
    )

    figura.update_yaxes(title='', automargin=True)

    return figura


def ejecutar_flujo_aplicacion(
    registro, solicitar_explicacion=False, objetivos_explicacion=None, solicitar_documentacion=False
):
    """
    Ejecuta el flujo LangGraph desde una solicitud de aplicación.
    """
    objetivos_explicacion = (
        targets if solicitar_explicacion and objetivos_explicacion is None
        else objetivos_explicacion or []
    )

    estado = aplicacion_flujo.invoke({
        'registro': registro,
        'solicitar_explicacion': solicitar_explicacion,
        'objetivos_explicacion': objetivos_explicacion,
        'solicitar_documentacion': solicitar_documentacion,
        'trazabilidad': []
    })

    return estado


def ejecutar_informe_aplicacion(registro):
    """
    Ejecuta el flujo completo necesario para generar un informe.
    """
    estado = ejecutar_flujo_aplicacion(
        registro, solicitar_explicacion=True, objetivos_explicacion=targets,
        solicitar_documentacion=True
    )

    contexto = preparar_contexto_generativo(
        estado, n_variables_shap=3, max_caracteres_fragmento=900
    )

    generacion = generar_informe_mistral(
        modelo_mistral_estructurado, contexto, prompt_sistema_informe
    )

    informe = construir_informe_preventivo(generacion['salida'], contexto)

    detalle = construir_detalle_explicabilidad(
        contexto['factores_ampliados'], generacion['salida'].traducciones_factores
    )

    comprobaciones = evaluar_guardrails_informe(
        informe, contexto['descripciones_objetivos'], contexto['factores_permitidos'],
        contexto['fuentes_permitidas'], contexto['predicciones'], frases_informe
    )

    if not comprobaciones['resultado'].all():
        raise ValueError('El informe generado no supera los guardrails.')

    salida = preparar_respuesta_aplicacion(
        estado, informe, detalle, modelo_mistral, version_prompt_informe
    )

    return salida


def validar_acceso_api(clave_esperada):
    """
    Comprueba una credencial de acceso sin exponer su valor.
    """
    clave_recibida = request.headers.get('X-API-Key', '')

    respuesta = bool(clave_esperada) and hmac.compare_digest(clave_recibida, clave_esperada)

    return respuesta


def cargar_registro_csv(archivo):
    """
    Recupera y valida un único registro desde un archivo CSV.
    """
    datos = pd.read_csv(archivo, encoding=encoding_csv)

    if len(datos) != 1:
        raise ValueError('El archivo CSV debe contener exactamente un registro.')

    if set(datos.columns) != set(vars_predictoras_modelado_dl):
        raise ValueError('Las columnas del CSV no coinciden con las 813 variables esperadas.')

    registro = datos[vars_predictoras_modelado_dl].iloc[0].to_dict()

    registro = {
        variable: None if pd.isna(valor) else valor for variable, valor in registro.items()
    }

    salida = validar_registro_entrada(registro, vars_predictoras_modelado_dl)

    return salida


def agrupar_factores_interfaz(factores):
    """
    Agrupa factores principales por variable objetivo.
    """
    grupos = {}

    for factor in factores:
        datos = (
            factor.model_dump(mode='json') if isinstance(factor, BaseModel) else factor
        )
        grupos.setdefault(datos['descripcion_objetivo'], []).append(datos)

    return grupos


def agrupar_detalle_interfaz(detalle):
    """
    Agrupa la explicación ampliada por objetivo y dirección.
    """
    grupos = {}

    for factor in detalle:
        objetivo = factor['descripcion_objetivo']
        tipo = ('riesgo' if 'riesgo' in factor['tipo_factor'].lower() else 'protector')
        grupos.setdefault(objetivo, {'riesgo': [], 'protector': []})[tipo].append(factor)

    return grupos


def guardar_informe_aplicacion(respuesta, archivo_origen=None):
    """
    Guarda un informe validado sin persistir el registro original.
    """
    nombre = Path(archivo_origen or 'registro').stem

    coincidencia = re.fullmatch(r'\d{2}_registro_entrada_(demo_\d{2})', nombre)

    id_informe = (
        coincidencia.group(1) if coincidencia else datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    )

    datos = {
        'id_informe': id_informe,
        'fecha': datetime.now().isoformat(timespec='seconds'),
        'archivo_origen': archivo_origen,
        'respuesta': respuesta
    }

    ruta = guardar_json(datos, f'informe_{id_informe}.json', ruta_informes)

    resultado = {
        'id_informe': id_informe,
        'ruta': ruta,
        'datos': datos
    }

    return resultado


def cargar_historial_informes():
    """
    Recupera únicamente los informes compatibles con el contrato vigente.
    """
    informes = []

    campos_respuesta = {
        'estado', 'fecha_ejecucion', 'version_predictiva', 'modelo_generativo', 'version_prompt',
        'predicciones', 'factores', 'detalle_explicabilidad', 'informe', 'trazabilidad'
    }

    campos_detalle = {
        'variable_objetivo', 'descripcion_objetivo', 'tipo_factor', 'descripcion'
    }

    for ruta in ruta_informes.glob('informe_*.json'):
        try:
            datos = json.loads(ruta.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue

        if not isinstance(datos, dict) or not {'id_informe', 'fecha', 'respuesta'}.issubset(datos):
            continue

        respuesta = datos['respuesta']

        if not isinstance(respuesta, dict) or set(respuesta) != campos_respuesta:
            continue

        try:
            predicciones = [
                ResultadoPredictivo.model_validate(resultado)
                for resultado in respuesta['predicciones']
            ]

            factores = [FactorInforme.model_validate(factor) for factor in respuesta['factores']]

            _ = InformePreventivo.model_validate(respuesta['informe'])

            detalle = respuesta['detalle_explicabilidad']
            trazabilidad = respuesta['trazabilidad']

        except (ValidationError, TypeError, KeyError):
            continue

        if (
            respuesta['estado'] != 'OK' or len(predicciones) != len(targets)
            or {resultado.variable_objetivo for resultado in predicciones} != set(targets)
            or len(factores) != 3 * len(targets) or not isinstance(detalle, list)
            or not isinstance(trazabilidad, list)
        ):
            continue

        if not all(
            isinstance(factor, dict) and campos_detalle.issubset(factor) for factor in detalle
        ):
            continue

        if {factor['variable_objetivo'] for factor in detalle} != set(targets):
            continue

        indicadores_superados = sum(resultado.clasificacion == 1 for resultado in predicciones)

        informes.append({
            'id_informe': datos['id_informe'],
            'fecha': datos['fecha'],
            'archivo_origen': datos.get('archivo_origen'),
            'indicadores_superados': indicadores_superados
        })

    informes.sort(key=lambda informe: informe['fecha'], reverse=True)

    return informes


def cargar_informe_aplicacion(id_informe):
    """
    Recupera un informe previamente generado.
    """
    if not re.fullmatch(r'[A-Za-z0-9_-]+', id_informe):
        raise ValueError('Identificador de informe no válido.')

    ruta = ruta_informes / f'informe_{id_informe}.json'

    if not ruta.is_file():
        raise FileNotFoundError(f'No existe el informe {id_informe}.')

    datos = json.loads(ruta.read_text(encoding='utf-8'))

    return datos


# -----------------------------------------------------------------------------
# CONTROL DEL SERVIDOR DE DEMOSTRACIÓN
# -----------------------------------------------------------------------------
def detener_servidor_demo(servidor):
    """
    Detiene de forma controlada el servidor utilizado en la demostración.
    """
    try:
        servidor.shutdown()
        servidor.server_close()
    except (OSError, RuntimeError):
        pass


### Resultados

Quedan centralizadas las funciones generales de guardado, serialización, comprobación de contratos, validación predictiva y comunicación controlada con `TFM_ML`.

La recuperación documental dispone de funciones comunes para localizar, identificar y recuperar documentos mediante coincidencia exacta y similitud semántica. La presentación incorpora además la visualización completa de tablas y la transformación determinista de predicciones y explicaciones SHAP en texto legible.

La lógica reutilizable de LangGraph queda separada de la definición del estado y de la construcción visible del grafo. También quedan disponibles las funciones que prepararán el contexto generativo, ejecutarán Mistral mediante una salida estructurada y aplicarán guardrails deterministas sobre el informe.

En este apartado únicamente se definen funciones auxiliares; las capacidades se ejecutan y validan posteriormente en sus correspondientes secciones.

## Síntesis de la sección 0 (memoria)

El entorno `TFM_Agentes` queda preparado como capa independiente para agentes y productivización, manteniendo el aislamiento respecto a TensorFlow, XGBoost y SHAP, que permanecen exclusivamente en el entorno predictivo congelado `TFM_ML`. Las versiones comprobadas coinciden con las fijadas para el proyecto y PyTorch detecta correctamente la GPU disponible, manteniendo `cpu` como alternativa automática.

Las credenciales externas permanecen almacenadas únicamente en `.env`, con permisos restringidos y fuera del control de versiones. Los archivos de reproducción y del smoke test integral continúan disponibles; esta comprobación completa, ya superada durante la preparación del entorno, se reserva para el cierre definitivo del entorno `TFM_Agentes`.

La configuración inicial centraliza las rutas, etiquetas y funciones auxiliares necesarias para las siguientes fases. Se distinguen además tool local y MCP como mecanismos de acceso a la capacidad predictiva, cuya comparación funcional se realizará posteriormente sobre la misma inferencia real.

**Tabla candidata:** no; las versiones, comprobaciones del entorno y diccionarios completos se reservan para anexos.

**Figura candidata:** no.

**Destino:** ANEXO; la separación entre `TFM_ML` y `TFM_Agentes` se incorporará posteriormente a la arquitectura general de la memoria.

# 1. Recuperación y validación de la transferencia

La fase predictiva quedó definitivamente cerrada en el Notebook 04, que preparó una transferencia específica para permitir su utilización posterior sin reconstruir modelos, variables, preprocesamientos, umbrales ni decisiones de selección.

En esta sección se recuperan los objetos y tablas que definen dicho contrato y se comprueba su consistencia antes de construir cualquier interfaz de inferencia. La configuración transferida constituye la fuente de verdad para las cuatro variables objetivo, las familias y modelos seleccionados, los umbrales definitivos y las variables requeridas por Machine Learning y Deep Learning.

La referencia TRAIN preparada para las explicaciones SHAP permanece asociada al entorno predictivo `TFM_ML`. En esta fase únicamente se comprueba su disponibilidad; su recuperación y validación efectiva se realizarán cuando se implemente la explicación local bajo demanda, evitando introducir dependencias o lógica predictiva dentro de `TFM_Agentes`.

No se carga todavía ningún modelo predictivo ni se realiza inferencia.

## 1.1. Configuración de productivización

Se recupera la configuración `joblib` preparada por el Notebook 04, que contiene el contrato básico de la solución híbrida final: variables objetivo, familias, modelos, umbrales, variables de entrada y configuración del ensemble Deep Learning.

Antes de utilizar sus contenidos se comprueban las claves contractuales y la coherencia de los principales elementos transferidos. La referencia SHAP se mantiene disponible para su utilización posterior desde `TFM_ML`.

In [7]:
# =============================================================================
# CONFIGURACIÓN DE PRODUCTIVIZACIÓN
# =============================================================================
print('\nCONFIGURACIÓN DE PRODUCTIVIZACIÓN')

ruta_configuracion_productivizacion = archivos_transferencia['Configuración de productivización']
ruta_background_productivizacion = archivos_transferencia['Referencia SHAP']

configuracion_productivizacion = joblib.load(ruta_configuracion_productivizacion)

# -----------------------------------------------------------------------------
# CONTRATO DE LA CONFIGURACIÓN
# -----------------------------------------------------------------------------
claves_configuracion_necesarias = {
    'version',
    'fecha_ejecucion',
    'targets',
    'titulos_targets',
    'familias_finales',
    'modelos_finales',
    'umbrales',
    'variables_ml',
    'variables_dl',
    'columnas_entrada_ml',
    'modelos_ensemble_dl',
    'pesos_modelos_dl',
    'vocabulario_dl',
    'semilla',
    'uso_test'
}

_ = comprobar_claves(
    configuracion_productivizacion, claves_configuracion_necesarias,
    'configuracion_productivizacion'
)

# -----------------------------------------------------------------------------
# RECUPERACIÓN DE OBJETOS
# -----------------------------------------------------------------------------
version_productivizacion = configuracion_productivizacion['version']

targets = list(configuracion_productivizacion['targets'])
titulos_targets = dict(configuracion_productivizacion['titulos_targets'])
familias_finales = dict(configuracion_productivizacion['familias_finales'])
modelos_finales = dict(configuracion_productivizacion['modelos_finales'])
umbrales = dict(configuracion_productivizacion['umbrales'])

vars_predictoras_modelado_ml = list(configuracion_productivizacion['variables_ml'])
vars_predictoras_modelado_dl = list(configuracion_productivizacion['variables_dl'])
columnas_entrada_ml = list(configuracion_productivizacion['columnas_entrada_ml'])

modelos_ensemble_dl = list(configuracion_productivizacion['modelos_ensemble_dl'])
pesos_modelos_dl = dict(configuracion_productivizacion['pesos_modelos_dl'])
vocabulario_total_final_dl = int(configuracion_productivizacion['vocabulario_dl'])

# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
resumen_configuracion_transferida = pd.DataFrame({
    'elemento': [
        'Versión de productivización',
        'Variables objetivo',
        'Variables originales Machine Learning',
        'Variables de entrada Machine Learning',
        'Variables originales Deep Learning',
        'Modelos del ensemble Deep Learning',
        'Vocabulario Deep Learning',
        'Referencia SHAP disponible'
    ],
    'valor': [
        version_productivizacion,
        len(targets),
        len(vars_predictoras_modelado_ml),
        len(columnas_entrada_ml),
        len(vars_predictoras_modelado_dl),
        len(modelos_ensemble_dl),
        vocabulario_total_final_dl,
        ruta_background_productivizacion.is_file()
    ]
})

tabla_configuracion_transferida = resumen_configuracion_transferida.rename(columns=renombrado_comun)

solucion_transferida = pd.DataFrame({
    'target': targets,
    'descripcion': [titulos_targets[target] for target in targets],
    'familia': [familias_finales[target] for target in targets],
    'modelo': [modelos_finales[target] for target in targets],
    'umbral': [umbrales[target] for target in targets]
})

tabla_solucion_transferida = solucion_transferida.rename(columns=renombrado_comun)
tabla_solucion_transferida = tabla_solucion_transferida.round({'Umbral': 3})

print('\nRESUMEN DE LA CONFIGURACIÓN TRANSFERIDA')
display(tabla_configuracion_transferida)

print('\nSOLUCIÓN PREDICTIVA TRANSFERIDA')
display(tabla_solucion_transferida)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
diccionarios_targets_correctos = all(
    set(diccionario) == set(targets)
    for diccionario in [titulos_targets, familias_finales, modelos_finales, umbrales]
)

comprobaciones_configuracion_transferida = pd.DataFrame({
    'comprobacion': [
        'La configuración corresponde a la versión v1',
        'Se conservan las cuatro variables objetivo',
        'Los diccionarios por objetivo contienen las mismas variables',
        'Las familias seleccionadas son ML o DL',
        'Machine Learning utiliza 59 variables originales',
        'Machine Learning utiliza 67 variables de entrada',
        'Deep Learning utiliza 813 variables originales',
        'El ensemble Deep Learning contiene dos modelos',
        'Los pesos del ensemble corresponden a los dos modelos',
        'Los pesos del ensemble permanecen en 0,50 / 0,50',
        'El vocabulario Deep Learning contiene 4.625 índices',
        'La referencia SHAP está disponible',
        'Las variables objetivo no forman parte de las entradas',
        'La semilla coincide con la configuración común del proyecto'
    ],
    'resultado': [
        version_productivizacion == 'v1',
        len(targets) == 4,
        diccionarios_targets_correctos,
        set(familias_finales.values()).issubset({'ML', 'DL'}),
        len(vars_predictoras_modelado_ml) == 59,
        len(columnas_entrada_ml) == 67,
        len(vars_predictoras_modelado_dl) == 813,
        len(modelos_ensemble_dl) == 2,
        set(pesos_modelos_dl) == set(modelos_ensemble_dl),
        all(np.isclose(peso, 0.5) for peso in pesos_modelos_dl.values()),
        vocabulario_total_final_dl == 4625,
        ruta_background_productivizacion.is_file(),
        (
            set(targets).isdisjoint(vars_predictoras_modelado_dl) and
            set(targets).isdisjoint(vars_predictoras_modelado_ml)
        ),
        configuracion_productivizacion['semilla'] == semilla
    ]
})

tabla_comprobaciones_configuracion = comprobaciones_configuracion_transferida.rename(
    columns=renombrado_comun
)

print('\nCOMPROBACIONES DE LA CONFIGURACIÓN TRANSFERIDA')
display(tabla_comprobaciones_configuracion)

if not comprobaciones_configuracion_transferida['resultado'].all():
    raise ValueError('La configuración transferida no es correcta.')

print('\nConfiguración de productivización recuperada correctamente.')
print('La referencia SHAP permanece disponible para su uso posterior en TFM_ML.')


CONFIGURACIÓN DE PRODUCTIVIZACIÓN

RESUMEN DE LA CONFIGURACIÓN TRANSFERIDA


,Elemento,Valor
0,Versión de productivización,v1
1,Variables objetivo,4
2,Variables originales Machine Learning,59
3,Variables de entrada Machine Learning,67
4,Variables originales Deep Learning,813
5,Modelos del ensemble Deep Learning,2
6,Vocabulario Deep Learning,4625
7,Referencia SHAP disponible,True



SOLUCIÓN PREDICTIVA TRANSFERIDA


,Variable objetivo,Descripción,Familia,Modelo,Umbral
0,IRAMDEYR,Episodio depresivo mayor,ML,XGBoost,0.500
1,IRSUICTHNK,Ideación suicida,DL,Ensemble — ramas 32 + sin BatchNormalization (...,0.509
2,IRSUIPLANYR,Planificación suicida,DL,Ensemble — ramas 32 + sin BatchNormalization (...,0.536
3,IRSUITRYYR,Intento suicida,DL,Ensemble — ramas 32 + sin BatchNormalization (...,0.502



COMPROBACIONES DE LA CONFIGURACIÓN TRANSFERIDA


,Comprobación,Resultado
0,La configuración corresponde a la versión v1,True
1,Se conservan las cuatro variables objetivo,True
2,Los diccionarios por objetivo contienen las mi...,True
3,Las familias seleccionadas son ML o DL,True
4,Machine Learning utiliza 59 variables originales,True
5,Machine Learning utiliza 67 variables de entrada,True
6,Deep Learning utiliza 813 variables originales,True
7,El ensemble Deep Learning contiene dos modelos,True
8,Los pesos del ensemble corresponden a los dos ...,True
9,"Los pesos del ensemble permanecen en 0,50 / 0,50",True



Configuración de productivización recuperada correctamente.
La referencia SHAP permanece disponible para su uso posterior en TFM_ML.


### Resultados

La configuración de productivización `v1` se recupera correctamente y conserva las cuatro variables objetivo, las familias y modelos seleccionados, los umbrales definitivos y la configuración común del proyecto.

Machine Learning mantiene 59 variables originales y 67 variables de entrada, mientras que Deep Learning utiliza 813 variables originales. El ensemble Deep Learning conserva sus dos modelos con pesos `0,50 / 0,50` y un vocabulario definitivo de 4.625 índices.

La solución híbrida transferida asigna XGBoost a episodio depresivo mayor con umbral `0,500`, y la solución Deep Learning a ideación, planificación e intento suicida con umbrales `0,509`, `0,536` y `0,502`, respectivamente.

Las 14 comprobaciones realizadas son correctas. La referencia SHAP está disponible para su utilización posterior desde `TFM_ML`, sin cargar modelos ni realizar inferencia en esta fase.

## 1.2. Tablas transferidas desde el Notebook 04

A continuación se recuperan las tablas que documentan la solución predictiva final, las predicciones utilizadas como referencia de trazabilidad, la explicabilidad global, los esquemas de entrada y salida, las limitaciones y las comprobaciones realizadas antes del cierre del Notebook 04.

La finalidad de esta recuperación no es repetir la evaluación realizada sobre TEST, sino disponer de evidencias con las que comprobar posteriormente que el servicio predictivo reproduce exactamente la solución cerrada. Las predicciones de TEST se utilizarán únicamente como referencia de comparación y nunca para reajustar modelos, umbrales o decisiones.

Los esquemas y limitaciones se conservarán con su nomenclatura original para construir posteriormente los contratos Pydantic y los guardrails de la aplicación.

In [8]:
# =============================================================================
# TABLAS TRANSFERIDAS DESDE EL NOTEBOOK 04
# =============================================================================
print('\nTABLAS TRANSFERIDAS DESDE EL NOTEBOOK 04')

# -----------------------------------------------------------------------------
# CARGA DE TABLAS
# -----------------------------------------------------------------------------
archivos_tablas_transferidas = {
    'seleccion_final': 'Selección final por objetivo',
    'predicciones_finales_test': 'Predicciones finales de TEST',
    'principales_variables_shap': 'Variables principales SHAP',
    'coincidencias_variables_shap': 'Coincidencias de variables SHAP',
    'esquema_entrada_productivizacion': 'Esquema de entrada',
    'esquema_salida_productivizacion': 'Esquema de salida',
    'limitaciones_modelo_final': 'Limitaciones',
    'archivos_transferidos': 'Archivos transferidos',
    'comprobaciones_transferencia_origen': 'Comprobaciones de transferencia',
    'comprobaciones_cierre_origen': 'Comprobaciones de cierre'
}

tablas_transferidas = {
    nombre: pd.read_csv(archivos_transferencia[archivo], encoding=encoding_csv)
    for nombre, archivo in archivos_tablas_transferidas.items()
}

seleccion_final = tablas_transferidas['seleccion_final']
predicciones_finales_test = tablas_transferidas['predicciones_finales_test']
principales_variables_shap = tablas_transferidas['principales_variables_shap']
coincidencias_variables_shap = tablas_transferidas['coincidencias_variables_shap']

esquema_entrada_productivizacion = tablas_transferidas['esquema_entrada_productivizacion']
esquema_salida_productivizacion = tablas_transferidas['esquema_salida_productivizacion']
limitaciones_modelo_final = tablas_transferidas['limitaciones_modelo_final']
archivos_transferidos = tablas_transferidas['archivos_transferidos']

comprobaciones_transferencia_origen = tablas_transferidas['comprobaciones_transferencia_origen']
comprobaciones_cierre_origen = tablas_transferidas['comprobaciones_cierre_origen']

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
columnas_tablas_necesarias = {
    'seleccion_final': ['target', 'descripcion', 'familia', 'modelo', 'umbral'],
    'predicciones_finales_test': [
        'QUESTID2', 'particion', 'familia', 'modelo', 'target', 'descripcion', 'y_true',
        'y_score', 'umbral', 'y_pred'
    ],
    'principales_variables_shap': [
        'target', 'descripcion', 'familia', 'variable', 'importancia_shap', 'etiqueta_variable',
        'modulo', 'posicion'
    ],
    'coincidencias_variables_shap': [
        'variable', 'objetivos_principales', 'posicion_media', 'etiqueta_variable',
        'modulo', *targets
    ],
    'esquema_entrada_productivizacion': [
        'orden', 'variable', 'etiqueta_variable', 'modulo', 'tipo_dato', 'usa_ml', 'usa_dl'
    ],
    'esquema_salida_productivizacion': ['campo', 'descripcion', 'obligatorio'],
    'limitaciones_modelo_final': ['limitacion', 'detalle'],
    'archivos_transferidos': ['elemento', 'ruta', 'uso'],
    'comprobaciones_transferencia_origen': ['comprobacion', 'resultado'],
    'comprobaciones_cierre_origen': ['comprobacion', 'resultado']
}

for nombre, columnas in columnas_tablas_necesarias.items():
    _ = comprobar_columnas(tablas_transferidas[nombre], columnas, nombre)

# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
resumen_tablas_transferidas = pd.DataFrame({
    'elemento': [
        'Soluciones finales por objetivo',
        'Predicciones finales de TEST',
        'Variables principales SHAP',
        'Variables compartidas SHAP',
        'Variables del esquema de entrada',
        'Campos del esquema de salida',
        'Limitaciones documentadas',
        'Archivos declarados en la transferencia',
        'Comprobaciones de transferencia de origen',
        'Comprobaciones de cierre del Notebook 04'
    ],
    'valor': [
        len(seleccion_final),
        len(predicciones_finales_test),
        len(principales_variables_shap),
        len(coincidencias_variables_shap),
        len(esquema_entrada_productivizacion),
        len(esquema_salida_productivizacion),
        len(limitaciones_modelo_final),
        len(archivos_transferidos),
        len(comprobaciones_transferencia_origen),
        len(comprobaciones_cierre_origen)
    ]
})

tabla_resumen_transferencia = resumen_tablas_transferidas.rename(columns=renombrado_comun)

print('\nRESUMEN DE LAS TABLAS TRANSFERIDAS')
display(tabla_resumen_transferencia)

print('\nTablas transferidas recuperadas correctamente.')


TABLAS TRANSFERIDAS DESDE EL NOTEBOOK 04



RESUMEN DE LAS TABLAS TRANSFERIDAS


,Elemento,Valor
0,Soluciones finales por objetivo,4
1,Predicciones finales de TEST,37840
2,Variables principales SHAP,60
3,Variables compartidas SHAP,17
4,Variables del esquema de entrada,813
5,Campos del esquema de salida,10
6,Limitaciones documentadas,10
7,Archivos declarados en la transferencia,16
8,Comprobaciones de transferencia de origen,9
9,Comprobaciones de cierre del Notebook 04,22



Tablas transferidas recuperadas correctamente.


### Resultados

Las diez tablas de transferencia se recuperan correctamente y mantienen la estructura preparada durante el cierre del Notebook 04.

La información disponible incluye las cuatro soluciones predictivas finales, 37.840 predicciones de TEST para trazabilidad, 60 registros correspondientes a variables principales SHAP y 17 variables compartidas entre indicadores. El esquema de productivización contiene 813 variables de entrada y 10 campos de salida.

También se recuperan las 10 limitaciones documentadas de la solución, los 16 archivos declarados para la siguiente fase, las 9 comprobaciones específicas de transferencia y las 22 comprobaciones del cierre definitivo del Notebook 04.

Estos resultados se utilizarán únicamente para validar contratos y reproducibilidad. Las predicciones de TEST no se emplearán para modificar modelos, variables, umbrales ni decisiones previamente cerradas.

## 1.3. Integridad y consistencia de la transferencia

Una vez recuperados los archivos se contrasta la información almacenada en la configuración de productivización con las tablas transferidas y con las comprobaciones realizadas durante el cierre del Notebook 04.

Se verifica que las cuatro variables objetivo mantienen la misma familia, modelo y umbral; que el esquema de entrada reproduce las variables originales utilizadas por Machine Learning y Deep Learning; que el esquema de salida contiene los campos obligatorios; y que las predicciones de TEST conservan la estructura utilizada como referencia de trazabilidad.

También se comprueba la disponibilidad de las variables principales SHAP, las limitaciones de interpretación y los archivos declarados para esta fase. Las comprobaciones originales del Notebook 04 se utilizan como evidencia de integridad para aquellos elementos que permanecen asociados al entorno `TFM_ML`, como la referencia TRAIN utilizada por SHAP.

Este control cierra la recuperación de la transferencia antes de definir los contratos estructurados de entrada y salida.

In [9]:
# =============================================================================
# INTEGRIDAD Y CONSISTENCIA DE LA TRANSFERENCIA
# =============================================================================
print('\nINTEGRIDAD Y CONSISTENCIA DE LA TRANSFERENCIA')

# -----------------------------------------------------------------------------
# INFORMACIÓN POR VARIABLE OBJETIVO
# -----------------------------------------------------------------------------
familias_seleccion_final = seleccion_final.set_index('target')['familia'].to_dict()
modelos_seleccion_final = seleccion_final.set_index('target')['modelo'].to_dict()
umbrales_seleccion_final = seleccion_final.set_index('target')['umbral'].to_dict()

familias_predicciones = predicciones_finales_test.groupby('target')['familia'].first().to_dict()
modelos_predicciones = predicciones_finales_test.groupby('target')['modelo'].first().to_dict()
umbrales_predicciones = predicciones_finales_test.groupby('target')['umbral'].first().to_dict()

# -----------------------------------------------------------------------------
# ESQUEMAS Y ARCHIVOS
# -----------------------------------------------------------------------------
vars_entrada_ml = esquema_entrada_productivizacion.loc[
    esquema_entrada_productivizacion['usa_ml'], 'variable'
].tolist()

vars_entrada_dl = esquema_entrada_productivizacion.loc[
    esquema_entrada_productivizacion['usa_dl'], 'variable'
].tolist()

campos_salida_obligatorios = set(esquema_salida_productivizacion.loc[
    esquema_salida_productivizacion['obligatorio'], 'campo'
])

campos_salida_necesarios = {
    'variable_objetivo',
    'descripcion',
    'probabilidad',
    'umbral',
    'clasificacion',
    'familia',
    'modelo',
    'advertencias'
}

rutas_archivos_transferidos = [ruta_proyecto / ruta for ruta in archivos_transferidos['ruta']]

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_recuperacion_transferencia = pd.DataFrame({
    'comprobacion': [
        'La selección contiene exactamente las cuatro variables objetivo',
        'Las familias seleccionadas coinciden con la configuración',
        'Los modelos seleccionados coinciden con la configuración',
        'Los umbrales seleccionados coinciden con la configuración',
        'El esquema contiene las 59 variables Machine Learning',
        'El esquema contiene las 813 variables Deep Learning',
        'Las variables objetivo no aparecen entre las entradas',
        'El esquema de salida contiene los ocho campos obligatorios',
        'Las predicciones contienen 9.460 registros por variable objetivo',
        'No existen duplicados por identificador y variable objetivo',
        'Las predicciones utilizan las familias seleccionadas',
        'Las predicciones utilizan los modelos seleccionados',
        'Las predicciones utilizan los umbrales definitivos',
        'Cada variable objetivo dispone de 15 variables SHAP principales',
        'Las 10 limitaciones transferidas contienen descripción',
        'Las comprobaciones de transferencia del Notebook 04 son correctas',
        'Las comprobaciones de cierre del Notebook 04 son correctas',
        'Los 16 archivos declarados continúan disponibles'
    ],
    'resultado': [
        set(seleccion_final['target']) == set(targets) and len(seleccion_final) == 4,
        familias_seleccion_final == familias_finales,
        modelos_seleccion_final == modelos_finales,
        all(np.isclose(umbrales_seleccion_final[target], umbrales[target]) for target in targets),
        set(vars_entrada_ml) == set(vars_predictoras_modelado_ml),
        vars_entrada_dl == vars_predictoras_modelado_dl,
        set(targets).isdisjoint(esquema_entrada_productivizacion['variable']),
        campos_salida_necesarios.issubset(campos_salida_obligatorios),
        predicciones_finales_test.groupby('target').size().eq(9460).all(),
        not predicciones_finales_test.duplicated(['QUESTID2', 'target']).any(),
        familias_predicciones == familias_finales,
        modelos_predicciones == modelos_finales,
        all(np.isclose(umbrales_predicciones[target], umbrales[target]) for target in targets),
        principales_variables_shap.groupby('target').size().eq(15).all(),
        len(limitaciones_modelo_final) == 10 and
        limitaciones_modelo_final['detalle'].notna().all(),
        comprobaciones_transferencia_origen['resultado'].all(),
        comprobaciones_cierre_origen['resultado'].all(),
        len(rutas_archivos_transferidos) == 16 and
        all(ruta.is_file() for ruta in rutas_archivos_transferidos)
    ]
})

tabla_comprobaciones_transferencia = (
    comprobaciones_recuperacion_transferencia.rename(columns=renombrado_comun)
)

print('\nCOMPROBACIONES DE LA RECUPERACIÓN')
display(tabla_comprobaciones_transferencia)

if not comprobaciones_recuperacion_transferencia['resultado'].all():
    raise ValueError('La recuperación de la transferencia no es correcta.')

# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
resumen_transferencia_recuperada = pd.DataFrame({
    'elemento': [
        'Variables objetivo',
        'Variables requeridas por el sistema',
        'Variables utilizadas por Machine Learning',
        'Variables utilizadas por Deep Learning',
        'Predicciones de TEST para trazabilidad',
        'Variables SHAP principales por objetivo',
        'Campos del esquema de salida',
        'Campos obligatorios de salida',
        'Limitaciones documentadas',
        'Archivos transferidos disponibles'
    ],
    'valor': [
        len(targets),
        len(esquema_entrada_productivizacion),
        len(vars_entrada_ml),
        len(vars_entrada_dl),
        len(predicciones_finales_test),
        int(principales_variables_shap.groupby('target').size().min()),
        len(esquema_salida_productivizacion),
        len(campos_salida_obligatorios),
        len(limitaciones_modelo_final),
        sum(ruta.is_file() for ruta in rutas_archivos_transferidos)
    ]
})

tabla_resumen_transferencia = resumen_transferencia_recuperada.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA TRANSFERENCIA RECUPERADA')
display(tabla_resumen_transferencia)

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_transferencia_recuperada, '01_resumen_transferencia_recuperada')
_ = guardar_csv(
    comprobaciones_recuperacion_transferencia, '02_comprobaciones_recuperacion_transferencia'
)

print('\nTransferencia recuperada y validada correctamente.')


INTEGRIDAD Y CONSISTENCIA DE LA TRANSFERENCIA

COMPROBACIONES DE LA RECUPERACIÓN


,Comprobación,Resultado
0,La selección contiene exactamente las cuatro v...,True
1,Las familias seleccionadas coinciden con la co...,True
2,Los modelos seleccionados coinciden con la con...,True
3,Los umbrales seleccionados coinciden con la co...,True
4,El esquema contiene las 59 variables Machine L...,True
5,El esquema contiene las 813 variables Deep Lea...,True
6,Las variables objetivo no aparecen entre las e...,True
7,El esquema de salida contiene los ocho campos ...,True
8,Las predicciones contienen 9.460 registros por...,True
9,No existen duplicados por identificador y vari...,True



RESUMEN DE LA TRANSFERENCIA RECUPERADA


,Elemento,Valor
0,Variables objetivo,4
1,Variables requeridas por el sistema,813
2,Variables utilizadas por Machine Learning,59
3,Variables utilizadas por Deep Learning,813
4,Predicciones de TEST para trazabilidad,37840
5,Variables SHAP principales por objetivo,15
6,Campos del esquema de salida,10
7,Campos obligatorios de salida,8
8,Limitaciones documentadas,10
9,Archivos transferidos disponibles,16



Transferencia recuperada y validada correctamente.


### Resultados

Las 18 comprobaciones de integridad y consistencia resultan correctas. La selección recuperada mantiene exactamente las cuatro variables objetivo, sus familias y modelos definitivos y los umbrales establecidos durante el cierre predictivo.

El esquema general conserva las 813 variables originales requeridas por el sistema, de las cuales 59 son utilizadas también por Machine Learning. Las variables objetivo permanecen excluidas de la entrada y el esquema de salida contiene los 10 campos definidos para productivización, con 8 de ellos obligatorios.

Las 37.840 predicciones de TEST se conservan exclusivamente como referencia de trazabilidad, con 9.460 registros por variable objetivo, sin duplicados por identificador y objetivo y manteniendo las familias, modelos y umbrales definitivos. También permanecen disponibles 15 variables SHAP principales por objetivo, las 10 limitaciones documentadas y los 16 archivos declarados para la transferencia.

## Síntesis de la sección 1 (memoria)

La solución predictiva cerrada se ha transferido correctamente a la fase de productivización mediante un contrato explícito y reproducible. La entrada general queda formada por 813 variables originales, incluyendo las 59 requeridas por Machine Learning, mientras que la salida mantiene 10 campos estructurados, 8 de ellos obligatorios. La configuración conserva las cuatro soluciones definitivas, sus umbrales y la información necesaria para utilizar posteriormente la explicabilidad local sin reabrir ninguna decisión predictiva.

La transferencia incluye además 37.840 predicciones de TEST exclusivamente para comprobaciones predictivas, 15 variables SHAP principales por objetivo, 10 limitaciones documentadas y 16 archivos destinados a la siguiente fase. Las comprobaciones realizadas confirman que la información recuperada es consistente con el cierre del Notebook 04 y que `TFM_ML` permanece aislado y sin modificaciones.

**Tabla candidata:** resumen de la transferencia recuperada, únicamente para ANEXO.

**Figura candidata:** no.

**Destino:** MEMORIA + ANEXO. En la memoria se conservará únicamente la descripción del contrato y la separación entre la capa predictiva y la capa de productivización; las comprobaciones completas y esquemas detallados se trasladarán al anexo.

# 2. Contratos estructurados y validación de entrada

La comunicación entre la aplicación, la capa de agentes y el servicio predictivo requiere estructuras explícitas que permitan controlar la información intercambiada entre las distintas partes del sistema. Para ello se utiliza Pydantic como mecanismo de validación estructural, manteniendo separados el registro de entrada, el resultado predictivo y el informe preventivo.

Los contratos se construyen a partir de los esquemas recuperados en la sección anterior. No se reproducen manualmente las 813 variables como atributos independientes, sino que el registro mantiene los nombres originales y se valida frente a la lista transferida desde el Notebook 04.

La validación se limita en esta fase a las restricciones que pueden comprobarse de forma objetiva: estructura, nombres de variables, tipos básicos, campos requeridos y coherencia con la configuración predictiva cerrada. Los códigos concretos de las variables NSDUH solo se validarán cuando exista respaldo documental suficiente.

## 2.1. Contrato de entrada

La entrada para inferencia se representa mediante un registro estructurado con las 813 variables originales utilizadas por Deep Learning. Las 59 variables requeridas por Machine Learning forman parte de este mismo registro y no necesitan una entrada independiente.

Se incorpora además un indicador opcional para solicitar explicación local. Esta petición únicamente determinará si posteriormente se solicita SHAP al entorno predictivo y no modifica el cálculo de probabilidades, umbrales o clasificaciones.

Los nombres y valores del registro se comprobarán mediante la función de validación común definida al inicio del notebook.

In [10]:
# =============================================================================
# CONTRATO DE ENTRADA
# =============================================================================
class EntradaPredictiva(BaseModel):
    """
    Entrada estructurada para una inferencia predictiva.
    """
    registro: dict[str, Any]
    solicitar_explicacion: bool = False

    model_config = {'extra': 'forbid'}


# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
resumen_contrato_entrada = pd.DataFrame({
    'elemento': [
        'Campos del contrato',
        'Variables requeridas',
        'Variables utilizadas por Machine Learning',
        'Variables utilizadas por Deep Learning',
        'Variables objetivo permitidas',
        'Explicación local opcional',
        'Campos adicionales permitidos'
    ],
    'valor': [
        len(EntradaPredictiva.model_fields),
        len(vars_predictoras_modelado_dl),
        len(vars_predictoras_modelado_ml),
        len(vars_predictoras_modelado_dl),
        0,
        True,
        False
    ]
})

tabla_contrato_entrada = resumen_contrato_entrada.rename(columns=renombrado_comun)

print('\nCONTRATO DE ENTRADA')
display(tabla_contrato_entrada)


CONTRATO DE ENTRADA


,Elemento,Valor
0,Campos del contrato,2
1,Variables requeridas,813
2,Variables utilizadas por Machine Learning,59
3,Variables utilizadas por Deep Learning,813
4,Variables objetivo permitidas,0
5,Explicación local opcional,True
6,Campos adicionales permitidos,False


### Resultados

El contrato `EntradaPredictiva` queda definido mediante dos campos: el registro con las variables originales y el indicador opcional para solicitar explicación local.

La entrada general requiere las 813 variables utilizadas por Deep Learning, dentro de las cuales se encuentran las 59 variables necesarias para Machine Learning. Las variables objetivo no forman parte del registro de entrada y no se admiten campos adicionales en el contrato.

La solicitud de explicación se mantiene como una opción independiente de la predicción, por lo que no modifica la selección de modelos, las probabilidades ni los umbrales previamente cerrados.

## 2.2. Contrato de salida predictiva

La salida predictiva se representa mediante los 10 campos definidos durante la transferencia desde el Notebook 04. Ocho de ellos son obligatorios en toda predicción, mientras que las variables principales y la diferencia de reconstrucción se incorporan únicamente cuando se solicita explicabilidad.

Pydantic controla los tipos básicos y restringe probabilidades y umbrales al intervalo `[0, 1]`, así como la clasificación a los valores binarios admitidos. La coherencia con la solución predictiva definitiva se comprueba posteriormente mediante la función determinista común, que contrasta variable objetivo, descripción, familia, modelo, umbral y clasificación con la configuración transferida.

La estructura interna de las variables SHAP locales se mantendrá abierta hasta observar la salida real del bridge. De este modo se evita definir por anticipado un contrato que pueda no coincidir exactamente con el servicio predictivo.

In [11]:
# =============================================================================
# CONTRATO DE SALIDA PREDICTIVA
# =============================================================================
class ResultadoPredictivo(BaseModel):
    """
    Salida estructurada para una variable objetivo.
    """
    variable_objetivo: str = Field(min_length=1)
    descripcion: str = Field(min_length=1)
    probabilidad: float = Field(ge=0, le=1)
    umbral: float = Field(ge=0, le=1)
    clasificacion: Literal[0, 1]
    familia: str = Field(min_length=1)
    modelo: str = Field(min_length=1)
    variables_principales: list[dict[str, Any]] | None = None
    diferencia_reconstruccion: float | None = None
    advertencias: list[str]

    model_config = {'extra': 'forbid'}


# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
campos_contrato_salida = set(ResultadoPredictivo.model_fields)

campos_obligatorios_contrato = {
    nombre for nombre, campo in ResultadoPredictivo.model_fields.items() if campo.is_required()
}

comprobaciones_contrato_salida = pd.DataFrame({
    'comprobacion': [
        'El contrato contiene los 10 campos transferidos',
        'Los ocho campos obligatorios coinciden con el esquema',
        'Las variables principales son opcionales',
        'La diferencia de reconstrucción es opcional'
    ],
    'resultado': [
        campos_contrato_salida == set(esquema_salida_productivizacion['campo']),
        campos_obligatorios_contrato == campos_salida_obligatorios,
        not ResultadoPredictivo.model_fields['variables_principales'].is_required(),
        not ResultadoPredictivo.model_fields['diferencia_reconstruccion'].is_required()
    ]
})

tabla_comprobaciones_salida = comprobaciones_contrato_salida.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DEL CONTRATO DE SALIDA')
display(tabla_comprobaciones_salida)

if not comprobaciones_contrato_salida['resultado'].all():
    raise ValueError('El contrato predictivo no coincide con el esquema transferido.')


COMPROBACIONES DEL CONTRATO DE SALIDA


,Comprobación,Resultado
0,El contrato contiene los 10 campos transferidos,True
1,Los ocho campos obligatorios coinciden con el ...,True
2,Las variables principales son opcionales,True
3,La diferencia de reconstrucción es opcional,True


### Resultados

El contrato `ResultadoPredictivo` reproduce correctamente los 10 campos definidos en el esquema de salida transferido desde el Notebook 04.

Los ocho campos obligatorios coinciden exactamente con el contrato de productivización, mientras que `variables_principales` y `diferencia_reconstruccion` permanecen como campos opcionales asociados a la explicabilidad local.

Las cuatro comprobaciones estructurales resultan correctas. Las probabilidades y umbrales quedan restringidos al intervalo `[0, 1]`, la clasificación a los valores `0` y `1`, y no se permiten campos adicionales fuera del contrato.

## 2.3. Contrato del informe preventivo

La capa generativa producirá posteriormente un informe estructurado independiente de la salida predictiva. El contrato controla la forma de la respuesta antes de incorporar Mistral y facilita su validación mediante guardrails deterministas.

Los factores explicativos se estructuran individualmente y se vinculan con la variable objetivo cuya predicción explican. Para cada indicador se conservarán tres factores principales procedentes exclusivamente de su explicación SHAP local.

La dirección de la contribución se presenta mediante las expresiones `Factor de riesgo para la predicción` y `Factor protector para la predicción`. Estas denominaciones tienen una finalidad comunicativa: corresponden respectivamente a contribuciones SHAP que incrementan o reducen la probabilidad estimada y no deben interpretarse como relaciones causales o como factores clínicos de riesgo o protección.

La variable original se mantiene internamente para garantizar la trazabilidad, mientras que el informe visible utiliza una descripción comprensible y el significado de la respuesta registrada. Este significado se obtiene de forma determinista a partir del codebook oficial, evitando presentar al usuario los códigos internos, las frecuencias o los porcentajes asociados a cada categoría.

Cuando una variable ha sido recodificada, combinada o derivada por NSDUH se incorpora además una aclaración breve sobre su procedencia, evitando presentarla como si correspondiera necesariamente a una pregunta independiente del cuestionario.

El informe conserva siete bloques principales: resumen, interpretación, factores relevantes por indicador, orientación preventiva, limitaciones, fuentes y advertencia de uso. Las probabilidades, umbrales, clasificaciones y valores SHAP continúan fuera de la generación y permanecen bajo control determinista.

In [12]:
# =============================================================================
# CONTRATO DEL INFORME PREVENTIVO
# =============================================================================
class FactorInforme(BaseModel):
    """
    Factor explicativo vinculado a una variable objetivo.
    """
    variable_objetivo: str = Field(min_length=1)
    descripcion_objetivo: str = Field(min_length=1)
    tipo_factor: Literal[
        'Factor de riesgo para la predicción',
        'Factor protector para la predicción'
    ]
    variable_origen: str = Field(min_length=1)
    descripcion: str = Field(min_length=1)
    respuesta_observada: str = Field(min_length=1)
    nota_variable: str | None = None
    posicion_factor: int = Field(ge=1, le=3)

    model_config = {'extra': 'forbid'}


class TraduccionFactor(BaseModel):
    """
    Traducción visible de una variable explicativa.
    """
    variable_origen: str = Field(min_length=1)
    descripcion: str = Field(min_length=1)

    model_config = {'extra': 'forbid'}


class SalidaGenerativa(BaseModel):
    """
    Contenido que Mistral puede generar libremente.
    """
    resumen: str = Field(min_length=1)
    interpretacion: str = Field(min_length=1)
    traducciones_factores: list[TraduccionFactor]

    model_config = {'extra': 'forbid'}


class InformePreventivo(BaseModel):
    """
    Informe preventivo generado a partir de información validada.
    """
    resumen: str = Field(min_length=1)
    interpretacion: str = Field(min_length=1)
    factores_relevantes: list[FactorInforme]
    orientacion_preventiva: list[str]
    limitaciones: list[str]
    fuentes: list[str]
    advertencia_uso: str = Field(min_length=1)

    model_config = {'extra': 'forbid'}

### Resultados

El contrato `InformePreventivo` queda definido mediante siete campos diferenciados para resumen, interpretación, factores relevantes, orientación preventiva, limitaciones, fuentes y advertencia de uso.

Cada factor conserva además la respuesta observada en una forma comprensible para el usuario y admite una aclaración opcional cuando la variable ha sido construida o recodificada por NSDUH. Esta información forma parte de la estructura interna del factor sin modificar los siete campos principales de `InformePreventivo`.

La estructura mantiene separado el contenido generado de la salida predictiva. Las probabilidades, umbrales, clasificaciones y contribuciones SHAP no forman parte del contenido generado por Mistral, sino que permanecen asociados a sus fuentes deterministas y se incorporan posteriormente mediante la reconstrucción controlada del informe.

Los tres contratos principales del sistema quedan así definidos con 2 campos para la entrada, 10 para la salida predictiva y 7 para el informe preventivo.

## 2.4. Validación de los contratos

Los tres contratos se someten a pruebas deterministas antes de utilizarlos en el bridge predictivo.

Para la entrada se construye un registro exclusivamente estructural con los 813 nombres esperados. Se comprueba que la estructura completa es aceptada y que la ausencia de una variable, la incorporación de una variable desconocida o la inclusión de un campo adicional en el contrato producen errores controlados.

La salida predictiva se valida a partir de una predicción ya existente para cada variable objetivo. Estas observaciones se utilizan únicamente como referencia de estructura y trazabilidad, sin realizar ninguna nueva evaluación sobre TEST. Se comprueba además el rechazo de probabilidades fuera del rango permitido, campos desconocidos y clasificaciones incoherentes.

Finalmente se valida la estructura del informe preventivo sin realizar ninguna llamada a Mistral ni a otros servicios externos.

In [13]:
# =============================================================================
# VALIDACIÓN DE LOS CONTRATOS
# =============================================================================
print('\nVALIDACIÓN DE LOS CONTRATOS')

# -----------------------------------------------------------------------------
# ENTRADA
# -----------------------------------------------------------------------------
registro_prueba = {variable: None for variable in vars_predictoras_modelado_dl}

entrada_prueba = EntradaPredictiva(registro=registro_prueba, solicitar_explicacion=False)

_ = validar_registro_entrada(entrada_prueba.registro, vars_predictoras_modelado_dl)

registro_incompleto = registro_prueba.copy()
registro_incompleto.pop(vars_predictoras_modelado_dl[0])

try:
    validar_registro_entrada(registro_incompleto, vars_predictoras_modelado_dl)
    entrada_incompleta_rechazada = False
except ValueError:
    entrada_incompleta_rechazada = True

registro_desconocido = registro_prueba.copy()
registro_desconocido['VARIABLE_DESCONOCIDA'] = 0

try:
    validar_registro_entrada(registro_desconocido, vars_predictoras_modelado_dl)
    entrada_desconocida_rechazada = False
except ValueError:
    entrada_desconocida_rechazada = True

try:
    EntradaPredictiva(registro=registro_prueba, campo_desconocido=True)
    campo_entrada_desconocido_rechazado = False
except ValidationError:
    campo_entrada_desconocido_rechazado = True

# -----------------------------------------------------------------------------
# SALIDA PREDICTIVA
# -----------------------------------------------------------------------------
filas_prediccion_prueba = predicciones_finales_test.groupby(
    'target', sort=False
).head(1).set_index('target').loc[targets].reset_index()

advertencias_prueba = limitaciones_modelo_final['detalle'].astype(str).tolist()

resultados_predictivos_prueba = []

for _, fila in filas_prediccion_prueba.iterrows():
    datos_resultado = {
        'variable_objetivo': fila['target'],
        'descripcion': fila['descripcion'],
        'probabilidad': float(fila['y_score']),
        'umbral': float(fila['umbral']),
        'clasificacion': int(fila['y_pred']),
        'familia': fila['familia'],
        'modelo': fila['modelo'],
        'advertencias': advertencias_prueba
    }

    resultado_pydantic = ResultadoPredictivo(**datos_resultado)
    resultado_validado = validar_resultado_predictivo(
        resultado_pydantic.model_dump(), configuracion_productivizacion
    )

    resultados_predictivos_prueba.append(resultado_validado)

# -----------------------------------------------------------------------------
# EJEMPLO DE SALIDA PREDICTIVA
# -----------------------------------------------------------------------------
ejemplo_salida_predictiva = pd.DataFrame([resultados_predictivos_prueba[0]]).copy()
ejemplo_salida_predictiva['advertencias'] = ejemplo_salida_predictiva['advertencias'].apply(len)

tabla_ejemplo_salida = ejemplo_salida_predictiva.rename(columns={
    **renombrado_comun, 'advertencias': 'Número de advertencias'
})

print('\nEJEMPLO DE SALIDA PREDICTIVA DE REFERENCIA')
display(tabla_ejemplo_salida)

vista_previa_contrato = pd.DataFrame({
    'salida_textual': [resumir_resultado_predictivo(resultados_predictivos_prueba[0])]
})

tabla_vista_previa = vista_previa_contrato.rename(columns={
    'salida_textual': 'Vista previa textual'
})

print('\nVISTA PREVIA TEXTUAL DE LA SALIDA')
mostrar_tabla_completa(tabla_vista_previa)

datos_probabilidad_invalida = resultados_predictivos_prueba[0].copy()
datos_probabilidad_invalida['probabilidad'] = 1.1

try:
    ResultadoPredictivo(**datos_probabilidad_invalida)
    probabilidad_invalida_rechazada = False
except ValidationError:
    probabilidad_invalida_rechazada = True

datos_campo_desconocido = resultados_predictivos_prueba[0].copy()
datos_campo_desconocido['campo_desconocido'] = True

try:
    ResultadoPredictivo(**datos_campo_desconocido)
    campo_salida_desconocido_rechazado = False
except ValidationError:
    campo_salida_desconocido_rechazado = True

datos_clasificacion_incorrecta = resultados_predictivos_prueba[0].copy()
datos_clasificacion_incorrecta['clasificacion'] = (
    1 - datos_clasificacion_incorrecta['clasificacion']
)

try:
    validar_resultado_predictivo(datos_clasificacion_incorrecta, configuracion_productivizacion)
    clasificacion_incorrecta_rechazada = False
except ValueError:
    clasificacion_incorrecta_rechazada = True

# -----------------------------------------------------------------------------
# INFORME PREVENTIVO
# -----------------------------------------------------------------------------
factor_prueba = FactorInforme(
    variable_objetivo=targets[0],
    descripcion_objetivo=titulos_targets[targets[0]],
    tipo_factor=frases_informe['tipos_factor']['positiva'],
    variable_origen='VARIABLE_PRUEBA',
    descripcion='Variable explicativa de prueba',
    respuesta_observada='Respuesta de prueba', posicion_factor=1
)

informe_prueba = InformePreventivo(
    resumen='Resumen estructural de prueba.',
    interpretacion='Interpretación estructural de prueba.',
    factores_relevantes=[factor_prueba],
    orientacion_preventiva=[frases_informe['orientacion_preventiva'][0]],
    limitaciones=frases_informe['limitaciones_obligatorias'],
    fuentes=['Fuente documental de prueba'],
    advertencia_uso=frases_informe['advertencia_uso']
)

informe_serializable = isinstance(informe_prueba.model_dump(mode='json'), dict)

respuesta_factor_serializada = (
    informe_prueba.model_dump(mode='json')['factores_relevantes'][0]['respuesta_observada']
    == 'Respuesta de prueba'
)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_contratos = pd.DataFrame({
    'comprobacion': [
        'La entrada estructural acepta las 813 variables esperadas',
        'Una entrada incompleta es rechazada',
        'Una variable desconocida es rechazada',
        'Un campo adicional en la entrada es rechazado',
        'Las cuatro salidas predictivas de referencia son válidas',
        'Una probabilidad fuera de rango es rechazada',
        'Un campo adicional en la salida es rechazado',
        'Una clasificación incoherente es rechazada',
        'El informe preventivo cumple el contrato estructurado',
        'Los factores conservan la respuesta observada'
    ],
    'resultado': [
        len(entrada_prueba.registro) == 813,
        entrada_incompleta_rechazada,
        entrada_desconocida_rechazada,
        campo_entrada_desconocido_rechazado,
        len(resultados_predictivos_prueba) == 4,
        probabilidad_invalida_rechazada,
        campo_salida_desconocido_rechazado,
        clasificacion_incorrecta_rechazada,
        informe_serializable,
        respuesta_factor_serializada
    ]
})

tabla_comprobaciones_contratos = comprobaciones_contratos.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LOS CONTRATOS')
display(tabla_comprobaciones_contratos)

if not comprobaciones_contratos['resultado'].all():
    raise ValueError('Los contratos estructurados no son correctos.')

# -----------------------------------------------------------------------------
# RESUMEN FINAL DE LOS CONTRATOS
# -----------------------------------------------------------------------------
resumen_contratos = pd.DataFrame({
    'elemento': [
        'EntradaPredictiva',
        'ResultadoPredictivo',
        'InformePreventivo'
    ],
    'finalidad': [
        'Validar el registro recibido para inferencia',
        'Estructurar y proteger la salida predictiva',
        'Estructurar el informe preventivo generado'
    ],
    'campos': [
        len(EntradaPredictiva.model_fields),
        len(ResultadoPredictivo.model_fields),
        len(InformePreventivo.model_fields)
    ],
    'campos_obligatorios': [
        sum(campo.is_required() for campo in EntradaPredictiva.model_fields.values()),
        sum(campo.is_required() for campo in ResultadoPredictivo.model_fields.values()),
        sum(campo.is_required() for campo in InformePreventivo.model_fields.values())
    ],
    'validacion': [
        '813 variables exactas y rechazo de entradas incompletas o desconocidas',
        'Coherencia de probabilidades, umbrales, clasificación, familia y modelo',
        'Estructura cerrada y serialización JSON'
    ],
    'resultado': [
        (
            len(entrada_prueba.registro) == 813 and entrada_incompleta_rechazada and
            entrada_desconocida_rechazada and campo_entrada_desconocido_rechazado
        ),
        (
            len(resultados_predictivos_prueba) == 4 and probabilidad_invalida_rechazada and
            campo_salida_desconocido_rechazado and clasificacion_incorrecta_rechazada
        ),
        informe_serializable
    ]
})

tabla_resumen_contratos = resumen_contratos.rename(columns=renombrado_comun)

print('\nRESUMEN FINAL DE LOS CONTRATOS')
display(tabla_resumen_contratos)

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_contratos, '03_resumen_contratos_estructurados')
_ = guardar_csv(comprobaciones_contratos, '04_comprobaciones_contratos_estructurados')

print('\nContratos estructurados definidos y validados correctamente.')


VALIDACIÓN DE LOS CONTRATOS

EJEMPLO DE SALIDA PREDICTIVA DE REFERENCIA


,Variable objetivo,Descripción,Probabilidad,Umbral,Clasificación,Familia,Modelo,Variables principales,Diferencia de reconstrucción,Número de advertencias
0,IRAMDEYR,Episodio depresivo mayor,0.826969,0.5,1,ML,XGBoost,None,None,10



VISTA PREVIA TEXTUAL DE LA SALIDA


,Vista previa textual
0,"Episodio depresivo mayor: la probabilidad estimada es 0.827 y el umbral validado es 0.500; la probabilidad supera el umbral, por lo que la clasificación estructurada es 1. La predicción procede de XGBoost (Machine Learning)."



COMPROBACIONES DE LOS CONTRATOS


,Comprobación,Resultado
0,La entrada estructural acepta las 813 variable...,True
1,Una entrada incompleta es rechazada,True
2,Una variable desconocida es rechazada,True
3,Un campo adicional en la entrada es rechazado,True
4,Las cuatro salidas predictivas de referencia s...,True
5,Una probabilidad fuera de rango es rechazada,True
6,Un campo adicional en la salida es rechazado,True
7,Una clasificación incoherente es rechazada,True
8,El informe preventivo cumple el contrato estru...,True
9,Los factores conservan la respuesta observada,True



RESUMEN FINAL DE LOS CONTRATOS


,Elemento,Finalidad,Campos,Campos obligatorios,Validación,Resultado
0,EntradaPredictiva,Validar el registro recibido para inferencia,2,1,813 variables exactas y rechazo de entradas in...,True
1,ResultadoPredictivo,Estructurar y proteger la salida predictiva,10,8,"Coherencia de probabilidades, umbrales, clasif...",True
2,InformePreventivo,Estructurar el informe preventivo generado,7,7,Estructura cerrada y serialización JSON,True



Contratos estructurados definidos y validados correctamente.


### Resultados

Las diez comprobaciones realizadas sobre los contratos estructurados resultan correctas.

El contrato de entrada acepta el registro con las 813 variables esperadas y rechaza correctamente registros incompletos, variables desconocidas y campos adicionales. De este modo se controla la estructura antes de enviar información al servicio predictivo.

Las cuatro salidas de TEST utilizadas como referencia cumplen el contrato predictivo y mantienen la coherencia con la configuración cerrada. También se rechazan probabilidades fuera del rango permitido, campos de salida no definidos y clasificaciones incompatibles con la probabilidad y el umbral correspondiente.

El informe preventivo puede serializarse correctamente mediante Pydantic y los factores conservan la respuesta observada como parte de su estructura. Todas estas comprobaciones se realizan sin ejecutar inferencia, SHAP, Mistral ni otros servicios externos.

## Síntesis de la sección 2 (memoria)

La interfaz entre la aplicación y la solución predictiva se ha formalizado mediante tres contratos Pydantic independientes para la entrada, la predicción y el informe preventivo. La entrada conserva las 813 variables originales requeridas por el sistema, incluyendo las 59 utilizadas por Machine Learning, y rechaza registros incompletos o con variables no contempladas en el esquema cerrado.

La salida predictiva reproduce los 10 campos definidos durante la transferencia, con 8 campos obligatorios y dos campos opcionales vinculados a la explicabilidad. Además de la validación estructural, se mantiene una comprobación determinista de la correspondencia entre variable objetivo, familia, modelo, umbral y clasificación, evitando que las capas posteriores puedan alterar decisiones predictivas previamente cerradas.

El informe preventivo se mantiene separado de la salida numérica y dispone de una estructura específica para síntesis, interpretación, factores relevantes, orientación, limitaciones, fuentes y advertencia de uso. Cada factor conserva además el significado comprensible de la respuesta registrada y, cuando procede, una aclaración sobre su carácter recodificado o combinado. Las diez pruebas realizadas confirman que los contratos aceptan estructuras válidas y rechazan entradas, probabilidades, clasificaciones o campos incompatibles.

**Tabla candidata:** resumen de los tres contratos y comprobaciones de validación, para ANEXO.

**Figura candidata:** no.

**Destino:** MEMORIA + ANEXO. En la memoria se describirá el papel de Pydantic como frontera estructurada y mecanismo de control entre las capas del sistema; las definiciones completas y las pruebas individuales se reservarán para el anexo.

# 3. Bridge e inferencia predictiva real

La solución predictiva definitiva permanece aislada en el entorno `TFM_ML`, que continúa cerrado y congelado. El Notebook 05 no carga directamente TensorFlow, XGBoost ni SHAP, sino que se comunica con ese entorno mediante un subproceso controlado y un intercambio estructurado en formato JSON.

El bridge debe reproducir exactamente el preprocesamiento y la inferencia cerrados durante los Notebooks 02–04. Machine Learning utiliza el pipeline final ya ajustado para `IRAMDEYR`, mientras que Deep Learning recupera los dos modelos Keras del ensemble final, su preprocesamiento numérico, el encoder categórico y los índices de inicio guardados.

La explicación local se mantiene como una operación opcional. Cuando se solicite, se calculará exclusivamente dentro de `TFM_ML` utilizando la misma estrategia SHAP validada durante la evaluación final, sin modificar la predicción ni recalcular resultados globales.

## 3.1. Archivos y configuración del bridge predictivo

Antes de crear el módulo de inferencia se resuelven las rutas de los archivos declarados durante la transferencia del Notebook 04.

Se recuperan los modelos Machine Learning y Deep Learning, el preprocesamiento final, los metadatos, los rangos documentales, las etiquetas y el conjunto de referencia SHAP. La base premodelado se utilizará únicamente para comprobar posteriormente que el bridge reproduce las predicciones ya cerradas sobre registros conocidos de TEST.

El código del servicio se ubicará en `src/productivizacion/servicio_predictivo.py`. Esta separación permite ejecutar la lógica predictiva con `TFM_ML` sin incorporar sus dependencias al entorno `TFM_Agentes`.

In [14]:
# =============================================================================
# ARCHIVOS Y CONFIGURACIÓN DEL BRIDGE PREDICTIVO
# =============================================================================
# -----------------------------------------------------------------------------
# ARCHIVOS TRANSFERIDOS
# -----------------------------------------------------------------------------
rutas_transferidas = {
    fila.elemento: ruta_proyecto / fila.ruta
    for fila in archivos_transferidos[['elemento', 'ruta']].itertuples(index=False)
}

elementos_bridge = [
    'Configuración de productivización',
    'Referencia SHAP',
    'Modelos finales Machine Learning',
    'Preprocesamiento Deep Learning',
    'Metadatos Deep Learning',
    'Rangos de variables numéricas',
    'Roles y etiquetas'
]

rutas_bridge = {elemento: rutas_transferidas[elemento] for elemento in elementos_bridge}

for clave in modelos_ensemble_dl:
    elemento = f'Modelo Deep Learning — {clave}'
    rutas_bridge[elemento] = rutas_transferidas[elemento]

rutas_bridge['Base premodelado'] = ruta_base_premodelado

# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
archivos_bridge = pd.DataFrame([
    {
        'elemento': elemento,
        'ruta': ruta_relativa(ruta),
        'disponible': ruta.is_file()
    }
    for elemento, ruta in rutas_bridge.items()
])

tabla_archivos_bridge = archivos_bridge.rename(columns=renombrado_comun)

print('\nARCHIVOS DEL BRIDGE PREDICTIVO')
display(tabla_archivos_bridge)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_archivos_bridge = pd.DataFrame({
    'comprobacion': [
        'Todos los archivos requeridos por el bridge están disponibles',
        'Se han localizado los dos modelos del ensemble Deep Learning',
        'La base premodelado está disponible para comprobar las predicciones',
        'La carpeta de productivización está preparada',
        'El módulo predictivo está disponible antes de la primera ejecución'
    ],
    'resultado': [
        archivos_bridge['disponible'].all(),
        all(f'Modelo Deep Learning — {clave}' in rutas_bridge for clave in modelos_ensemble_dl),
        ruta_base_premodelado.is_file(),
        ruta_productivizacion.is_dir(),
        ruta_servicio_predictivo.is_file()
    ]
})

tabla_comprobaciones_archivos_bridge = comprobaciones_archivos_bridge.rename(
    columns=renombrado_comun
)

print('\nCOMPROBACIONES DE LOS ARCHIVOS DEL BRIDGE')
display(tabla_comprobaciones_archivos_bridge)

if not comprobaciones_archivos_bridge['resultado'].all():
    raise FileNotFoundError(
        'No están disponibles todos los archivos requeridos por el bridge predictivo.'
    )

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(archivos_bridge, '05_archivos_bridge_predictivo')
_ = guardar_csv(comprobaciones_archivos_bridge, '06_comprobaciones_archivos_bridge_predictivo')

print('\nArchivos del bridge predictivo validados correctamente.')


ARCHIVOS DEL BRIDGE PREDICTIVO


,Elemento,Ruta,Disponible
0,Configuración de productivización,results/tables/04_evaluacion_explicabilidad/co...,True
1,Referencia SHAP,results/tables/04_evaluacion_explicabilidad/ba...,True
2,Modelos finales Machine Learning,models/02_modelado_ml/modelos_ml_finales.pkl,True
3,Preprocesamiento Deep Learning,models/03_modelado_dl/preprocesamiento_final_d...,True
4,Metadatos Deep Learning,models/03_modelado_dl/metadatos_modelo_final_d...,True
5,Rangos de variables numéricas,results/tables/01_datos_eda/30_resumen_numeric...,True
6,Roles y etiquetas,results/tables/01_datos_eda/28_roles_base_prem...,True
7,Modelo Deep Learning — multisalida_focal_ramas_32,models/03_modelado_dl/modelo_final_focal_ramas...,True
8,Modelo Deep Learning — multisalida_focal_sin_b...,models/03_modelado_dl/modelo_final_focal_sin_b...,True
9,Base premodelado,data/interim/NSDUH_2024_adultos_premodelado.pa...,True



COMPROBACIONES DE LOS ARCHIVOS DEL BRIDGE


,Comprobación,Resultado
0,Todos los archivos requeridos por el bridge es...,True
1,Se han localizado los dos modelos del ensemble...,True
2,La base premodelado está disponible para compr...,True
3,La carpeta de productivización está preparada,True
4,El módulo predictivo está disponible antes de ...,True



Archivos del bridge predictivo validados correctamente.


### Resultados

Los 10 archivos necesarios para construir y comprobar el bridge predictivo se encuentran disponibles. Se recuperan la configuración de productivización, la referencia TRAIN para SHAP, los modelos finales Machine Learning, el preprocesamiento y los metadatos Deep Learning, los rangos documentales y los dos modelos que forman el ensemble definitivo.

También está disponible la base premodelado, que se utilizará exclusivamente para seleccionar registros conocidos y comprobar posteriormente que el servicio reproduce las predicciones ya cerradas en el Notebook 04.

Las cinco comprobaciones realizadas resultan correctas. El archivo del servicio predictivo está disponible antes de la primera ejecución y la carpeta `src/productivizacion` se encuentra preparada para utilizarlo. `TFM_ML` permanece cerrado y no se modifica desde este notebook.

## 3.2. Validación del servicio predictivo

La lógica predictiva se encapsula en un módulo funcional independiente ejecutado exclusivamente mediante el entorno congelado `TFM_ML`. El módulo no entrena, reajusta ni modifica ningún objeto, sino que recupera los modelos y preprocesamientos definitivos y los utiliza únicamente en modo de inferencia.

El servicio reproduce la preparación validada en el Notebook 04. Para las variables numéricas se distinguen los valores comprendidos en el rango documental y los estados asociados a valores ausentes o códigos especiales. Machine Learning reconstruye sus 67 variables de entrada en el orden original, mientras Deep Learning aplica el preprocesamiento numérico y el encoder categórico ya ajustados antes de ejecutar el ensemble formado por los dos modelos finales.

La selección de familia se realiza automáticamente a partir de la configuración transferida: XGBoost se utiliza para episodio depresivo mayor y el ensemble Deep Learning para los otros tres indicadores. Las probabilidades se transforman en clasificaciones mediante los umbrales definitivos, sin recalibración ni decisiones nuevas.

El módulo admite inicialmente dos operaciones: `health`, destinada a comprobar la disponibilidad y consistencia de los recursos predictivos, y `predict`, que recibe un registro con las 813 variables originales y devuelve las cuatro salidas estructuradas. La explicación SHAP se incorporará únicamente después de validar que esta inferencia reproduce las predicciones de referencia.

In [15]:
# =============================================================================
# VALIDACIÓN DEL SERVICIO PREDICTIVO
# =============================================================================
print('\nVALIDACIÓN DEL SERVICIO PREDICTIVO', flush=True)

ruta_conda = Path.home() / 'miniconda3' / 'bin' / 'conda'
solicitud_health = {'operacion': 'health'}

print('[1/4] Iniciando el servicio en TFM_ML...', flush=True)
print('      Recuperación: configuración, modelos ML/DL y referencia SHAP', flush=True)

inicio = perf_counter()

proceso_health = subprocess.run(
    [
        str(ruta_conda), 'run', '--no-capture-output', '-n', 'TFM_ML', 'python',
        str(ruta_servicio_predictivo)
    ],
    input=json.dumps(solicitud_health, ensure_ascii=False), capture_output=True, text=True,
    check=False, timeout=180
)

tiempo_health = perf_counter() - inicio

lineas_health = [linea.strip() for linea in proceso_health.stdout.splitlines() if linea.strip()]

print(f'      Servicio iniciado en {tiempo_health:.2f} s', flush=True)
print('[2/4] Recuperando el estado y la configuración del servicio...', flush=True)

if proceso_health.returncode != 0:
    raise RuntimeError(f'No se ha podido ejecutar el servicio predictivo:\n{proceso_health.stderr}')

if not lineas_health:
    raise RuntimeError('El servicio predictivo no ha devuelto ninguna respuesta.')

respuesta_health = json.loads(lineas_health[-1])

print(
    f"      {respuesta_health.get('modelos_ml')} modelos ML | "
    f"{respuesta_health.get('modelos_dl')} modelos DL | "
    f"{respuesta_health.get('variables_entrada')} variables de entrada", flush=True
)
print('[3/4] Comprobando recursos y contratos transferidos...', flush=True)

# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
resumen_servicio_predictivo = pd.DataFrame({
    'elemento': [
        'Estado',
        'Entorno',
        'Versión de productivización',
        'Variables objetivo',
        'Modelos Machine Learning',
        'Modelos Deep Learning',
        'Variables generales de entrada',
        'Variables originales Machine Learning',
        'Variables de entrada Machine Learning',
        'Registros de referencia SHAP',
        'Variables de referencia SHAP',
        'Tiempo de inicialización (s)'
    ],
    'valor': [
        respuesta_health.get('estado'),
        respuesta_health.get('entorno'),
        respuesta_health.get('version'),
        respuesta_health.get('targets'),
        respuesta_health.get('modelos_ml'),
        respuesta_health.get('modelos_dl'),
        respuesta_health.get('variables_entrada'),
        respuesta_health.get('variables_ml'),
        respuesta_health.get('columnas_ml'),
        respuesta_health.get('background', [None, None])[0],
        respuesta_health.get('background', [None, None])[1],
        round(tiempo_health, 3)
    ]
})

tabla_resumen_servicio = resumen_servicio_predictivo.rename(columns=renombrado_comun)

tabla_resumen_servicio = tabla_resumen_servicio.round({'Valor': 3})

print('\nRESUMEN DEL SERVICIO PREDICTIVO')
display(tabla_resumen_servicio)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_servicio_predictivo = pd.DataFrame({
    'comprobacion': [
        'El archivo del servicio predictivo está disponible',
        'La ejecución mediante TFM_ML finaliza correctamente',
        'El servicio devuelve estado OK',
        'El servicio se ejecuta dentro de TFM_ML',
        'La configuración corresponde a la versión v1',
        'Se recuperan las cuatro variables objetivo',
        'Se recuperan los cuatro modelos Machine Learning',
        'Se recuperan los dos modelos Deep Learning',
        'La entrada general contiene 813 variables',
        'Machine Learning conserva 59 variables originales',
        'Machine Learning conserva 67 variables de entrada',
        'La referencia SHAP conserva 20 registros y 813 variables',
        'Todas las comprobaciones internas del servicio son correctas'
    ],
    'resultado': [
        ruta_servicio_predictivo.is_file(),
        proceso_health.returncode == 0,
        respuesta_health.get('estado') == 'OK',
        respuesta_health.get('entorno') == 'TFM_ML',
        respuesta_health.get('version') == 'v1',
        respuesta_health.get('targets') == 4,
        respuesta_health.get('modelos_ml') == 4,
        respuesta_health.get('modelos_dl') == 2,
        respuesta_health.get('variables_entrada') == 813,
        respuesta_health.get('variables_ml') == 59,
        respuesta_health.get('columnas_ml') == 67,
        respuesta_health.get('background') == [20, 813],
        all(respuesta_health.get('comprobaciones', {}).values())
    ]
})

tabla_comprobaciones_servicio = comprobaciones_servicio_predictivo.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DEL SERVICIO PREDICTIVO')
display(tabla_comprobaciones_servicio)

if not comprobaciones_servicio_predictivo['resultado'].all():
    raise ValueError('El servicio predictivo no está correctamente preparado.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
print('[4/4] Guardando el resumen de validación...', flush=True)
_ = guardar_csv(resumen_servicio_predictivo, '07_resumen_servicio_predictivo')
_ = guardar_csv(comprobaciones_servicio_predictivo, '08_comprobaciones_servicio_predictivo')

print('\nServicio predictivo recuperado correctamente desde TFM_ML.')


VALIDACIÓN DEL SERVICIO PREDICTIVO

[1/4] Iniciando el servicio en TFM_ML...


      Recuperación: configuración, modelos ML/DL y referencia SHAP


      Servicio iniciado en 13.00 s


[2/4] Recuperando el estado y la configuración del servicio...


...

,Elemento,Valor
0,Estado,OK
1,Entorno,TFM_ML
2,Versión de productivización,v1
3,Variables objetivo,4
4,Modelos Machine Learning,4
5,Modelos Deep Learning,2
6,Variables generales de entrada,813
7,Variables originales Machine Learning,59
8,Variables de entrada Machine Learning,67
9,Registros de referencia SHAP,20



COMPROBACIONES DEL SERVICIO PREDICTIVO


,Comprobación,Resultado
0,El archivo del servicio predictivo está dispon...,True
1,La ejecución mediante TFM_ML finaliza correcta...,True
2,El servicio devuelve estado OK,True
3,El servicio se ejecuta dentro de TFM_ML,True
4,La configuración corresponde a la versión v1,True
5,Se recuperan las cuatro variables objetivo,True
6,Se recuperan los cuatro modelos Machine Learning,True
7,Se recuperan los dos modelos Deep Learning,True
8,La entrada general contiene 813 variables,True
9,Machine Learning conserva 59 variables originales,True


[4/4] Guardando el resumen de validación...



Servicio predictivo recuperado correctamente desde TFM_ML.


### Resultados

El servicio predictivo se recupera correctamente mediante un subproceso controlado ejecutado en el entorno `TFM_ML`. El estado devuelto es `OK` y las 13 comprobaciones externas e internas resultan correctas.

El servicio conserva la configuración de productivización `v1`, las cuatro variables objetivo, los cuatro modelos Machine Learning y los dos modelos Deep Learning del ensemble definitivo. La entrada general mantiene las 813 variables originales, Machine Learning conserva sus 59 variables originales y las 67 variables de entrada reconstruidas, y la referencia TRAIN para SHAP mantiene sus 20 registros y 813 variables.

El tiempo de inicialización se registra en la tabla de resultados e incluye la recuperación de modelos y la puesta en marcha de los recursos predictivos, por lo que no representa únicamente el coste de una inferencia.

La validación se completa sin enviar todavía ningún registro para predicción y sin modificar objetos del entorno predictivo.

## 3.3. Primera inferencia real y comprobación de predicciones

Una vez recuperado el servicio se realiza una inferencia completa a través del bridge y se compara su salida con las predicciones cerradas en el Notebook 04.

Para disponer de una referencia reproducible se selecciona un identificador perteneciente a TEST con resultados disponibles para las cuatro variables objetivo. El registro original se recupera desde la base premodelado exclusivamente dentro de `TFM_ML`, evitando incorporar al entorno de agentes las dependencias necesarias para leer el archivo Parquet.

Las 813 variables recuperadas se devuelven a `TFM_Agentes`, se validan mediante `EntradaPredictiva` y se envían de nuevo al servicio mediante la operación `predict`. Para reproducir numéricamente la inferencia cerrada, cada registro individual se evalúa dentro de un grupo técnico fijo de 10 copias idénticas y se conserva la primera salida. Esta convención no modifica variables, modelos, pesos, umbrales ni clasificaciones y se comprobará posteriormente sobre varios registros durante la evaluación extremo a extremo.

Las cuatro salidas se validan con Pydantic y con la configuración predictiva cerrada. Finalmente se comparan probabilidades, umbrales, clasificaciones, familias y modelos con los resultados almacenados durante el Notebook 04. TEST se utiliza únicamente como referencia para esta comprobación y no interviene en ningún reajuste o decisión predictiva.

In [16]:
# =============================================================================
# PRIMERA INFERENCIA REAL Y COMPROBACIÓN DE PREDICCIONES
# =============================================================================
print('\nPRIMERA INFERENCIA REAL Y COMPROBACIÓN DE PREDICCIONES', flush=True)

tolerancia_probabilidad_bridge = 1e-6

# -----------------------------------------------------------------------------
# REGISTRO DE REFERENCIA
# -----------------------------------------------------------------------------
print('[1/5] Seleccionando un registro TEST con las cuatro variables objetivo...', flush=True)

conteo_targets_id = predicciones_finales_test.groupby('QUESTID2')['target'].nunique()
ids_validos_referencia = conteo_targets_id[conteo_targets_id == len(targets)].index
questid2_referencia = int(ids_validos_referencia[0])

print(f'      QUESTID2 seleccionado: {questid2_referencia}', flush=True)
print('[2/5] Recuperando sus 813 variables originales desde TFM_ML...', flush=True)

solicitud_referencia = {
    'operacion': 'reference',
    'QUESTID2': questid2_referencia
}

respuesta_referencia = ejecutar_bridge_predictivo(
    solicitud_referencia, ruta_conda, ruta_servicio_predictivo
)

if respuesta_referencia.get('estado') != 'OK':
    raise RuntimeError(
        respuesta_referencia.get('mensaje', 'No se ha podido recuperar el registro de referencia.')
    )

registro_referencia = respuesta_referencia['registro']

print(f'      Variables recuperadas: {len(registro_referencia)}', flush=True)
print('[3/5] Validando el contrato estructurado de entrada...', flush=True)

entrada_referencia = EntradaPredictiva(registro=registro_referencia, solicitar_explicacion=False)

_ = validar_registro_entrada(entrada_referencia.registro, vars_predictoras_modelado_dl)

print('      Contrato de entrada correcto', flush=True)

# -----------------------------------------------------------------------------
# INFERENCIA
# -----------------------------------------------------------------------------
print('[4/5] Ejecutando la solución predictiva híbrida...', flush=True)

for target in targets:
    print(f"      {target}: {familias_finales[target]} | {modelos_finales[target]}", flush=True)

solicitud_prediccion = {
    'operacion': 'predict',
    'registro': entrada_referencia.registro
}

inicio = perf_counter()

respuesta_prediccion = ejecutar_bridge_predictivo(
    solicitud_prediccion, ruta_conda, ruta_servicio_predictivo
)

tiempo_prediccion_bridge = perf_counter() - inicio

print(f'      Inferencia completada en {tiempo_prediccion_bridge:.2f} s', flush=True)
print('[5/5] Validando salidas y comparando con el Notebook 04...', flush=True)

if respuesta_prediccion.get('estado') != 'OK':
    raise RuntimeError(
        respuesta_prediccion.get('mensaje', 'No se ha podido realizar la inferencia.')
    )

resultados_bridge = []

for resultado in respuesta_prediccion['resultados']:
    resultado_pydantic = ResultadoPredictivo(**resultado)

    resultado_validado = validar_resultado_predictivo(
        resultado_pydantic.model_dump(), configuracion_productivizacion
    )

    resultados_bridge.append(resultado_validado)

predicciones_bridge = pd.DataFrame(resultados_bridge)

# -----------------------------------------------------------------------------
# REFERENCIA DEL NOTEBOOK 04
# -----------------------------------------------------------------------------
predicciones_referencia = predicciones_finales_test.loc[
    predicciones_finales_test['QUESTID2'] == questid2_referencia
].copy()

predicciones_referencia = predicciones_referencia.set_index('target')
predicciones_bridge = predicciones_bridge.set_index('variable_objetivo')

# -----------------------------------------------------------------------------
# COMPARACIÓN
# -----------------------------------------------------------------------------
filas_comparacion = []

for target in targets:
    referencia = predicciones_referencia.loc[target]
    bridge = predicciones_bridge.loc[target]

    diferencia = abs(float(referencia['y_score']) - float(bridge['probabilidad']))

    filas_comparacion.append({
        'target': target,
        'descripcion': titulos_targets[target],
        'familia': bridge['familia'],
        'modelo': bridge['modelo'],
        'probabilidad_referencia': float(referencia['y_score']),
        'probabilidad_bridge': float(bridge['probabilidad']),
        'diferencia_probabilidad': diferencia,
        'umbral_referencia': float(referencia['umbral']),
        'umbral_bridge': float(bridge['umbral']),
        'clasificacion_referencia': int(referencia['y_pred']),
        'clasificacion_bridge': int(bridge['clasificacion']),
        'resultado': (diferencia <= tolerancia_probabilidad_bridge and
                      np.isclose(float(referencia['umbral']), float(bridge['umbral'])) and
                      int(referencia['y_pred']) == int(bridge['clasificacion']) and
                      referencia['familia'] == bridge['familia'] and
                      referencia['modelo'] == bridge['modelo'])
    })

comparacion_predicciones_bridge = pd.DataFrame(filas_comparacion)

# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
resumen_inferencia_bridge = pd.DataFrame({
    'elemento': [
        'QUESTID2 de referencia',
        'Variables enviadas al servicio',
        'Variables objetivo obtenidas',
        'Resultados Pydantic válidos',
        'Diferencia máxima de probabilidad',
        'Tolerancia de probabilidad',
        'Clasificaciones coincidentes',
        'Tiempo total del bridge (s)'
    ],
    'valor': [
        str(questid2_referencia),
        len(entrada_referencia.registro),
        len(resultados_bridge),
        len(resultados_bridge),
        comparacion_predicciones_bridge['diferencia_probabilidad'].max(),
        tolerancia_probabilidad_bridge,
        (comparacion_predicciones_bridge['clasificacion_referencia'] ==
         comparacion_predicciones_bridge['clasificacion_bridge']).sum(),
        round(tiempo_prediccion_bridge, 3)
    ]
})

tabla_resumen_inferencia = resumen_inferencia_bridge.rename(columns=renombrado_comun)

renombrado_comparacion_predicciones = {
    **renombrado_comun,
    'probabilidad_referencia': 'Probabilidad Notebook 04',
    'probabilidad_bridge': 'Probabilidad bridge',
    'diferencia_probabilidad': 'Diferencia',
    'umbral_referencia': 'Umbral Notebook 04',
    'umbral_bridge': 'Umbral bridge',
    'clasificacion_referencia': 'Clasificación Notebook 04',
    'clasificacion_bridge': 'Clasificación bridge'
}

tabla_comparacion_predicciones = comparacion_predicciones_bridge.rename(
    columns=renombrado_comparacion_predicciones
)

tabla_comparacion_predicciones = tabla_comparacion_predicciones.round({
    'Probabilidad Notebook 04': 6, 'Probabilidad bridge': 6, 'Diferencia': 8,
    'Umbral Notebook 04': 3, 'Umbral bridge': 3
})

print('\nRESUMEN DE LA PRIMERA INFERENCIA')
display(tabla_resumen_inferencia)

print('\nCOMPARACIÓN CON EL NOTEBOOK 04')
display(tabla_comparacion_predicciones)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_inferencia_bridge = pd.DataFrame({
    'comprobacion': [
        'El registro de referencia contiene las 813 variables',
        'La entrada supera el contrato Pydantic',
        'El servicio devuelve las cuatro variables objetivo',
        'Las cuatro salidas superan el contrato predictivo',
        'Las probabilidades coinciden con el Notebook 04 dentro de la tolerancia',
        'Los umbrales coinciden con los valores definitivos',
        'Las cuatro clasificaciones coinciden',
        'Las familias coinciden con la selección definitiva',
        'Los modelos coinciden con la selección definitiva'
    ],
    'resultado': [
        len(entrada_referencia.registro) == 813,
        isinstance(entrada_referencia, EntradaPredictiva),
        len(resultados_bridge) == 4,
        len(resultados_bridge) == 4,
        (comparacion_predicciones_bridge['diferencia_probabilidad'] <=
         tolerancia_probabilidad_bridge).all(),
        np.isclose(comparacion_predicciones_bridge['umbral_referencia'],
                   comparacion_predicciones_bridge['umbral_bridge']).all(),
        (comparacion_predicciones_bridge['clasificacion_referencia'] ==
         comparacion_predicciones_bridge['clasificacion_bridge']).all(),
        (comparacion_predicciones_bridge['familia'].to_numpy() ==
         np.array([familias_finales[target] for target in targets])).all(),
        (comparacion_predicciones_bridge['modelo'].to_numpy() ==
         np.array([modelos_finales[target] for target in targets])).all()
    ]
})

tabla_comprobaciones_inferencia = comprobaciones_inferencia_bridge.rename(
    columns=renombrado_comun
)

print('\nCOMPROBACIONES DE LA PRIMERA INFERENCIA')
display(tabla_comprobaciones_inferencia)

if not comprobaciones_inferencia_bridge['resultado'].all():
    raise ValueError('La inferencia del bridge no coincide con la solución predictiva cerrada.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_inferencia_bridge, '09_resumen_primera_inferencia_bridge')
_ = guardar_csv(comparacion_predicciones_bridge, '10_comparacion_predicciones_bridge')
_ = guardar_csv(comprobaciones_inferencia_bridge, '11_comprobaciones_primera_inferencia_bridge')

print('\nPrimera inferencia real completada correctamente.')
print('La salida coincide con la solución predictiva cerrada del Notebook 04.')


PRIMERA INFERENCIA REAL Y COMPROBACIÓN DE PREDICCIONES


[1/5] Seleccionando un registro TEST con las cuatro variables objetivo...


      QUESTID2 seleccionado: 10004548


[2/5] Recuperando sus 813 variables originales desde TFM_ML...


      Variables recuperadas: 813


...


COMPARACIÓN CON EL NOTEBOOK 04


,Variable objetivo,Descripción,Familia,Modelo,Probabilidad Notebook 04,Probabilidad bridge,Diferencia,Umbral Notebook 04,Umbral bridge,Clasificación Notebook 04,Clasificación bridge,Resultado
0,IRAMDEYR,Episodio depresivo mayor,ML,XGBoost,0.826969,0.826969,0.0,0.500,0.500,1,1,True
1,IRSUICTHNK,Ideación suicida,DL,Ensemble — ramas 32 + sin BatchNormalization (...,0.565728,0.565728,0.0,0.509,0.509,1,1,True
2,IRSUIPLANYR,Planificación suicida,DL,Ensemble — ramas 32 + sin BatchNormalization (...,0.525724,0.525724,0.0,0.536,0.536,0,0,True
3,IRSUITRYYR,Intento suicida,DL,Ensemble — ramas 32 + sin BatchNormalization (...,0.524104,0.524104,0.0,0.502,0.502,1,1,True



COMPROBACIONES DE LA PRIMERA INFERENCIA


,Comprobación,Resultado
0,El registro de referencia contiene las 813 var...,True
1,La entrada supera el contrato Pydantic,True
2,El servicio devuelve las cuatro variables obje...,True
3,Las cuatro salidas superan el contrato predictivo,True
4,Las probabilidades coinciden con el Notebook 0...,True
5,Los umbrales coinciden con los valores definit...,True
6,Las cuatro clasificaciones coinciden,True
7,Las familias coinciden con la selección defini...,True
8,Los modelos coinciden con la selección definitiva,True



Primera inferencia real completada correctamente.
La salida coincide con la solución predictiva cerrada del Notebook 04.


### Resultados

La primera inferencia real mediante el bridge se completa correctamente a partir de un registro conocido de TEST y sus 813 variables originales.

El servicio devuelve las cuatro variables objetivo mediante una salida JSON que cumple tanto el contrato Pydantic como las comprobaciones deterministas de la configuración cerrada. Las probabilidades obtenidas coinciden exactamente con las almacenadas en el Notebook 04, con una diferencia máxima de `0,0`, inferior a la tolerancia numérica de `1e-6`.

También coinciden los cuatro umbrales, las cuatro clasificaciones, las familias seleccionadas y los modelos definitivos. La solución híbrida mantiene por tanto sus resultados sin realizar reajustes ni modificaciones sobre `TFM_ML`.

El tiempo completo del bridge se registra en la tabla anterior e incluye el inicio del subproceso, la recuperación de los modelos, la inferencia Machine Learning y Deep Learning y la transferencia de la respuesta estructurada.

## 3.4. Explicación local SHAP y validación final del bridge

Una vez comprobada la coincidencia de las predicciones, se incorpora la explicación local bajo demanda utilizando el mismo procedimiento SHAP aplicado durante el Notebook 04.

La explicación se calcula exclusivamente dentro de `TFM_ML`, donde permanecen disponibles SHAP, los modelos finales y el conjunto TRAIN de referencia. Se utiliza `shap.Explainer` con el algoritmo `permutation` sobre las 813 variables originales y el background de 20 registros procedente de TRAIN, manteniendo la semilla, el número mínimo de evaluaciones y el tamaño de bloque utilizados durante la evaluación final.

La función explicada reutiliza la misma preparación Machine Learning y Deep Learning del servicio predictivo, incluida la convención técnica de ejecución individual validada en el apartado anterior. Para comprobar las dos ramas de la solución híbrida se explican dos resultados del mismo registro de referencia: uno perteneciente a Machine Learning y otro a Deep Learning.

En cada explicación se recuperan las 15 variables con mayor contribución SHAP absoluta, conservando el código de la variable, su descripción documentada, el valor observado y el signo de la contribución. La diferencia entre la probabilidad reconstruida mediante SHAP y la predicción explicada se mantiene como información técnica y la tolerancia de `0,0075` se utiliza como referencia descriptiva, sin modificar la predicción.

Finalmente se comprueba la respuesta controlada del servicio ante una entrada incompleta, una variable objetivo desconocida y una operación no reconocida. Las contribuciones SHAP describen la predicción del modelo y no deben interpretarse como relaciones causales.

In [17]:
# =============================================================================
# EXPLICACIÓN LOCAL SHAP Y VALIDACIÓN FINAL DEL BRIDGE
# =============================================================================
print('\nEXPLICACIÓN LOCAL SHAP Y VALIDACIÓN FINAL DEL BRIDGE', flush=True)

tolerancia_reconstruccion_shap = 0.0075
n_variables_explicacion = 15
n_evaluaciones_esperadas_shap = 2 * len(vars_predictoras_modelado_dl) + 1

print(
    f'[1/4] Configuración SHAP: {len(vars_predictoras_modelado_dl)} variables | '
    f'20 registros TRAIN | {n_evaluaciones_esperadas_shap:,} evaluaciones'.replace(',', '.'),
    flush=True
)

target_ml_explicacion = next(target for target in targets if familias_finales[target] == 'ML')
target_dl_explicacion = next(target for target in targets if familias_finales[target] == 'DL')
targets_explicacion = [target_ml_explicacion, target_dl_explicacion]

# -----------------------------------------------------------------------------
# EXPLICACIONES LOCALES
# -----------------------------------------------------------------------------
explicaciones_bridge = []
filas_variables_shap = []

for numero, target in enumerate(targets_explicacion, start=1):
    familia = familias_finales[target]

    print(
        f'\n[2/4] Explicación {numero}/{len(targets_explicacion)}: '
        f'{target} — {titulos_targets[target]}', flush=True
    )
    print(f'      Familia: {familia} | Modelo: {modelos_finales[target]}', flush=True)
    print(
        f'      SHAP permutation: {n_evaluaciones_esperadas_shap} evaluaciones '
        f'→ selección de {n_variables_explicacion} variables', flush=True
    )

    solicitud_explicacion = {
        'operacion': 'explain',
        'registro': entrada_referencia.registro,
        'variable_objetivo': target
    }

    respuesta_explicacion = ejecutar_bridge_predictivo(
        solicitud_explicacion, ruta_conda, ruta_servicio_predictivo, timeout=600
    )

    if respuesta_explicacion.get('estado') != 'OK':
        raise RuntimeError(
            respuesta_explicacion.get('mensaje', f'No se ha podido explicar {target}.')
        )

    explicacion = respuesta_explicacion['explicacion']

    print(
        f"      Completada en {explicacion['tiempo_s']:.2f} s | "
        f"Diferencia de reconstrucción: {explicacion['diferencia_reconstruccion']:.6f}", flush=True
    )

    explicaciones_bridge.append({
        'target': target,
        'descripcion': explicacion['descripcion'],
        'familia': explicacion['familia'],
        'modelo': explicacion['modelo'],
        'probabilidad': explicacion['probabilidad'],
        'valor_base': explicacion['valor_base'],
        'reconstruccion': explicacion['reconstruccion'],
        'diferencia_reconstruccion': explicacion['diferencia_reconstruccion'],
        'n_variables': len(explicacion['variables_principales']),
        'n_evaluaciones_shap': explicacion['n_evaluaciones_shap'],
        'tiempo_s': explicacion['tiempo_s']
    })

    for posicion, variable_shap in enumerate(explicacion['variables_principales'], start=1):
        filas_variables_shap.append({
            'target': target,
            'descripcion': explicacion['descripcion'],
            'familia': explicacion['familia'],
            'posicion': posicion,
            'variable': variable_shap['variable'],
            'etiqueta_variable': variable_shap['etiqueta_variable'],
            'valor_observado': variable_shap['valor_observado'],
            'valor_shap': variable_shap['valor_shap'],
            'signo_contribucion': variable_shap['signo_contribucion']
        })

resumen_explicaciones_bridge = pd.DataFrame(explicaciones_bridge)
variables_shap_bridge = pd.DataFrame(filas_variables_shap)

# -----------------------------------------------------------------------------
# COMPARACIÓN CON LA PREDICCIÓN DEL BRIDGE
# -----------------------------------------------------------------------------
probabilidades_bridge = {
    target: float(predicciones_bridge.loc[target, 'probabilidad']) for target in targets_explicacion
}

resumen_explicaciones_bridge['diferencia_probabilidad'] = [
    abs(fila.probabilidad - probabilidades_bridge[fila.target])
    for fila in resumen_explicaciones_bridge.itertuples()
]

resumen_explicaciones_bridge['reconstruccion_dentro_tolerancia'] = (
    resumen_explicaciones_bridge['diferencia_reconstruccion'] <= tolerancia_reconstruccion_shap
)

# -----------------------------------------------------------------------------
# ERRORES CONTROLADOS
# -----------------------------------------------------------------------------
print('\n[3/4] Comprobando respuestas ante entradas no válidas...', flush=True)

registro_incompleto = entrada_referencia.registro.copy()
registro_incompleto.pop(vars_predictoras_modelado_dl[0])

print('      Entrada incompleta...', end=' ', flush=True)
respuesta_entrada_incompleta = ejecutar_bridge_predictivo(
    {
        'operacion': 'predict',
        'registro': registro_incompleto
    },
    ruta_conda, ruta_servicio_predictivo
)
print(
    'OK' if respuesta_entrada_incompleta.get('estado') == 'ERROR' else 'ERROR',
    flush=True
)

print('      Variable objetivo desconocida...', end=' ', flush=True)
respuesta_target_desconocido = ejecutar_bridge_predictivo(
    {
        'operacion': 'explain',
        'registro': entrada_referencia.registro,
        'variable_objetivo': 'TARGET_DESCONOCIDO'
    },
    ruta_conda, ruta_servicio_predictivo
)
print(
    'OK' if respuesta_target_desconocido.get('estado') == 'ERROR' else 'ERROR', flush=True
)

print('      Operación desconocida...', end=' ', flush=True)
respuesta_operacion_desconocida = ejecutar_bridge_predictivo(
    {
        'operacion': 'OPERACION_DESCONOCIDA'
    },
    ruta_conda, ruta_servicio_predictivo
)
print(
    'OK' if respuesta_operacion_desconocida.get('estado') == 'ERROR' else 'ERROR', flush=True
)

print('\n[4/4] Consolidando resultados y comprobaciones...', flush=True)
# -----------------------------------------------------------------------------
# PRESENTACIÓN
# -----------------------------------------------------------------------------
tabla_resumen_explicaciones = resumen_explicaciones_bridge.rename(
    columns={
        **renombrado_comun, 'valor_base': 'Valor base', 'reconstruccion': 'Reconstrucción',
        'diferencia_reconstruccion': 'Diferencia de reconstrucción',
        'n_variables': 'Variables principales',
        'n_evaluaciones_shap': 'Evaluaciones SHAP', 'tiempo_s': 'Tiempo (s)',
        'diferencia_probabilidad': 'Diferencia de probabilidad',
        'reconstruccion_dentro_tolerancia': 'Dentro de tolerancia'
    }
)

tabla_resumen_explicaciones = tabla_resumen_explicaciones.round({
    'Probabilidad': 6, 'Valor base': 6, 'Reconstrucción': 6, 'Diferencia de reconstrucción': 6,
    'Diferencia de probabilidad': 8, 'Tiempo (s)': 2
})

tabla_variables_shap = variables_shap_bridge.rename(
    columns={
        **renombrado_comun, 'valor_observado': 'Valor observado', 'valor_shap': 'Valor SHAP',
        'signo_contribucion': 'Contribución'
    }
)

tabla_variables_shap = tabla_variables_shap.round({'Valor SHAP': 6})

print('\nRESUMEN DE LAS EXPLICACIONES LOCALES')
display(tabla_resumen_explicaciones)

print('\nVARIABLES PRINCIPALES DE LAS EXPLICACIONES')
display(tabla_variables_shap)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
claves_variables_shap = {
    'variable',
    'etiqueta_variable',
    'valor_observado',
    'valor_shap',
    'signo_contribucion'
}

comprobaciones_bridge_final = pd.DataFrame({
    'comprobacion': [
        'Se obtiene una explicación Machine Learning',
        'Se obtiene una explicación Deep Learning',
        'Las explicaciones corresponden a los objetivos solicitados',
        'Las explicaciones corresponden a los modelos definitivos',
        'Cada explicación contiene 15 variables principales',
        'Las variables principales mantienen la estructura esperada',
        'Todas las variables explicativas pertenecen al esquema de entrada',
        'Los valores SHAP son finitos',
        'Las explicaciones utilizan 1.627 evaluaciones SHAP',
        'Las probabilidades explicadas coinciden con las predicciones del bridge',
        'Las diferencias de reconstrucción son finitas',
        'La entrada incompleta genera un error controlado',
        'La variable objetivo desconocida genera un error controlado',
        'La operación desconocida genera un error controlado'
    ],
    'resultado': [
        resumen_explicaciones_bridge['familia'].eq('ML').sum() == 1,
        resumen_explicaciones_bridge['familia'].eq('DL').sum() == 1,
        set(resumen_explicaciones_bridge['target']) == set(targets_explicacion),
        all(fila.modelo == modelos_finales[fila.target]
            for fila in resumen_explicaciones_bridge.itertuples()),
        resumen_explicaciones_bridge['n_variables'].eq(n_variables_explicacion).all(),
        set(claves_variables_shap).issubset(variables_shap_bridge.columns),
        set(variables_shap_bridge['variable']).issubset(set(vars_predictoras_modelado_dl)),
        np.isfinite(variables_shap_bridge['valor_shap']).all(),
        resumen_explicaciones_bridge['n_evaluaciones_shap'].eq(n_evaluaciones_esperadas_shap).all(),
        (resumen_explicaciones_bridge['diferencia_probabilidad'] <=
         tolerancia_probabilidad_bridge).all(),
        np.isfinite(resumen_explicaciones_bridge['diferencia_reconstruccion']).all(),
        respuesta_entrada_incompleta.get('estado') == 'ERROR',
        respuesta_target_desconocido.get('estado') == 'ERROR',
        respuesta_operacion_desconocida.get('estado') == 'ERROR'
    ]
})

tabla_comprobaciones_bridge_final = comprobaciones_bridge_final.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES FINALES DEL BRIDGE')
display(tabla_comprobaciones_bridge_final)

if not comprobaciones_bridge_final['resultado'].all():
    raise ValueError('La validación final del bridge predictivo no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_explicaciones_bridge, '12_resumen_explicaciones_locales_bridge')
_ = guardar_csv(variables_shap_bridge, '13_variables_principales_shap_bridge')
_ = guardar_csv(comprobaciones_bridge_final, '14_comprobaciones_finales_bridge')

print('\nBridge predictivo y explicación local validados correctamente.')
print('La capa predictiva permanece aislada dentro de TFM_ML.')


EXPLICACIÓN LOCAL SHAP Y VALIDACIÓN FINAL DEL BRIDGE


[1/4] Configuración SHAP: 813 variables | 20 registros TRAIN | 1.627 evaluaciones



[2/4] Explicación 1/2: IRAMDEYR — Episodio depresivo mayor


      Familia: ML | Modelo: XGBoost


      SHAP permutation: 1627 evaluaciones → selección de 15 variables


...


VARIABLES PRINCIPALES DE LAS EXPLICACIONES


,Variable objetivo,Descripción,Familia,Posición,Variable,Descripción variable,Valor observado,Valor SHAP,Contribución
0,IRAMDEYR,Episodio depresivo mayor,ML,1,CAMHPROB2,RC-PERCEIVED EVER HAD A MENTAL HEALTH ISSUE,1.0,0.352954,Positiva
1,IRAMDEYR,Episodio depresivo mayor,ML,2,RCVYMHPRB,RC-PERCEIVED RECOVERY FROM MENTAL HEALTH ISSUE,1.0,-0.062648,Negativa
2,IRAMDEYR,Episodio depresivo mayor,ML,3,LVLDIFMEM2,LEVEL OF DIFFICULTY REMEMBERING OR CONCENTRATING,2.0,0.058497,Positiva
3,IRAMDEYR,Episodio depresivo mayor,ML,4,LVLDIFCARE2,LEVEL OF DIFFICULTY WITH SELF-CARE,2.0,0.057277,Positiva
4,IRAMDEYR,Episodio depresivo mayor,ML,5,HEALTH2,RC-OVERALL HEALTH RECODE,4.0,0.050178,Positiva
5,IRAMDEYR,Episodio depresivo mayor,ML,6,SVYRSUDANY,"RC-SUBSTANCE USE DISORDER SEVERITY, PY USERS",1.0,0.047322,Positiva
6,IRAMDEYR,Episodio depresivo mayor,ML,7,PREGST,RC-PREGNANCY STATUS OF FEMALES AGED 12-50,2.0,0.034099,Positiva
7,IRAMDEYR,Episodio depresivo mayor,ML,8,TQSDANYFLG,RC-ANY TRANQUILIZERS OR SEDATIVES - EVER USED,1.0,0.016946,Positiva
8,IRAMDEYR,Episodio depresivo mayor,ML,9,NMERTMT2,# OF TIMES BEEN TREATED IN EMER ROOM PAST 12 MOS,0.0,-0.012164,Negativa
9,IRAMDEYR,Episodio depresivo mayor,ML,10,MRJAGLST,HOW OLD WERE YOU THE LAST TIME USED MARIJUANA/...,999.0,-0.011229,Negativa



COMPROBACIONES FINALES DEL BRIDGE


,Comprobación,Resultado
0,Se obtiene una explicación Machine Learning,True
1,Se obtiene una explicación Deep Learning,True
2,Las explicaciones corresponden a los objetivos...,True
3,Las explicaciones corresponden a los modelos d...,True
4,Cada explicación contiene 15 variables princip...,True
5,Las variables principales mantienen la estruct...,True
6,Todas las variables explicativas pertenecen al...,True
7,Los valores SHAP son finitos,True
8,Las explicaciones utilizan 1.627 evaluaciones ...,True
9,Las probabilidades explicadas coinciden con la...,True



Bridge predictivo y explicación local validados correctamente.
La capa predictiva permanece aislada dentro de TFM_ML.


### Resultados

La explicación local se ejecuta correctamente para las dos ramas de la solución híbrida: XGBoost para episodio depresivo mayor y el ensemble Deep Learning para ideación suicida.

Cada explicación utiliza las 813 variables originales, el background de 20 registros procedente de TRAIN y las 1.627 evaluaciones requeridas por el algoritmo `permutation`. En ambos casos se recuperan las 15 variables con mayor contribución SHAP absoluta junto con su descripción, valor observado y signo de la contribución.

Las probabilidades explicadas coinciden exactamente con las obtenidas previamente por el bridge. La reconstrucción SHAP presenta una diferencia de `0,0` en ambas explicaciones, permaneciendo dentro de la tolerancia técnica de `0,0075`.

Los tiempos de las explicaciones se registran en la tabla anterior. La rama Deep Learning presenta un coste computacional claramente superior al de Machine Learning, lo que respalda mantener la explicación SHAP como operación opcional bajo demanda.

Las 14 comprobaciones finales resultan correctas, incluyendo la estructura de las explicaciones y la gestión controlada de entradas incompletas, variables objetivo desconocidas y operaciones no reconocidas. La capa predictiva y la explicabilidad permanecen aisladas dentro de `TFM_ML`.

## Síntesis de la sección 3 (memoria)

La solución predictiva cerrada ha quedado accesible desde `TFM_Agentes` mediante un bridge JSON que ejecuta un servicio de solo lectura dentro del entorno congelado `TFM_ML`. El servicio recupera los modelos y preprocesamientos definitivos sin importar TensorFlow, XGBoost o SHAP en la capa de agentes y mantiene así la separación técnica establecida entre ambos entornos.

La primera inferencia completa recibe las 813 variables originales y devuelve las cuatro variables objetivo mediante una salida estructurada validada con Pydantic. Las probabilidades, umbrales, clasificaciones, familias y modelos coinciden exactamente con los resultados almacenados durante el cierre del Notebook 04, con una diferencia máxima de probabilidad de `0,0`.

La explicabilidad local se incorpora como operación opcional y permanece también dentro de `TFM_ML`. Las pruebas realizadas sobre una rama Machine Learning y otra Deep Learning recuperan 15 variables principales por explicación mediante SHAP sobre las variables originales. En ambos casos la probabilidad explicada y su reconstrucción coinciden con la salida predictiva, mientras que el mayor tiempo requerido por Deep Learning confirma que la explicación debe mantenerse bajo demanda.

El bridge responde además de forma controlada ante entradas incompletas, variables objetivo desconocidas y operaciones no reconocidas. De este modo queda establecida una interfaz predictiva determinista y validada que puede ser reutilizada posteriormente por tools, MCP, LangGraph y Flask sin duplicar la lógica de los modelos.

**Tabla candidata:** comparación de las cuatro predicciones del bridge con el Notebook 04 y resumen de las dos explicaciones locales SHAP.

**Figura candidata:** no; las visualizaciones de importancia SHAP global ya fueron desarrolladas en el Notebook 04.

**Destino:** MEMORIA + ANEXO. La memoria incluirá la arquitectura del bridge, la coincidencia de predicciones y la validación de las dos ramas de explicabilidad; las comprobaciones completas y las 30 contribuciones SHAP locales analizadas se reservarán para el anexo.

# 4. Tools y MCP

Una vez validado el bridge predictivo se encapsula su capacidad mediante tools que puedan ser utilizadas posteriormente por el flujo con LangGraph.

Se comparan dos mecanismos sobre exactamente la misma operación predictiva. La tool local utiliza directamente la función de comunicación con `TFM_ML`, mientras que MCP incorpora una capa cliente-servidor mediante transporte `stdio` y descubrimiento dinámico de herramientas. Ambos mecanismos deben recibir el mismo registro de 813 variables y devolver las mismas cuatro predicciones cerradas.

La finalidad de esta comparación no es sustituir por defecto la solución local por MCP. El sistema predictivo pertenece al propio proyecto y la simplicidad, trazabilidad y aislamiento de `TFM_ML` continúan siendo requisitos prioritarios. MCP se evalúa como mecanismo estandarizado de exposición y consumo de tools, tal como se trabajó en el Máster y se validó previamente en el entorno `TFM_Agentes`.

La integración real con LangGraph se realizará posteriormente sobre el mecanismo seleccionado, evitando construir dos grafos equivalentes únicamente para repetir la misma prueba.

## 4.1. Tool predictiva local

El bridge validado se encapsula como una tool de LangChain sin modificar su lógica interna. La herramienta recibe el registro original, aplica los contratos de entrada y salida ya definidos y delega la inferencia en `TFM_ML`.

La tool no conoce los modelos ni sus preprocesamientos y tampoco recalcula probabilidades, umbrales o clasificaciones. Su responsabilidad se limita a validar la solicitud, utilizar el bridge y devolver las cuatro salidas predictivas estructuradas.

Para comprobar su funcionamiento se utiliza el mismo registro de referencia empleado en la sección anterior y se comparan las probabilidades obtenidas con las ya validadas directamente mediante el bridge.

In [18]:
# =============================================================================
# TOOL PREDICTIVA LOCAL
# =============================================================================
print('\nTOOL PREDICTIVA LOCAL')


@tool('predecir_indicadores')
def tool_predictiva_local(registro: dict[str, Any]) -> dict[str, Any]:
    """Obtiene las cuatro predicciones definitivas a partir de un registro NSDUH."""
    entrada = EntradaPredictiva(registro=registro)
    _ = validar_registro_entrada(entrada.registro, vars_predictoras_modelado_dl)

    respuesta = ejecutar_bridge_predictivo(
        {
            'operacion': 'predict',
            'registro': entrada.registro
        }, ruta_conda, ruta_servicio_predictivo
    )

    if respuesta.get('estado') != 'OK':
        raise RuntimeError(respuesta.get('mensaje', 'Error en el servicio predictivo.'))

    resultados = []
    for resultado in respuesta['resultados']:
        resultado_pydantic = ResultadoPredictivo(**resultado)
        _ = validar_resultado_predictivo(
            resultado_pydantic.model_dump(), configuracion_productivizacion
        )
        resultados.append(resultado_pydantic.model_dump(mode='json'))

    return {'estado': 'OK', 'resultados': resultados}


# -----------------------------------------------------------------------------
# EJECUCIÓN
# -----------------------------------------------------------------------------
inicio = perf_counter()
respuesta_tool_local = tool_predictiva_local.invoke({'registro': entrada_referencia.registro})
tiempo_tool_local = perf_counter() - inicio

predicciones_tool_local = pd.DataFrame(respuesta_tool_local['resultados'])
predicciones_tool_local = predicciones_tool_local.set_index('variable_objetivo')

# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
resumen_tool_local = pd.DataFrame({
    'elemento': [
        'Nombre de la tool',
        'Variables de entrada',
        'Variables objetivo devueltas',
        'Estado',
        'Tiempo de ejecución (s)'
    ],
    'valor': [
        tool_predictiva_local.name,
        len(entrada_referencia.registro),
        len(predicciones_tool_local),
        respuesta_tool_local['estado'],
        round(tiempo_tool_local, 3)
    ]
})

tabla_resumen_tool_local = resumen_tool_local.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA TOOL LOCAL')
display(tabla_resumen_tool_local)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
probabilidades_local_correctas = all(
    np.isclose(
        predicciones_tool_local.loc[target, 'probabilidad'],
        predicciones_bridge.loc[target, 'probabilidad'],
        atol=tolerancia_probabilidad_bridge, rtol=0
    )
    for target in targets
)

comprobaciones_tool_local = pd.DataFrame({
    'comprobacion': [
        'La tool utiliza el nombre predictivo definido',
        'El esquema de entrada contiene el registro',
        'La ejecución devuelve estado OK',
        'Se obtienen las cuatro variables objetivo',
        'Las probabilidades coinciden con el bridge',
        'Las salidas mantienen los contratos predictivos'
    ],
    'resultado': [
        tool_predictiva_local.name == 'predecir_indicadores',
        'registro' in tool_predictiva_local.args_schema.model_fields,
        respuesta_tool_local['estado'] == 'OK',
        len(predicciones_tool_local) == 4,
        probabilidades_local_correctas,
        set(predicciones_tool_local.index) == set(targets)
    ]
})

tabla_comprobaciones_tool_local = comprobaciones_tool_local.rename(
    columns=renombrado_comun
)

print('\nCOMPROBACIONES DE LA TOOL LOCAL')
display(tabla_comprobaciones_tool_local)

if not comprobaciones_tool_local['resultado'].all():
    raise ValueError('La tool predictiva local no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_tool_local, '15_resumen_tool_local')
_ = guardar_csv(comprobaciones_tool_local, '16_comprobaciones_tool_local')

print('\nTool predictiva local validada correctamente.')


TOOL PREDICTIVA LOCAL



RESUMEN DE LA TOOL LOCAL


,Elemento,Valor
0,Nombre de la tool,predecir_indicadores
1,Variables de entrada,813
2,Variables objetivo devueltas,4
3,Estado,OK
4,Tiempo de ejecución (s),7.833



COMPROBACIONES DE LA TOOL LOCAL


,Comprobación,Resultado
0,La tool utiliza el nombre predictivo definido,True
1,El esquema de entrada contiene el registro,True
2,La ejecución devuelve estado OK,True
3,Se obtienen las cuatro variables objetivo,True
4,Las probabilidades coinciden con el bridge,True
5,Las salidas mantienen los contratos predictivos,True



Tool predictiva local validada correctamente.


### Resultados

La capacidad predictiva se encapsula correctamente como una tool local de LangChain denominada `predecir_indicadores`.

La herramienta recibe las 813 variables originales, valida la entrada mediante el contrato definido y delega la inferencia en el bridge hacia `TFM_ML`. Como resultado devuelve las cuatro variables objetivo mediante las mismas salidas estructuradas utilizadas en la sección anterior.

Las seis comprobaciones realizadas resultan correctas. El esquema de entrada contiene el registro esperado, la ejecución devuelve estado `OK`, se obtienen las cuatro variables objetivo y sus probabilidades coinciden con las generadas directamente mediante el bridge.

El tiempo de ejecución observado se registra en la tabla anterior. Incluye el inicio del subproceso predictivo en `TFM_ML`, por lo que no representa únicamente el coste añadido por la tool de LangChain.

## 4.2. Tool predictiva mediante MCP

La misma capacidad predictiva se expone mediante un servidor MCP local ejecutado en `TFM_Agentes`. El servidor no contiene modelos ni replica la preparación de los datos: su única tool delega nuevamente la inferencia en el bridge validado hacia `TFM_ML`.

El cliente utiliza transporte `stdio` y descubre dinámicamente las herramientas disponibles. La tool recuperada se adapta automáticamente a la interfaz de LangChain mediante `langchain-mcp-adapters`, de forma que pueda utilizarse posteriormente dentro del mismo flujo con LangGraph que una tool local.

La prueba utiliza de nuevo el registro de referencia de 813 variables. Se comprueba el descubrimiento de la herramienta, su esquema de entrada, la ejecución asíncrona y la coincidencia de las cuatro predicciones con la salida directa del bridge.

In [19]:
# =============================================================================
# TOOL PREDICTIVA MEDIANTE MCP
# =============================================================================
print('\nTOOL PREDICTIVA MEDIANTE MCP', flush=True)

cliente_mcp = MultiServerMCPClient({
    'predictivo': {
        'command': sys.executable,
        'args': [str(ruta_servidor_mcp)],
        'transport': 'stdio'
    }
})

print('[1/3] Descubriendo tools MCP...', flush=True)
inicio = perf_counter()

tools_mcp = await cliente_mcp.get_tools()

tiempo_descubrimiento_mcp = perf_counter() - inicio

nombres_tools_mcp = [tool_mcp.name for tool_mcp in tools_mcp]

tool_predictiva_mcp = next(
    tool_mcp for tool_mcp in tools_mcp if tool_mcp.name == 'predecir_indicadores'
)

print(f'      {nombres_tools_mcp} | {tiempo_descubrimiento_mcp:.2f} s', flush=True)

print('[2/3] Ejecutando la tool predictiva...', flush=True)
inicio = perf_counter()

resultado_mcp = await tool_predictiva_mcp.ainvoke(
    {'registro': entrada_referencia.registro}
)

tiempo_tool_mcp = perf_counter() - inicio

print(f'      Respuesta recibida en {tiempo_tool_mcp:.2f} s', flush=True)

print('[3/3] Validando respuesta y comparación con el bridge...', flush=True)
bloque_texto_mcp = next(
    bloque for bloque in resultado_mcp
    if isinstance(bloque, dict) and bloque.get('type') == 'text'
)

respuesta_tool_mcp = json.loads(bloque_texto_mcp['text'])

resultados_tool_mcp = []

for resultado in respuesta_tool_mcp['resultados']:
    resultado_pydantic = ResultadoPredictivo(**resultado)

    _ = validar_resultado_predictivo(
        resultado_pydantic.model_dump(), configuracion_productivizacion
    )

    resultados_tool_mcp.append(resultado_pydantic.model_dump(mode='json'))

predicciones_tool_mcp = pd.DataFrame(resultados_tool_mcp).set_index('variable_objetivo')

probabilidades_mcp_correctas = all(
    np.isclose(
        predicciones_tool_mcp.loc[target, 'probabilidad'],
        predicciones_bridge.loc[target, 'probabilidad'],
        atol=tolerancia_probabilidad_bridge, rtol=0
    )
    for target in targets
)

resumen_tool_mcp = pd.DataFrame({
    'elemento': [
        'Servidor MCP', 'Tools descubiertas', 'Tool predictiva',
        'Variables de entrada', 'Variables objetivo devueltas', 'Estado',
        'Tiempo de descubrimiento (s)', 'Tiempo de ejecución (s)'
    ],
    'valor': [
        'predictivo', len(tools_mcp), tool_predictiva_mcp.name,
        len(entrada_referencia.registro), len(predicciones_tool_mcp),
        respuesta_tool_mcp.get('estado'), round(tiempo_descubrimiento_mcp, 3),
        round(tiempo_tool_mcp, 3)
    ]
})

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_tool_mcp = pd.DataFrame({
    'comprobacion': [
        'El archivo del servidor MCP está disponible',
        'La tool predictiva se descubre dinámicamente',
        'El esquema MCP contiene el registro de entrada',
        'La ejecución devuelve estado OK',
        'Se obtienen las cuatro variables objetivo',
        'Las probabilidades coinciden con el bridge',
        'Las salidas mantienen los contratos predictivos'
    ],
    'resultado': [
        ruta_servidor_mcp.is_file(),
        'predecir_indicadores' in nombres_tools_mcp,
        'registro' in tool_predictiva_mcp.args_schema.get('properties', {}),
        respuesta_tool_mcp.get('estado') == 'OK',
        len(predicciones_tool_mcp) == 4,
        probabilidades_mcp_correctas,
        set(predicciones_tool_mcp.index) == set(targets)
    ]
})

tabla_resumen = resumen_tool_mcp.rename(columns=renombrado_comun)
tabla_comprobaciones = comprobaciones_tool_mcp.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA TOOL MCP')
display(tabla_resumen)

print('\nCOMPROBACIONES DE LA TOOL MCP')
display(tabla_comprobaciones)

if not comprobaciones_tool_mcp['resultado'].all():
    raise ValueError('La tool predictiva MCP no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_tool_mcp, '17_resumen_tool_mcp')
_ = guardar_csv(comprobaciones_tool_mcp, '18_comprobaciones_tool_mcp')

print('\nTool predictiva MCP validada correctamente.')


TOOL PREDICTIVA MEDIANTE MCP


[1/3] Descubriendo tools MCP...


      ['predecir_indicadores'] | 0.47 s


[2/3] Ejecutando la tool predictiva...


      Respuesta recibida en 8.51 s


...


RESUMEN DE LA TOOL MCP


,Elemento,Valor
0,Servidor MCP,predictivo
1,Tools descubiertas,1
2,Tool predictiva,predecir_indicadores
3,Variables de entrada,813
4,Variables objetivo devueltas,4
5,Estado,OK
6,Tiempo de descubrimiento (s),0.473
7,Tiempo de ejecución (s),8.514



COMPROBACIONES DE LA TOOL MCP


,Comprobación,Resultado
0,El archivo del servidor MCP está disponible,True
1,La tool predictiva se descubre dinámicamente,True
2,El esquema MCP contiene el registro de entrada,True
3,La ejecución devuelve estado OK,True
4,Se obtienen las cuatro variables objetivo,True
5,Las probabilidades coinciden con el bridge,True
6,Las salidas mantienen los contratos predictivos,True



Tool predictiva MCP validada correctamente.


### Resultados

El servidor MCP se inicia correctamente mediante transporte `stdio` y permite descubrir dinámicamente la tool `predecir_indicadores`.

La herramienta recibe las 813 variables originales y devuelve mediante MCP las cuatro salidas de la solución predictiva. La respuesta se recibe como una lista de contenidos MCP de tipo `text`, cuyo JSON se recupera y valida posteriormente mediante los mismos contratos Pydantic utilizados por la tool local.

Las siete comprobaciones realizadas resultan correctas. Se valida el descubrimiento dinámico, el esquema de entrada, el estado de ejecución, las cuatro variables objetivo, los contratos predictivos y la coincidencia de las probabilidades con el bridge.

Los tiempos de descubrimiento y ejecución se registran en la tabla anterior y pueden variar entre ejecuciones debido principalmente al inicio de los procesos necesarios. MCP incorpora correctamente una capa estandarizada de descubrimiento y ejecución sobre la misma capacidad predictiva, sin modificar sus resultados.

## 4.3. Comparación de los mecanismos

Una vez ejecutadas ambas alternativas se realiza una comparación funcional y arquitectónica de los mecanismos de acceso a la capacidad predictiva.

En primer lugar se comprueba la equivalencia de las cuatro respuestas generadas mediante tool local y MCP. La validación incluye los diez campos definidos por el contrato `ResultadoPredictivo`, de manera que ninguna decisión arquitectónica depende de diferencias en las predicciones.

La comparación técnica incorpora los criterios establecidos en la planificación: complejidad, trazabilidad, interoperabilidad, descubrimiento, integración con LangGraph y mantenibilidad. Se añaden además el transporte, la necesidad de servidor y sesión, la reutilización por varios clientes y la responsabilidad sobre el mantenimiento de la capacidad.

Este último aspecto resulta especialmente relevante en este proyecto. MCP proporciona ventajas claras cuando un servidor externo expone y mantiene capacidades reutilizables. En el prototipo actual, sin embargo, tanto la tool local como el servidor MCP y el bridge predictivo pertenecen al propio proyecto. MCP aporta por tanto estandarización e interoperabilidad, pero no elimina la responsabilidad de mantener la capacidad predictiva.

Los tiempos observados se muestran únicamente como información descriptiva, ya que pueden variar con el inicio del servicio predictivo y no constituyen por sí solos un criterio de selección.

In [20]:
# =============================================================================
# COMPARACIÓN DE LOS MECANISMOS
# =============================================================================
print('\nCOMPARACIÓN DE LOS MECANISMOS')

resultados_local = {
    resultado['variable_objetivo']: resultado for resultado in respuesta_tool_local['resultados']
}
resultados_mcp = {
    resultado['variable_objetivo']: resultado for resultado in respuesta_tool_mcp['resultados']
}
campos_salida_tools = list(ResultadoPredictivo.model_fields)

# -----------------------------------------------------------------------------
# COMPARACIÓN DE PREDICCIONES
# -----------------------------------------------------------------------------
filas_comparacion_tools = []

for target in targets:
    local, mcp = resultados_local[target], resultados_mcp[target]
    diferencia = abs(local['probabilidad'] - mcp['probabilidad'])
    campos_coincidentes = all(local[campo] == mcp[campo] for campo in campos_salida_tools)

    filas_comparacion_tools.append({
        'target': target, 'descripcion': titulos_targets[target],
        'probabilidad_local': local['probabilidad'], 'probabilidad_mcp': mcp['probabilidad'],
        'diferencia_probabilidad': diferencia, 'umbral_local': local['umbral'],
        'umbral_mcp': mcp['umbral'], 'clasificacion_local': local['clasificacion'],
        'clasificacion_mcp': mcp['clasificacion'], 'campos_coincidentes': campos_coincidentes,
        'resultado': (
            diferencia <= tolerancia_probabilidad_bridge
            and np.isclose(local['umbral'], mcp['umbral'])
            and local['clasificacion'] == mcp['clasificacion']
            and local['familia'] == mcp['familia']
            and local['modelo'] == mcp['modelo']
            and campos_coincidentes
        )
    })

comparacion_predicciones_tools = pd.DataFrame(filas_comparacion_tools)

tabla_predicciones = comparacion_predicciones_tools.rename(columns={
    **renombrado_comun,
    'probabilidad_local': 'Probabilidad tool local', 'probabilidad_mcp': 'Probabilidad MCP',
    'diferencia_probabilidad': 'Diferencia', 'umbral_local': 'Umbral tool local',
    'umbral_mcp': 'Umbral MCP', 'clasificacion_local': 'Clasificación tool local',
    'clasificacion_mcp': 'Clasificación MCP', 'campos_coincidentes': 'Campos coincidentes'
}).round({
    'Probabilidad tool local': 6, 'Probabilidad MCP': 6, 'Diferencia': 8,
    'Umbral tool local': 3, 'Umbral MCP': 3
})

print('\nCOMPARACIÓN DE LAS PREDICCIONES')
display(tabla_predicciones)

# -----------------------------------------------------------------------------
# COMPARACIÓN DE LOS CAMPOS DE SALIDA
# -----------------------------------------------------------------------------
comparacion_campos_tools = pd.DataFrame([
    {
        'campo': campo,
        'coincidencias': sum(
            resultados_local[target][campo] == resultados_mcp[target][campo] for target in targets
        ),
        'total': len(targets)
    }
    for campo in campos_salida_tools
])
comparacion_campos_tools['resultado'] = comparacion_campos_tools[
    'coincidencias'
].eq(comparacion_campos_tools['total'])

tabla_campos = comparacion_campos_tools.rename(columns={
    'campo': 'Campo', 'coincidencias': 'Coincidencias',
    'total': 'Resultados comparados', 'resultado': 'Resultado'
})

print('\nCOMPARACIÓN DE LOS CAMPOS DE SALIDA')
display(tabla_campos)

# -----------------------------------------------------------------------------
# SALIDA REAL PARA LOS CUATRO OBJETIVOS
# -----------------------------------------------------------------------------
respuesta_real_tools = pd.DataFrame(respuesta_tool_local['resultados']).copy()
respuesta_real_tools['diferencia_umbral'] = (
    respuesta_real_tools['probabilidad'] - respuesta_real_tools['umbral']
)
respuesta_real_tools['decision_umbral'] = np.where(
    respuesta_real_tools['clasificacion'].eq(1), 'Supera el umbral', 'No supera el umbral'
)
respuesta_real_tools['numero_advertencias'] = respuesta_real_tools['advertencias'].apply(len)
respuesta_real_tools['coincide_mcp'] = [
    resultados_local[target] == resultados_mcp[target]
    for target in respuesta_real_tools['variable_objetivo']
]

tabla_respuesta_real = respuesta_real_tools[[
    'variable_objetivo', 'descripcion', 'probabilidad', 'umbral', 'diferencia_umbral',
    'clasificacion', 'decision_umbral', 'familia', 'modelo', 'variables_principales',
    'diferencia_reconstruccion', 'numero_advertencias', 'coincide_mcp'
]].rename(columns={
    **renombrado_comun,
    'diferencia_umbral': 'Diferencia respecto al umbral',
    'decision_umbral': 'Decisión según umbral',
    'numero_advertencias': 'Número de advertencias',
    'coincide_mcp': 'Coincide con MCP'
})

tabla_respuesta_real = tabla_respuesta_real.round({
    'Probabilidad': 6, 'Umbral': 3, 'Diferencia respecto al umbral': 6
})

print(f'\nSALIDA REAL PARA LOS CUATRO OBJETIVOS — QUESTID2 {questid2_referencia}')
display(tabla_respuesta_real)

vista_previa_resultados = pd.DataFrame({
    'variable_objetivo': [
        resultado['variable_objetivo'] for resultado in respuesta_tool_local['resultados']
    ],
    'salida_textual': [
        resumir_resultado_predictivo(resultado) for resultado in respuesta_tool_local['resultados']
    ]
})

tabla_vista_previa = vista_previa_resultados.rename(columns={
    **renombrado_comun, 'salida_textual': 'Vista previa textual'
})

print('\nVISTA PREVIA TEXTUAL DE LOS CUATRO OBJETIVOS')
mostrar_tabla_completa(tabla_vista_previa)

advertencias_comunes = all(
    resultados_local[target]['advertencias'] == resultados_local[targets[0]]['advertencias']
    for target in targets
)

if advertencias_comunes:
    print('\nADVERTENCIAS DEVUELTAS POR LA TOOL')
    for numero, advertencia in enumerate(resultados_local[targets[0]]['advertencias'], start=1):
        print(f'{numero}. {advertencia}')

# -----------------------------------------------------------------------------
# COMPARACIÓN TÉCNICA
# -----------------------------------------------------------------------------
comparacion_mecanismos_tools = pd.DataFrame({
    'criterio': [
        'Resultado predictivo',
        'Tiempo observado',
        'Descubrimiento',
        'Ejecución',
        'Servidor y sesión',
        'Transporte',
        'Esquema de argumentos',
        'Interoperabilidad',
        'Reutilización por varios clientes',
        'Mantenimiento de la capacidad',
        'Trazabilidad',
        'Integración con LangGraph',
        'Adecuación al caso actual'
    ],
    'tool_local': [
        'Equivalente',
        f'{tiempo_tool_local:.3f} s',
        'Directo',
        'Síncrona',
        'No necesarios',
        'Llamada interna',
        'Tool LangChain / contrato local',
        'Limitada al proyecto',
        'Menor',
        'Proyecto propio',
        'Directa',
        'Directa',
        'Alta'
    ],
    'mcp': [
        'Equivalente',
        f'{tiempo_tool_mcp:.3f} s',
        'Dinámico',
        'Asíncrona',
        'Cliente, servidor y sesión',
        'stdio',
        'Schema descubierto mediante MCP',
        'Protocolo estándar',
        'Alta',
        'Servidor MCP del propio proyecto',
        'Buena, con una capa adicional',
        'Mediante adaptador LangChain',
        'Alta si existen varios clientes'
    ],
    'lectura': [
        'Empate',
        'Información descriptiva',
        'Ventaja MCP',
        'Depende del escenario',
        'Ventaja tool local',
        'Ventaja tool local en simplicidad',
        'Ventaja MCP',
        'Ventaja MCP',
        'Ventaja MCP',
        'Sin ventaja externa en este TFM',
        'Ventaja tool local',
        'Ambas válidas',
        'Ventaja tool local'
    ]
})

tabla_mecanismos = comparacion_mecanismos_tools.rename(columns=renombrado_comun)

print('\nCOMPARACIÓN ARQUITECTÓNICA')
mostrar_tabla_completa(tabla_mecanismos)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_comparacion_tools = pd.DataFrame({
    'comprobacion': [
        'Ambos mecanismos devuelven las cuatro variables objetivo',
        'La tool local conserva exactamente los 10 campos de salida',
        'La tool MCP conserva exactamente los 10 campos de salida',
        'Los 10 campos coinciden para las cuatro variables objetivo',
        'Las probabilidades coinciden entre ambos mecanismos',
        'Los umbrales coinciden entre ambos mecanismos',
        'Las clasificaciones coinciden entre ambos mecanismos',
        'Las familias y modelos coinciden',
        'Las advertencias coinciden entre ambos mecanismos',
        'La tool local dispone de interfaz LangChain',
        'La tool MCP dispone de interfaz LangChain',
        'MCP proporciona descubrimiento dinámico',
        'La comparación incluye interoperabilidad, mantenibilidad y trazabilidad'
    ],
    'resultado': [
        len(resultados_local) == len(resultados_mcp) == 4,
        all(set(resultado) == set(campos_salida_tools) for resultado in resultados_local.values()),
        all(set(resultado) == set(campos_salida_tools) for resultado in resultados_mcp.values()),
        comparacion_campos_tools['resultado'].all(),
        comparacion_predicciones_tools['diferencia_probabilidad']
        .le(tolerancia_probabilidad_bridge).all(),
        np.isclose(comparacion_predicciones_tools['umbral_local'],
                   comparacion_predicciones_tools['umbral_mcp']).all(),
        comparacion_predicciones_tools['clasificacion_local']
        .eq(comparacion_predicciones_tools['clasificacion_mcp']).all(),
        comparacion_predicciones_tools['resultado'].all(),
        all(resultados_local[target]['advertencias'] == resultados_mcp[target]['advertencias']
            for target in targets),
        hasattr(tool_predictiva_local, 'invoke'),
        hasattr(tool_predictiva_mcp, 'ainvoke'),
        'predecir_indicadores' in nombres_tools_mcp,
        {'Interoperabilidad', 'Mantenimiento de la capacidad', 'Trazabilidad'}
        .issubset(set(comparacion_mecanismos_tools['criterio']))
    ]
})

tabla_comprobaciones = comprobaciones_comparacion_tools.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LA COMPARACIÓN')
display(tabla_comprobaciones)

if not comprobaciones_comparacion_tools['resultado'].all():
    raise ValueError('La comparación de los mecanismos de tools no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(comparacion_predicciones_tools, '19_comparacion_predicciones_tools')
_ = guardar_csv(comparacion_campos_tools, '20_comparacion_campos_tools')
_ = guardar_csv(comparacion_mecanismos_tools, '21_comparacion_mecanismos_tools')
_ = guardar_csv(comprobaciones_comparacion_tools, '22_comprobaciones_comparacion_tools')

print('\nTool local y MCP comparados correctamente.')


COMPARACIÓN DE LOS MECANISMOS

COMPARACIÓN DE LAS PREDICCIONES


,Variable objetivo,Descripción,Probabilidad tool local,Probabilidad MCP,Diferencia,Umbral tool local,Umbral MCP,Clasificación tool local,Clasificación MCP,Campos coincidentes,Resultado
0,IRAMDEYR,Episodio depresivo mayor,0.826969,0.826969,0.0,0.500,0.500,1,1,True,True
1,IRSUICTHNK,Ideación suicida,0.565728,0.565728,0.0,0.509,0.509,1,1,True,True
2,IRSUIPLANYR,Planificación suicida,0.525724,0.525724,0.0,0.536,0.536,0,0,True,True
3,IRSUITRYYR,Intento suicida,0.524104,0.524104,0.0,0.502,0.502,1,1,True,True



COMPARACIÓN DE LOS CAMPOS DE SALIDA


,Campo,Coincidencias,Resultados comparados,Resultado
0,variable_objetivo,4,4,True
1,descripcion,4,4,True
2,probabilidad,4,4,True
3,umbral,4,4,True
4,clasificacion,4,4,True
5,familia,4,4,True
6,modelo,4,4,True
7,variables_principales,4,4,True
8,diferencia_reconstruccion,4,4,True
9,advertencias,4,4,True



SALIDA REAL PARA LOS CUATRO OBJETIVOS — QUESTID2 10004548


...


ADVERTENCIAS DEVUELTAS POR LA TOOL
1. La salida apoya priorización preventiva y requiere interpretación humana.
2. La calibración descriptiva impide interpretar la probabilidad como riesgo clínico directo.
3. Planificación e intento presentan una prevalencia reducida y una Precision limitada.
4. La prioridad de Recall incrementa los falsos positivos, especialmente en intento suicida.
5. Las contribuciones SHAP describen asociaciones aprendidas por el modelo y no causalidad.
6. La muestra 50/50 utilizada para SHAP no reproduce la prevalencia natural de TEST.
7. Trece de 100 explicaciones superan la tolerancia técnica de reconstrucción establecida.
8. Algunas variables originales y recodificadas representan contenidos estrechamente relacionados.
9. Los códigos de salto o respuesta deben conservar exactamente el tratamiento definido en el proyecto.
10. La solución híbrida requiere validación posterior sobre datos independientes o nuevas ediciones.

COMPARACIÓN ARQUITECTÓNICA


,Criterio,Tool local,MCP,Lectura
0,Resultado predictivo,Equivalente,Equivalente,Empate
1,Tiempo observado,7.833 s,8.514 s,Información descriptiva
2,Descubrimiento,Directo,Dinámico,Ventaja MCP
3,Ejecución,Síncrona,Asíncrona,Depende del escenario
4,Servidor y sesión,No necesarios,"Cliente, servidor y sesión",Ventaja tool local
5,Transporte,Llamada interna,stdio,Ventaja tool local en simplicidad
6,Esquema de argumentos,Tool LangChain / contrato local,Schema descubierto mediante MCP,Ventaja MCP
7,Interoperabilidad,Limitada al proyecto,Protocolo estándar,Ventaja MCP
8,Reutilización por varios clientes,Menor,Alta,Ventaja MCP
9,Mantenimiento de la capacidad,Proyecto propio,Servidor MCP del propio proyecto,Sin ventaja externa en este TFM



COMPROBACIONES DE LA COMPARACIÓN


,Comprobación,Resultado
0,Ambos mecanismos devuelven las cuatro variable...,True
1,La tool local conserva exactamente los 10 camp...,True
2,La tool MCP conserva exactamente los 10 campos...,True
3,Los 10 campos coinciden para las cuatro variab...,True
4,Las probabilidades coinciden entre ambos mecan...,True
5,Los umbrales coinciden entre ambos mecanismos,True
6,Las clasificaciones coinciden entre ambos meca...,True
7,Las familias y modelos coinciden,True
8,Las advertencias coinciden entre ambos mecanismos,True
9,La tool local dispone de interfaz LangChain,True



Tool local y MCP comparados correctamente.


### Resultados

La tool local y MCP reproducen exactamente la misma capacidad predictiva sobre el registro de referencia. Para las cuatro variables objetivo coinciden las probabilidades, los umbrales, las clasificaciones, las familias y los modelos, sin diferencias numéricas entre ambos mecanismos.

La equivalencia se mantiene también en los 10 campos del contrato de salida. Por tanto, MCP no introduce una lógica predictiva alternativa, sino una capa estandarizada de acceso a la misma capacidad alojada en `TFM_ML`.

Los tiempos observados son del mismo orden, varios segundos por ejecución, y están condicionados principalmente por el inicio del subproceso predictivo. La diferencia temporal entre ambos mecanismos es pequeña y se considera únicamente descriptiva, no un criterio suficiente para seleccionar la arquitectura.

La comparación arquitectónica muestra diferencias más relevantes en otros aspectos. La tool local proporciona ejecución directa, menor número de capas, trazabilidad más sencilla y no requiere servidor, sesión ni transporte adicional. MCP aporta descubrimiento dinámico, publicación del esquema de argumentos, protocolo estándar y mayor capacidad de reutilización por distintos clientes.

En este prototipo ambos mecanismos son mantenidos por el propio proyecto. Por ello no se materializa una de las ventajas más relevantes de MCP en escenarios empresariales: que un proveedor externo publique, describa y mantenga sus propias capacidades.

Las trece comprobaciones de la comparación resultan correctas y confirman tanto la equivalencia predictiva como la disponibilidad de interfaces compatibles con LangChain para los dos mecanismos.

## 4.4. Selección del mecanismo principal

Tras comprobar que ambos mecanismos reproducen exactamente la misma capacidad predictiva, se selecciona la tool local como mecanismo principal para la integración posterior con LangGraph.

La decisión no responde a diferencias en probabilidades, umbrales, clasificaciones, familias o modelos. Los tiempos tampoco se utilizan como criterio determinante, ya que dependen de las condiciones de ejecución.

La tool local resulta más adecuada para el escenario actual porque la capacidad predictiva pertenece al propio proyecto y es consumida por una única aplicación controlada. Permite reducir capas de comunicación, evitar un servidor y una sesión adicionales y mantener una trazabilidad directa entre LangGraph y el bridge predictivo.

MCP aporta descubrimiento dinámico, autodescripción del esquema e interoperabilidad mediante un protocolo estándar. Estas ventajas adquirirían mayor importancia si la capacidad predictiva tuviera que ser consumida por varios clientes, agentes o aplicaciones independientes.

En este prototipo el servidor MCP también es desarrollado y mantenido por el propio proyecto, por lo que no se materializa la ventaja asociada al consumo de una tool mantenida por un proveedor externo.

Se mantiene por tanto la **tool local como mecanismo principal** y MCP como **alternativa interoperable validada y preparada para una posible evolución multicliente**.

In [21]:
# =============================================================================
# SELECCIÓN DEL MECANISMO PRINCIPAL
# =============================================================================
print('\nSELECCIÓN DEL MECANISMO PRINCIPAL')

mecanismo_principal = 'local'
mecanismo_alternativo = 'mcp'

decision_mecanismo_tools = pd.DataFrame({
    'elemento': [
        'Mecanismo principal',
        'Mecanismo alternativo validado',
        'Coincidencia predictiva',
        'Escenario actual',
        'Mantenimiento de la capacidad',
        'Integración posterior',
        'Motivo principal',
        'Evolución prevista'
    ],
    'valor': [
        nombres_mecanismos_tools[mecanismo_principal],
        nombres_mecanismos_tools[mecanismo_alternativo],
        'Completa',
        'Una aplicación interna controlada',
        'Ambos mecanismos son mantenidos por el proyecto',
        'LangGraph mediante tool local',
        'Menor complejidad y trazabilidad más directa',
        'MCP si aparecen varios clientes o consumidores externos'
    ]
})

tabla_decision = decision_mecanismo_tools.rename(columns=renombrado_comun)

print('\nDECISIÓN SOBRE EL MECANISMO DE TOOLS')
display(tabla_decision)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_decision_tools = pd.DataFrame({
    'comprobacion': [
        'La comparación previa está completamente validada',
        'Ambos mecanismos producen las mismas predicciones',
        'La tool local dispone de interfaz LangChain',
        'MCP queda validado como mecanismo interoperable',
        'El mecanismo principal pertenece a los mecanismos definidos'
    ],
    'resultado': [
        comprobaciones_comparacion_tools['resultado'].all(),
        comparacion_predicciones_tools['resultado'].all(),
        hasattr(tool_predictiva_local, 'invoke'),
        hasattr(tool_predictiva_mcp, 'ainvoke'),
        mecanismo_principal in nombres_mecanismos_tools
    ]
})
tabla_comprobaciones = comprobaciones_decision_tools.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LA DECISIÓN')
display(tabla_comprobaciones)

if not comprobaciones_decision_tools['resultado'].all():
    raise ValueError('La selección del mecanismo de tools no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(decision_mecanismo_tools, '23_decision_mecanismo_tools')
_ = guardar_csv(comprobaciones_decision_tools, '24_comprobaciones_decision_tools')

print('\nTool local seleccionada como mecanismo principal.')
print('MCP permanece validado como mecanismo alternativo interoperable.')


SELECCIÓN DEL MECANISMO PRINCIPAL

DECISIÓN SOBRE EL MECANISMO DE TOOLS


,Elemento,Valor
0,Mecanismo principal,Tool local
1,Mecanismo alternativo validado,MCP
2,Coincidencia predictiva,Completa
3,Escenario actual,Una aplicación interna controlada
4,Mantenimiento de la capacidad,Ambos mecanismos son mantenidos por el proyecto
5,Integración posterior,LangGraph mediante tool local
6,Motivo principal,Menor complejidad y trazabilidad más directa
7,Evolución prevista,MCP si aparecen varios clientes o consumidores...



COMPROBACIONES DE LA DECISIÓN


,Comprobación,Resultado
0,La comparación previa está completamente validada,True
1,Ambos mecanismos producen las mismas predicciones,True
2,La tool local dispone de interfaz LangChain,True
3,MCP queda validado como mecanismo interoperable,True
4,El mecanismo principal pertenece a los mecanis...,True



Tool local seleccionada como mecanismo principal.
MCP permanece validado como mecanismo alternativo interoperable.


### Resultados

Se selecciona la **tool local** como mecanismo principal de acceso a la capacidad predictiva y se mantiene **MCP como alternativa interoperable validada**.

La decisión no responde a diferencias en las predicciones, que son completamente equivalentes, sino a la adecuación arquitectónica al caso de uso actual. El prototipo dispone de una única aplicación interna controlada, por lo que la tool local reduce capas intermedias, simplifica la trazabilidad y se integra directamente con LangGraph.

MCP queda validado funcionalmente mediante arquitectura cliente-servidor, transporte `stdio`, ejecución asíncrona, descubrimiento dinámico de la tool y recuperación de su esquema. Estas capacidades resultarían especialmente útiles si la solución evolucionara hacia varios clientes, consumidores externos o servicios mantenidos independientemente.

En el escenario actual, el servidor MCP también pertenece al propio proyecto y no aporta una separación real de mantenimiento respecto a la tool local. Por este motivo se evita convertir MCP en una dependencia obligatoria de la arquitectura sin una necesidad funcional que lo justifique.

Las cinco comprobaciones finales de la decisión resultan correctas.

## Síntesis de la sección 4 (memoria)

La capacidad predictiva se ha expuesto mediante dos mecanismos distintos sobre una misma implementación: una tool local de LangChain y una tool publicada mediante MCP. Ambos reciben las 813 variables originales y reproducen exactamente las cuatro salidas de la solución predictiva cerrada, manteniendo probabilidades, umbrales, clasificaciones, familias, modelos y el resto de campos del contrato sin diferencias.

MCP se ha validado mediante una arquitectura cliente-servidor real con transporte `stdio`, sesión asíncrona, descubrimiento dinámico de tools y recuperación de su esquema. De este modo se demuestra su capacidad de interoperabilidad sin trasladar artificialmente a MCP otros elementos del proyecto, como el RAG o los prompts, que permanecen bajo control local.

La comparación arquitectónica muestra que MCP aporta principalmente estandarización, descubrimiento y mayor potencial de reutilización por varios clientes. La tool local ofrece una integración más directa, menor número de capas y una trazabilidad más sencilla. Los tiempos observados son similares y se consideran únicamente descriptivos.

Para el prototipo actual se selecciona la **tool local como mecanismo principal**, ya que existe una única aplicación controlada y tanto la tool como el servidor MCP serían mantenidos por el propio proyecto. **MCP permanece validado como alternativa interoperable**, especialmente adecuada si la solución evolucionara hacia varios consumidores o capacidades mantenidas externamente.

**Tabla candidata:** comparación arquitectónica entre tool local y MCP.

**Figura candidata:** no; la diferencia entre mecanismos se representa con mayor claridad mediante la tabla comparativa y el esquema global de arquitectura.

**Destino:** MEMORIA + ANEXO. La memoria recogerá la decisión arquitectónica y sus motivos; la implementación MCP, sus contratos y comprobaciones detalladas se reservarán para el anexo.

# 5. Base documental y RAG controlado

La capa RAG se incorpora para resolver una necesidad diferente de la inferencia predictiva. Las probabilidades, umbrales, clasificaciones, modelos y explicaciones SHAP proceden exclusivamente del servicio validado en `TFM_ML` y no deben reconstruirse mediante recuperación documental.

El RAG se utiliza para recuperar información que no debe depender del conocimiento interno del modelo generativo: definiciones de variables, documentación oficial NSDUH, limitaciones del sistema y procedencia de la información utilizada durante la interpretación.

Se construye deliberadamente un corpus pequeño y controlado. Las variables incluidas se limitan a las cuatro variables objetivo y a aquellas que aparecen entre las variables SHAP globalmente relevantes transferidas desde el Notebook 04. Los valores de importancia SHAP no se incorporan al índice; únicamente se utilizan para decidir qué variables necesitan documentación.

La documentación oficial se obtiene del codebook NSDUH 2024. Para cada variable se selecciona el fragmento más representativo de su definición, conservando como metadatos la variable, la página, el tipo de documento y la fuente. Las limitaciones cerradas del Notebook 04 se incorporan como documentos independientes.

## 5.1. Justificación y preparación del corpus RAG

Antes de generar embeddings se comprueba qué información necesita recuperación documental y se construye el corpus trazable que utilizará el índice vectorial.

Los documentos ya se generan como fragmentos breves y específicos, por lo que no se aplica posteriormente una división adicional de texto. Esto permite mantener una correspondencia directa entre cada fragmento, su variable o limitación y su fuente original.

La misma lectura del codebook conserva también su contenido textual para resolver posteriormente, de forma determinista, el significado de los valores observados en las variables explicativas. Esta utilización es independiente del índice RAG: el codebook completo no se envía a Mistral ni se incorpora íntegramente al índice vectorial.

In [22]:
# =============================================================================
# JUSTIFICACIÓN Y PREPARACIÓN DEL CORPUS RAG
# =============================================================================
print('\nJUSTIFICACIÓN Y PREPARACIÓN DEL CORPUS RAG', flush=True)

# -----------------------------------------------------------------------------
# NECESIDAD DEL RAG
# -----------------------------------------------------------------------------
necesidad_rag = pd.DataFrame({
    'informacion': [
        'Probabilidad, umbral y clasificación',
        'Explicación SHAP local',
        'Definición y codificación de variables',
        'Limitaciones del sistema',
        'Procedencia documental'
    ],
    'origen': [
        'Servicio predictivo',
        'Servicio predictivo',
        'RAG',
        'RAG',
        'RAG'
    ],
    'rag_necesario': [False, False, True, True, True]
})

tabla_necesidad = necesidad_rag.rename(columns={
    'informacion': 'Información', 'origen': 'Origen', 'rag_necesario': 'RAG necesario'
})

print('\nNECESIDAD DE RECUPERACIÓN DOCUMENTAL')
display(tabla_necesidad)

# -----------------------------------------------------------------------------
# DOCUMENTACIÓN OFICIAL
# -----------------------------------------------------------------------------
nombre_codebook = 'nsduh-2024-ds0001-info-codebook_v1.pdf'

if ruta_codebook_nsduh.is_file():
    lector_codebook = PdfReader(ruta_codebook_nsduh)
    origen_codebook = str(ruta_codebook_nsduh.relative_to(ruta_proyecto))
elif ruta_zip_documentacion_nsduh.is_file():
    with ZipFile(ruta_zip_documentacion_nsduh) as archivo_zip:
        lector_codebook = PdfReader(BytesIO(archivo_zip.read(nombre_codebook)))
    origen_codebook = (
        f'{ruta_zip_documentacion_nsduh.relative_to(ruta_proyecto)}::{nombre_codebook}'
    )
else:
    raise FileNotFoundError('No se encuentra el codebook oficial NSDUH 2024.')

variables_documentadas = list(dict.fromkeys(
    targets + principales_variables_shap.sort_values(['target', 'posicion'])['variable'].tolist()
))

etiquetas_variables = (
    esquema_entrada_productivizacion.drop_duplicates('variable')
    .set_index('variable')['etiqueta_variable'].to_dict()
)
etiquetas_variables.update(titulos_targets)

patrones_variables = {
    variable: re.compile(rf'(?<![A-Z0-9_]){re.escape(variable)}(?![A-Z0-9_])', re.IGNORECASE)
    for variable in variables_documentadas
}

print(f'\n[1/3] Analizando {len(lector_codebook.pages)} páginas del codebook...', flush=True)

mejores_fragmentos, texto_codebook_completo = localizar_fragmentos_codebook(
    lector_codebook, patrones_variables
)

variables_no_localizadas = [
    variable for variable in variables_documentadas if variable not in mejores_fragmentos
]

print(f'[2/3] Variables seleccionadas: {len(variables_documentadas)}', flush=True)
print(f'      Variables documentadas: {len(mejores_fragmentos)}', flush=True)

if variables_no_localizadas:
    print(f'      No localizadas: {variables_no_localizadas}', flush=True)

# -----------------------------------------------------------------------------
# CONSTRUCCIÓN DEL CORPUS
# -----------------------------------------------------------------------------
documentos_rag = []

for variable in variables_documentadas:
    if variable not in mejores_fragmentos:
        continue

    info = mejores_fragmentos[variable]
    tipo = 'variable_objetivo' if variable in targets else 'variable_relevante'
    etiqueta = etiquetas_variables.get(variable, variable)

    documentos_rag.append(Document(
        text=(
            f'Variable NSDUH: {variable}. Descripción: {etiqueta}. '
            f'Extracto oficial del codebook NSDUH 2024: {info["fragmento"]}'
        ),
        metadata={
            'fuente': 'NSDUH 2024 Codebook', 'documento': nombre_codebook,
            'tipo': tipo, 'variable': variable, 'pagina': info['pagina']
        }
    ))

for indice, fila in limitaciones_modelo_final.iterrows():
    documentos_rag.append(Document(
        text=f'Limitación del modelo: {fila["limitacion"]}. {fila["detalle"]}',
        metadata={
            'fuente': 'Notebook 04', 'documento': '77_limitaciones_modelo_final.csv',
            'tipo': 'limitacion', 'variable': None, 'pagina': None, 'registro': int(indice) + 1
        }
    ))

print(f'[3/3] Documentos del corpus generados: {len(documentos_rag)}', flush=True)

inventario_corpus_rag = pd.DataFrame([
    {
        'documento': numero, **documento.metadata,
        'caracteres': len(documento.get_content())
    }
    for numero, documento in enumerate(documentos_rag, start=1)
])

resumen_corpus_rag = (
    inventario_corpus_rag.groupby(['fuente', 'tipo'], dropna=False).size()
    .reset_index(name='documentos')
)

tabla_resumen = resumen_corpus_rag.rename(columns={
    **renombrado_comun, 'tipo': 'Tipo', 'documentos': 'Documentos'
})

print('\nRESUMEN DEL CORPUS RAG')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_corpus_rag = pd.DataFrame({
    'comprobacion': [
        'La documentación oficial NSDUH está disponible',
        'Las cuatro variables objetivo forman parte del corpus',
        'Todas las variables seleccionadas se localizan en el codebook',
        'Se incorporan las 10 limitaciones cerradas',
        'Existe un único documento documental por variable seleccionada',
        'Todos los documentos conservan fuente y tipo',
        'Las predicciones individuales de TEST quedan excluidas'
    ],
    'resultado': [
        bool(origen_codebook),
        set(targets).issubset(set(inventario_corpus_rag['variable'].dropna())),
        len(variables_no_localizadas) == 0,
        inventario_corpus_rag['tipo'].eq('limitacion').sum() == 10,
        inventario_corpus_rag['tipo'].isin(['variable_objetivo', 'variable_relevante']).sum()
        == len(variables_documentadas),
        inventario_corpus_rag[['fuente', 'tipo']].notna().all().all(),
        not inventario_corpus_rag['fuente'].astype(str).str.contains('TEST', case=False).any()
    ]
})

tabla_comprobaciones = comprobaciones_corpus_rag.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DEL CORPUS RAG')
display(tabla_comprobaciones)

if not comprobaciones_corpus_rag['resultado'].all():
    raise ValueError('La preparación del corpus RAG no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(necesidad_rag, '25_necesidad_rag')
_ = guardar_csv(inventario_corpus_rag, '26_inventario_corpus_rag')
_ = guardar_csv(resumen_corpus_rag, '27_resumen_corpus_rag')
_ = guardar_csv(comprobaciones_corpus_rag, '28_comprobaciones_corpus_rag')

print('\nCorpus documental RAG preparado correctamente.')


JUSTIFICACIÓN Y PREPARACIÓN DEL CORPUS RAG



NECESIDAD DE RECUPERACIÓN DOCUMENTAL


,Información,Origen,RAG necesario
0,"Probabilidad, umbral y clasificación",Servicio predictivo,False
1,Explicación SHAP local,Servicio predictivo,False
2,Definición y codificación de variables,RAG,True
3,Limitaciones del sistema,RAG,True
4,Procedencia documental,RAG,True



[1/3] Analizando 684 páginas del codebook...


      Páginas revisadas: 100/684


...


RESUMEN DEL CORPUS RAG


,Fuente,Tipo,Documentos
0,NSDUH 2024 Codebook,variable_objetivo,4
1,NSDUH 2024 Codebook,variable_relevante,28
2,Notebook 04,limitacion,10



COMPROBACIONES DEL CORPUS RAG


,Comprobación,Resultado
0,La documentación oficial NSDUH está disponible,True
1,Las cuatro variables objetivo forman parte del...,True
2,Todas las variables seleccionadas se localizan...,True
3,Se incorporan las 10 limitaciones cerradas,True
4,Existe un único documento documental por varia...,True
5,Todos los documentos conservan fuente y tipo,True
6,Las predicciones individuales de TEST quedan e...,True



Corpus documental RAG preparado correctamente.


### Resultados

El codebook oficial NSDUH 2024 se analiza sobre sus 684 páginas y permite localizar correctamente las 32 variables seleccionadas para documentación.

El corpus final contiene 42 documentos controlados: 4 corresponden a las variables objetivo, 28 a variables relevantes seleccionadas a partir de los resultados SHAP globales y 10 a las limitaciones transferidas desde el Notebook 04.

Cada documento conserva su fuente, tipo y metadatos de trazabilidad. Las predicciones individuales de TEST quedan expresamente excluidas del corpus y los valores SHAP no se utilizan como contenido documental.

Las siete comprobaciones realizadas resultan correctas, por lo que el corpus queda preparado para la representación mediante embeddings.

## 5.2. Embeddings e índice vectorial

El corpus controlado se representa mediante embeddings locales utilizando el modelo multilingüe definido en la configuración del entorno. La generación se realiza dentro de `TFM_Agentes` y no requiere acceder a `TFM_ML` ni a servicios externos de embeddings.

LlamaIndex utiliza estos vectores para construir un índice local en memoria. No se introduce una base vectorial adicional, ya que el volumen documental es reducido y el almacenamiento simple resulta suficiente para la finalidad del prototipo.

Se valida la dimensión esperada de los embeddings, la construcción del índice y una primera recuperación semántica sobre una limitación conocida del sistema.

In [23]:
# =============================================================================
# EMBEDDINGS E ÍNDICE VECTORIAL
# =============================================================================
print('\nEMBEDDINGS E ÍNDICE VECTORIAL', flush=True)

print('[1/3] Cargando el modelo de embeddings...', flush=True)
inicio = perf_counter()

Settings.embed_model = HuggingFaceEmbedding(
    model_name=modelo_embeddings, device=dispositivo_embeddings
)

tiempo_carga_embeddings = perf_counter() - inicio

embedding_prueba = Settings.embed_model.get_text_embedding(
    'El sistema tiene finalidad preventiva y no diagnóstica.'
)

print(f'      Modelo cargado en {tiempo_carga_embeddings:.2f} s', flush=True)
print(f'      Dimensión obtenida: {len(embedding_prueba)}', flush=True)

print('[2/3] Construyendo el índice vectorial...', flush=True)
inicio = perf_counter()

indice_rag = VectorStoreIndex.from_documents(documentos_rag, show_progress=True)

tiempo_indice_rag = perf_counter() - inicio

retriever_rag = indice_rag.as_retriever(similarity_top_k=3)

print(f'      Índice construido en {tiempo_indice_rag:.2f} s', flush=True)

print('[3/3] Ejecutando una recuperación de control...', flush=True)
consulta_prueba_rag = '¿Los valores SHAP permiten afirmar causalidad?'

recuperados_prueba_rag = retriever_rag.retrieve(consulta_prueba_rag)

recuperacion_prueba_rag = pd.DataFrame([
    {
        'posicion': numero, 'fuente': nodo.metadata.get('fuente'),
        'tipo': nodo.metadata.get('tipo'), 'variable': nodo.metadata.get('variable'),
        'pagina': nodo.metadata.get('pagina'), 'similitud': float(nodo.score or 0.0),
        'fragmento': nodo.get_content()
    }
    for numero, nodo in enumerate(recuperados_prueba_rag, start=1)
])

tabla_recuperacion = recuperacion_prueba_rag.copy()
tabla_recuperacion['fragmento'] = tabla_recuperacion['fragmento'].str.slice(0, 240) + '...'
tabla_recuperacion = tabla_recuperacion.rename(columns={
    **renombrado_comun, 'tipo': 'Tipo', 'pagina': 'Página'
})

print('\nRECUPERACIÓN SEMÁNTICA DE CONTROL')
display(tabla_recuperacion)

resumen_indice_rag = pd.DataFrame({
    'elemento': [
        'Modelo de embeddings',
        'Dispositivo',
        'Dimensión',
        'Documentos indexados',
        'Documentos recuperados',
        'Tiempo de carga del modelo (s)',
        'Tiempo de indexación (s)'
    ],
    'valor': [
        modelo_embeddings,
        dispositivo_embeddings,
        len(embedding_prueba),
        len(documentos_rag),
        len(recuperados_prueba_rag),
        round(tiempo_carga_embeddings, 3),
        round(tiempo_indice_rag, 3)
    ]
})

tabla_resumen = resumen_indice_rag.rename(columns=renombrado_comun)

print('\nRESUMEN DEL ÍNDICE VECTORIAL')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
causalidad_recuperada = any(
    fila.tipo == 'limitacion' and 'causal' in fila.fragmento.lower()
    for fila in recuperacion_prueba_rag.itertuples()
)

comprobaciones_indice_rag = pd.DataFrame({
    'comprobacion': [
        'El embedding tiene 384 dimensiones',
        'El índice contiene el corpus documental preparado',
        'El retriever devuelve tres documentos',
        'Todos los resultados conservan metadatos de fuente',
        'La consulta sobre causalidad recupera una limitación relacionada'
    ],
    'resultado': [
        len(embedding_prueba) == dimension_embeddings,
        len(documentos_rag) == len(inventario_corpus_rag),
        len(recuperados_prueba_rag) == 3,
        recuperacion_prueba_rag['fuente'].notna().all(),
        causalidad_recuperada
    ]
})

tabla_comprobaciones = comprobaciones_indice_rag.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DEL ÍNDICE VECTORIAL')
display(tabla_comprobaciones)

if not comprobaciones_indice_rag['resultado'].all():
    raise ValueError('El índice vectorial RAG no es correcto.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_indice_rag, '29_resumen_indice_rag')
_ = guardar_csv(recuperacion_prueba_rag, '30_recuperacion_prueba_rag')
_ = guardar_csv(comprobaciones_indice_rag, '31_comprobaciones_indice_rag')

print('\nEmbeddings e índice vectorial validados correctamente.')


EMBEDDINGS E ÍNDICE VECTORIAL


[1/3] Cargando el modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  46%|████▌     | 91/199 [00:00<00:00, 903.93it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1376.25it/s]

...


RESUMEN DEL ÍNDICE VECTORIAL


,Elemento,Valor
0,Modelo de embeddings,sentence-transformers/paraphrase-multilingual-...
1,Dispositivo,cpu
2,Dimensión,384
3,Documentos indexados,42
4,Documentos recuperados,3
5,Tiempo de carga del modelo (s),8.285
6,Tiempo de indexación (s),0.987



COMPROBACIONES DEL ÍNDICE VECTORIAL


,Comprobación,Resultado
0,El embedding tiene 384 dimensiones,True
1,El índice contiene el corpus documental preparado,True
2,El retriever devuelve tres documentos,True
3,Todos los resultados conservan metadatos de fu...,True
4,La consulta sobre causalidad recupera una limi...,True



Embeddings e índice vectorial validados correctamente.


### Resultados

El modelo multilingüe de embeddings se carga correctamente en `TFM_Agentes` utilizando CPU y genera vectores de 384 dimensiones, coincidiendo con la configuración definida para el proyecto. Esta elección prioriza la estabilidad de la capa RAG frente al uso de GPU, cuyo beneficio resulta reducido dado el pequeño volumen del corpus documental.

Los 42 documentos del corpus se incorporan correctamente al índice vectorial de LlamaIndex. La construcción se realiza localmente y no requiere acceder al entorno predictivo ni utilizar servicios externos de embeddings.

La consulta de control sobre causalidad recupera tres limitaciones procedentes del Notebook 04. El primer resultado corresponde a la limitación sobre explicabilidad no causal, confirmando que la búsqueda semántica identifica correctamente el contenido metodológico relacionado.

Las cinco comprobaciones realizadas resultan correctas y el índice queda disponible para la capa de recuperación documental.

## 5.3. Recuperación documental y tool RAG

La recuperación documental combina dos criterios complementarios. Cuando una consulta contiene explícitamente el código de una variable incluida en el corpus, su documentación se recupera prioritariamente mediante los metadatos asociados. Los resultados restantes se completan mediante similitud semántica sobre el índice vectorial.

Esta estrategia evita depender exclusivamente de los embeddings para interpretar identificadores alfanuméricos específicos de NSDUH, manteniendo al mismo tiempo la recuperación semántica para preguntas conceptuales. Cada resultado identifica el método utilizado, la fuente, el tipo de documento, la variable asociada cuando existe y la página correspondiente.

La capacidad se encapsula como una tool local de LangChain denominada `consultar_documentacion`. La tool no recibe registros NSDUH completos ni puede acceder a probabilidades, umbrales, clasificaciones o valores SHAP.

Se prueban tres necesidades diferentes: una limitación metodológica, la definición de una variable objetivo y la definición de una variable seleccionada por su relevancia SHAP.

In [24]:
# =============================================================================
# RECUPERACIÓN DOCUMENTAL Y TOOL RAG
# =============================================================================
print('\nRECUPERACIÓN DOCUMENTAL Y TOOL RAG')


@tool('consultar_documentacion')
def tool_rag(consulta: str) -> list[dict[str, Any]]:
    """
    Recupera definiciones y limitaciones desde la documentación controlada.
    """
    return recuperar_documentacion(consulta, top_k=3)


variable_shap_prueba = (
    principales_variables_shap.sort_values(['target', 'posicion']).iloc[0]['variable']
)

consultas_ejemplo_rag = [
    '¿Los valores SHAP permiten afirmar causalidad?',
    f'¿Cómo define NSDUH la variable {targets[0]}?',
    f'¿Cómo define NSDUH la variable {variable_shap_prueba}?'
]

print(f'\nEjecutando {len(consultas_ejemplo_rag)} consultas documentales...', flush=True)

filas_recuperaciones_rag = []

for numero, consulta in enumerate(consultas_ejemplo_rag, start=1):
    print(f'[{numero}/{len(consultas_ejemplo_rag)}] {consulta}', flush=True)
    recuperados = tool_rag.invoke({
        'consulta': consulta
    })

    for posicion, resultado in enumerate(recuperados, start=1):
        filas_recuperaciones_rag.append({
            'consulta': consulta,
            'posicion': posicion,
            **resultado
        })

recuperaciones_tool_rag = pd.DataFrame(filas_recuperaciones_rag)

tabla_recuperaciones = recuperaciones_tool_rag.copy()
tabla_recuperaciones['fragmento'] = tabla_recuperaciones['fragmento'].str.slice(0, 220) + '...'
tabla_recuperaciones = tabla_recuperaciones.rename(columns={
    **renombrado_comun, 'metodo': 'Método', 'tipo': 'Tipo', 'pagina': 'Página',
    'registro': 'Registro'
})

print('\nEJEMPLOS DE RECUPERACIÓN DOCUMENTAL')
display(tabla_recuperaciones)

ejemplos_salida_rag = (
    recuperaciones_tool_rag.groupby('consulta', sort=False).head(1).iloc[:2].copy()
)

tabla_ejemplos_rag = ejemplos_salida_rag[[
    'consulta', 'metodo', 'fuente', 'tipo', 'variable', 'pagina', 'registro', 'fragmento'
]].rename(columns={
    **renombrado_comun, 'metodo': 'Método', 'tipo': 'Tipo', 'pagina': 'Página',
    'registro': 'Registro'
})

print('\nRESPUESTAS REALES DE LA TOOL RAG')
mostrar_tabla_completa(tabla_ejemplos_rag)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
consulta_causalidad = recuperaciones_tool_rag[
    recuperaciones_tool_rag['consulta'].eq(consultas_ejemplo_rag[0])
]
consulta_target = recuperaciones_tool_rag[
    recuperaciones_tool_rag['consulta'].eq(consultas_ejemplo_rag[1])
]
consulta_variable = recuperaciones_tool_rag[
    recuperaciones_tool_rag['consulta'].eq(consultas_ejemplo_rag[2])
]

comprobaciones_tool_rag = pd.DataFrame({
    'comprobacion': [
        'La tool utiliza el nombre documental definido',
        'Cada consulta devuelve tres documentos',
        'Todos los resultados incluyen fuente y tipo',
        'La consulta SHAP recupera una limitación sobre causalidad',
        'La variable objetivo se recupera en primera posición',
        'La variable SHAP se recupera en primera posición',
        'Las variables explícitas utilizan coincidencia exacta',
        'La recuperación no utiliza predicciones individuales de TEST'
    ],
    'resultado': [
        tool_rag.name == 'consultar_documentacion',
        recuperaciones_tool_rag.groupby('consulta').size().eq(3).all(),
        recuperaciones_tool_rag[['fuente', 'tipo']].notna().all().all(),
        any(fila.tipo == 'limitacion' and 'causal' in fila.fragmento.lower()
            for fila in consulta_causalidad.itertuples()),
        consulta_target.iloc[0]['variable'] == targets[0],
        consulta_variable.iloc[0]['variable'] == variable_shap_prueba,
        consulta_target.iloc[0]['metodo'] == consulta_variable.iloc[0]['metodo'] ==
        'Coincidencia exacta',
        not recuperaciones_tool_rag['fuente'].astype(str).str.contains('TEST', case=False).any()
    ]
})

tabla_comprobaciones = comprobaciones_tool_rag.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LA TOOL RAG')
display(tabla_comprobaciones)

if not comprobaciones_tool_rag['resultado'].all():
    raise ValueError('La recuperación documental mediante la tool RAG no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(recuperaciones_tool_rag, '32_ejemplos_recuperacion_rag')
_ = guardar_csv(comprobaciones_tool_rag, '33_comprobaciones_tool_rag')

print('\nTool RAG validada correctamente.')


RECUPERACIÓN DOCUMENTAL Y TOOL RAG

Ejecutando 3 consultas documentales...


[1/3] ¿Los valores SHAP permiten afirmar causalidad?


[2/3] ¿Cómo define NSDUH la variable IRAMDEYR?


[3/3] ¿Cómo define NSDUH la variable CAMHPROB2?



EJEMPLOS DE RECUPERACIÓN DOCUMENTAL


...


RESPUESTAS REALES DE LA TOOL RAG


,Consulta,Método,Fuente,Tipo,Variable,Página,Registro,Fragmento
0,¿Los valores SHAP permiten afirmar causalidad?,Semántica,Notebook 04,limitacion,NaN,NaN,5.0,Limitación del modelo: Explicabilidad no causal. Las contribuciones SHAP describen asociaciones aprendidas por el modelo y no causalidad.
3,¿Cómo define NSDUH la variable IRAMDEYR?,Coincidencia exacta,NSDUH 2024 Codebook,variable_objetivo,IRAMDEYR,507.0,NaN,Variable NSDUH: IRAMDEYR. Descripción: Episodio depresivo mayor. Extracto oficial del codebook NSDUH 2024: IAMDELT Len : 1 ADULT: LIFETIME MAJOR DEPRESSIVE EPISODE (MDE) - IMP IND Freq Pct 1 = Questionnaire data ........................................................................................................ 45747 78.02 3 = Statistically imputed data .............................................................................................. 1552 2.65 9 = LEGITIMATE SKIP..................................................................................................... 11334 19.33 (AMDEYR) IRAMDEYR Len : 1 ADULT: PAST YEAR MAJOR DEPRESSIVE EPISODE (MDE) - IMP REV Freq Pct . = Aged 12-17 .................................................................................................................... 11334 19.33 0 = No ................................................................................................................................. 42094 71.79 1 = Yes ................................................................................................................................ 5205 8.88 (AMDEYR) IIAMDEYR Len : 1 ADULT: PAST YEAR MAJOR DEPRESSIVE EPISODE (MDE) - IMP IND Freq Pct 1 = Questionnaire data ........................................................................................................ 45659 77.87 3 = Statistically imputed data .............................................................................................. 1640 2.80 9 = LEGITIMATE SKIP..................................................................................................... 11334 19.33 (AMDEIMP) IRAMDEIMP Len : 1 ADULT: MDE WITH SEVERE ROLE IMPAIRMENT - IMP REV Freq Pct . = Aged 12-17 .................................................................................................................... 11334 19.33 0 = No ................................................................................................................................. 43556 74.29 1 = Yes ................................................................................................................................ 3743 6.38 (AMDEIMP) IIAMDEIMP Len : 1 ADULT: MDE WITH SEVERE ROLE IMPAIRMENT - IMP IND Freq Pct 1 = Questionnaire data ........................................................................................................ 45617 77.80 3 = Statistically impu



COMPROBACIONES DE LA TOOL RAG


,Comprobación,Resultado
0,La tool utiliza el nombre documental definido,True
1,Cada consulta devuelve tres documentos,True
2,Todos los resultados incluyen fuente y tipo,True
3,La consulta SHAP recupera una limitación sobre...,True
4,La variable objetivo se recupera en primera po...,True
5,La variable SHAP se recupera en primera posición,True
6,Las variables explícitas utilizan coincidencia...,True
7,La recuperación no utiliza predicciones indivi...,True



Tool RAG validada correctamente.


### Resultados

La capacidad documental queda encapsulada mediante la tool local `consultar_documentacion`, que combina coincidencia exacta por metadatos y recuperación semántica.

Las tres consultas de ejemplo devuelven tres documentos cada una. La pregunta sobre causalidad utiliza recuperación semántica y sitúa en primera posición la limitación sobre explicabilidad no causal del Notebook 04.

Las consultas que contienen códigos explícitos de variables utilizan coincidencia exacta. `IRAMDEYR` se recupera en primera posición desde el codebook oficial, en la página 507, y `CAMHPROB2` se recupera igualmente en primera posición desde la página 573. Los resultados restantes se completan mediante similitud semántica.

Las ocho comprobaciones realizadas resultan correctas. La tool conserva la procedencia documental y no utiliza predicciones individuales de TEST, probabilidades, clasificaciones ni valores SHAP como contenido recuperado.

## 5.4. Evaluación de la recuperación

La recuperación documental se evalúa mediante un conjunto de consultas con documentos esperados conocidos a partir del corpus controlado.

Se incluyen las cuatro variables objetivo, dos variables seleccionadas entre las principales variables SHAP y tres limitaciones metodológicas especialmente relevantes: el carácter no causal de SHAP, el uso preventivo y no diagnóstico del sistema y la imposibilidad de interpretar las probabilidades como riesgo clínico.

Para las variables se comprueba su identificación mediante los metadatos de tipo y código. Para las limitaciones se utiliza el identificador del registro transferido desde el Notebook 04, evitando depender de la aparición literal de determinadas palabras dentro del fragmento recuperado.

Una recuperación se considera correcta cuando el documento esperado aparece entre los tres primeros resultados. Adicionalmente se comprueba que las consultas que contienen códigos de variables priorizan la coincidencia exacta, mientras que las preguntas conceptuales utilizan recuperación semántica.

Esta evaluación valora exclusivamente la recuperación documental. La utilidad del RAG sobre la generación se evaluará posteriormente cuando esté disponible la capa generativa.

In [25]:
# =============================================================================
# EVALUACIÓN DE LA RECUPERACIÓN
# =============================================================================
print('\nEVALUACIÓN DE LA RECUPERACIÓN', flush=True)

variables_shap_evaluacion = (
    principales_variables_shap.sort_values(['target', 'posicion']).drop_duplicates('variable')
    .head(2)['variable'].tolist()
)

pruebas_rag = [
    {
        'consulta': f'¿Cómo define NSDUH la variable {target}?',
        'tipo_esperado': 'variable_objetivo',
        'variable_esperada': target,
        'registro_esperado': None
    }
    for target in targets
]

pruebas_rag += [
    {
        'consulta': f'¿Cómo define NSDUH la variable {variable}?',
        'tipo_esperado': 'variable_relevante',
        'variable_esperada': variable,
        'registro_esperado': None
    }
    for variable in variables_shap_evaluacion
]

pruebas_rag += [
    {
        'consulta': '¿Los valores SHAP permiten afirmar causalidad?',
        'tipo_esperado': 'limitacion',
        'variable_esperada': None,
        'registro_esperado': 5
    },
    {
        'consulta': '¿Puede utilizarse el sistema con finalidad diagnóstica?',
        'tipo_esperado': 'limitacion',
        'variable_esperada': None,
        'registro_esperado': 1
    },
    {
        'consulta': '¿La probabilidad estimada equivale a riesgo clínico?',
        'tipo_esperado': 'limitacion',
        'variable_esperada': None,
        'registro_esperado': 2
    }
]

firma_predicciones_antes_rag = json.dumps(respuesta_tool_local, sort_keys=True, ensure_ascii=False)

filas_evaluacion_rag = []

for numero, prueba in enumerate(pruebas_rag, start=1):
    print(f'[{numero}/{len(pruebas_rag)}] {prueba["consulta"]}', flush=True)
    recuperados = recuperar_documentacion(prueba['consulta'], top_k=3)

    coincidencias = [
        posicion for posicion, documento in enumerate(recuperados, start=1)
        if (
            documento['tipo'] == prueba['tipo_esperado'] and (
                prueba['variable_esperada'] is None
                or documento['variable'] == prueba['variable_esperada']
            ) and (
                prueba['registro_esperado'] is None
                or documento['registro'] == prueba['registro_esperado']
            )
        )
    ]

    filas_evaluacion_rag.append({
        'consulta': prueba['consulta'],
        'tipo_esperado': prueba['tipo_esperado'],
        'variable_esperada': prueba['variable_esperada'],
        'registro_esperado': prueba['registro_esperado'],
        'posicion_recuperacion': coincidencias[0] if coincidencias else np.nan,
        'fuente_primer_resultado': recuperados[0]['fuente'],
        'metodo_primer_resultado': recuperados[0]['metodo'],
        'registro_primer_resultado': recuperados[0]['registro'],
        'similitud_primer_resultado': recuperados[0]['similitud'],
        'documentos_recuperados': len(recuperados),
        'resultado': bool(coincidencias)
    })

evaluacion_recuperacion_rag = pd.DataFrame(filas_evaluacion_rag)

tabla_evaluacion = evaluacion_recuperacion_rag.rename(columns={
    **renombrado_comun, 'tipo_esperado': 'Tipo esperado', 'variable_esperada': 'Variable esperada',
    'registro_esperado': 'Registro esperado', 'posicion_recuperacion': 'Posición recuperación',
    'fuente_primer_resultado': 'Fuente primer resultado',
    'metodo_primer_resultado': 'Método primer resultado',
    'registro_primer_resultado': 'Registro primer resultado',
    'similitud_primer_resultado': 'Similitud primer resultado'
})
tabla_evaluacion['Similitud primer resultado'] = tabla_evaluacion[
    'Similitud primer resultado'
].round(4)

print('\nRESULTADOS DE LA EVALUACIÓN RAG')
display(tabla_evaluacion)

resumen_evaluacion_rag = pd.DataFrame({
    'elemento': [
        'Consultas evaluadas',
        'Consultas correctas',
        'Recuperación correcta top-3 (%)',
        'Variables objetivo evaluadas',
        'Variables SHAP relevantes evaluadas',
        'Limitaciones evaluadas'
    ],
    'valor': [
        len(evaluacion_recuperacion_rag),
        int(evaluacion_recuperacion_rag['resultado'].sum()),
        round(100 * evaluacion_recuperacion_rag['resultado'].mean(), 1),
        len(targets),
        len(variables_shap_evaluacion),
        3
    ]
})

tabla_resumen = resumen_evaluacion_rag.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA EVALUACIÓN RAG')
display(tabla_resumen)

firma_predicciones_despues_rag = json.dumps(
    respuesta_tool_local, sort_keys=True, ensure_ascii=False
)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
consultas_variables = evaluacion_recuperacion_rag['variable_esperada'].notna()
consultas_limitaciones = evaluacion_recuperacion_rag['tipo_esperado'].eq('limitacion')

comprobaciones_evaluacion_rag = pd.DataFrame({
    'comprobacion': [
        'Se evalúan las cuatro variables objetivo',
        'Se evalúan al menos dos variables relevantes seleccionadas por SHAP',
        'Se evalúan tres limitaciones metodológicas',
        'Cada consulta recupera tres documentos',
        'Todos los documentos esperados aparecen en el top-3',
        'Las consultas con variables priorizan coincidencia exacta',
        'Las consultas conceptuales utilizan recuperación semántica',
        'Las predicciones permanecen inalteradas durante la recuperación'
    ],
    'resultado': [
        evaluacion_recuperacion_rag['variable_esperada'].isin(targets).sum() == len(targets),
        len(variables_shap_evaluacion) >= 2,
        consultas_limitaciones.sum() == 3,
        evaluacion_recuperacion_rag['documentos_recuperados'].eq(3).all(),
        evaluacion_recuperacion_rag['resultado'].all(),
        evaluacion_recuperacion_rag.loc[
            consultas_variables, 'metodo_primer_resultado'].eq('Coincidencia exacta').all(),
        evaluacion_recuperacion_rag.loc[
            consultas_limitaciones, 'metodo_primer_resultado'].eq('Semántica').all(),
        firma_predicciones_antes_rag == firma_predicciones_despues_rag
    ]
})

tabla_comprobaciones = comprobaciones_evaluacion_rag.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LA EVALUACIÓN RAG')
display(tabla_comprobaciones)

if not comprobaciones_evaluacion_rag['resultado'].all():
    raise ValueError('La evaluación de la recuperación RAG no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(evaluacion_recuperacion_rag, '34_evaluacion_recuperacion_rag')
_ = guardar_csv(resumen_evaluacion_rag, '35_resumen_evaluacion_rag')
_ = guardar_csv(comprobaciones_evaluacion_rag, '36_comprobaciones_evaluacion_rag')

print('\nRecuperación documental RAG evaluada correctamente.')


EVALUACIÓN DE LA RECUPERACIÓN


[1/9] ¿Cómo define NSDUH la variable IRAMDEYR?


[2/9] ¿Cómo define NSDUH la variable IRSUICTHNK?


[3/9] ¿Cómo define NSDUH la variable IRSUIPLANYR?


[4/9] ¿Cómo define NSDUH la variable IRSUITRYYR?


...


RESUMEN DE LA EVALUACIÓN RAG


,Elemento,Valor
0,Consultas evaluadas,9.0
1,Consultas correctas,9.0
2,Recuperación correcta top-3 (%),100.0
3,Variables objetivo evaluadas,4.0
4,Variables SHAP relevantes evaluadas,2.0
5,Limitaciones evaluadas,3.0



COMPROBACIONES DE LA EVALUACIÓN RAG


,Comprobación,Resultado
0,Se evalúan las cuatro variables objetivo,True
1,Se evalúan al menos dos variables relevantes s...,True
2,Se evalúan tres limitaciones metodológicas,True
3,Cada consulta recupera tres documentos,True
4,Todos los documentos esperados aparecen en el ...,True
5,Las consultas con variables priorizan coincide...,True
6,Las consultas conceptuales utilizan recuperaci...,True
7,Las predicciones permanecen inalteradas durant...,True



Recuperación documental RAG evaluada correctamente.


### Resultados

La recuperación documental se evalúa mediante nueve consultas con documentos esperados conocidos: las cuatro variables objetivo, dos variables relevantes seleccionadas por SHAP y tres limitaciones metodológicas.

Las nueve consultas resultan correctas y todos los documentos esperados aparecen en primera posición, de modo que el documento esperado aparece entre los tres primeros resultados en el `100 %` de las consultas.

Las seis consultas que incluyen códigos de variables priorizan correctamente la coincidencia exacta. Las cuatro variables objetivo y las variables `CAMHPROB2` y `LVLDIFMEM2` recuperan en primer lugar su documentación específica del codebook NSDUH 2024.

Las tres consultas conceptuales utilizan recuperación semántica y recuperan como primer resultado las limitaciones esperadas: explicabilidad no causal, uso preventivo y no diagnóstico y probabilidades no equivalentes a riesgo clínico.

Las ocho comprobaciones finales resultan correctas. La firma de las predicciones permanece inalterada antes y después de la evaluación, confirmando que la capa RAG no modifica la solución predictiva.

## Síntesis de la sección 5 (memoria)

Se ha construido una capa RAG controlada cuya responsabilidad queda separada de la inferencia predictiva. Las probabilidades, umbrales, clasificaciones y explicaciones SHAP continúan procediendo exclusivamente del servicio alojado en `TFM_ML`, mientras el RAG proporciona definiciones, documentación oficial y limitaciones necesarias para contextualizar posteriormente las respuestas generativas.

El corpus documental se construye a partir del codebook oficial NSDUH 2024 y de las limitaciones cerradas del Notebook 04. Incluye 42 documentos: las cuatro variables objetivo, 28 variables relevantes seleccionadas a partir del análisis SHAP global y 10 limitaciones metodológicas. Las predicciones individuales de TEST y los valores SHAP quedan excluidos del índice.

Los documentos se representan mediante embeddings multilingües locales de 384 dimensiones y se incorporan a un índice vectorial de LlamaIndex. La recuperación combina dos mecanismos: coincidencia exacta mediante metadatos cuando la consulta contiene un código de variable y similitud semántica para preguntas conceptuales. Esta combinación evita depender de los embeddings para interpretar identificadores alfanuméricos propios de NSDUH.

La evaluación final incluye nueve consultas y alcanza un `100 %` de acierto considerando los tres primeros resultados de cada consulta. Las seis consultas sobre variables recuperan su documento específico mediante coincidencia exacta y las tres preguntas metodológicas recuperan semánticamente las limitaciones esperadas. Las predicciones permanecen inalteradas durante todo el proceso, confirmando la independencia entre recuperación documental e inferencia.

La capacidad queda finalmente expuesta mediante la tool `consultar_documentacion`, preparada para integrarse junto con `predecir_indicadores` dentro de LangGraph.

**Tabla candidata:** evaluación final de las nueve consultas RAG y ejemplos de recuperación exacta y semántica.

**Figura candidata:** no; el tamaño reducido del corpus y la naturaleza de la evaluación quedan representados de forma más clara mediante tablas y ejemplos documentales.

**Destino:** MEMORIA + ANEXO. La memoria recogerá el diseño del corpus, la recuperación híbrida y el resultado global de evaluación; el inventario documental, las consultas completas y las comprobaciones detalladas se reservarán para el anexo.

# 6. Flujo con LangGraph

Una vez validadas las capacidades predictiva, explicativa y documental se construye el flujo que coordinará su utilización.

LangGraph se utiliza como mecanismo de orquestación, pero el sistema se presenta mediante un flujo único y explícito. No se construye una arquitectura multiagente artificial: cada nodo representa una responsabilidad concreta y reutiliza capacidades ya comprobadas en las secciones anteriores.

Además del grafo, la solución dispone de una infraestructura de ejecución y control del agente, concepto habitualmente denominado *agent harness*. No constituye un modelo adicional ni un nuevo agente, sino el conjunto de mecanismos que rodean al modelo generativo y controlan su interacción con el resto del sistema.

En este proyecto, esta infraestructura está formada por el estado compartido de LangGraph, las tools disponibles, las decisiones de routing, los contratos Pydantic, el acceso controlado al servicio predictivo, la recuperación documental mediante RAG, los guardrails y la trazabilidad de la ejecución.

El *agent harness* no selecciona ni modifica los modelos predictivos. La familia, el modelo y el umbral correspondientes a cada indicador permanecen fijados por la evaluación cerrada del Notebook 04. Su función consiste en controlar qué capacidades se ejecutan, en qué orden y bajo qué validaciones.

Aunque el Máster presenta también patrones como ReAct, en este caso se utiliza deliberadamente un flujo LangGraph controlado. Esta decisión reduce las decisiones libres del modelo generativo en etapas deterministas y favorece la reproducibilidad, la trazabilidad y el control.

## 6.1. Estado y nodos del flujo

El estado compartido almacena únicamente la información necesaria para trasladar una solicitud entre las diferentes etapas.

La entrada contiene el registro original y dos decisiones independientes: solicitar una explicación SHAP y solicitar documentación adicional. Cuando se requiere explicación se especifican también las variables objetivo que deben explicarse, evitando calcular SHAP de forma indiscriminada.

El flujo utiliza cinco nodos:

1. validación del registro;
2. predicción mediante la tool local seleccionada;
3. explicación SHAP opcional mediante el bridge hacia `TFM_ML`;
4. recuperación documental opcional mediante la tool RAG;
5. preparación del contexto para la futura capa generativa.

La predicción, SHAP y RAG conservan así sus responsabilidades independientes y LangGraph únicamente controla el orden y las decisiones entre ellas.

In [26]:
# =============================================================================
# ESTADO Y NODOS DEL FLUJO
# =============================================================================
print('\nESTADO Y NODOS DEL FLUJO')

n_variables_contexto_shap = 3


class EstadoFlujo(TypedDict, total=False):
    registro: dict[str, Any]
    solicitar_explicacion: bool
    objetivos_explicacion: list[str]
    solicitar_documentacion: bool
    predicciones: list[dict[str, Any]]
    explicaciones: list[dict[str, Any]]
    variables_explicacion: list[str]
    consultas_documentales: list[str]
    contexto_documental: list[dict[str, Any]]
    contexto_generacion: dict[str, Any]
    vista_previa_informe: str
    trazabilidad: list[dict[str, Any]]


resumen_nodos_flujo = pd.DataFrame({
    'nodo': [
        'validar_entrada',
        'predecir_indicadores',
        'explicar_resultados',
        'recuperar_documentacion',
        'preparar_contexto'
    ],
    'capacidad': [
        'Validación Pydantic',
        'Tool predictiva local',
        'SHAP mediante bridge',
        'Tool RAG',
        'Consolidación determinista'
    ],
    'finalidad': [
        'Validar las 813 variables antes de inferencia',
        'Obtener las cuatro predicciones cerradas',
        'Explicar únicamente las variables objetivo solicitadas',
        'Recuperar documentación controlada cuando proceda',
        'Preparar la información para la siguiente etapa'
    ]
})

tabla_resumen = resumen_nodos_flujo.rename(columns={
    'nodo': 'Nodo', 'capacidad': 'Capacidad', 'finalidad': 'Finalidad'
})

print('\nNODOS DEL FLUJO')
display(tabla_resumen)

campos_estado_necesarios = {
    'registro', 'solicitar_explicacion', 'objetivos_explicacion', 'solicitar_documentacion',
    'predicciones', 'explicaciones', 'variables_explicacion', 'contexto_documental',
    'contexto_generacion', 'vista_previa_informe', 'trazabilidad'
}

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_nodos_flujo = pd.DataFrame({
    'comprobacion': [
        'El estado contiene los campos necesarios',
        'Los cinco nodos están disponibles',
        'La predicción utiliza la tool local seleccionada',
        'SHAP permanece delegado al servicio predictivo',
        'La recuperación utiliza la tool RAG validada',
        'La generación con Mistral no se incorpora todavía'
    ],
    'resultado': [
        campos_estado_necesarios.issubset(EstadoFlujo.__annotations__),
        all(callable(funcion) for funcion in [
            nodo_validar_entrada, nodo_predecir_indicadores, nodo_explicar_resultados,
            nodo_recuperar_documentacion, nodo_preparar_contexto
        ]),
        mecanismo_principal == 'local' and tool_predictiva_local.name == 'predecir_indicadores',
        ruta_servicio_predictivo.is_file(),
        tool_rag.name == 'consultar_documentacion',
        'informe' not in EstadoFlujo.__annotations__
    ]
})

tabla_comprobaciones = comprobaciones_nodos_flujo.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LOS NODOS')
display(tabla_comprobaciones)

if not comprobaciones_nodos_flujo['resultado'].all():
    raise ValueError('Los nodos del flujo LangGraph no son correctos.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_nodos_flujo, '37_resumen_nodos_flujo')
_ = guardar_csv(comprobaciones_nodos_flujo, '38_comprobaciones_nodos_flujo')

print('\nEstado y nodos del flujo definidos correctamente.')


ESTADO Y NODOS DEL FLUJO

NODOS DEL FLUJO


,Nodo,Capacidad,Finalidad
0,validar_entrada,Validación Pydantic,Validar las 813 variables antes de inferencia
1,predecir_indicadores,Tool predictiva local,Obtener las cuatro predicciones cerradas
2,explicar_resultados,SHAP mediante bridge,Explicar únicamente las variables objetivo sol...
3,recuperar_documentacion,Tool RAG,Recuperar documentación controlada cuando proceda
4,preparar_contexto,Consolidación determinista,Preparar la información para la siguiente etapa



COMPROBACIONES DE LOS NODOS


,Comprobación,Resultado
0,El estado contiene los campos necesarios,True
1,Los cinco nodos están disponibles,True
2,La predicción utiliza la tool local seleccionada,True
3,SHAP permanece delegado al servicio predictivo,True
4,La recuperación utiliza la tool RAG validada,True
5,La generación con Mistral no se incorpora todavía,True



Estado y nodos del flujo definidos correctamente.


### Resultados

El estado compartido de LangGraph queda definido con los campos necesarios para trasladar el registro, las decisiones opcionales, las predicciones, las explicaciones SHAP, la documentación recuperada, el contexto final y la trazabilidad entre nodos.

El flujo dispone de cinco nodos con responsabilidades diferenciadas: validación de la entrada, predicción, explicación SHAP, recuperación documental y preparación del contexto.

La predicción utiliza la tool local seleccionada en la sección anterior, SHAP permanece delegado al servicio predictivo de `TFM_ML` y la documentación se obtiene mediante la tool RAG validada.

Las seis comprobaciones realizadas resultan correctas. La capa Mistral permanece todavía fuera del estado, evitando incorporar generación antes de validar el flujo determinista.

## 6.2. Construcción del grafo y rutas

Los cinco nodos definidos para el flujo se conectan mediante un único `StateGraph`.

Después de validar la entrada y obtener las cuatro predicciones, el grafo aplica una primera decisión determinista. Si se ha solicitado explicación SHAP, el flujo continúa hacia el nodo de explicación. Si no se requiere explicación pero sí documentación, pasa directamente a la recuperación RAG. Cuando no se solicita ninguna capacidad opcional, continúa hacia la preparación del contexto.

Tras la explicación SHAP se aplica una segunda decisión: recuperar documentación o preparar directamente la información para la siguiente etapa.

Esta estructura permite representar las distintas necesidades de ejecución mediante un único grafo, sin crear flujos independientes ni agentes adicionales. LangGraph actúa únicamente como mecanismo de control del estado y de las transiciones entre capacidades previamente validadas.

<div style="max-width:1000px;margin:20px auto;font-family:Arial,sans-serif;text-align:center;">

<div style="display:flex;align-items:center;justify-content:center;gap:8px;flex-wrap:wrap;">

<div style="background:#176b70;color:white;padding:12px 16px;border-radius:9px;">
<strong>Entrada</strong><br><small>813 variables</small>
</div>

<div style="font-size:24px;color:#2b9f99;">→</div>

<div style="background:#dcf7f4;padding:12px 16px;border-radius:9px;">
<strong>Validar entrada</strong>
</div>

<div style="font-size:24px;color:#2b9f99;">→</div>

<div style="background:#dcf7f4;padding:12px 16px;border-radius:9px;">
<strong>Predecir indicadores</strong>
</div>

</div>

<div style="margin:14px 0;font-size:16px;color:#2b9f99;">↓ routing determinista</div>

<div style="display:flex;gap:16px;justify-content:center;align-items:stretch;">

<div style="flex:1;background:#f4fcfb;border:2px solid #81d8d0;padding:15px;border-radius:11px;">
<strong>Con SHAP</strong><br><br>
Explicar resultados<br>
↓<br>
Recuperar documentación
</div>

<div style="flex:1;background:#f4fcfb;border:2px solid #81d8d0;padding:15px;border-radius:11px;">
<strong>Solo documentación</strong><br><br>
Recuperar documentación
</div>

<div style="flex:1;background:#f7f7f7;border:1px dashed #52656a;padding:15px;border-radius:11px;">
<strong>Sin capacidades opcionales</strong><br><br>
Continuar directamente
</div>

</div>

<div style="margin:16px 0;font-size:24px;color:#2b9f99;">↓</div>

<div style="background:#0e4f55;color:white;padding:13px 18px;border-radius:10px;">
<strong>Preparar contexto</strong><br>
<small>Salida común del flujo LangGraph</small>
</div>

</div>

In [27]:
# =============================================================================
# CONSTRUCCIÓN DEL GRAFO Y RUTAS
# =============================================================================
print('\nCONSTRUCCIÓN DEL GRAFO Y RUTAS')

grafo_flujo = StateGraph(EstadoFlujo)

# -----------------------------------------------------------------------------
# NODOS
# -----------------------------------------------------------------------------
grafo_flujo.add_node('validar_entrada', nodo_validar_entrada)
grafo_flujo.add_node('predecir_indicadores', nodo_predecir_indicadores)
grafo_flujo.add_node('explicar_resultados', nodo_explicar_resultados)
grafo_flujo.add_node('recuperar_documentacion', nodo_recuperar_documentacion)
grafo_flujo.add_node('preparar_contexto', nodo_preparar_contexto)

# -----------------------------------------------------------------------------
# TRANSICIONES
# -----------------------------------------------------------------------------
grafo_flujo.add_edge(START, 'validar_entrada')
grafo_flujo.add_edge('validar_entrada', 'predecir_indicadores')

grafo_flujo.add_conditional_edges(
    'predecir_indicadores', decidir_despues_prediccion,
    {
        'explicar': 'explicar_resultados',
        'documentar': 'recuperar_documentacion',
        'preparar': 'preparar_contexto'
    }
)

grafo_flujo.add_conditional_edges(
    'explicar_resultados', decidir_despues_explicacion,
    {
        'documentar': 'recuperar_documentacion',
        'preparar': 'preparar_contexto'
    }
)

grafo_flujo.add_edge('recuperar_documentacion', 'preparar_contexto')
grafo_flujo.add_edge('preparar_contexto', END)

aplicacion_flujo = grafo_flujo.compile()

# -----------------------------------------------------------------------------
# RESUMEN DE TRANSICIONES
# -----------------------------------------------------------------------------
transiciones_flujo = pd.DataFrame({
    'origen': [
        'START',
        'validar_entrada',
        'predecir_indicadores',
        'predecir_indicadores',
        'predecir_indicadores',
        'explicar_resultados',
        'explicar_resultados',
        'recuperar_documentacion',
        'preparar_contexto'
    ],
    'destino': [
        'validar_entrada',
        'predecir_indicadores',
        'explicar_resultados',
        'recuperar_documentacion',
        'preparar_contexto',
        'recuperar_documentacion',
        'preparar_contexto',
        'preparar_contexto',
        'END'
    ],
    'condicion': [
        'Inicio',
        'Entrada válida',
        'Explicación solicitada',
        'Sin explicación y con documentación',
        'Sin explicación ni documentación',
        'Documentación solicitada',
        'Sin documentación',
        'Documentación recuperada',
        'Contexto preparado'
    ]
})

tabla_transiciones = transiciones_flujo.rename(columns={
    'origen': 'Origen', 'destino': 'Destino', 'condicion': 'Condición'
})

print('\nTRANSICIONES DEL GRAFO')
display(tabla_transiciones)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_grafo_flujo = pd.DataFrame({
    'comprobacion': [
        'El grafo se compila correctamente',
        'La explicación solicitada dirige a SHAP',
        'La documentación sin explicación dirige a RAG',
        'La ausencia de capacidades opcionales dirige a preparación',
        'Tras SHAP puede continuarse hacia RAG',
        'Tras SHAP puede omitirse RAG',
        'El grafo mantiene una única arquitectura'
    ],
    'resultado': [
        hasattr(aplicacion_flujo, 'invoke'),
        decidir_despues_prediccion(
            {'solicitar_explicacion': True, 'objetivos_explicacion': [targets[0]],
             'solicitar_documentacion': True}) == 'explicar',
        decidir_despues_prediccion({
            'solicitar_explicacion': False, 'objetivos_explicacion': [],
            'solicitar_documentacion': True}) == 'documentar',
        decidir_despues_prediccion({
            'solicitar_explicacion': False, 'objetivos_explicacion': [],
            'solicitar_documentacion': False}) == 'preparar',
        decidir_despues_explicacion({'solicitar_documentacion': True}) == 'documentar',
        decidir_despues_explicacion({'solicitar_documentacion': False}) == 'preparar',
        len(transiciones_flujo) == 9
    ]
})

tabla_comprobaciones = comprobaciones_grafo_flujo.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DEL GRAFO')
display(tabla_comprobaciones)

if not comprobaciones_grafo_flujo['resultado'].all():
    raise ValueError('La construcción del flujo LangGraph no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(transiciones_flujo, '39_transiciones_flujo_langgraph')
_ = guardar_csv(comprobaciones_grafo_flujo, '40_comprobaciones_grafo_langgraph')

print('\nGrafo LangGraph construido y compilado correctamente.')


CONSTRUCCIÓN DEL GRAFO Y RUTAS



TRANSICIONES DEL GRAFO


,Origen,Destino,Condición
0,START,validar_entrada,Inicio
1,validar_entrada,predecir_indicadores,Entrada válida
2,predecir_indicadores,explicar_resultados,Explicación solicitada
3,predecir_indicadores,recuperar_documentacion,Sin explicación y con documentación
4,predecir_indicadores,preparar_contexto,Sin explicación ni documentación
5,explicar_resultados,recuperar_documentacion,Documentación solicitada
6,explicar_resultados,preparar_contexto,Sin documentación
7,recuperar_documentacion,preparar_contexto,Documentación recuperada
8,preparar_contexto,END,Contexto preparado



COMPROBACIONES DEL GRAFO


,Comprobación,Resultado
0,El grafo se compila correctamente,True
1,La explicación solicitada dirige a SHAP,True
2,La documentación sin explicación dirige a RAG,True
3,La ausencia de capacidades opcionales dirige a...,True
4,Tras SHAP puede continuarse hacia RAG,True
5,Tras SHAP puede omitirse RAG,True
6,El grafo mantiene una única arquitectura,True



Grafo LangGraph construido y compilado correctamente.


### Resultados

El `StateGraph` se construye y compila correctamente a partir de los cinco nodos definidos.

Se establecen nueve transiciones que permiten representar mediante una única arquitectura la ejecución completa y las variantes que omiten SHAP, RAG o ambas capacidades opcionales.

Después de la predicción el flujo puede dirigirse hacia la explicación SHAP, la recuperación documental o directamente hacia la preparación del contexto. Tras SHAP se mantiene una segunda decisión que permite continuar hacia RAG o finalizar la preparación sin documentación adicional.

Las siete comprobaciones realizadas resultan correctas, confirmando el funcionamiento de las rutas condicionales y la utilización de un único grafo para todas las combinaciones.

## 6.3. Ejecución real y trazabilidad del flujo

El grafo se ejecuta con el mismo registro TEST utilizado como referencia en las secciones anteriores.

Para demostrar las capacidades opcionales sin repetir cálculos innecesarios, se solicita una explicación SHAP sobre la variable objetivo perteneciente a Machine Learning y se activa la recuperación documental. La explicación Deep Learning ya fue validada previamente y no necesita recalcularse para comprobar el funcionamiento de LangGraph.

Esta ejecución parte nuevamente de las 813 variables originales y recorre de forma efectiva la validación, la predicción, SHAP, RAG y la preparación del contexto.

Cada nodo registra una descripción de su entrada, su salida y su tiempo de ejecución. Se muestran además las predicciones, las variables SHAP principales, la documentación que llegará a la siguiente etapa y una vista textual completa de la información preparada.

La vista textual todavía no constituye un informe generado. Es una construcción determinista que permite observar de forma directa qué información recibirá posteriormente Mistral.

In [28]:
# =============================================================================
# EJECUCIÓN REAL Y TRAZABILIDAD DEL FLUJO
# =============================================================================
print('\nEJECUCIÓN REAL DEL FLUJO LANGGRAPH', flush=True)

estado_inicial_flujo = {
    'registro': entrada_referencia.registro,
    'solicitar_explicacion': True,
    'objetivos_explicacion': [target_ml_explicacion],
    'solicitar_documentacion': True,
    'trazabilidad': []
}

inicio = perf_counter()
estado_final_flujo = aplicacion_flujo.invoke(estado_inicial_flujo)
tiempo_flujo = perf_counter() - inicio

print(f'\nEjecución completa: {tiempo_flujo:.2f} s', flush=True)

# -----------------------------------------------------------------------------
# TRAZABILIDAD
# -----------------------------------------------------------------------------
traza_flujo = pd.DataFrame(estado_final_flujo['trazabilidad'])

tabla_traza = traza_flujo.rename(columns={
    'paso': 'Paso', 'nodo': 'Nodo', 'entrada': 'Entrada', 'salida': 'Salida',
    'tiempo_s': 'Tiempo (s)'
})

print('\nTRAZA REAL DEL FLUJO')
mostrar_tabla_completa(tabla_traza)

# -----------------------------------------------------------------------------
# PREDICCIONES
# -----------------------------------------------------------------------------
predicciones_flujo = pd.DataFrame(estado_final_flujo['predicciones'])

tabla_predicciones = predicciones_flujo[[
    'variable_objetivo', 'descripcion', 'probabilidad', 'umbral', 'clasificacion',
    'familia', 'modelo'
]].rename(columns=renombrado_comun)

tabla_predicciones = tabla_predicciones.round({'Probabilidad': 6, 'Umbral': 3})

print('\nPREDICCIONES RECIBIDAS POR EL FLUJO')
display(tabla_predicciones)

# -----------------------------------------------------------------------------
# EXPLICACIÓN SHAP
# -----------------------------------------------------------------------------
explicacion_flujo = estado_final_flujo['explicaciones'][0]

variables_explicacion_flujo = pd.DataFrame(explicacion_flujo['variables_principales'])
variables_explicacion_flujo.insert(0, 'variable_objetivo', explicacion_flujo['variable_objetivo'])

if 'descripcion' not in variables_explicacion_flujo:
    variables_explicacion_flujo['descripcion'] = (
        variables_explicacion_flujo['etiqueta_variable']
        if 'etiqueta_variable' in variables_explicacion_flujo
        else variables_explicacion_flujo['variable']
    )

tabla_explicacion = variables_explicacion_flujo.head(5).copy()
tabla_explicacion = tabla_explicacion.rename(columns={
    **renombrado_comun, 'valor_observado': 'Valor observado', 'valor_shap': 'Valor SHAP',
    'signo_contribucion': 'Signo de la contribución'
})
tabla_explicacion['Valor SHAP'] = tabla_explicacion['Valor SHAP'].round(6)

print('\nPRINCIPALES VARIABLES SHAP RECIBIDAS POR EL FLUJO')
display(tabla_explicacion)

# -----------------------------------------------------------------------------
# CONTEXTO DOCUMENTAL
# -----------------------------------------------------------------------------
contexto_flujo = pd.DataFrame(estado_final_flujo['contexto_documental'])

tabla_contexto = contexto_flujo[[
    'consulta', 'metodo', 'fuente', 'tipo', 'variable', 'pagina', 'registro'
]].rename(columns={
    **renombrado_comun, 'metodo': 'Método', 'tipo': 'Tipo', 'pagina': 'Página',
    'registro': 'Registro'
})

print('\nDOCUMENTACIÓN ENTREGADA A LA SIGUIENTE ETAPA')
display(tabla_contexto)

# -----------------------------------------------------------------------------
# VISTA PREVIA COMPLETA
# -----------------------------------------------------------------------------
vista_previa_flujo = pd.DataFrame({'salida': [estado_final_flujo['vista_previa_informe']]})

tabla_vista_previa = vista_previa_flujo.rename(columns={
    'salida': 'Vista previa estructurada — todavía no generativa'
})

print('\nVISTA PREVIA DEL CONTEXTO QUE RECIBIRÁ MISTRAL')
mostrar_tabla_completa(tabla_vista_previa)

# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
resumen_ejecucion_flujo = pd.DataFrame({
    'elemento': [
        'Variables de entrada', 'Predicciones obtenidas', 'Explicaciones SHAP',
        'Variables SHAP incorporadas al contexto', 'Consultas documentales',
        'Documentos principales', 'Nodos recorridos', 'Tiempo total (s)'
    ],
    'valor': [
        len(estado_final_flujo['registro']),
        len(estado_final_flujo['predicciones']),
        len(estado_final_flujo['explicaciones']),
        len(estado_final_flujo['variables_explicacion']),
        len(estado_final_flujo['consultas_documentales']),
        len(estado_final_flujo['contexto_documental']),
        len(estado_final_flujo['trazabilidad']),
        round(tiempo_flujo, 3)
    ]
})

tabla_resumen = resumen_ejecucion_flujo.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA EJECUCIÓN')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
predicciones_grafo = predicciones_flujo.set_index('variable_objetivo')

probabilidades_grafo_correctas = all(
    np.isclose(
        predicciones_grafo.loc[target, 'probabilidad'],
        predicciones_tool_local.loc[target, 'probabilidad'],
        atol=tolerancia_probabilidad_bridge, rtol=0
    )
    for target in targets
)

targets_contexto = set(
    contexto_flujo.loc[contexto_flujo['tipo'].eq('variable_objetivo'), 'variable'].dropna()
)

registros_limitaciones_contexto = set(
    contexto_flujo.loc[contexto_flujo['tipo'].eq('limitacion'), 'registro'].dropna().astype(int)
)

orden_nodos_esperado = [
    'validar_entrada', 'predecir_indicadores', 'explicar_resultados', 'recuperar_documentacion',
    'preparar_contexto'
]

comprobaciones_ejecucion_flujo = pd.DataFrame({
    'comprobacion': [
        'El flujo recibe las 813 variables originales',
        'El flujo devuelve las cuatro predicciones',
        'Las probabilidades coinciden con la tool local',
        'Se obtiene la explicación SHAP solicitada',
        'La reconstrucción SHAP permanece dentro de tolerancia',
        'Se seleccionan tres variables SHAP para contexto documental',
        'El contexto contiene las cuatro variables objetivo',
        'El contexto contiene las tres limitaciones esperadas',
        'Cada consulta aporta un documento principal',
        'Los cinco nodos se recorren en el orden previsto',
        'La vista previa contiene los cuatro objetivos',
        'La vista previa incorpora la explicación SHAP'
    ],
    'resultado': [
        len(estado_final_flujo['registro']) == 813,
        len(predicciones_flujo) == 4,
        probabilidades_grafo_correctas,
        explicacion_flujo['variable_objetivo'] == target_ml_explicacion,
        explicacion_flujo['diferencia_reconstruccion'] <= tolerancia_reconstruccion_shap,
        len(estado_final_flujo['variables_explicacion']) == n_variables_contexto_shap,
        targets_contexto == set(targets),
        {1, 2, 5}.issubset(registros_limitaciones_contexto),
        len(contexto_flujo) == len(estado_final_flujo['consultas_documentales']),
        traza_flujo['nodo'].tolist() == orden_nodos_esperado,
        all(titulos_targets[target] in estado_final_flujo['vista_previa_informe']
            for target in targets),
        explicacion_flujo['descripcion'] in estado_final_flujo['vista_previa_informe']
    ]
})

tabla_comprobaciones = comprobaciones_ejecucion_flujo.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LA EJECUCIÓN')
display(tabla_comprobaciones)

if not comprobaciones_ejecucion_flujo['resultado'].all():
    raise ValueError('La ejecución real del flujo LangGraph no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(traza_flujo, '41_traza_flujo_langgraph')
_ = guardar_csv(variables_explicacion_flujo, '42_variables_explicacion_flujo_langgraph')
_ = guardar_csv(contexto_flujo, '43_contexto_documental_flujo_langgraph')
_ = guardar_csv(resumen_ejecucion_flujo, '44_resumen_ejecucion_flujo_langgraph')
_ = guardar_csv(comprobaciones_ejecucion_flujo, '45_comprobaciones_ejecucion_flujo_langgraph')

print('\nEjecución real del flujo LangGraph validada correctamente.')


EJECUCIÓN REAL DEL FLUJO LANGGRAPH


[1/5] Validando el registro de entrada...


[2/5] Ejecutando la tool predictiva local...


[3/5] Generando 1 explicaciones SHAP...


      IRAMDEYR — Episodio depresivo mayor


...


RESUMEN DE LA EJECUCIÓN


,Elemento,Valor
0,Variables de entrada,813.000
1,Predicciones obtenidas,4.000
2,Explicaciones SHAP,1.000
3,Variables SHAP incorporadas al contexto,3.000
4,Consultas documentales,10.000
5,Documentos principales,10.000
6,Nodos recorridos,5.000
7,Tiempo total (s),23.744



COMPROBACIONES DE LA EJECUCIÓN


,Comprobación,Resultado
0,El flujo recibe las 813 variables originales,True
1,El flujo devuelve las cuatro predicciones,True
2,Las probabilidades coinciden con la tool local,True
3,Se obtiene la explicación SHAP solicitada,True
4,La reconstrucción SHAP permanece dentro de tol...,True
5,Se seleccionan tres variables SHAP para contex...,True
6,El contexto contiene las cuatro variables obje...,True
7,El contexto contiene las tres limitaciones esp...,True
8,Cada consulta aporta un documento principal,True
9,Los cinco nodos se recorren en el orden previsto,True



Ejecución real del flujo LangGraph validada correctamente.


### Resultados

La ejecución real de LangGraph recorre correctamente los cinco nodos sobre el registro TEST de referencia con sus 813 variables originales.

El flujo obtiene las cuatro predicciones cerradas mediante la tool local y genera bajo demanda una explicación SHAP para episodio depresivo mayor. Las tres variables utilizadas posteriormente como contexto documental son `CAMHPROB2`, `RCVYMHPRB` y `LVLDIFMEM2`, que constituyen las principales contribuciones locales de esta explicación.

A partir de las cuatro variables objetivo, las tres variables SHAP seleccionadas y las tres limitaciones metodológicas se realizan 10 consultas documentales. Cada consulta aporta un documento principal y conserva su método de recuperación y procedencia.

La traza muestra de forma explícita el paso desde la entrada validada hasta cuatro predicciones, una explicación SHAP, 10 documentos recuperados y un contexto estructurado preparado para la generación. Los tiempos registrados confirman que el coste del flujo se concentra fundamentalmente en la predicción y, especialmente, en la explicación SHAP.

La vista previa textual reúne las cuatro decisiones predictivas, las principales contribuciones SHAP y la procedencia documental en una única salida legible. Esta representación sigue siendo determinista y no constituye todavía un informe generado.

Las doce comprobaciones realizadas resultan correctas. Las probabilidades coinciden con la tool local, la reconstrucción SHAP se mantiene dentro de tolerancia, las fuentes y limitaciones esperadas están presentes y los cinco nodos se recorren en el orden establecido.

## 6.4. Validación de las rutas del flujo

La ejecución anterior demuestra la ruta más completa del grafo: predicción, explicación SHAP, recuperación documental y preparación del contexto.

Para cerrar la sección se comprueban también las restantes combinaciones de decisiones sin repetir innecesariamente la inferencia predictiva. Se valida que el flujo pueda utilizar RAG sin SHAP, omitir RAG después de una explicación y continuar directamente cuando no se solicita ninguna capacidad opcional.

Los errores de entrada no se delegan a un agente ni a un modelo generativo. Las validaciones deterministas detienen el flujo antes de ejecutar la predicción, manteniendo la misma protección establecida en las secciones anteriores.

La comparación de informes generados con y sin SHAP o RAG se realizará posteriormente, una vez incorporada la capa Mistral.

In [29]:
# =============================================================================
# VALIDACIÓN DE LAS RUTAS DEL FLUJO
# =============================================================================
print('\nVALIDACIÓN DE LAS RUTAS DEL FLUJO')

casos_rutas_flujo = [
    {
        'caso': 'SHAP + RAG',
        'solicitar_explicacion': True,
        'objetivos_explicacion': [target_ml_explicacion],
        'solicitar_documentacion': True
    },
    {
        'caso': 'Solo RAG',
        'solicitar_explicacion': False,
        'objetivos_explicacion': [],
        'solicitar_documentacion': True
    },
    {
        'caso': 'Solo SHAP',
        'solicitar_explicacion': True,
        'objetivos_explicacion': [target_ml_explicacion],
        'solicitar_documentacion': False
    },
    {
        'caso': 'Solo predicción',
        'solicitar_explicacion': False,
        'objetivos_explicacion': [],
        'solicitar_documentacion': False
    }
]

rutas_flujo = pd.DataFrame([
    {
        **caso,
        'ruta_tras_prediccion': decidir_despues_prediccion(caso),
        'ruta_tras_explicacion': (
            decidir_despues_explicacion(caso) if caso['solicitar_explicacion'] else 'No procede'
        )
    }
    for caso in casos_rutas_flujo
])

tabla_rutas = rutas_flujo.rename(columns={
    'caso': 'Caso',
    'solicitar_explicacion': 'Solicitar explicación',
    'objetivos_explicacion': 'Variables objetivo para explicación',
    'solicitar_documentacion': 'Solicitar documentación',
    'ruta_tras_prediccion': 'Ruta tras predicción',
    'ruta_tras_explicacion': 'Ruta tras explicación'
})

print('\nRUTAS DISPONIBLES')
display(tabla_rutas)

# -----------------------------------------------------------------------------
# ENTRADA NO VÁLIDA
# -----------------------------------------------------------------------------
registro_invalido_flujo = dict(entrada_referencia.registro)
registro_invalido_flujo.pop(vars_predictoras_modelado_dl[0])

try:
    nodo_validar_entrada({
        'registro': registro_invalido_flujo,
        'solicitar_explicacion': False,
        'trazabilidad': []
    })
    entrada_invalida_controlada = False

except (ValidationError, ValueError):
    entrada_invalida_controlada = True

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
campos_contexto_necesarios = {
    'predicciones', 'explicaciones', 'documentacion', 'advertencias', 'vista_previa'
}

comprobaciones_rutas_flujo = pd.DataFrame({
    'comprobacion': [
        'SHAP y RAG dirigen primero a la explicación',
        'La ruta con solo RAG dirige directamente a documentación',
        'La ruta con solo SHAP omite RAG después de explicar',
        'La ruta con solo predicción continúa directamente a preparación',
        'Una entrada incompleta se detiene antes de inferencia',
        'El contexto final contiene los campos necesarios',
        'La trazabilidad conserva los cinco nodos de la ejecución completa',
        'La capa generativa todavía no forma parte del estado'
    ],
    'resultado': [
        rutas_flujo.loc[rutas_flujo['caso'].eq('SHAP + RAG'), 'ruta_tras_prediccion'].iloc[0]
        == 'explicar',
        rutas_flujo.loc[rutas_flujo['caso'].eq('Solo RAG'), 'ruta_tras_prediccion'].iloc[0]
        == 'documentar',
        rutas_flujo.loc[rutas_flujo['caso'].eq('Solo SHAP'), 'ruta_tras_explicacion'].iloc[0]
        == 'preparar',
        rutas_flujo.loc[rutas_flujo['caso'].eq('Solo predicción'), 'ruta_tras_prediccion'].iloc[0]
        == 'preparar',
        entrada_invalida_controlada,
        campos_contexto_necesarios.issubset(estado_final_flujo['contexto_generacion']),
        len(traza_flujo) == 5,
        'informe' not in estado_final_flujo
    ]
})

tabla_comprobaciones = comprobaciones_rutas_flujo.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LAS RUTAS')
display(tabla_comprobaciones)

if not comprobaciones_rutas_flujo['resultado'].all():
    raise ValueError('La validación de las rutas LangGraph no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(rutas_flujo, '46_rutas_flujo_langgraph')
_ = guardar_csv(comprobaciones_rutas_flujo, '47_comprobaciones_rutas_flujo_langgraph')

print('\nRutas del flujo LangGraph validadas correctamente.')


VALIDACIÓN DE LAS RUTAS DEL FLUJO

RUTAS DISPONIBLES


,Caso,Solicitar explicación,Variables objetivo para explicación,Solicitar documentación,Ruta tras predicción,Ruta tras explicación
0,SHAP + RAG,True,[IRAMDEYR],True,explicar,documentar
1,Solo RAG,False,[],True,documentar,No procede
2,Solo SHAP,True,[IRAMDEYR],False,explicar,preparar
3,Solo predicción,False,[],False,preparar,No procede


[1/5] Validando el registro de entrada...



COMPROBACIONES DE LAS RUTAS


,Comprobación,Resultado
0,SHAP y RAG dirigen primero a la explicación,True
1,La ruta con solo RAG dirige directamente a doc...,True
2,La ruta con solo SHAP omite RAG después de exp...,True
3,La ruta con solo predicción continúa directame...,True
4,Una entrada incompleta se detiene antes de inf...,True
5,El contexto final contiene los campos necesarios,True
6,La trazabilidad conserva los cinco nodos de la...,True
7,La capa generativa todavía no forma parte del ...,True



Rutas del flujo LangGraph validadas correctamente.


### Resultados

Las cuatro combinaciones previstas de ejecución quedan correctamente representadas: predicción con SHAP y RAG, predicción con solo RAG, predicción con solo SHAP y predicción sin capacidades opcionales.

La ruta completa dirige inicialmente hacia SHAP y posteriormente hacia RAG. Cuando únicamente se solicita documentación se accede directamente a la recuperación documental, mientras que la ruta con solo SHAP omite RAG una vez obtenida la explicación. La predicción sin capacidades adicionales continúa directamente hacia la preparación del contexto.

También se comprueba que un registro incompleto se detiene durante la validación de entrada antes de alcanzar la inferencia predictiva.

Las ocho comprobaciones realizadas resultan correctas. El contexto final contiene los elementos previstos, la traza completa conserva los cinco nodos y la capa generativa continúa separada del flujo validado.

## Síntesis de la sección 6 (memoria)

LangGraph se ha incorporado como mecanismo de control de un flujo único que coordina capacidades previamente validadas sin modificar su lógica interna. El estado compartido permite trasladar el registro, las decisiones opcionales, las predicciones, las explicaciones, la documentación y la trazabilidad entre cinco nodos diferenciados.

El grafo contiene nueve transiciones y dos decisiones condicionales. Esta estructura permite utilizar SHAP y RAG conjuntamente, emplear únicamente una de estas capacidades o continuar directamente después de la predicción, evitando construir agentes o grafos independientes para cada combinación.

La ejecución real parte de las 813 variables originales y recorre validación, predicción, explicación SHAP, recuperación documental y preparación del contexto. Se conservan las cuatro predicciones cerradas, se obtiene una explicación local para episodio depresivo mayor y se incorporan las tres variables SHAP principales a la recuperación documental. El contexto final contiene 10 documentos: cuatro correspondientes a las variables objetivo, tres a las variables explicativas seleccionadas y tres a limitaciones metodológicas.

La trazabilidad permite seguir de forma explícita qué recibe y qué devuelve cada nodo. La vista previa final demuestra además cómo las salidas estructuradas pueden convertirse en información legible antes de llegar al modelo generativo, manteniendo todavía separada la generación con Mistral.

Las rutas alternativas y el rechazo de una entrada incompleta se validan de forma determinista. LangGraph queda así preparado para proporcionar a la siguiente capa únicamente información predictiva, explicativa y documental previamente controlada.

**Tabla candidata:** traza real de los cinco nodos y resumen de las cuatro rutas posibles del flujo.

**Figura candidata:** esquema general del flujo desde la entrada NSDUH hasta la generación del informe, destacando la separación entre `TFM_Agentes` y `TFM_ML`.

**Destino:** MEMORIA + ANEXO. La memoria incluirá la arquitectura del flujo, sus decisiones condicionales y una traza resumida; las transiciones completas y comprobaciones individuales se reservarán para el anexo.

# 7. Generación estructurada con Mistral y guardrails

Una vez validado el flujo determinista se incorpora la capa de generación de lenguaje natural.

Mistral no recibe el registro NSDUH completo ni accede directamente a `TFM_ML`. Su entrada procede exclusivamente del contexto preparado por LangGraph y contiene las decisiones predictivas, las principales variables de la explicación solicitada, la documentación recuperada y las limitaciones de interpretación.

Los valores numéricos de probabilidades, umbrales y contribuciones SHAP se excluyen deliberadamente del contexto generativo. Estos valores permanecen bajo control de la aplicación y podrán mostrarse posteriormente mediante bloques deterministas. De este modo el modelo generativo no puede recalcularlos, redondearlos de forma diferente ni sustituirlos.

La respuesta generativa se solicita mediante el contrato Pydantic `SalidaGenerativa`. A partir de este contenido, la aplicación construye de forma determinista el contrato final `InformePreventivo`, conservando bajo control de la aplicación los factores SHAP, las orientaciones, las limitaciones, las fuentes y la advertencia de uso. Tras la generación se aplican guardrails deterministas independientes del modelo para comprobar objetivos, factores utilizados, fuentes, advertencias, ausencia de causalidad o diagnóstico y respeto de las restricciones establecidas.

La salida generativa se utiliza exclusivamente para organizar e interpretar información ya validada y no constituye una valoración clínica.

## 7.1. Modelo y contexto generativo controlado

La generación utiliza Mistral mediante la integración de LangChain y una salida estructurada contra el contrato `SalidaGenerativa`. Este contrato limita la generación a resumen, interpretación y traducciones de los factores; posteriormente, la aplicación reconstruye de forma determinista el contrato final `InformePreventivo`.

Antes de realizar una llamada externa se prepara un contexto reducido a la información que el modelo necesita para redactar. Las probabilidades, umbrales y valores SHAP se eliminan de esta entrada, conservando únicamente la decisión respecto al umbral, la procedencia del modelo, las principales variables explicativas, su signo de contribución y los fragmentos documentales recuperados.

El prompt establece explícitamente las restricciones de interpretación: no realizar diagnósticos, no atribuir causalidad, no introducir cifras predictivas, utilizar únicamente los factores y fuentes disponibles y mantener una orientación preventiva no prescriptiva.

Las cuatro variables objetivo deben aparecer mediante sus descripciones exactas, permitiendo posteriormente comprobar de forma determinista que ninguna se ha omitido.

In [30]:
# =============================================================================
# MODELO Y CONTEXTO GENERATIVO CONTROLADO
# =============================================================================
print('\nMODELO Y CONTEXTO GENERATIVO CONTROLADO', flush=True)

# -----------------------------------------------------------------------------
# CONTEXTO COMPLETO PARA EL INFORME
# -----------------------------------------------------------------------------
print('[1/2] Obteniendo explicaciones para los cuatro indicadores...', flush=True)

inicio = perf_counter()

estado_informe_flujo = aplicacion_flujo.invoke({
    'registro': entrada_referencia.registro,
    'solicitar_explicacion': True,
    'objetivos_explicacion': targets,
    'solicitar_documentacion': True,
    'trazabilidad': []
})

tiempo_contexto_informe = perf_counter() - inicio

print(f'      Flujo completo preparado en {tiempo_contexto_informe:.2f} s', flush=True)

contexto_mistral = preparar_contexto_generativo(
    estado_informe_flujo, n_variables_shap=3, max_caracteres_fragmento=900
)

factores_contexto_mistral = pd.DataFrame(contexto_mistral['factores_permitidos'])

tabla_factores = factores_contexto_mistral.rename(columns=renombrado_comun)

print('\nFACTORES AUTORIZADOS PARA EL INFORME')
mostrar_tabla_completa(tabla_factores)

# -----------------------------------------------------------------------------
# PROMPT
# -----------------------------------------------------------------------------
print('[2/2] Configurando la generación estructurada...', flush=True)

modelo_mistral_chat = ChatMistralAI(
    model=modelo_mistral, temperature=temperatura_mistral, max_retries=2
)

modelo_mistral_estructurado = modelo_mistral_chat.with_structured_output(
    SalidaGenerativa, include_raw=True
)

factores_por_objetivo_contexto = factores_contexto_mistral.groupby(
    ['variable_objetivo', 'descripcion_objetivo'], sort=False
).size().reset_index(name='factores')

tabla_factores_objetivo = factores_por_objetivo_contexto.rename(columns={
    **renombrado_comun, 'factores': 'Factores'
})

print('\nFACTORES POR INDICADOR')
display(tabla_factores_objetivo)

resumen_contexto_mistral = pd.DataFrame({
    'elemento': [
        'Modelo generativo',
        'Versión del prompt',
        'Predicciones disponibles',
        'Explicaciones SHAP',
        'Factores SHAP permitidos',
        'Documentos recuperados',
        'Fuentes permitidas',
        'Frases de orientación disponibles',
        'Registro original incluido',
        'Tiempo de preparación del contexto (s)'
    ],
    'valor': [
        modelo_mistral,
        version_prompt_informe,
        len(contexto_mistral['predicciones']),
        len(contexto_mistral['explicaciones']),
        len(contexto_mistral['factores_permitidos']),
        len(contexto_mistral['documentacion']),
        len(contexto_mistral['fuentes_permitidas']),
        len(contexto_mistral['orientaciones_permitidas']),
        False,
        round(tiempo_contexto_informe, 3)
    ]
})

tabla_resumen = resumen_contexto_mistral.rename(columns=renombrado_comun)

print('\nRESUMEN DEL CONTEXTO GENERATIVO')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_contexto_mistral = pd.DataFrame({
    'comprobacion': [
        'La credencial de Mistral está configurada',
        'El contexto contiene las cuatro predicciones',
        'Existen explicaciones SHAP para los cuatro indicadores',
        'Se conservan exactamente tres factores por indicador',
        'El contexto contiene 12 factores SHAP',
        'La documentación coincide con la recuperada por el flujo',
        'El banco de frases está incorporado al contexto',
        'El registro original de 813 variables no se envía a Mistral',
        'El contexto no contiene probabilidades ni umbrales predictivos',
        'La salida estructurada utiliza SalidaGenerativa'
    ],
    'resultado': [
        bool(os.getenv('MISTRAL_API_KEY')),
        len(contexto_mistral['predicciones']) == 4,
        len(contexto_mistral['explicaciones']) == 4,
        factores_por_objetivo_contexto['factores'].eq(3).all(),
        len(contexto_mistral['factores_permitidos']) == 12,
        len(contexto_mistral['documentacion']) == len(estado_informe_flujo['contexto_documental']),
        contexto_mistral['orientaciones_permitidas'] == frases_informe['orientacion_preventiva'] and
        contexto_mistral['limitaciones_obligatorias'] == frases_informe['limitaciones_obligatorias']
        and contexto_mistral['advertencia_obligatoria'] == frases_informe['advertencia_uso'] and
        contexto_mistral['nota_factores'] == frases_informe['nota_factores'],
        'registro' not in contexto_mistral,
        all('probabilidad' not in resultado and 'umbral' not in resultado
            for resultado in contexto_mistral['predicciones']),
        set(SalidaGenerativa.model_fields) == {'resumen', 'interpretacion', 'traducciones_factores'}
    ]
})

tabla_comprobaciones = comprobaciones_contexto_mistral.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DEL CONTEXTO GENERATIVO')
display(tabla_comprobaciones)

if not comprobaciones_contexto_mistral['resultado'].all():
    raise ValueError('El contexto generativo para Mistral no es correcto.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_contexto_mistral, '48_configuracion_generacion_mistral')
_ = guardar_json(contexto_mistral, '49_contexto_generativo_mistral.json')

print('\nContexto generativo preparado correctamente.')


MODELO Y CONTEXTO GENERATIVO CONTROLADO


[1/2] Obteniendo explicaciones para los cuatro indicadores...


[1/5] Validando el registro de entrada...


[2/5] Ejecutando la tool predictiva local...


[3/5] Generando 4 explicaciones SHAP...


...


RESUMEN DEL CONTEXTO GENERATIVO


,Elemento,Valor
0,Modelo generativo,mistral-small-latest
1,Versión del prompt,v5
2,Predicciones disponibles,4
3,Explicaciones SHAP,4
4,Factores SHAP permitidos,12
5,Documentos recuperados,13
6,Fuentes permitidas,2
7,Frases de orientación disponibles,5
8,Registro original incluido,False
9,Tiempo de preparación del contexto (s),196.831



COMPROBACIONES DEL CONTEXTO GENERATIVO


,Comprobación,Resultado
0,La credencial de Mistral está configurada,True
1,El contexto contiene las cuatro predicciones,True
2,Existen explicaciones SHAP para los cuatro ind...,True
3,Se conservan exactamente tres factores por ind...,True
4,El contexto contiene 12 factores SHAP,True
5,La documentación coincide con la recuperada po...,True
6,El banco de frases está incorporado al contexto,True
7,El registro original de 813 variables no se en...,True
8,El contexto no contiene probabilidades ni umbr...,True
9,La salida estructurada utiliza SalidaGenerativa,True



Contexto generativo preparado correctamente.


### Resultados

El contexto generativo se prepara a partir de una ejecución completa del flujo para los cuatro indicadores. Se obtienen cuatro explicaciones SHAP y se seleccionan exactamente tres factores contractuales por indicador, formando los 12 factores autorizados que podrán incorporarse al informe preventivo.

La recuperación documental proporciona 13 documentos principales procedentes de dos fuentes autorizadas: el codebook oficial NSDUH 2024 y las limitaciones metodológicas transferidas desde el Notebook 04.

El contexto entregado a Mistral se minimiza deliberadamente. No contiene el registro original con las 813 variables ni los valores numéricos de probabilidades, umbrales o contribuciones SHAP. Sí conserva la información necesaria para redactar: indicadores, decisiones respecto al umbral, factores autorizados, dirección de las contribuciones, documentación y banco controlado de frases.

La preparación completa del contexto requiere varios minutos porque incluye las cuatro explicaciones SHAP, que concentran la mayor parte del coste computacional.

Se utiliza `mistral-small-latest` con la versión `v5` del prompt y el contrato `SalidaGenerativa`. Las diez comprobaciones realizadas resultan correctas, confirmando la minimización del contexto, la presencia de los 12 factores permitidos y la separación entre información determinista y generación de lenguaje natural.

## 7.2. Generación estructurada del informe preventivo

Se realiza la primera llamada real a Mistral utilizando exclusivamente el contexto validado en el apartado anterior.

La respuesta generativa se solicita mediante el contrato `SalidaGenerativa`, que contiene únicamente el resumen, la interpretación y las traducciones necesarias para presentar los factores de forma comprensible. Mistral no genera probabilidades, umbrales, clasificaciones, posiciones SHAP, dirección de las contribuciones, fuentes, limitaciones, orientaciones ni advertencias.

A partir de esta salida reducida, la aplicación reconstruye de forma determinista `InformePreventivo`. Los 12 factores contractuales se obtienen directamente de SHAP y las orientaciones, limitaciones, fuentes y advertencia de uso proceden de estructuras previamente controladas.

Además del informe contractual se prepara una explicación ampliada destinada a la interfaz. Para cada uno de los cuatro indicadores se muestran hasta tres factores que aumentan la predicción y hasta dos que la reducen, siempre que existan entre las contribuciones SHAP disponibles. Esta explicación ampliada se mantiene independiente de que el indicador supere o no su umbral.

La salida generativa se valida primero mediante Pydantic y posteriormente se somete a los guardrails deterministas del siguiente apartado.

In [31]:
# =============================================================================
# GENERACIÓN ESTRUCTURADA DEL INFORME PREVENTIVO
# =============================================================================
print('\nGENERACIÓN ESTRUCTURADA DEL INFORME PREVENTIVO', flush=True)

print('[1/3] Enviando el contexto validado a Mistral...', flush=True)

resultado_generacion_mistral = generar_informe_mistral(
    modelo_mistral_estructurado, contexto_mistral, prompt_sistema_informe
)

salida_generativa = resultado_generacion_mistral['salida']

salida_generativa = SalidaGenerativa.model_validate(salida_generativa)

informe_generado = construir_informe_preventivo(salida_generativa, contexto_mistral)

detalle_explicabilidad_generado = construir_detalle_explicabilidad(
    contexto_mistral['factores_ampliados'], salida_generativa.traducciones_factores
)

detalle_explicabilidad = pd.DataFrame(detalle_explicabilidad_generado)

tabla_detalle_explicabilidad = detalle_explicabilidad.rename(columns=renombrado_comun)

print('\nEXPLICACIÓN AMPLIADA PARA LA INTERFAZ')
mostrar_tabla_completa(tabla_detalle_explicabilidad)

tiempo_generacion_mistral = resultado_generacion_mistral['tiempo_s']

uso_tokens_mistral = resultado_generacion_mistral['uso_tokens']

print(f'      Respuesta estructurada recibida en {tiempo_generacion_mistral:.2f} s', flush=True)

print('[2/3] Validando el contrato Pydantic...', flush=True)
informe_generado = InformePreventivo.model_validate(informe_generado)

traducciones_generadas = pd.DataFrame([
    {
        'variable_origen': traduccion.variable_origen,
        'descripcion': limpiar_descripcion_factor(
            traduccion.descripcion, traduccion.variable_origen
        )
    }
    for traduccion in salida_generativa.traducciones_factores
])

tabla_traducciones = traducciones_generadas.rename(columns=renombrado_comun)

print('\nTRADUCCIONES GENERADAS PARA LOS FACTORES')
mostrar_tabla_completa(tabla_traducciones)

factores_informe_generado = pd.DataFrame([
    factor.model_dump(mode='json') for factor in informe_generado.factores_relevantes
])

tabla_factores = factores_informe_generado.rename(columns=renombrado_comun)

print('\nFACTORES GENERADOS POR INDICADOR')
mostrar_tabla_completa(tabla_factores)

tabla_informe_generado = pd.DataFrame({
    'campo': list(InformePreventivo.model_fields),
    'contenido': [getattr(informe_generado, campo) for campo in InformePreventivo.model_fields]
})

print('\nINFORME ESTRUCTURADO GENERADO')
mostrar_tabla_completa(tabla_informe_generado)

print('[3/3] Preparando una representación legible...', flush=True)

texto_informe_generado = formatear_informe_preventivo(informe_generado)

print('\nINFORME PREVENTIVO — RECONSTRUCCIÓN DETERMINISTA\n')
print(texto_informe_generado)

resumen_generacion_mistral = pd.DataFrame({
    'elemento': [
        'Modelo generativo',
        'Versión del prompt',
        'Campos generados',
        'Factores relevantes',
        'Orientaciones preventivas',
        'Limitaciones',
        'Fuentes',
        'Tiempo de generación (s)',
        'Tokens de entrada',
        'Tokens de salida',
        'Tokens totales'
    ],
    'valor': [
        modelo_mistral,
        version_prompt_informe,
        len(InformePreventivo.model_fields),
        len(informe_generado.factores_relevantes),
        len(informe_generado.orientacion_preventiva),
        len(informe_generado.limitaciones),
        len(informe_generado.fuentes),
        round(tiempo_generacion_mistral, 3),
        uso_tokens_mistral.get('input_tokens'),
        uso_tokens_mistral.get('output_tokens'),
        uso_tokens_mistral.get('total_tokens')
    ]
})

tabla_resumen = resumen_generacion_mistral.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA GENERACIÓN')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
variables_traducidas = {
    traduccion.variable_origen for traduccion in salida_generativa.traducciones_factores
}

variables_esperadas = {
    variable['variable_origen'] for variable in contexto_mistral['variables_traducir']
}

numero_traducciones_esperadas = len(variables_esperadas)

factores_por_objetivo = pd.Series([
    factor.variable_objetivo for factor in informe_generado.factores_relevantes
]).value_counts()

comprobaciones_generacion_mistral = pd.DataFrame({
    'comprobacion': [
        'La respuesta cumple el contrato SalidaGenerativa',
        'Se generan exactamente las variables esperadas',
        'No existen traducciones duplicadas',
        'Todas las traducciones contienen información',
        'El informe final cumple InformePreventivo',
        'Se construyen exactamente 12 factores',
        'Se conservan exactamente tres factores por indicador',
        'Los cuatro indicadores conservan factores independientemente de su clasificación',
        'La explicación ampliada procede exclusivamente de factores SHAP permitidos',
        'Las descripciones ampliadas no muestran códigos técnicos',
        'El prompt de sistema v5 está configurado'
    ],
    'resultado': [
        isinstance(salida_generativa, SalidaGenerativa),
        variables_traducidas == variables_esperadas,
        len(salida_generativa.traducciones_factores) == numero_traducciones_esperadas,
        all(traduccion.descripcion.strip()
            for traduccion in salida_generativa.traducciones_factores),
        isinstance(informe_generado, InformePreventivo),
        len(informe_generado.factores_relevantes) == 12,
        (set(factores_por_objetivo.index) == set(targets) and factores_por_objetivo.eq(3).all()),
        set(factores_por_objetivo.index) == set(targets),
        set(factor['variable_origen'] for factor in detalle_explicabilidad_generado)
        .issubset(variables_traducidas),
        all(factor['variable_origen'].casefold() not in factor['descripcion'].casefold()
            for factor in detalle_explicabilidad_generado),
        bool(prompt_sistema_informe.strip()) and version_prompt_informe == 'v5'
    ]
})

tabla_comprobaciones = comprobaciones_generacion_mistral.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES ESTRUCTURALES DE LA GENERACIÓN')
display(tabla_comprobaciones)

if not comprobaciones_generacion_mistral['resultado'].all():
    raise ValueError('La generación estructurada mediante Mistral no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_generacion_mistral, '50_resumen_generacion_mistral')

print('\nGeneración estructurada completada correctamente.')


GENERACIÓN ESTRUCTURADA DEL INFORME PREVENTIVO


[1/3] Enviando el contexto validado a Mistral...



EXPLICACIÓN AMPLIADA PARA LA INTERFAZ


,Variable objetivo,Indicador,Variable original,Descripción original,Tipo de factor,Posición,Valor observado,Valor SHAP,Descripción,Respuesta observada,Aclaración de la variable
0,IRAMDEYR,Episodio depresivo mayor,CAMHPROB2,RC-PERCEIVED EVER HAD A MENTAL HEALTH ISSUE,Factor de riesgo para la predicción,1,1.0,0.352954,Percepción de haber tenido alguna vez un problema de salud mental,Sí,NSDUH obtiene esta variable recodificando en Sí/No la respuesta original sobre si la persona considera haber tenido alguna vez un problema de salud mental; no corresponde a una pregunta independiente.
1,IRAMDEYR,Episodio depresivo mayor,LVLDIFMEM2,LEVEL OF DIFFICULTY REMEMBERING OR CONCENTRATING,Factor de riesgo para la predicción,2,2.0,0.058497,Nivel de dificultad para recordar o concentrarse,Alguna dificultad,NaN
2,IRAMDEYR,Episodio depresivo mayor,LVLDIFCARE2,LEVEL OF DIFFICULTY WITH SELF-CARE,Factor de riesgo para la predicción,3,2.0,0.057277,Nivel de dificultad con el autocuidado,Alguna dificultad,NaN
3,IRAMDEYR,Episodio depresivo mayor,RCVYMHPRB,RC-PERCEIVED RECOVERY FROM MENTAL HEALTH ISSUE,Factor protector para la predicción,1,1.0,-0.062648,Percepción de recuperación de un problema de salud mental,Sí,NSDUH obtiene esta variable combinando la respuesta sobre haber tenido un problema de salud mental con la respuesta sobre recuperación; no corresponde a una pregunta independiente.
4,IRAMDEYR,Episodio depresivo mayor,NMERTMT2,# OF TIMES BEEN TREATED IN EMER ROOM PAST 12 MOS,Factor protector para la predicción,2,0.0,-0.012164,Número de veces que ha sido atendido en sala de emergencias en los últimos 12 meses,0 veces,NaN
5,IRSUICTHNK,Ideación suicida,CAMHPROB,THINK EVER HAD PROBLEM WITH OWN MENTAL HEALTH,Factor de riesgo para la predicción,1,1.0,0.045312,Percepción de haber tenido alguna vez un problema de salud mental,Sí,NaN
6,IRSUICTHNK,Ideación suicida,CAMHPROB2,RC-PERCEIVED EVER HAD A MENTAL HEALTH ISSUE,Factor de riesgo para la predicción,2,1.0,0.033840,Percepción de haber tenido alguna vez un problema de salud mental,Sí,NSDUH obtiene esta variable recodificando en Sí/No la respuesta original sobre si la persona considera haber tenido alguna vez un problema de salud mental; no corresponde a una pregunta independiente.
7,IRSUICTHNK,Ideación suicida,LVLDIFCARE2,LEVEL OF DIFFICULTY WITH SELF-CARE,Factor de riesgo para la predicción,3,2.0,0.027407,Nivel de dificultad con el autocuidado,Alguna dificultad,NaN
8,IRSUICTHNK,Ideación suicida,RCVYMHPRB,RC-PERCEIVED RECOVERY FROM MENTAL HEALTH ISSUE,Factor protector para la predicción,1,1.0,-0.011296,Percepción de recuperación de un problema de salud mental,Sí,NSDUH obtiene esta variable combinando la respuesta sobre haber tenido un problema de salud mental con la respuesta sobre recuperación; no corresponde a una pregunta independiente.
9,IRSUICTHNK,Ideación suicida,MRJYDAYS,RC-# OF DAYS USED MARIJUANA IN PAST YEAR,Factor protector para la predicción,2,5.0,-0.007429,Número de días que consumió marihuana en el último año,300–365 días,NSDUH obtiene esta variable recodificando una respuesta original según las reglas de su documentación oficial; no corresponde a una pregunta independiente.


      Respuesta estructurada recibida en 8.09 s


...


INFORME PREVENTIVO — RECONSTRUCCIÓN DETERMINISTA

RESUMEN
En esta evaluación, tres de los cuatro indicadores evaluados superan sus umbrales validados: episodio depresivo mayor, ideación suicida e intento suicida. El indicador de planificación suicida no supera su umbral validado.

INTERPRETACIÓN
Se evaluaron cuatro indicadores: episodio depresivo mayor, ideación suicida, planificación suicida e intento suicida. El indicador de episodio depresivo mayor supera su umbral validado. El indicador de ideación suicida supera su umbral validado. El indicador de planificación suicida no supera su umbral validado. El indicador de intento suicida supera su umbral validado. Este resultado señala la conveniencia de una revisión preventiva adicional.

FACTORES RELEVANTES POR INDICADOR

Episodio depresivo mayor
Factores de riesgo para la predicción:
- Percepción de haber tenido alguna vez un problema de salud mental
- Nivel de dificultad para recordar o concentrarse
Factores protectores para la predi

,Elemento,Valor
0,Modelo generativo,mistral-small-latest
1,Versión del prompt,v5
2,Campos generados,7
3,Factores relevantes,12
4,Orientaciones preventivas,3
5,Limitaciones,3
6,Fuentes,2
7,Tiempo de generación (s),8.094
8,Tokens de entrada,5630
9,Tokens de salida,523



COMPROBACIONES ESTRUCTURALES DE LA GENERACIÓN


,Comprobación,Resultado
0,La respuesta cumple el contrato SalidaGenerativa,True
1,Se generan exactamente las variables esperadas,True
2,No existen traducciones duplicadas,True
3,Todas las traducciones contienen información,True
4,El informe final cumple InformePreventivo,True
5,Se construyen exactamente 12 factores,True
6,Se conservan exactamente tres factores por ind...,True
7,Los cuatro indicadores conservan factores inde...,True
8,La explicación ampliada procede exclusivamente...,True
9,Las descripciones ampliadas no muestran código...,True



Generación estructurada completada correctamente.


### Resultados

Mistral devuelve correctamente una salida compatible con `SalidaGenerativa` utilizando el modelo `mistral-small-latest` y el prompt `v5`. La generación requiere varios segundos y utiliza aproximadamente cinco mil tokens de entrada y medio millar de salida.

Se generan las traducciones necesarias para las 10 variables explicativas distintas utilizadas en el informe y en la explicación ampliada. Tras limpiar los prefijos técnicos, las descripciones visibles no contienen los códigos internos de las variables.

El significado de las respuestas observadas no se delega al modelo generativo. La aplicación lo recupera posteriormente de forma determinista desde el codebook oficial y elimina de la presentación los códigos internos, las frecuencias y los porcentajes utilizados en la documentación estadística de NSDUH.

La aplicación reconstruye posteriormente el contrato `InformePreventivo` con sus siete campos. El informe contractual contiene exactamente 12 factores, tres para cada uno de los cuatro indicadores, y estos factores proceden de las explicaciones SHAP previamente autorizadas.

La explicación ampliada contiene, para este registro, cinco factores por indicador: hasta tres contribuciones positivas y dos negativas. En total se presentan 20 factores explicativos ampliados, manteniendo para cada uno el valor observado y la dirección de su contribución. Esta información se genera también para planificación suicida, aunque su probabilidad no supera el umbral validado.

Las diez comprobaciones estructurales resultan correctas: se valida `SalidaGenerativa`, se reconstruye correctamente `InformePreventivo`, se mantienen los 12 factores contractuales, los cuatro indicadores conservan explicación y la capa ampliada utiliza exclusivamente factores SHAP permitidos.

## 7.3. Guardrails deterministas sobre el informe

El cumplimiento del esquema Pydantic garantiza la estructura de la respuesta, pero no es suficiente para aceptar su contenido.

Por este motivo se aplica una segunda capa de validación completamente determinista e independiente de Mistral. Los guardrails comprueban que aparezcan las cuatro variables objetivo, que los factores se limiten a las variables SHAP previamente autorizadas, que las fuentes correspondan a la documentación recuperada y que se mantengan las advertencias de uso.

También se rechazan afirmaciones diagnósticas o causales, la introducción de cifras predictivas en el texto generado y las recomendaciones farmacológicas.

El informe solo se considera válido cuando todas estas comprobaciones resultan correctas.

In [32]:
# =============================================================================
# GUARDRAILS DETERMINISTAS SOBRE EL INFORME
# =============================================================================
print('\nGUARDRAILS DETERMINISTAS SOBRE EL INFORME')

guardrails_informe = evaluar_guardrails_informe(
    informe_generado, contexto_mistral['descripciones_objetivos'],
    contexto_mistral['factores_permitidos'], contexto_mistral['fuentes_permitidas'],
    contexto_mistral['predicciones'], frases_informe
)

tabla_guardrails = guardrails_informe.rename(columns=renombrado_comun)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
print('\nCOMPROBACIONES DE LOS GUARDRAILS')
display(tabla_guardrails)

if not guardrails_informe['resultado'].all():
    raise ValueError('El informe generado no supera todos los guardrails.')

informe_validado = informe_generado
texto_informe_validado = formatear_informe_preventivo(informe_validado)

resumen_guardrails = pd.DataFrame({
    'elemento': [
        'Guardrails evaluados',
        'Guardrails superados',
        'Informe estructuralmente válido',
        'Informe aceptado'
    ],
    'valor': [
        len(guardrails_informe),
        int(guardrails_informe['resultado'].sum()),
        isinstance(informe_validado, InformePreventivo),
        guardrails_informe['resultado'].all()
    ]
})

tabla_resumen = resumen_guardrails.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA VALIDACIÓN DEL INFORME')
display(tabla_resumen)

print('\nINFORME PREVENTIVO VALIDADO\n')
print(texto_informe_validado)

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(guardrails_informe, '51_guardrails_informe_mistral')
_ = guardar_json(informe_validado, '01_informe_preventivo_mistral.json', ruta_informes)
_ = guardar_texto(texto_informe_validado, '02_informe_preventivo_mistral.txt')

print('\nInforme preventivo validado y guardado correctamente.')


GUARDRAILS DETERMINISTAS SOBRE EL INFORME

COMPROBACIONES DE LOS GUARDRAILS


,Comprobación,Resultado
0,El informe mantiene los siete campos contractu...,True
1,Las cuatro variables objetivo aparecen en la i...,True
2,Se generan exactamente tres factores por varia...,True
3,Los factores proceden exclusivamente de SHAP,True
4,El tipo de factor coincide con la dirección SHAP,True
5,Las descripciones visibles se presentan traduc...,True
6,Las fuentes coinciden con las fuentes document...,True
7,La orientación utiliza únicamente el banco de ...,True
8,Las limitaciones obligatorias se mantienen exa...,True
9,La advertencia de uso coincide con el banco de...,True



RESUMEN DE LA VALIDACIÓN DEL INFORME


,Elemento,Valor
0,Guardrails evaluados,15
1,Guardrails superados,15
2,Informe estructuralmente válido,True
3,Informe aceptado,True



INFORME PREVENTIVO VALIDADO

RESUMEN
En esta evaluación, tres de los cuatro indicadores evaluados superan sus umbrales validados: episodio depresivo mayor, ideación suicida e intento suicida. El indicador de planificación suicida no supera su umbral validado.

INTERPRETACIÓN
Se evaluaron cuatro indicadores: episodio depresivo mayor, ideación suicida, planificación suicida e intento suicida. El indicador de episodio depresivo mayor supera su umbral validado. El indicador de ideación suicida supera su umbral validado. El indicador de planificación suicida no supera su umbral validado. El indicador de intento suicida supera su umbral validado. Este resultado señala la conveniencia de una revisión preventiva adicional.

FACTORES RELEVANTES POR INDICADOR

Episodio depresivo mayor
Factores de riesgo para la predicción:
- Percepción de haber tenido alguna vez un problema de salud mental
- Nivel de dificultad para recordar o concentrarse
Factores protectores para la predicción:
- Percepción d

### Resultados

El informe generado supera las 15 comprobaciones deterministas establecidas.

Se confirma el mantenimiento de los siete campos contractuales, la presencia de los cuatro indicadores, la asignación de exactamente tres factores por variable objetivo y la correspondencia de estos factores con las explicaciones SHAP.

La dirección de cada contribución se conserva mediante las categorías `Factor de riesgo para la predicción` y `Factor protector para la predicción`, mientras que las descripciones visibles se presentan mediante las traducciones generadas.

Las fuentes se limitan a la documentación autorizada y las orientaciones, limitaciones y advertencia de uso permanecen vinculadas al banco de frases controlado.

No se detectan afirmaciones causales o diagnósticas, cifras predictivas introducidas por la capa generativa ni recomendaciones farmacológicas. También se comprueba específicamente que los indicadores que no superan su umbral mantienen sus factores explicativos.

Los 15 guardrails son superados y el informe queda aceptado.

## 7.4. Pruebas controladas de los guardrails

Una validación correcta del informe generado demuestra que las reglas aceptan una respuesta válida, pero también es necesario comprobar que rechazan contenidos incompatibles.

Para ello se crean copias controladas del informe ya validado y se introduce de forma aislada una infracción conocida: una afirmación diagnóstica, una relación causal, un valor predictivo numérico, una fuente no autorizada, un factor no procedente de la explicación SHAP o una modificación incorrecta de la dirección del factor.

Estas respuestas no se generan mediante nuevas llamadas a Mistral y no forman parte del informe final. Se utilizan exclusivamente para comprobar de forma determinista que cada guardrail actúa ante el tipo de error para el que fue definido.

In [33]:
# =============================================================================
# PRUEBAS CONTROLADAS DE LOS GUARDRAILS
# =============================================================================
print('\nPRUEBAS CONTROLADAS DE LOS GUARDRAILS')

casos_guardrails = []

# -----------------------------------------------------------------------------
# DIAGNÓSTICO
# -----------------------------------------------------------------------------
informe_diagnostico = informe_validado.model_copy(deep=True)
informe_diagnostico.resumen += ('El registro confirma un diagnóstico de depresión.')

guardrails_diagnostico = evaluar_guardrails_informe(
    informe_diagnostico, contexto_mistral['descripciones_objetivos'],
    contexto_mistral['factores_permitidos'], contexto_mistral['fuentes_permitidas'],
    contexto_mistral['predicciones'], frases_informe
)

casos_guardrails.append({
    'caso': 'Afirmación diagnóstica',
    'guardrail': 'No se formulan afirmaciones diagnósticas',
    'rechazado': not guardrails_diagnostico.loc[
        guardrails_diagnostico['comprobacion'].eq('No se formulan afirmaciones diagnósticas'),
        'resultado'
    ].iloc[0]
})

# -----------------------------------------------------------------------------
# CAUSALIDAD
# -----------------------------------------------------------------------------
informe_causal = informe_validado.model_copy(deep=True)
informe_causal.interpretacion += (' Esta variable causa el resultado observado.')

guardrails_causal = evaluar_guardrails_informe(
    informe_causal, contexto_mistral['descripciones_objetivos'],
    contexto_mistral['factores_permitidos'], contexto_mistral['fuentes_permitidas'],
    contexto_mistral['predicciones'], frases_informe
)

casos_guardrails.append({
    'caso': 'Afirmación causal',
    'guardrail': 'No se formulan relaciones causales',
    'rechazado': not guardrails_causal.loc[
        guardrails_causal['comprobacion'].eq('No se formulan relaciones causales'),
        'resultado'
    ].iloc[0]
})

# -----------------------------------------------------------------------------
# CIFRA PREDICTIVA
# -----------------------------------------------------------------------------
informe_numerico = informe_validado.model_copy(deep=True)
informe_numerico.resumen += (' La probabilidad estimada es 0.827.')

guardrails_numerico = evaluar_guardrails_informe(
    informe_numerico, contexto_mistral['descripciones_objetivos'],
    contexto_mistral['factores_permitidos'], contexto_mistral['fuentes_permitidas'],
    contexto_mistral['predicciones'], frases_informe
)

casos_guardrails.append({
    'caso': 'Cifra predictiva generada',
    'guardrail': 'La narración no introduce cifras predictivas',
    'rechazado': not guardrails_numerico.loc[
        guardrails_numerico['comprobacion'].eq('La narración no introduce cifras predictivas'),
        'resultado'
    ].iloc[0]
})

# -----------------------------------------------------------------------------
# FUENTE NO AUTORIZADA
# -----------------------------------------------------------------------------
informe_fuente = informe_validado.model_copy(deep=True)
informe_fuente.fuentes = ['Fuente no autorizada']

guardrails_fuente = evaluar_guardrails_informe(
    informe_fuente, contexto_mistral['descripciones_objetivos'],
    contexto_mistral['factores_permitidos'], contexto_mistral['fuentes_permitidas'],
    contexto_mistral['predicciones'], frases_informe
)

casos_guardrails.append({
    'caso': 'Fuente no autorizada',
    'guardrail': ('Las fuentes coinciden con las fuentes documentales permitidas'),
    'rechazado': not guardrails_fuente.loc[
        guardrails_fuente['comprobacion']
        .eq('Las fuentes coinciden con las fuentes documentales permitidas'), 'resultado'
    ].iloc[0]
})

# -----------------------------------------------------------------------------
# FACTOR NO AUTORIZADO
# -----------------------------------------------------------------------------
informe_factor = informe_validado.model_copy(deep=True)
informe_factor.factores_relevantes[0].variable_origen = ('VARIABLE_INVENTADA')

guardrails_factor = evaluar_guardrails_informe(
    informe_factor, contexto_mistral['descripciones_objetivos'],
    contexto_mistral['factores_permitidos'], contexto_mistral['fuentes_permitidas'],
    contexto_mistral['predicciones'], frases_informe
)

casos_guardrails.append({
    'caso': 'Factor no autorizado',
    'guardrail': 'Los factores proceden exclusivamente de SHAP',
    'rechazado': not guardrails_factor.loc[
        guardrails_factor['comprobacion'].eq('Los factores proceden exclusivamente de SHAP'),
        'resultado'
    ].iloc[0]
})

# -----------------------------------------------------------------------------
# TIPO DE FACTOR INCOHERENTE
# -----------------------------------------------------------------------------
informe_tipo_factor = informe_validado.model_copy(deep=True)
factor_modificado = informe_tipo_factor.factores_relevantes[0]

factor_modificado.tipo_factor = (
    'Factor protector para la predicción'
    if factor_modificado.tipo_factor == 'Factor de riesgo para la predicción'
    else 'Factor de riesgo para la predicción'
)

guardrails_tipo_factor = evaluar_guardrails_informe(
    informe_tipo_factor, contexto_mistral['descripciones_objetivos'],
    contexto_mistral['factores_permitidos'], contexto_mistral['fuentes_permitidas'],
    contexto_mistral['predicciones'], frases_informe
)

casos_guardrails.append({
    'caso': 'Tipo de factor incoherente',
    'guardrail': 'El tipo de factor coincide con la dirección SHAP',
    'rechazado': not guardrails_tipo_factor.loc[
        guardrails_tipo_factor['comprobacion']
        .eq('El tipo de factor coincide con la dirección SHAP'), 'resultado'
    ].iloc[0]
})

# -----------------------------------------------------------------------------
# RESUMEN DE PRUEBAS
# -----------------------------------------------------------------------------
pruebas_guardrails_controladas = pd.DataFrame(casos_guardrails)

tabla_pruebas = pruebas_guardrails_controladas.rename(columns={
    'caso': 'Caso', 'guardrail': 'Guardrail esperado', 'rechazado': 'Rechazado correctamente'
})

print('\nRESPUESTAS NO VÁLIDAS CONTROLADAS')
display(tabla_pruebas)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_finales_generacion = pd.DataFrame({
    'comprobacion': [
        'El informe real supera todos los guardrails',
        'La afirmación diagnóstica es rechazada',
        'La afirmación causal es rechazada',
        'La cifra predictiva generada es rechazada',
        'La fuente no autorizada es rechazada',
        'El factor no autorizado es rechazado',
        'El tipo de factor incoherente es rechazado'
    ],
    'resultado': [
        guardrails_informe['resultado'].all(),
        pruebas_guardrails_controladas.loc[
            pruebas_guardrails_controladas['caso']
            .eq('Afirmación diagnóstica'), 'rechazado'].iloc[0],
        pruebas_guardrails_controladas.loc[
            pruebas_guardrails_controladas['caso']
            .eq('Afirmación causal'), 'rechazado'].iloc[0],
        pruebas_guardrails_controladas.loc[
            pruebas_guardrails_controladas['caso']
            .eq('Cifra predictiva generada'), 'rechazado'].iloc[0],
        pruebas_guardrails_controladas.loc[
            pruebas_guardrails_controladas['caso']
            .eq('Fuente no autorizada'), 'rechazado'].iloc[0],
        pruebas_guardrails_controladas.loc[
            pruebas_guardrails_controladas['caso']
            .eq('Factor no autorizado'), 'rechazado'].iloc[0],
        pruebas_guardrails_controladas.loc[
            pruebas_guardrails_controladas['caso']
            .eq('Tipo de factor incoherente'), 'rechazado'].iloc[0]
    ]
})

tabla_comprobaciones = comprobaciones_finales_generacion.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES FINALES DE LA GENERACIÓN')
display(tabla_comprobaciones)

if not comprobaciones_finales_generacion['resultado'].all():
    raise ValueError('La validación controlada de los guardrails no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(pruebas_guardrails_controladas, '52_pruebas_guardrails_controladas')

print('\nGeneración y guardrails validados correctamente.')


PRUEBAS CONTROLADAS DE LOS GUARDRAILS

RESPUESTAS NO VÁLIDAS CONTROLADAS


,Caso,Guardrail esperado,Rechazado correctamente
0,Afirmación diagnóstica,No se formulan afirmaciones diagnósticas,True
1,Afirmación causal,No se formulan relaciones causales,True
2,Cifra predictiva generada,La narración no introduce cifras predictivas,True
3,Fuente no autorizada,Las fuentes coinciden con las fuentes document...,True
4,Factor no autorizado,Los factores proceden exclusivamente de SHAP,True
5,Tipo de factor incoherente,El tipo de factor coincide con la dirección SHAP,True



COMPROBACIONES FINALES DE LA GENERACIÓN


,Comprobación,Resultado
0,El informe real supera todos los guardrails,True
1,La afirmación diagnóstica es rechazada,True
2,La afirmación causal es rechazada,True
3,La cifra predictiva generada es rechazada,True
4,La fuente no autorizada es rechazada,True
5,El factor no autorizado es rechazado,True
6,El tipo de factor incoherente es rechazado,True



Generación y guardrails validados correctamente.


### Resultados

Los guardrails se someten a seis pruebas adversas controladas mediante modificaciones artificiales del informe ya validado, sin realizar nuevas llamadas a Mistral.

Se introducen de forma independiente una afirmación diagnóstica, una afirmación causal, una cifra predictiva generada, una fuente no autorizada, un factor no procedente de SHAP y una modificación incorrecta del tipo de factor.

Los seis contenidos incompatibles son rechazados por el guardrail previsto para cada caso.

Las siete comprobaciones finales resultan correctas: el informe real es aceptado y las seis alteraciones controladas son detectadas. La capa de seguridad queda así validada tanto para la aceptación de contenido permitido como para el rechazo de contenido incompatible.

## Síntesis de la sección 7 (memoria)

La capa generativa se ha diseñado para reducir deliberadamente la libertad del modelo de lenguaje en un dominio sensible. Mistral no recibe el registro original de 813 variables ni controla probabilidades, umbrales, clasificaciones, factores SHAP, fuentes, limitaciones, orientaciones o advertencias.

El flujo completo obtiene cuatro explicaciones SHAP y selecciona tres factores contractuales por indicador. El contexto generativo incorpora 12 factores autorizados y 13 documentos controlados procedentes del codebook NSDUH 2024 y del Notebook 04, eliminando los valores numéricos predictivos antes de realizar la llamada externa.

Mistral, mediante `mistral-small-latest` y el prompt `v5`, genera exclusivamente resumen, interpretación y traducciones mediante el contrato `SalidaGenerativa`. La aplicación utiliza posteriormente estas salidas para reconstruir de forma determinista `InformePreventivo`, que conserva siete campos y exactamente 12 factores, tres por indicador.

Las descripciones en lenguaje natural generadas por Mistral se complementan con el significado de la respuesta observada, obtenido determinísticamente del codebook. De este modo se mantiene separada la traducción lingüística de la variable de la interpretación documental de su valor.

La interfaz dispone además de una explicación ampliada basada directamente en SHAP. Para el registro evaluado se muestran cinco factores por indicador —hasta tres contribuciones positivas y dos negativas—, incluidos los indicadores que no superan su umbral. Las denominaciones `Factor de riesgo para la predicción` y `Factor protector para la predicción` se limitan estrictamente a la dirección de la contribución sobre la predicción y no implican causalidad ni interpretación clínica.

El informe final supera los 15 guardrails deterministas. Además, seis alteraciones adversas introducidas de forma controlada —diagnóstico, causalidad, cifra predictiva, fuente no autorizada, factor no autorizado y dirección incoherente— son correctamente rechazadas.

**Tabla candidata:** resumen del contrato generativo, factores permitidos y validación de guardrails.

**Figura candidata:** esquema de separación entre contenido determinista, contenido generado y guardrails.

**Destino:** MEMORIA + ANEXO. La memoria recogerá el principio de generación restringida y la validación global; el prompt completo, traducciones, pruebas adversas y comprobaciones detalladas se reservarán para el anexo.

# 8. Productivización con Flask y Plotly

La solución validada se expone mediante una capa sencilla de aplicación.

La aplicación no implementa lógica predictiva propia. Flask actúa como punto de entrada y utiliza el mismo flujo LangGraph, las mismas tools, los mismos contratos Pydantic y los mismos guardrails ya comprobados.

La entrada de la aplicación se plantea mediante JSON estructurado. No se construye un formulario con 813 campos, ya que resultaría poco práctico y no aportaría valor al prototipo.

La presentación combina dos tipos de información claramente diferenciados: los valores predictivos deterministas —probabilidad, umbral, clasificación, familia y modelo— y el informe preventivo validado generado por Mistral.

## 8.1. Salida integrada para la aplicación

Antes de construir la API se consolida la respuesta que recibirá la aplicación a partir de la ejecución ya validada.

Esta estructura conserva las cuatro predicciones, los factores explicativos de los cuatro indicadores, el informe preventivo y la trazabilidad del flujo. Los códigos internos de las variables se mantienen únicamente para trazabilidad. La presentación destinada al usuario utiliza descripciones comprensibles, el significado de la respuesta registrada y, cuando procede, una aclaración sobre el carácter recodificado o combinado de la variable.

In [34]:
# =============================================================================
# SALIDA INTEGRADA PARA LA APLICACIÓN
# =============================================================================
print('\nSALIDA INTEGRADA PARA LA APLICACIÓN')

respuesta_demo_aplicacion = preparar_respuesta_aplicacion(
    estado_informe_flujo, informe_validado, detalle_explicabilidad_generado,
    modelo_mistral, version_prompt_informe
)

predicciones_aplicacion = pd.DataFrame(respuesta_demo_aplicacion['predicciones'])
factores_aplicacion = preparar_tabla_factores_informe(informe_validado)

tabla_predicciones = predicciones_aplicacion[[
    'descripcion', 'probabilidad', 'umbral', 'clasificacion', 'familia', 'modelo'
]].rename(columns=renombrado_comun).round({'Probabilidad': 6, 'Umbral': 3})

print('\nPREDICCIONES DE LA APLICACIÓN')
display(tabla_predicciones)

tabla_factores = factores_aplicacion.rename(columns=renombrado_comun)

print('\nFACTORES DEL INFORME POR INDICADOR')
mostrar_tabla_completa(tabla_factores)

print('\nINFORME PREVENTIVO INTEGRADO\n')
print(texto_informe_validado)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
factores_por_objetivo_aplicacion = factores_aplicacion.groupby('variable_objetivo').size()

comprobaciones_respuesta_aplicacion = pd.DataFrame({
    'comprobacion': [
        'La respuesta integrada devuelve estado OK',
        'Se conservan las cuatro predicciones',
        'Se conservan los cuatro indicadores',
        'Se presentan tres factores por indicador',
        'La aplicación contiene 12 factores explicativos',
        'El informe mantiene los siete campos contractuales',
        'El informe integrado ha superado todos los guardrails',
        'El registro original de 813 variables no forma parte de la respuesta',
        'La respuesta contiene la explicación ampliada de los cuatro indicadores'
    ],
    'resultado': [
        respuesta_demo_aplicacion['estado'] == 'OK',
        len(predicciones_aplicacion) == 4,
        set(predicciones_aplicacion['variable_objetivo']) == set(targets),
        factores_por_objetivo_aplicacion.eq(3).all(),
        len(factores_aplicacion) == 12,
        len(InformePreventivo.model_fields) == 7,
        guardrails_informe['resultado'].all(),
        'registro' not in respuesta_demo_aplicacion,
        set(factor['variable_objetivo']
            for factor in respuesta_demo_aplicacion['detalle_explicabilidad']) == set(targets)
    ]
})

tabla_comprobaciones = comprobaciones_respuesta_aplicacion.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LA RESPUESTA INTEGRADA')
display(tabla_comprobaciones)

if not comprobaciones_respuesta_aplicacion['resultado'].all():
    raise ValueError('La respuesta integrada de la aplicación no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_json(respuesta_demo_aplicacion, '53_respuesta_demo_aplicacion.json')
_ = guardar_csv(predicciones_aplicacion, '54_predicciones_aplicacion')
_ = guardar_csv(factores_aplicacion, '55_factores_aplicacion')
_ = guardar_csv(comprobaciones_respuesta_aplicacion, '56_comprobaciones_respuesta_aplicacion')

print('\nRespuesta integrada preparada correctamente.')


SALIDA INTEGRADA PARA LA APLICACIÓN

PREDICCIONES DE LA APLICACIÓN


,Descripción,Probabilidad,Umbral,Clasificación,Familia,Modelo
0,Episodio depresivo mayor,0.826969,0.500,1,ML,XGBoost
1,Ideación suicida,0.565728,0.509,1,DL,Ensemble — ramas 32 + sin BatchNormalization (...
2,Planificación suicida,0.525724,0.536,0,DL,Ensemble — ramas 32 + sin BatchNormalization (...
3,Intento suicida,0.524104,0.502,1,DL,Ensemble — ramas 32 + sin BatchNormalization (...



FACTORES DEL INFORME POR INDICADOR


,Variable objetivo,Indicador,Tipo de factor,Variable original,Descripción,Respuesta observada,Aclaración de la variable,Posición
0,IRAMDEYR,Episodio depresivo mayor,Factor de riesgo para la predicción,CAMHPROB2,Percepción de haber tenido alguna vez un problema de salud mental,Sí,NSDUH obtiene esta variable recodificando en Sí/No la respuesta original sobre si la persona considera haber tenido alguna vez un problema de salud mental; no corresponde a una pregunta independiente.,1
1,IRAMDEYR,Episodio depresivo mayor,Factor protector para la predicción,RCVYMHPRB,Percepción de recuperación de un problema de salud mental,Sí,NSDUH obtiene esta variable combinando la respuesta sobre haber tenido un problema de salud mental con la respuesta sobre recuperación; no corresponde a una pregunta independiente.,2
2,IRAMDEYR,Episodio depresivo mayor,Factor de riesgo para la predicción,LVLDIFMEM2,Nivel de dificultad para recordar o concentrarse,Alguna dificultad,NaN,3
3,IRSUICTHNK,Ideación suicida,Factor de riesgo para la predicción,CAMHPROB,Percepción de haber tenido alguna vez un problema de salud mental,Sí,NaN,1
4,IRSUICTHNK,Ideación suicida,Factor de riesgo para la predicción,CAMHPROB2,Percepción de haber tenido alguna vez un problema de salud mental,Sí,NSDUH obtiene esta variable recodificando en Sí/No la respuesta original sobre si la persona considera haber tenido alguna vez un problema de salud mental; no corresponde a una pregunta independiente.,2
5,IRSUICTHNK,Ideación suicida,Factor de riesgo para la predicción,LVLDIFCARE2,Nivel de dificultad con el autocuidado,Alguna dificultad,NaN,3
6,IRSUIPLANYR,Planificación suicida,Factor de riesgo para la predicción,CAMHPROB,Percepción de haber tenido alguna vez un problema de salud mental,Sí,NaN,1
7,IRSUIPLANYR,Planificación suicida,Factor de riesgo para la predicción,COPDAGE,Edad en la que se diagnosticó por primera vez la EPOC,44 años,NaN,2
8,IRSUIPLANYR,Planificación suicida,Factor de riesgo para la predicción,LVLDIFCARE2,Nivel de dificultad con el autocuidado,Alguna dificultad,NaN,3
9,IRSUITRYYR,Intento suicida,Factor de riesgo para la predicción,CAMHPROB,Percepción de haber tenido alguna vez un problema de salud mental,Sí,NaN,1



INFORME PREVENTIVO INTEGRADO

RESUMEN
En esta evaluación, tres de los cuatro indicadores evaluados superan sus umbrales validados: episodio depresivo mayor, ideación suicida e intento suicida. El indicador de planificación suicida no supera su umbral validado.

INTERPRETACIÓN
Se evaluaron cuatro indicadores: episodio depresivo mayor, ideación suicida, planificación suicida e intento suicida. El indicador de episodio depresivo mayor supera su umbral validado. El indicador de ideación suicida supera su umbral validado. El indicador de planificación suicida no supera su umbral validado. El indicador de intento suicida supera su umbral validado. Este resultado señala la conveniencia de una revisión preventiva adicional.

FACTORES RELEVANTES POR INDICADOR

Episodio depresivo mayor
Factores de riesgo para la predicción:
- Percepción de haber tenido alguna vez un problema de salud mental
- Nivel de dificultad para recordar o concentrarse
Factores protectores para la predicción:
- Percepción 

,Comprobación,Resultado
0,La respuesta integrada devuelve estado OK,True
1,Se conservan las cuatro predicciones,True
2,Se conservan los cuatro indicadores,True
3,Se presentan tres factores por indicador,True
4,La aplicación contiene 12 factores explicativos,True
5,El informe mantiene los siete campos contractu...,True
6,El informe integrado ha superado todos los gua...,True
7,El registro original de 813 variables no forma...,True
8,La respuesta contiene la explicación ampliada ...,True



Respuesta integrada preparada correctamente.


### Resultados

La salida integrada reúne correctamente las cuatro predicciones, los 12 factores contractuales, el informe preventivo, la explicación ampliada y la trazabilidad necesaria para la aplicación.

Para el registro de referencia se mantienen las probabilidades y umbrales ya validados. Episodio depresivo mayor, ideación suicida e intento suicida superan sus respectivos umbrales, mientras que planificación suicida permanece por debajo del suyo.

Los 12 factores principales se presentan mediante descripciones comprensibles, incorporan el significado de la respuesta registrada y conservan internamente la variable original para trazabilidad. Las variables recodificadas o combinadas pueden incluir además una aclaración breve sobre su procedencia documental.

Las nueve comprobaciones de integración resultan correctas. La salida mantiene los cuatro indicadores, tres factores principales por indicador, el contrato completo del informe, la validación de guardrails y la explicación ampliada destinada a la interfaz.

## 8.2. Visualización con Plotly

La visualización principal compara, para cada indicador, la probabilidad estimada y el umbral validado utilizado para obtener la clasificación.

Los valores proceden directamente de la salida predictiva y permanecen separados del informe generativo. La representación utiliza una gama cromática turquesa coherente con la interfaz final, manteniendo el mismo criterio visual en toda la capa de productivización.

El gráfico permite identificar de forma inmediata la posición relativa entre probabilidad y umbral para los cuatro indicadores, incluidos aquellos cuya probabilidad no alcanza el punto de clasificación validado.

In [35]:
# =============================================================================
# VISUALIZACIÓN CON PLOTLY
# =============================================================================
print('\nVISUALIZACIÓN CON PLOTLY')

figura_predicciones = crear_figura_predicciones(respuesta_demo_aplicacion['predicciones'])

figura_predicciones_html = figura_predicciones.to_html(
    include_plotlyjs='cdn', full_html=False, default_width='100%', default_height='500px'
)

vista_figura_predicciones = HTML(figura_predicciones_html)

print('\nGRÁFICO DE PROBABILIDAD Y UMBRAL')
display(vista_figura_predicciones)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_visualizacion = pd.DataFrame({
    'comprobacion': [
        'La visualización contiene los cuatro indicadores',
        'Se representan probabilidad y umbral',
        'Todos los valores representados están entre 0 y 1',
        'El fragmento HTML de la visualización está disponible'
    ],
    'resultado': [
        predicciones_aplicacion['descripcion'].nunique() == 4,
        {'probabilidad', 'umbral'}.issubset(predicciones_aplicacion),
        predicciones_aplicacion[['probabilidad', 'umbral']].apply(
            lambda columna: columna.between(0, 1).all()).all(),
        bool(figura_predicciones_html)
    ]
})

tabla_comprobaciones = comprobaciones_visualizacion.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LA VISUALIZACIÓN')
display(tabla_comprobaciones)

if not comprobaciones_visualizacion['resultado'].all():
    raise ValueError('La visualización Plotly no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
ruta_figura_predicciones = ruta_figuras / '01_predicciones_umbral_aplicacion.html'
figura_predicciones.write_html(ruta_figura_predicciones, include_plotlyjs='cdn', full_html=True)

_ = guardar_csv(comprobaciones_visualizacion, '57_comprobaciones_visualizacion_plotly')

print('\nVisualización Plotly preparada correctamente.')


VISUALIZACIÓN CON PLOTLY



GRÁFICO DE PROBABILIDAD Y UMBRAL



COMPROBACIONES DE LA VISUALIZACIÓN


,Comprobación,Resultado
0,La visualización contiene los cuatro indicadores,True
1,Se representan probabilidad y umbral,True
2,Todos los valores representados están entre 0 y 1,True
3,El fragmento HTML de la visualización está dis...,True



Visualización Plotly preparada correctamente.


### Resultados

La visualización Plotly representa conjuntamente los cuatro indicadores y compara para cada uno la probabilidad estimada con su umbral validado.

La figura utiliza la identidad visual turquesa definida para la aplicación y conserva los valores predictivos como información determinista, independiente de la narración generada.

Las cuatro comprobaciones realizadas resultan correctas: aparecen los cuatro indicadores, se representan probabilidad y umbral, todos los valores permanecen en el intervalo esperado y el fragmento HTML queda disponible para integrarse en Flask.

## 8.3. API e interfaz Flask

Flask expone la capacidad validada mediante una API sencilla versionada.

Se definen endpoints diferenciados para comprobar el estado del servicio, consultar el esquema de entrada, obtener predicciones y solicitar el informe completo. El endpoint de informe activa las cuatro explicaciones SHAP, RAG, Mistral y guardrails utilizando el mismo flujo ya validado.

La ruta principal presenta una demostración HTML basada en el caso previamente ejecutado. Esta interfaz no sustituye la API: su finalidad es mostrar de forma comprensible las probabilidades, umbrales, clasificaciones, factores explicativos e informe final.

No se inicia un servidor persistente desde el notebook. La aplicación se valida localmente mediante el cliente de pruebas de Flask.

In [36]:
# =============================================================================
# API E INTERFAZ FLASK
# =============================================================================
print('\nAPI E INTERFAZ FLASK')

# -----------------------------------------------------------------------------
# CONFIGURACIÓN DE LA APLICACIÓN
# -----------------------------------------------------------------------------
app = Flask('TFM_NSDUH_2024')

app.secret_key = os.getenv('FLASK_SECRET_KEY', 'TFM_NSDUH_2024_DEMO')

# Credencial exclusivamente demostrativa para el prototipo
app_api_key = 'CAM'

# -----------------------------------------------------------------------------
# INTERFAZ DE RESULTADOS
# -----------------------------------------------------------------------------
factores_interfaz = agrupar_factores_interfaz(respuesta_demo_aplicacion['factores'])

detalle_interfaz = agrupar_detalle_interfaz(respuesta_demo_aplicacion['detalle_explicabilidad'])


# -----------------------------------------------------------------------------
# RUTAS DE INTERFAZ
# -----------------------------------------------------------------------------
@app.get('/')
def interfaz_principal():
    return render_template_string(
        plantilla_interfaz, predicciones=respuesta_demo_aplicacion['predicciones'],
        factores=factores_interfaz, detalle=detalle_interfaz,
        informe=informe_validado.model_dump(mode='json'),
        grafico=figura_predicciones_html, nota_factores=frases_informe['nota_factores'],
        advertencia=frases_informe['advertencia_uso']
    )


@app.route('/acceso', methods=['GET', 'POST'])
def acceso_usuario():
    autorizado = bool(session.get('autorizado'))

    if request.method == 'GET':
        return render_template_string(plantilla_acceso, error=None, autorizado=autorizado)

    if request.form.get('accion') == 'finalizar':
        session.clear()

        temporizador = globals().get('temporizador_servidor_demo')
        if temporizador is not None:
            temporizador.cancel()

        servidor = globals().get('servidor_demo')
        if servidor is not None:
            cierre = threading.Timer(0.5, detener_servidor_demo, args=(servidor,))
            cierre.daemon = True
            cierre.start()

        return plantilla_salida

    archivo = request.files.get('archivo')

    if not autorizado:
        clave = request.form.get('api_key', '')

        if not hmac.compare_digest(clave, app_api_key):
            return render_template_string(
                plantilla_acceso, error='Credencial no válida.', autorizado=False
            ), 401

        session['autorizado'] = True
        autorizado = True

        if archivo is None:
            return render_template_string(plantilla_acceso, error=None, autorizado=True)

    if archivo is None or not archivo.filename.lower().endswith('.csv'):
        return render_template_string(
            plantilla_acceso, error='Debe seleccionar un archivo CSV válido.', autorizado=True
        ), 400

    try:
        registro = cargar_registro_csv(archivo)
        respuesta = ejecutar_informe_aplicacion(registro)
        _ = guardar_informe_aplicacion(respuesta, archivo.filename)

        factores = agrupar_factores_interfaz(respuesta['factores'])
        detalle = agrupar_detalle_interfaz(respuesta['detalle_explicabilidad'])

        grafico = crear_figura_predicciones(
            respuesta['predicciones']
        ).to_html(include_plotlyjs='cdn', full_html=False)

        return render_template_string(
            plantilla_interfaz, predicciones=respuesta['predicciones'], factores=factores,
            detalle=detalle, informe=respuesta['informe'], grafico=grafico,
            nota_factores=frases_informe['nota_factores'],
            advertencia=frases_informe['advertencia_uso']
        )

    except (TypeError, ValidationError, ValueError) as error:
        return render_template_string(plantilla_acceso, error=str(error), autorizado=True), 400

    except RuntimeError:
        return render_template_string(
            plantilla_acceso, error='No ha sido posible completar el informe.', autorizado=True
        ), 500


@app.get('/informes')
def informes_usuario():
    if not session.get('autorizado'):
        return redirect(url_for('acceso_usuario'))

    historial = cargar_historial_informes()

    return render_template_string(plantilla_historial, informes=historial)


@app.get('/informe/<id_informe>')
def informe_guardado(id_informe):
    if not session.get('autorizado'):
        return redirect(url_for('acceso_usuario'))

    try:
        datos = cargar_informe_aplicacion(id_informe)
        respuesta = datos['respuesta']

        factores = agrupar_factores_interfaz(respuesta['factores'])
        detalle = agrupar_detalle_interfaz(respuesta.get('detalle_explicabilidad', []))

        grafico = crear_figura_predicciones(
            respuesta['predicciones']
        ).to_html(include_plotlyjs='cdn', full_html=False)

        return render_template_string(
            plantilla_interfaz, predicciones=respuesta['predicciones'], factores=factores,
            detalle=detalle, informe=respuesta['informe'], grafico=grafico,
            nota_factores=frases_informe['nota_factores'],
            advertencia=frases_informe['advertencia_uso']
        )

    except (ValueError, FileNotFoundError) as error:
        return str(error), 404


# -----------------------------------------------------------------------------
# ENDPOINTS DE SERVICIO
# -----------------------------------------------------------------------------
@app.get('/api/v1/health')
def api_health():
    return jsonify({
        'estado': 'OK',
        'version_predictiva': configuracion_productivizacion['version'],
        'modelo_generativo': modelo_mistral,
        'version_prompt': version_prompt_informe
    })


@app.get('/api/v1/esquema')
def api_esquema():
    return jsonify({
        'variables_entrada': len(vars_predictoras_modelado_dl),
        'variables_objetivo': targets,
        'campos_prediccion': list(ResultadoPredictivo.model_fields),
        'campos_informe': list(InformePreventivo.model_fields)
    })


@app.post('/api/v1/predict')
def api_predict():
    datos = request.get_json(silent=True) or {}

    if 'registro' not in datos:
        return jsonify({
            'estado': 'ERROR',
            'mensaje': 'Debe proporcionarse un registro.'
        }), 400

    try:
        solicitar_explicacion = bool(datos.get('solicitar_explicacion', False))
        objetivos = datos.get('objetivos_explicacion', targets if solicitar_explicacion else [])

        estado = ejecutar_flujo_aplicacion(
            datos['registro'], solicitar_explicacion=solicitar_explicacion,
            objetivos_explicacion=objetivos,
            solicitar_documentacion=bool(datos.get('solicitar_documentacion', False))
        )

        respuesta = {
            'estado': 'OK',
            'predicciones': estado['predicciones'],
            'explicaciones': estado.get('explicaciones', []),
            'trazabilidad': estado.get('trazabilidad', [])
        }

        respuesta = json.loads(json.dumps(respuesta, ensure_ascii=False, default=convertir_json))

        return jsonify(respuesta)

    except (ValidationError, ValueError) as error:
        return jsonify({
            'estado': 'ERROR',
            'mensaje': str(error)
        }), 400

    except RuntimeError as error:
        return jsonify({
            'estado': 'ERROR',
            'mensaje': str(error)
        }), 500


@app.post('/api/v1/report')
def api_report():
    datos = request.get_json(silent=True) or {}

    if 'registro' not in datos:
        return jsonify({
            'estado': 'ERROR',
            'mensaje': 'Debe proporcionarse un registro.'
        }), 400

    try:
        respuesta = ejecutar_informe_aplicacion(datos['registro'])
        respuesta = json.loads(json.dumps(respuesta, ensure_ascii=False, default=convertir_json))

        return jsonify(respuesta)

    except (ValidationError, ValueError) as error:
        return jsonify({
            'estado': 'ERROR',
            'mensaje': str(error)
        }), 400

    except RuntimeError as error:
        return jsonify({
            'estado': 'ERROR',
            'mensaje': str(error)
        }), 500


# -----------------------------------------------------------------------------
# ENDPOINTS PROTEGIDOS
# -----------------------------------------------------------------------------
@app.get('/api/v1/access')
def api_access():
    if not validar_acceso_api(app_api_key):
        return jsonify({
            'estado': 'ERROR',
            'mensaje': 'Acceso no autorizado.'
        }), 401

    return jsonify({
        'estado': 'OK',
        'acceso': 'autorizado'
    })


@app.post('/api/v1/report-file')
def api_report_file():
    if not validar_acceso_api(app_api_key):
        return jsonify({
            'estado': 'ERROR',
            'mensaje': 'Acceso no autorizado.'
        }), 401

    archivo = request.files.get('archivo')

    if archivo is None or not archivo.filename.lower().endswith('.csv'):
        return jsonify({
            'estado': 'ERROR',
            'mensaje': 'Debe proporcionarse un archivo CSV.'
        }), 400

    try:
        registro = cargar_registro_csv(archivo)

        respuesta = ejecutar_informe_aplicacion(registro)
        respuesta = json.loads(json.dumps(respuesta, ensure_ascii=False, default=convertir_json))

        return jsonify(respuesta)

    except (TypeError, ValidationError, ValueError) as error:
        return jsonify({
            'estado': 'ERROR',
            'mensaje': str(error)
        }), 400

    except RuntimeError as error:
        return jsonify({
            'estado': 'ERROR',
            'mensaje': str(error)
        }), 500


# -----------------------------------------------------------------------------
# RESUMEN
# -----------------------------------------------------------------------------
endpoints_flask = pd.DataFrame([
    {
        'ruta': regla.rule,
        'metodos': ', '.join(sorted(
            metodo for metodo in regla.methods if metodo not in {'HEAD', 'OPTIONS'}
        ))
    }
    for regla in app.url_map.iter_rules() if regla.endpoint != 'static'
])

tabla_endpoints = endpoints_flask.rename(columns=renombrado_comun)

print('\nENDPOINTS DE LA APLICACIÓN')
display(tabla_endpoints)

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(endpoints_flask, '58_endpoints_flask')

print('\nAPI Flask preparada correctamente.')


API E INTERFAZ FLASK

ENDPOINTS DE LA APLICACIÓN


,Ruta,Métodos
0,/,GET
1,/acceso,"GET, POST"
2,/informes,GET
3,/informe/<id_informe>,GET
4,/api/v1/health,GET
5,/api/v1/esquema,GET
6,/api/v1/predict,POST
7,/api/v1/report,POST
8,/api/v1/access,GET
9,/api/v1/report-file,POST



API Flask preparada correctamente.


### Resultados

La aplicación Flask queda configurada con 10 endpoints que cubren navegación web, acceso autorizado, historial de informes, consulta de informes almacenados y consumo programático de las capacidades predictivas.

La API incluye endpoints de estado, esquema, predicción e informe completo, además del acceso protegido mediante archivo. Las rutas web permiten acceder a la pantalla de autorización, consultar el historial y abrir posteriormente un informe persistido.

La aplicación reutiliza los contratos, tools, flujo LangGraph y guardrails ya validados y no incorpora una segunda implementación de la lógica predictiva.

La API y la interfaz quedan preparadas sin iniciar todavía un servidor persistente desde el notebook.

## 8.4. Validación local de la aplicación

La aplicación se valida mediante el cliente de pruebas de Flask sin iniciar un servidor persistente desde JupyterLab.

Se comprueban la página principal, el estado del servicio, el esquema de entrada y el rechazo controlado de una solicitud incompleta. También se valida que la interfaz incorpore los cuatro indicadores, los doce factores traducidos, los estados comprensibles de clasificación y la identidad visual definida para el prototipo.

El endpoint de informe permanece disponible, pero no se ejecuta nuevamente para evitar repetir las cuatro explicaciones SHAP y una llamada adicional a Mistral. La ejecución completa a través de la API se reserva para la evaluación extremo a extremo.

In [37]:
# =============================================================================
# VALIDACIÓN LOCAL DE LA APLICACIÓN
# =============================================================================
print('\nVALIDACIÓN LOCAL DE LA APLICACIÓN')

cliente_flask = app.test_client()

respuesta_historial_sin_acceso = cliente_flask.get('/informes', follow_redirects=False)

with cliente_flask.session_transaction() as sesion:
    sesion['autorizado'] = True

respuesta_interfaz = cliente_flask.get('/')

respuesta_acceso = cliente_flask.get('/acceso')

respuesta_historial = cliente_flask.get('/informes')

html_acceso = respuesta_acceso.get_data(as_text=True)

respuesta_health = cliente_flask.get('/api/v1/health')

respuesta_esquema = cliente_flask.get('/api/v1/esquema')

respuesta_invalida = cliente_flask.post('/api/v1/predict', json={})

health_flask = respuesta_health.get_json()
esquema_flask = respuesta_esquema.get_json()

html_respuesta = respuesta_interfaz.get_data(as_text=True)

resumen_respuestas_flask = pd.DataFrame({
    'solicitud': [
        'GET /', 'GET /api/v1/health', 'GET /api/v1/esquema',
        'POST /api/v1/predict sin registro'
    ],
    'codigo_http': [
        respuesta_interfaz.status_code, respuesta_health.status_code,
        respuesta_esquema.status_code, respuesta_invalida.status_code
    ]
})

tabla_resumen = resumen_respuestas_flask.rename(columns=renombrado_comun)

print('\nRESPUESTAS DEL CLIENTE FLASK')
display(tabla_resumen)

print('\nDEMOSTRACIÓN DE LA INTERFAZ')
mostrar_html_aislado(html_respuesta)

rutas_disponibles = set(endpoints_flask['ruta'])

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_flask = pd.DataFrame({
    'comprobacion': [
        'La página principal responde correctamente',
        'El endpoint health devuelve estado OK',
        'El esquema declara 813 variables de entrada',
        'La entrada incompleta devuelve HTTP 400',
        'Los endpoints predict y report están disponibles',
        'La interfaz contiene los cuatro indicadores',
        'La interfaz contiene los 12 factores traducidos',
        'Se muestran estados comprensibles de clasificación',
        'El historial requiere una sesión autorizada',
        'Las credenciales no aparecen en las respuestas',
        'La pantalla de acceso responde correctamente',
        'La pantalla de historial responde correctamente',
        'La pantalla de acceso contiene el estado de procesamiento',
        'Los endpoints de historial están disponibles',
        'La interfaz utiliza la identidad visual elegida',
        'La interfaz incorpora la explicación ampliada'
    ],
    'resultado': [
        respuesta_interfaz.status_code == 200,
        respuesta_health.status_code == 200 and health_flask['estado'] == 'OK',
        respuesta_esquema.status_code == 200 and esquema_flask['variables_entrada'] == 813,
        respuesta_invalida.status_code == 400,
        {'/api/v1/predict', '/api/v1/report'}.issubset(rutas_disponibles),
        all(descripcion in html_respuesta for descripcion in titulos_targets.values()),
        all(factor.descripcion in html_respuesta
            for factor in informe_validado.factores_relevantes),
        'SUPERA EL UMBRAL' in html_respuesta and 'NO SUPERA EL UMBRAL' in html_respuesta,
        respuesta_historial_sin_acceso.status_code == 302,
        'MISTRAL_API_KEY' not in html_respuesta and 'HF_TOKEN' not in html_respuesta,
        respuesta_acceso.status_code == 200,
        respuesta_historial.status_code == 200,
        'Generando informe preventivo' in html_acceso,
        {'/informes', '/informe/<id_informe>'}.issubset(rutas_disponibles),
        '#81d8d0' in html_respuesta.lower(),
        'Explicabilidad ampliada' in html_respuesta
    ]
})

tabla_comprobaciones = comprobaciones_flask.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE LA APLICACIÓN')
display(tabla_comprobaciones)

if not comprobaciones_flask['resultado'].all():
    raise ValueError('La validación local de la aplicación Flask no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_respuestas_flask, '59_resumen_respuestas_flask')
_ = guardar_csv(comprobaciones_flask, '60_comprobaciones_aplicacion_flask')

print('\nAplicación Flask validada correctamente.')


VALIDACIÓN LOCAL DE LA APLICACIÓN

RESPUESTAS DEL CLIENTE FLASK


,Solicitud,Código HTTP
0,GET /,200
1,GET /api/v1/health,200
2,GET /api/v1/esquema,200
3,POST /api/v1/predict sin registro,400



DEMOSTRACIÓN DE LA INTERFAZ


/home/cam/miniconda3/envs/TFM_Agentes/lib/python3.11/site-packages/IPython/core/display.py:452: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")



COMPROBACIONES DE LA APLICACIÓN


,Comprobación,Resultado
0,La página principal responde correctamente,True
1,El endpoint health devuelve estado OK,True
2,El esquema declara 813 variables de entrada,True
3,La entrada incompleta devuelve HTTP 400,True
4,Los endpoints predict y report están disponibles,True
5,La interfaz contiene los cuatro indicadores,True
6,La interfaz contiene los 12 factores traducidos,True
7,Se muestran estados comprensibles de clasifica...,True
8,El historial requiere una sesión autorizada,True
9,Las credenciales no aparecen en las respuestas,True



Aplicación Flask validada correctamente.


### Resultados

La validación mediante el cliente de pruebas de Flask confirma el funcionamiento de las rutas principales sin necesidad de iniciar un servidor persistente.

La página principal, el endpoint de estado y el esquema responden correctamente, mientras que una solicitud predictiva sin registro es rechazada mediante HTTP 400. El esquema publicado mantiene las 813 variables originales de entrada.

La interfaz contiene los cuatro indicadores, los 12 factores principales traducidos, estados de clasificación comprensibles, la explicación ampliada y la identidad visual turquesa definida para el prototipo.

También se valida la protección del historial, la ausencia de credenciales en las respuestas, la pantalla de acceso y la disponibilidad de las rutas destinadas a recuperar informes.

Las 16 comprobaciones realizadas resultan correctas.

## 8.5. Acceso autorizado y escenario de despliegue

El prototipo desarrollado se ejecuta localmente y no se expone directamente a Internet. No obstante, la arquitectura permite representar el proceso que seguiría una persona autorizada si el sistema se desplegara posteriormente dentro de una organización.

En un despliegue real, el acceso debería realizarse mediante HTTPS y una capa de autenticación y autorización situada antes de Flask. En un despliegue real sería preferible utilizar un proveedor corporativo de identidad mediante OAuth 2.0/OpenID Connect, Microsoft Entra ID, Keycloak u otra solución equivalente.

Para demostrar el principio de control de acceso sin ampliar innecesariamente el alcance del TFM, se utiliza la credencial ficticia `CAM`, definida exclusivamente para este prototipo académico. La aplicación dispone tanto de una ruta web `/acceso`, destinada a una persona autorizada, como de endpoints protegidos para representar la integración programática con otros sistemas.

La entrada tampoco requiere completar manualmente las 813 variables. Se generan tres archivos CSV independientes, cada uno formado por un registro completo y las 813 variables originales requeridas por el sistema. Los tres registros se recuperan de TEST exclusivamente con finalidad demostrativa, se excluye el registro utilizado previamente como referencia del notebook y la selección se realiza de forma determinista entre perfiles de clasificación diferentes.

De este modo se dispone de varios archivos plausibles para una demostración de usuario sin reutilizar el caso empleado durante el desarrollo y la evaluación del bridge. Tras superar la identificación, cada CSV puede seleccionarse desde el panel del usuario autorizado y pasa nuevamente por la misma validación estructural que una entrada recibida mediante JSON.

Los tres archivos se utilizan para ejecutar tres demostraciones completas e independientes. Cada ejecución genera un informe persistido y permite comprobar posteriormente que el historial puede consultarse sin repetir predicción, SHAP, recuperación documental ni generación.

<div style="max-width:1000px;margin:24px auto;font-family:Arial,sans-serif;text-align:center;color:#24383c;">

<!-- ACCESO -->
<div style="background:#176b70;color:white;padding:14px 20px;border-radius:12px;font-weight:700;font-size:16px;">
PERSONA AUTORIZADA
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="background:#dcf7f4;border:2px solid #81d8d0;padding:13px 18px;border-radius:11px;">
<strong>IDENTIFICACIÓN</strong><br>
<code>/acceso</code><br>
<span style="font-size:13px;color:#52656a;">Credencial de demostración: <strong>CAM</strong></span>
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<!-- PANEL -->
<div style="background:#0e4f55;color:white;padding:13px 18px;border-radius:11px;font-weight:700;">
PANEL PRINCIPAL
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="display:flex;gap:14px;justify-content:center;align-items:stretch;flex-wrap:wrap;">

  <div style="flex:1;min-width:210px;background:#f4fcfb;border:2px solid #2b9f99;
              padding:14px;border-radius:11px;">
    <strong>GENERAR INFORME</strong><br>
    <span style="font-size:13px;color:#52656a;">Selección de archivo CSV</span>
  </div>

  <div style="flex:1;min-width:210px;background:#f4fcfb;border:2px solid #81d8d0;
              padding:14px;border-radius:11px;">
    <strong>INFORMES GUARDADOS</strong><br>
    <code>/informes</code>
  </div>

  <div style="flex:1;min-width:210px;background:#f7f7f7;border:1px solid #9aabad;
              padding:14px;border-radius:11px;">
    <strong>SALIR Y CERRAR</strong><br>
    <span style="font-size:13px;color:#52656a;">Finalización de la demostración</span>
  </div>

</div>

<!-- FLUJO DE GENERACIÓN -->
<div style="margin-top:7px;font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="background:#dcf7f4;border:2px solid #81d8d0;padding:12px 18px;border-radius:11px;">
<strong>VALIDACIÓN DEL CSV</strong><br>
<span style="font-size:13px;color:#52656a;">1 registro · 813 variables</span>
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="background:#176b70;color:white;padding:13px 18px;border-radius:11px;font-weight:700;">
LANGGRAPH
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="display:flex;gap:14px;justify-content:center;align-items:stretch;flex-wrap:wrap;">

  <div style="flex:1;min-width:180px;background:#f4fcfb;border:2px solid #2b9f99;
              padding:13px;border-radius:10px;">
    <strong>Predicción</strong><br>
    <span style="font-size:12px;color:#52656a;">4 indicadores</span>
  </div>

  <div style="flex:1;min-width:180px;background:#f4fcfb;border:2px solid #2b9f99;
              padding:13px;border-radius:10px;">
    <strong>SHAP</strong><br>
    <span style="font-size:12px;color:#52656a;">Explicabilidad</span>
  </div>

  <div style="flex:1;min-width:180px;background:#f4fcfb;border:2px solid #2b9f99;
              padding:13px;border-radius:10px;">
    <strong>RAG</strong><br>
    <span style="font-size:12px;color:#52656a;">Recuperación documental</span>
  </div>

</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="background:#dcf7f4;padding:12px 18px;border-radius:11px;">
<strong>CONTEXTO ESTRUCTURADO</strong>
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="background:#f4fcfb;border:2px solid #2b9f99;padding:12px 18px;border-radius:11px;">
<strong>MISTRAL</strong><br>
<span style="font-size:13px;color:#52656a;">Redacción generativa restringida</span>
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="background:#f4fcfb;border:2px solid #176b70;padding:12px 18px;border-radius:11px;">
<strong>GUARDRAILS DETERMINISTAS</strong>
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="background:#176b70;color:white;padding:14px 18px;border-radius:11px;font-weight:700;">
INFORME PREVENTIVO
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<!-- PERSISTENCIA -->
<div style="background:#dcf7f4;border:2px solid #81d8d0;padding:12px 18px;border-radius:11px;">
<strong>PERSISTENCIA CONTROLADA</strong><br>
<span style="font-size:13px;color:#52656a;">Sin volver a almacenar las 813 variables originales</span>
</div>

<div style="font-size:25px;color:#2b9f99;line-height:1.2;">↓</div>

<div style="display:flex;gap:14px;justify-content:center;align-items:center;flex-wrap:wrap;">

  <div style="background:#f4fcfb;border:2px solid #81d8d0;padding:12px 22px;border-radius:10px;">
    <strong>HISTORIAL</strong><br>
    <code>/informes</code>
  </div>

  <div style="font-size:25px;color:#2b9f99;">→</div>

  <div style="background:#f4fcfb;border:2px solid #81d8d0;padding:12px 22px;border-radius:10px;">
    <strong>INFORME GUARDADO</strong><br>
    <code>/informe/&lt;id_informe&gt;</code>
  </div>

</div>

</div>

Los informes persistidos conservan únicamente el identificador, fecha y hora, archivo de origen, respuesta validada, predicciones, factores, explicación ampliada, informe, trazabilidad, modelo generativo y versión del prompt. Las 813 variables originales utilizadas para la inferencia no se vuelven a almacenar en estos archivos.

Una aplicación externa podría realizar la misma operación mediante `POST /api/v1/report-file`, proporcionando una credencial autorizada y uno de los archivos CSV.

La credencial `CAM` no representa un mecanismo recomendado para producción. Su finalidad es hacer reproducible y visible el escenario de utilización. Un despliegue real debería incorporar HTTPS, autenticación corporativa, autorización por roles, gestión de sesiones o tokens, registro de accesos y políticas adecuadas de protección, minimización y conservación de los datos.

La demostración representa así dos modalidades de consumo de una misma arquitectura: interacción directa mediante interfaz web e integración programática mediante API protegida.

In [38]:
# =============================================================================
# ACCESO AUTORIZADO Y ESCENARIO DE DESPLIEGUE
# =============================================================================
print('\nACCESO AUTORIZADO Y ESCENARIO DE DESPLIEGUE')

# -----------------------------------------------------------------------------
# SELECCIÓN REPRODUCIBLE DE TRES REGISTROS DIFERENTES
# -----------------------------------------------------------------------------
perfiles_csv_demo = predicciones_finales_test.pivot(
    index='QUESTID2', columns='target', values='y_pred'
).reindex(columns=targets)

perfiles_csv_demo = perfiles_csv_demo.loc[
    perfiles_csv_demo.notna().all(axis=1) & (perfiles_csv_demo.index != questid2_referencia)
].astype(int)

perfiles_csv_demo['perfil_clasificacion'] = (
    perfiles_csv_demo[targets].astype(str).agg('-'.join, axis=1)
)

candidatos_csv_demo = (
    perfiles_csv_demo.reset_index().sort_values('QUESTID2').drop_duplicates('perfil_clasificacion')
)

if len(candidatos_csv_demo) < 3:
    raise ValueError('No existen tres perfiles de clasificación diferentes para la demostración.')

ids_csv_demo = candidatos_csv_demo.head(3)['QUESTID2'].astype(int).tolist()

# -----------------------------------------------------------------------------
# GENERACIÓN DE LOS TRES CSV
# -----------------------------------------------------------------------------
registros_csv_demo = {}
rutas_csv_demo = {}
filas_csv_demo = []

for numero, questid2 in enumerate(ids_csv_demo, start=1):
    respuesta_registro = ejecutar_bridge_predictivo(
        {'operacion': 'reference', 'QUESTID2': questid2}, ruta_conda, ruta_servicio_predictivo
    )

    if respuesta_registro.get('estado') != 'OK':
        raise RuntimeError(f'No se ha podido recuperar QUESTID2 {questid2}.')

    registro = validar_registro_entrada(
        respuesta_registro['registro'], vars_predictoras_modelado_dl
    )

    datos_csv = pd.DataFrame([registro], columns=vars_predictoras_modelado_dl)

    ruta_csv = guardar_csv(datos_csv, f'63_registro_entrada_demo_{numero:02d}.csv')

    registro_recuperado = cargar_registro_csv(ruta_csv)

    registros_csv_demo[questid2] = registro_recuperado
    rutas_csv_demo[questid2] = ruta_csv

    filas_csv_demo.append({
        'ejemplo': numero,
        'QUESTID2': questid2,
        'perfil_clasificacion': perfiles_csv_demo.loc[questid2, 'perfil_clasificacion'],
        'variables': len(registro_recuperado),
        'valores_ausentes': sum(valor is None for valor in registro_recuperado.values()),
        'archivo': ruta_csv.name
    })

resumen_csv_demo = pd.DataFrame(filas_csv_demo)

tabla_csv_demo = resumen_csv_demo.rename(columns=renombrado_comun)

print('\nARCHIVOS CSV DISPONIBLES PARA LA DEMOSTRACIÓN')
display(tabla_csv_demo)

# -----------------------------------------------------------------------------
# CASO DE USO SELECCIONADO
# -----------------------------------------------------------------------------
questid2_demo = ids_csv_demo[0]
ruta_csv_demo = rutas_csv_demo[questid2_demo]
registro_csv_demo = registros_csv_demo[questid2_demo]

caso_usuario_demo = pd.DataFrame({
    'elemento': [
        'Usuario',
        'Credencial',
        'QUESTID2 del ejemplo',
        'Archivo seleccionado',
        'Registros',
        'Variables'
    ],
    'valor': [
        'Usuario autorizado de demostración',
        'CAM',
        questid2_demo,
        ruta_csv_demo.name,
        1,
        len(registro_csv_demo)
    ]
})

tabla_demo = caso_usuario_demo.rename(columns=renombrado_comun)

print('\nCASO DE USO DEMOSTRATIVO')
display(tabla_demo)

# -----------------------------------------------------------------------------
# PANTALLA VISIBLE DE ACCESO
# -----------------------------------------------------------------------------
cliente_acceso = app.test_client()

respuesta_acceso_visible = cliente_acceso.get('/acceso')
html_acceso_visible = respuesta_acceso_visible.get_data(as_text=True)

print('\nPANTALLA DE ACCESO PARA USUARIO AUTORIZADO')
mostrar_html_aislado(html_acceso_visible)

respuesta_panel_visible = cliente_acceso.post(
    '/acceso',
    data={'api_key': 'CAM'}
)
html_panel_visible = respuesta_panel_visible.get_data(as_text=True)

print('\nPANEL DE USUARIO AUTORIZADO')
mostrar_html_aislado(html_panel_visible)

# -----------------------------------------------------------------------------
# VALIDACIÓN DEL CONTROL DE ACCESO
# -----------------------------------------------------------------------------
respuesta_sin_clave = cliente_flask.get('/api/v1/access')

respuesta_clave_incorrecta = cliente_flask.get(
    '/api/v1/access', headers={'X-API-Key': 'credencial-no-valida'}
)

respuesta_clave_correcta = cliente_flask.get('/api/v1/access', headers={'X-API-Key': app_api_key})

respuesta_csv_sin_clave = cliente_flask.post(
    '/api/v1/report-file', data={
        'archivo': (BytesIO(ruta_csv_demo.read_bytes()), ruta_csv_demo.name)
    }, content_type='multipart/form-data'
)

rutas_actuales = {
    regla.rule for regla in app.url_map.iter_rules() if regla.endpoint != 'static'
}

# -----------------------------------------------------------------------------
# DEMOSTRACIÓN COMPLETA DESDE LA INTERFAZ
# -----------------------------------------------------------------------------
print('\nGENERACIÓN DE LOS TRES INFORMES DE DEMOSTRACIÓN', flush=True)

ejecuciones_demo = []

for numero, questid2 in enumerate(ids_csv_demo, start=1):
    ruta_csv = rutas_csv_demo[questid2]

    print(f'[{numero}/3] {ruta_csv.name} — QUESTID2 {questid2}', flush=True)

    inicio = perf_counter()

    respuesta_demo = cliente_flask.post(
        '/acceso',
        data={'api_key': 'CAM', 'archivo': (BytesIO(ruta_csv.read_bytes()), ruta_csv.name)},
        content_type='multipart/form-data'
    )

    tiempo_s = perf_counter() - inicio

    ejecuciones_demo.append({
        'ejemplo': numero,
        'QUESTID2': questid2,
        'archivo': ruta_csv.name,
        'codigo_http': respuesta_demo.status_code,
        'tiempo_s': tiempo_s
    })

    if respuesta_demo.status_code != 200:
        raise RuntimeError(f'No se ha podido generar el informe {numero}.')

ejecuciones_informes_demo = pd.DataFrame(ejecuciones_demo)

informes_demo_guardados = [
    cargar_informe_aplicacion(f'demo_{numero:02d}') for numero in range(1, 4)
]

tabla_ejecuciones = ejecuciones_informes_demo.rename(
    columns=renombrado_comun
).round({'Tiempo (s)': 2})

print('\nINFORMES GENERADOS')
display(tabla_ejecuciones)

# -----------------------------------------------------------------------------
# HISTORIAL VISIBLE
# -----------------------------------------------------------------------------
respuesta_historial_demo = cliente_flask.get('/informes')
html_historial_demo = respuesta_historial_demo.get_data(as_text=True)

print('\nHISTORIAL DE INFORMES GENERADOS')
mostrar_html_aislado(html_historial_demo)

# -----------------------------------------------------------------------------
# APERTURA DE UN INFORME DESDE EL HISTORIAL
# -----------------------------------------------------------------------------
respuesta_informe_demo = cliente_flask.get('/informe/demo_02')
html_informe_demo = respuesta_informe_demo.get_data(as_text=True)

print('\nINFORME SELECCIONADO DESDE EL HISTORIAL')
mostrar_html_aislado(html_informe_demo)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_acceso_autorizado = pd.DataFrame({
    'comprobacion': [
        'Se generan tres archivos CSV diferentes',
        'Los tres registros son distintos del registro de referencia',
        'Los tres registros presentan perfiles de clasificación diferentes',
        'Cada CSV contiene exactamente un registro',
        'Cada CSV contiene las 813 variables esperadas',
        'Los tres CSV recuperados superan la validación de entrada',
        'Una solicitud sin credencial devuelve HTTP 401',
        'Una credencial incorrecta devuelve HTTP 401',
        'La credencial autorizada permite el acceso',
        'El endpoint protegido de informe está disponible',
        'La carga sin autorización se rechaza antes del modelado',
        'La pantalla de identificación responde correctamente',
        'La pantalla de identificación solicita una credencial protegida',
        'La pantalla identifica CAM como credencial de demostración',
        'Tras identificarse se muestran generación, historial y cierre',
        'Los tres CSV generan un informe correctamente',
        'Los tres informes de demostración quedan almacenados',
        'Los informes almacenados no contienen el registro original',
        'El historial visible contiene los tres informes',
        'Un informe puede recuperarse desde el historial',
        'El informe recuperado contiene los cuatro indicadores',
        'El informe recuperado contiene 12 factores principales',
        'El informe recuperado contiene explicación ampliada'
    ],
    'resultado': [
        len(rutas_csv_demo) == 3 and len(set(rutas_csv_demo.values())) == 3,
        all(questid2 != questid2_referencia for questid2 in ids_csv_demo),
        resumen_csv_demo['perfil_clasificacion'].nunique() == 3,
        all(len(pd.read_csv(ruta, encoding=encoding_csv)) == 1
            for ruta in rutas_csv_demo.values()),
        all(pd.read_csv(ruta, encoding=encoding_csv).shape[1] == 813
            for ruta in rutas_csv_demo.values()),
        all(len(registro) == 813 for registro in registros_csv_demo.values()),
        respuesta_sin_clave.status_code == 401,
        respuesta_clave_incorrecta.status_code == 401,
        respuesta_clave_correcta.status_code == 200
        and respuesta_clave_correcta.get_json()['acceso'] == 'autorizado',
        '/api/v1/report-file' in rutas_actuales,
        respuesta_csv_sin_clave.status_code == 401,
        respuesta_acceso_visible.status_code == 200,
        'type="password"' in html_acceso_visible,
        'Credencial del usuario autorizado:' in html_acceso_visible and
        '<strong>CAM</strong>' in html_acceso_visible,
        respuesta_panel_visible.status_code == 200 and 'type="file"' in html_panel_visible and
        'VER INFORMES GENERADOS' in html_panel_visible and
        'SALIR Y CERRAR DEMOSTRACIÓN' in html_panel_visible,
        ejecuciones_informes_demo['codigo_http'].eq(200).all(),
        all((ruta_informes / f'informe_demo_{numero:02d}.json').is_file()
            for numero in range(1, 4)),
        all('registro' not in datos['respuesta'] for datos in informes_demo_guardados),
        all(f'demo_{numero:02d}' in html_historial_demo for numero in range(1, 4)),
        respuesta_informe_demo.status_code == 200,
        all(descripcion in html_informe_demo for descripcion in titulos_targets.values()),
        len(cargar_informe_aplicacion('demo_02')['respuesta']['factores']) == 12,
        bool(cargar_informe_aplicacion('demo_02')['respuesta']['detalle_explicabilidad'])
    ]
})

tabla_comprobaciones = comprobaciones_acceso_autorizado.rename(
    columns=renombrado_comun
)

print('\nCOMPROBACIONES DEL ACCESO AUTORIZADO')
display(tabla_comprobaciones)

if not comprobaciones_acceso_autorizado['resultado'].all():
    raise ValueError('La demostración de acceso autorizado no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(comprobaciones_acceso_autorizado, '64_comprobaciones_acceso_autorizado')

print('\nAcceso autorizado y tres archivos CSV de ejemplo validados correctamente.')


ACCESO AUTORIZADO Y ESCENARIO DE DESPLIEGUE



ARCHIVOS CSV DISPONIBLES PARA LA DEMOSTRACIÓN


,Ejemplo,QUESTID2,Perfil de clasificación,Variables,Valores ausentes,Archivo
0,1,10004667,0-0-0-0,813,24,63_registro_entrada_demo_01.csv
1,2,10012085,0-0-0-1,813,24,63_registro_entrada_demo_02.csv
2,3,10015505,1-1-1-1,813,15,63_registro_entrada_demo_03.csv



CASO DE USO DEMOSTRATIVO


,Elemento,Valor
0,Usuario,Usuario autorizado de demostración
1,Credencial,CAM
2,QUESTID2 del ejemplo,10004667
3,Archivo seleccionado,63_registro_entrada_demo_01.csv
4,Registros,1
5,Variables,813


...


INFORME SELECCIONADO DESDE EL HISTORIAL


/home/cam/miniconda3/envs/TFM_Agentes/lib/python3.11/site-packages/IPython/core/display.py:452: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")



COMPROBACIONES DEL ACCESO AUTORIZADO


,Comprobación,Resultado
0,Se generan tres archivos CSV diferentes,True
1,Los tres registros son distintos del registro ...,True
2,Los tres registros presentan perfiles de clasi...,True
3,Cada CSV contiene exactamente un registro,True
4,Cada CSV contiene las 813 variables esperadas,True
5,Los tres CSV recuperados superan la validación...,True
6,Una solicitud sin credencial devuelve HTTP 401,True
7,Una credencial incorrecta devuelve HTTP 401,True
8,La credencial autorizada permite el acceso,True
9,El endpoint protegido de informe está disponible,True



Acceso autorizado y tres archivos CSV de ejemplo validados correctamente.


### Resultados

Se generan tres archivos CSV independientes, cada uno con un único registro y las 813 variables originales requeridas por el sistema. Los registros seleccionados son distintos del caso de referencia y representan tres perfiles de clasificación diferentes: `0-0-0-0`, `0-0-0-1` y `1-1-1-1`.

Los tres archivos superan la validación de entrada y producen correctamente un informe preventivo completo. Las tres ejecuciones requieren aproximadamente tres minutos cada una, en coherencia con el coste observado de las cuatro explicaciones SHAP.

Los informes se almacenan como `demo_01`, `demo_02` y `demo_03` y pueden recuperarse posteriormente desde el historial. La consulta de un informe persistido no vuelve a ejecutar `TFM_ML`, SHAP, RAG ni Mistral, sino que carga directamente la respuesta previamente validada.

Se comprueba además que los informes almacenados no contienen nuevamente el registro original con sus 813 variables, manteniendo el criterio de minimización de datos definido para el prototipo.

El acceso sin credencial o mediante una credencial incorrecta es rechazado con HTTP 401, mientras que la credencial académica `CAM` permite utilizar el flujo protegido. Las 23 comprobaciones realizadas resultan correctas.

## 8.6. Comprobación visual de la aplicación en navegador

Como última comprobación del prototipo se inicia temporalmente la aplicación Flask y se valida su disponibilidad mediante una petición real al endpoint de estado.

El servidor se ejecuta en segundo plano para no bloquear el notebook. Una vez comprobado que responde correctamente, se abre automáticamente una pestaña del navegador de Windows en `http://127.0.0.1:5000/acceso`.

La aplicación comienza en la pantalla de identificación. Tras introducir la credencial académica `CAM`, el usuario accede a un panel desde el que puede generar un nuevo informe a partir de uno de los CSV preparados, consultar los informes previamente almacenados o finalizar la demostración.

Esta comprobación complementa las pruebas realizadas mediante el cliente de Flask y la evaluación extremo a extremo con una validación visual en un navegador real.

Para evitar que el servicio permanezca activo accidentalmente, el servidor se cierra automáticamente después de diez minutos. También puede finalizarse de forma explícita desde el propio panel mediante la opción de cierre de la demostración.

In [39]:
# =============================================================================
# COMPROBACIÓN VISUAL DE LA APLICACIÓN EN NAVEGADOR
# =============================================================================
url_demo = 'http://127.0.0.1:5000/acceso'
url_health = 'http://127.0.0.1:5000/api/v1/health'
tiempo_servidor_demo = 600  # 10 minutos

# -----------------------------------------------------------------------------
# CIERRE DE UNA DEMOSTRACIÓN ANTERIOR
# -----------------------------------------------------------------------------
temporizador_anterior = globals().get('temporizador_servidor_demo')
servidor_anterior = globals().get('servidor_demo')

if temporizador_anterior is not None:
    temporizador_anterior.cancel()

if servidor_anterior is not None:
    detener_servidor_demo(servidor_anterior)

# -----------------------------------------------------------------------------
# INICIO DEL SERVIDOR
# -----------------------------------------------------------------------------
servidor_demo = make_server('127.0.0.1', 5000, app, threaded=True)

hilo_servidor_demo = threading.Thread(target=servidor_demo.serve_forever, daemon=True)
hilo_servidor_demo.start()

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
respuesta_servidor = None

for _ in range(20):
    try:
        respuesta_servidor = requests.get(url_health, timeout=1)

        if respuesta_servidor.status_code == 200:
            break

    except requests.RequestException:
        sleep(0.25)

if respuesta_servidor is None or respuesta_servidor.status_code != 200:
    detener_servidor_demo(servidor_demo)
    raise RuntimeError('No se ha podido iniciar correctamente la aplicación Flask.')

# -----------------------------------------------------------------------------
# CIERRE AUTOMÁTICO
# -----------------------------------------------------------------------------
temporizador_servidor_demo = threading.Timer(
    tiempo_servidor_demo, detener_servidor_demo, args=(servidor_demo,)
)
temporizador_servidor_demo.daemon = True
temporizador_servidor_demo.start()

# -----------------------------------------------------------------------------
# APERTURA AUTOMÁTICA EN EL NAVEGADOR
# -----------------------------------------------------------------------------
subprocess.Popen(
    ['cmd.exe', '/c', 'start', '', url_demo], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

print('\nCOMPROBACIÓN VISUAL DE LA APLICACIÓN')
print('Servidor Flask: OK')
print('Endpoint de estado: OK')
print(f'Pantalla inicial: {url_demo}')
print('Pestaña del navegador abierta automáticamente.')
print('El servidor se cerrará automáticamente en 10 minutos.')
print('También puede cerrarse desde el panel mediante FINALIZAR DEMOSTRACIÓN.')

127.0.0.1 - - [31/Aug/2026 21:23:12] "GET /api/v1/health HTTP/1.1" 200 -



COMPROBACIÓN VISUAL DE LA APLICACIÓN
Servidor Flask: OK
Endpoint de estado: OK
Pantalla inicial: http://127.0.0.1:5000/acceso
Pestaña del navegador abierta automáticamente.
El servidor se cerrará automáticamente en 10 minutos.
También puede cerrarse desde el panel mediante FINALIZAR DEMOSTRACIÓN.


### Resultados

La aplicación Flask se inicia correctamente en segundo plano y el endpoint de estado responde mediante HTTP 200.

Tras validar la disponibilidad del servicio, el notebook abre automáticamente una pestaña del navegador en la pantalla de identificación. La credencial académica permite acceder al panel principal, desde el que quedan disponibles la generación de informes, la consulta del historial y el cierre controlado de la demostración.

El servidor permanece disponible durante un máximo de diez minutos y puede finalizarse también desde la propia interfaz. Esta comprobación visual complementa las validaciones funcionales y extremo a extremo realizadas previamente sin modificar los resultados predictivos ya obtenidos.

## Síntesis de la sección 8 (memoria)

La solución se ha productivizado mediante una aplicación Flask que reutiliza íntegramente las capacidades validadas en las secciones anteriores. La capa de aplicación no reproduce lógica predictiva: las inferencias continúan ejecutándose en `TFM_ML`, mientras LangGraph, RAG, Mistral y los guardrails permanecen coordinados desde `TFM_Agentes`.

La respuesta integrada combina las cuatro predicciones, los 12 factores principales, una explicación ampliada para cada indicador, el informe preventivo y la trazabilidad. Plotly representa de forma determinista la comparación entre probabilidades y umbrales utilizando la misma identidad visual turquesa que la interfaz.

Flask expone 10 endpoints destinados tanto a interacción web como a integración programática. El prototipo incorpora una capa de acceso académico mediante la credencial ficticia `CAM`, mientras que un despliegue real requeriría HTTPS, autenticación corporativa mediante OAuth 2.0/OpenID Connect, autorización por roles y mecanismos adecuados de auditoría y gestión de sesiones.

La demostración utiliza tres CSV independientes, cada uno con un registro y 813 variables, seleccionados de forma determinista y con perfiles de clasificación distintos. Los tres generan correctamente informes completos. Estos informes quedan persistidos y pueden recuperarse desde un historial sin volver a ejecutar inferencia, SHAP, RAG o Mistral. Las variables originales del registro no se almacenan nuevamente en los informes.

Cada generación completa requiere aproximadamente tres minutos, debido principalmente al cálculo de las cuatro explicaciones SHAP. Como comprobación visual final, el notebook inicia temporalmente Flask, verifica su disponibilidad y abre automáticamente la pantalla de identificación en el navegador. El servicio se cierra de forma automática después de diez minutos o puede finalizarse desde el panel, evitando mantener un servidor local activo de forma indefinida.

**Tabla candidata:** tres casos de demostración, perfiles obtenidos y estado de generación de los informes.

**Figura candidata:** captura de la interfaz final y gráfico Plotly de probabilidades frente a umbrales.

**Destino:** MEMORIA + ANEXO. La memoria incluirá la arquitectura de productivización y una demostración visual resumida; endpoints, controles de acceso y pruebas completas se reservarán para el anexo.

# 9. Evaluación extremo a extremo

La última validación funcional comprueba el sistema desde el punto de entrada de la aplicación hasta la respuesta final que recibiría un sistema externo.

A diferencia de las validaciones parciales anteriores, esta evaluación no invoca directamente las funciones internas. La solicitud se realiza a través del endpoint Flask `/api/v1/report`, que debe recorrer validación de entrada, predicción, explicación SHAP de los cuatro indicadores, recuperación documental, generación estructurada, guardrails y construcción de la respuesta final.

Se utiliza nuevamente el registro de referencia para disponer de predicciones cerradas con las que comprobar la reproducibilidad de la API. Su utilización tiene exclusivamente finalidad confirmatoria: no se modifican modelos, variables, umbrales, prompts ni decisiones a partir de esta ejecución.

La evaluación se completa con solicitudes no válidas que deben ser rechazadas antes de alcanzar las etapas costosas del sistema.

## 9.1. Diseño y criterios de evaluación

Se definen los criterios que debe cumplir una ejecución completa. La evaluación combina reproducibilidad predictiva, integridad del informe, explicabilidad, trazabilidad, seguridad y comportamiento ante entradas incorrectas.

La llamada completa se realiza una única vez. Las comprobaciones posteriores reutilizan la respuesta obtenida para evitar repetir inferencia, SHAP o generación.

In [40]:
# =============================================================================
# DISEÑO Y CRITERIOS DE EVALUACIÓN
# =============================================================================
print('\nDISEÑO Y CRITERIOS DE EVALUACIÓN EXTREMO A EXTREMO')

plan_evaluacion_e2e = pd.DataFrame({
    'componente': [
        'Entrada', 'Predicción', 'Explicabilidad', 'Generación',
        'Trazabilidad', 'Robustez'
    ],
    'criterio': [
        'Registro completo con 813 variables',
        'Cuatro resultados coincidentes con la referencia',
        'Tres factores SHAP para cada indicador',
        'Informe estructurado aceptado por los guardrails',
        'Recorrido completo de los cinco nodos',
        'Rechazo controlado de entradas no válidas'
    ]
})

resumen_caso_e2e = pd.DataFrame({
    'elemento': [
        'QUESTID2 de referencia', 'Variables de entrada',
        'Variables objetivo', 'Factores esperados', 'Endpoint evaluado'
    ],
    'valor': [
        questid2_referencia, len(entrada_referencia.registro),
        len(targets), len(targets) * 3, '/api/v1/report'
    ]
})

tabla_plan = plan_evaluacion_e2e.rename(columns=renombrado_comun)

tabla_resumen = resumen_caso_e2e.rename(columns=renombrado_comun)

print('\nPLAN DE EVALUACIÓN')
display(tabla_plan)

print('\nCASO DE REFERENCIA')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(plan_evaluacion_e2e, '65_plan_evaluacion_e2e')

print('\nEvaluación extremo a extremo preparada correctamente.')


DISEÑO Y CRITERIOS DE EVALUACIÓN EXTREMO A EXTREMO

PLAN DE EVALUACIÓN


,Componente,Criterio
0,Entrada,Registro completo con 813 variables
1,Predicción,Cuatro resultados coincidentes con la referencia
2,Explicabilidad,Tres factores SHAP para cada indicador
3,Generación,Informe estructurado aceptado por los guardrails
4,Trazabilidad,Recorrido completo de los cinco nodos
5,Robustez,Rechazo controlado de entradas no válidas



CASO DE REFERENCIA


,Elemento,Valor
0,QUESTID2 de referencia,10004548
1,Variables de entrada,813
2,Variables objetivo,4
3,Factores esperados,12
4,Endpoint evaluado,/api/v1/report



Evaluación extremo a extremo preparada correctamente.


### Resultados

La evaluación extremo a extremo se define mediante seis dimensiones: entrada, predicción, explicabilidad, generación, trazabilidad y robustez.

Se utiliza el registro de referencia `QUESTID2 = 10004548`, formado por las 813 variables originales requeridas por el sistema. La evaluación debe devolver las cuatro variables objetivo, 12 factores explicativos —tres por indicador— y un informe estructurado a través del endpoint `/api/v1/report`.

La ejecución completa se realiza una única vez y sus resultados se reutilizan posteriormente para las comprobaciones de consistencia y robustez, evitando repetir operaciones costosas.

## 9.2. Ejecución real a través de la API

Se envía el registro completo al endpoint `/api/v1/report` utilizando el cliente de pruebas de Flask.

La solicitud atraviesa la misma interfaz HTTP que utilizaría una aplicación externa y activa el flujo completo: validación, predicción, cuatro explicaciones SHAP, recuperación documental, preparación del contexto, generación mediante Mistral, aplicación de guardrails y construcción de la respuesta.

La respuesta se conserva íntegramente para realizar posteriormente todas las comprobaciones sin repetir la ejecución.

In [41]:
# =============================================================================
# EJECUCIÓN REAL A TRAVÉS DE LA API
# =============================================================================
print('\nEJECUCIÓN EXTREMO A EXTREMO A TRAVÉS DE LA API', flush=True)

registro_e2e = json.loads(json.dumps(
    entrada_referencia.registro, ensure_ascii=False, default=convertir_json
))

print('[1/2] Ejecutando POST /api/v1/report...', flush=True)

inicio = perf_counter()

respuesta_http_e2e = cliente_flask.post('/api/v1/report', json={'registro': registro_e2e})

tiempo_e2e = perf_counter() - inicio
respuesta_e2e = respuesta_http_e2e.get_json(silent=True) or {}

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
if respuesta_http_e2e.status_code != 200:
    raise RuntimeError(
        f'La evaluación extremo a extremo ha fallado: {respuesta_e2e.get("mensaje", "sin detalle")}'
    )

print(f'      Respuesta recibida en {tiempo_e2e:.2f} s', flush=True)

print('[2/2] Recuperando los componentes de la respuesta...', flush=True)

predicciones_e2e = pd.DataFrame(respuesta_e2e['predicciones'])

factores_e2e = pd.DataFrame(respuesta_e2e['factores'])

traza_e2e = pd.DataFrame(respuesta_e2e['trazabilidad'])

informe_e2e = InformePreventivo.model_validate(respuesta_e2e['informe'])

tabla_predicciones = predicciones_e2e[[
    'descripcion', 'probabilidad', 'umbral', 'clasificacion', 'familia', 'modelo'
]].rename(columns=renombrado_comun).round({'Probabilidad': 6, 'Umbral': 3})

factores_por_objetivo_e2e = factores_e2e.groupby(
    ['variable_objetivo', 'descripcion_objetivo'], sort=False
).size().reset_index(name='factores')

tabla_factores = factores_por_objetivo_e2e.rename(columns=renombrado_comun)

tabla_traza = traza_e2e.rename(columns=renombrado_comun)

print('\nPREDICCIONES DEVUELTAS POR LA API')
display(tabla_predicciones)

print('\nFACTORES POR INDICADOR')
display(tabla_factores)

print('\nTRAZABILIDAD DE LA EJECUCIÓN EXTREMO A EXTREMO')
mostrar_tabla_completa(tabla_traza)

resumen_ejecucion_e2e = pd.DataFrame({
    'elemento': [
        'Código HTTP',
        'Estado',
        'Predicciones',
        'Factores',
        'Nodos recorridos',
        'Modelo generativo',
        'Versión del prompt',
        'Tiempo total extremo a extremo (s)'
    ],
    'valor': [
        respuesta_http_e2e.status_code,
        respuesta_e2e['estado'],
        len(predicciones_e2e),
        len(factores_e2e),
        len(traza_e2e),
        respuesta_e2e['modelo_generativo'],
        respuesta_e2e['version_prompt'],
        round(tiempo_e2e, 3)
    ]
})

tabla_resumen = resumen_ejecucion_e2e.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA EJECUCIÓN EXTREMO A EXTREMO')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_json(respuesta_e2e, '66_respuesta_e2e_api.json')
_ = guardar_csv(traza_e2e, '67_trazabilidad_e2e')

print('\nEjecución extremo a extremo completada.')


EJECUCIÓN EXTREMO A EXTREMO A TRAVÉS DE LA API


[1/2] Ejecutando POST /api/v1/report...


[1/5] Validando el registro de entrada...


[2/5] Ejecutando la tool predictiva local...


127.0.0.1 - - [31/Aug/2026 21:23:13] "GET /acceso HTTP/1.1" 200 -


...


TRAZABILIDAD DE LA EJECUCIÓN EXTREMO A EXTREMO


,Entrada,Nodo,Paso,Salida,Tiempo (s)
0,Registro recibido con 813 variables,validar_entrada,1,Entrada válida con 813 variables,0.001
1,Entrada validada con 813 variables,predecir_indicadores,2,4 predicciones estructuradas obtenidas,9.504
2,4 variables objetivo solicitadas,explicar_resultados,3,4 explicaciones y 6 variables principales,188.165
3,13 consultas documentales,recuperar_documentacion,4,13 documentos principales recuperados,0.252
4,"4 predicciones, 4 explicaciones y 13 documentos",preparar_contexto,5,Contexto estructurado preparado para la capa generativa,0.000



RESUMEN DE LA EJECUCIÓN EXTREMO A EXTREMO


,Elemento,Valor
0,Código HTTP,200
1,Estado,OK
2,Predicciones,4
3,Factores,12
4,Nodos recorridos,5
5,Modelo generativo,mistral-small-latest
6,Versión del prompt,v5
7,Tiempo total extremo a extremo (s),206.21



Ejecución extremo a extremo completada.


### Resultados

La ejecución real mediante `POST /api/v1/report` finaliza con HTTP 200 y estado `OK`, recorriendo el flujo completo desde la entrada de 813 variables hasta la construcción de la respuesta final.

La API devuelve las cuatro predicciones cerradas y exactamente 12 factores explicativos, tres por indicador. El modelo generativo utilizado es `mistral-small-latest` y la versión del prompt es `v5`.

La ejecución completa requiere aproximadamente tres minutos. La explicación SHAP de los cuatro indicadores concentra claramente la mayor parte del tiempo; la predicción requiere varios segundos, la recuperación documental menos de un segundo y la generación estructurada mediante Mistral del orden de varios segundos.

La trazabilidad conserva los cinco nodos del flujo: validación, predicción, explicación, recuperación documental y preparación del contexto. La respuesta obtenida se conserva para las comprobaciones siguientes sin repetir las operaciones costosas.

## 9.3. Consistencia y validación integral

La respuesta obtenida a través de Flask se compara con la salida de referencia previamente validada para el mismo registro.

La comparación predictiva exige conservar las cuatro variables objetivo, las probabilidades dentro de la tolerancia numérica establecida, los umbrales, las clasificaciones, las familias y los modelos seleccionados.

También se valida la estructura explicativa y generativa: 12 factores, tres por indicador, factores disponibles para los indicadores que no superan el umbral, contrato final del informe y recorrido completo de los cinco nodos.

Esta comprobación permite verificar que la capa de productivización no altera la lógica previamente cerrada.

In [42]:
# =============================================================================
# CONSISTENCIA Y VALIDACIÓN INTEGRAL
# =============================================================================
print('\nCONSISTENCIA Y VALIDACIÓN INTEGRAL')

referencia_e2e = pd.DataFrame(
    respuesta_demo_aplicacion['predicciones']
).set_index('variable_objetivo')

api_e2e = predicciones_e2e.set_index('variable_objetivo')

comparacion_predicciones_e2e = pd.DataFrame({
    'variable_objetivo': targets,
    'probabilidad_referencia': [referencia_e2e.loc[t, 'probabilidad'] for t in targets],
    'probabilidad_api': [api_e2e.loc[t, 'probabilidad'] for t in targets],
    'diferencia': [
        abs(referencia_e2e.loc[t, 'probabilidad'] - api_e2e.loc[t, 'probabilidad'])
        for t in targets
    ],
    'umbral': [api_e2e.loc[t, 'umbral'] for t in targets],
    'clasificacion': [api_e2e.loc[t, 'clasificacion'] for t in targets]
})

tabla_comparacion = comparacion_predicciones_e2e.rename(columns={
    **renombrado_comun, 'probabilidad_referencia': 'Probabilidad referencia',
    'probabilidad_api': 'Probabilidad API', 'diferencia': 'Diferencia'
}).round({
    'Probabilidad referencia': 6, 'Probabilidad API': 6, 'Diferencia': 8, 'Umbral': 3
})

print('\nCOMPARACIÓN DE PREDICCIONES')
display(tabla_comparacion)

conteo_factores_e2e = factores_e2e.groupby('variable_objetivo').size()

objetivos_no_superados_e2e = set(api_e2e.index[api_e2e['clasificacion'].eq(0)])

nodos_esperados_e2e = [
    'validar_entrada',
    'predecir_indicadores',
    'explicar_resultados',
    'recuperar_documentacion',
    'preparar_contexto'
]

respuesta_e2e_texto = json.dumps(respuesta_e2e, ensure_ascii=False, default=convertir_json)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_e2e = pd.DataFrame({
    'comprobacion': [
        'La API devuelve estado OK',
        'Se devuelven las cuatro variables objetivo',
        'Las probabilidades coinciden con la referencia',
        'Los umbrales coinciden con la referencia',
        'Las clasificaciones coinciden con la referencia',
        'Las familias y modelos coinciden con la referencia',
        'La respuesta contiene exactamente 12 factores',
        'Se mantienen tres factores por indicador',
        'Los indicadores que no superan el umbral conservan sus factores',
        'El informe mantiene los siete campos contractuales',
        'La trazabilidad recorre los cinco nodos en el orden previsto',
        'El registro original no se devuelve en la respuesta',
        'La respuesta no contiene credenciales',
        'La respuesta contiene explicación ampliada para los cuatro indicadores',
        'Las descripciones ampliadas no muestran códigos técnicos'
    ],
    'resultado': [
        respuesta_e2e['estado'] == 'OK',
        set(api_e2e.index) == set(targets),
        np.allclose(api_e2e.loc[targets, 'probabilidad'],
                    referencia_e2e.loc[targets, 'probabilidad'],
                    atol=tolerancia_probabilidad_bridge, rtol=0),
        np.allclose(api_e2e.loc[targets, 'umbral'], referencia_e2e.loc[targets, 'umbral'],
                    atol=0, rtol=0),
        api_e2e.loc[targets, 'clasificacion'].equals(referencia_e2e.loc[targets, 'clasificacion']),
        all(api_e2e.loc[t, ['familia', 'modelo']]
            .equals(referencia_e2e.loc[t, ['familia', 'modelo']]) for t in targets),
        len(factores_e2e) == 12,
        set(conteo_factores_e2e.index) == set(targets) and conteo_factores_e2e.eq(3).all(),
        objetivos_no_superados_e2e.issubset(set(conteo_factores_e2e.index)),
        set(informe_e2e.model_dump()) == set(InformePreventivo.model_fields),
        traza_e2e['nodo'].tolist() == nodos_esperados_e2e,
        'registro' not in respuesta_e2e,
        'MISTRAL_API_KEY' not in respuesta_e2e_texto and 'HF_TOKEN' not in respuesta_e2e_texto,
        set(factor['variable_objetivo']
            for factor in respuesta_e2e['detalle_explicabilidad']) == set(targets),
        all(factor['variable_origen'].casefold() not in factor['descripcion'].casefold()
            for factor in respuesta_e2e['detalle_explicabilidad'])
    ]
})

tabla_comprobaciones = comprobaciones_e2e.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES EXTREMO A EXTREMO')
display(tabla_comprobaciones)

if not comprobaciones_e2e['resultado'].all():
    raise ValueError('La validación integral extremo a extremo no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(comparacion_predicciones_e2e, '68_comparacion_predicciones_e2e')
_ = guardar_csv(comprobaciones_e2e, '69_comprobaciones_e2e')

print('\nConsistencia extremo a extremo validada correctamente.')


CONSISTENCIA Y VALIDACIÓN INTEGRAL

COMPARACIÓN DE PREDICCIONES


,Variable objetivo,Probabilidad referencia,Probabilidad API,Diferencia,Umbral,Clasificación
0,IRAMDEYR,0.826969,0.826969,0.0,0.500,1
1,IRSUICTHNK,0.565728,0.565728,0.0,0.509,1
2,IRSUIPLANYR,0.525724,0.525724,0.0,0.536,0
3,IRSUITRYYR,0.524104,0.524104,0.0,0.502,1



COMPROBACIONES EXTREMO A EXTREMO


,Comprobación,Resultado
0,La API devuelve estado OK,True
1,Se devuelven las cuatro variables objetivo,True
2,Las probabilidades coinciden con la referencia,True
3,Los umbrales coinciden con la referencia,True
4,Las clasificaciones coinciden con la referencia,True
5,Las familias y modelos coinciden con la refere...,True
6,La respuesta contiene exactamente 12 factores,True
7,Se mantienen tres factores por indicador,True
8,Los indicadores que no superan el umbral conse...,True
9,El informe mantiene los siete campos contractu...,True



Consistencia extremo a extremo validada correctamente.


### Resultados

Las cuatro probabilidades devueltas por la API coinciden exactamente con las utilizadas como referencia, con diferencia numérica `0,0`. También coinciden los cuatro umbrales, las clasificaciones, las familias y los modelos seleccionados.

La respuesta contiene exactamente 12 factores, tres para cada indicador. Planificación suicida conserva sus factores explicativos aunque su probabilidad no supera el umbral, confirmando que la explicabilidad no depende de la clasificación final.

El informe mantiene sus siete campos contractuales y la trazabilidad recorre los cinco nodos en el orden previsto. La respuesta tampoco devuelve el registro original ni expone credenciales.

La explicación ampliada está disponible para los cuatro indicadores y las descripciones visibles no incluyen los códigos técnicos de las variables.

Las 15 comprobaciones extremo a extremo resultan correctas.

## 9.4. Robustez y cierre de la evaluación

La evaluación se completa comprobando el comportamiento del endpoint de informe ante solicitudes incorrectas.

Se prueban dos errores diferenciados: ausencia completa del registro y presencia de un registro incompleto. En ambos casos la solicitud debe finalizar con HTTP 400 antes de alcanzar predicción, SHAP o generación.

Finalmente se comprueba que el servicio permanece operativo después de las solicitudes y se resume el resultado global de la evaluación extremo a extremo.

In [43]:
# =============================================================================
# ROBUSTEZ Y CIERRE DE LA EVALUACIÓN
# =============================================================================
print('\nROBUSTEZ Y CIERRE DE LA EVALUACIÓN')

respuesta_sin_registro = cliente_flask.post('/api/v1/report', json={})

registro_incompleto_e2e = dict(registro_e2e)
registro_incompleto_e2e.pop(vars_predictoras_modelado_dl[0])
respuesta_incompleta = cliente_flask.post(
    '/api/v1/report', json={'registro': registro_incompleto_e2e}
)

respuesta_health_final = cliente_flask.get('/api/v1/health')

mensaje_sin_registro = (respuesta_sin_registro.get_json() or {}).get('mensaje', '')
mensaje_incompleto = (respuesta_incompleta.get_json() or {}).get('mensaje', '')

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
comprobaciones_robustez_e2e = pd.DataFrame({
    'comprobacion': [
        'La ausencia de registro devuelve HTTP 400',
        'El registro incompleto devuelve HTTP 400',
        'Las dos respuestas proporcionan un mensaje controlado',
        'Los errores no exponen credenciales',
        'El servicio permanece operativo después de los errores',
        'Todas las comprobaciones funcionales extremo a extremo permanecen superadas'
    ],
    'resultado': [
        respuesta_sin_registro.status_code == 400,
        respuesta_incompleta.status_code == 400,
        bool(mensaje_sin_registro) and bool(mensaje_incompleto),
        all(
            clave not in f'{mensaje_sin_registro} {mensaje_incompleto}'
            for clave in ['MISTRAL_API_KEY', 'HF_TOKEN']
        ),
        respuesta_health_final.status_code == 200
        and respuesta_health_final.get_json()['estado'] == 'OK',
        comprobaciones_e2e['resultado'].all()
    ]
})

tabla_comprobaciones = comprobaciones_robustez_e2e.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES DE ROBUSTEZ')
display(tabla_comprobaciones)

resumen_evaluacion_e2e = pd.DataFrame({
    'elemento': [
        'Código HTTP de la ejecución completa',
        'Predicciones validadas',
        'Factores explicativos',
        'Comprobaciones funcionales superadas',
        'Comprobaciones de robustez superadas',
        'Tiempo total extremo a extremo (s)',
        'Resultado global'
    ],
    'valor': [
        respuesta_http_e2e.status_code,
        len(predicciones_e2e),
        len(factores_e2e),
        int(comprobaciones_e2e['resultado'].sum()),
        int(comprobaciones_robustez_e2e['resultado'].sum()),
        round(tiempo_e2e, 3),
        (
            'VALIDADO'
            if comprobaciones_e2e['resultado'].all()
            and comprobaciones_robustez_e2e['resultado'].all()
            else 'NO VALIDADO'
        )
    ]
})

tabla_resumen = resumen_evaluacion_e2e.rename(columns=renombrado_comun)

print('\nRESUMEN FINAL DE LA EVALUACIÓN')
display(tabla_resumen)

if not comprobaciones_robustez_e2e['resultado'].all():
    raise ValueError('La validación de robustez extremo a extremo no es correcta.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(comprobaciones_robustez_e2e, '70_comprobaciones_robustez_e2e')
_ = guardar_csv(resumen_evaluacion_e2e, '71_resumen_evaluacion_e2e')

print('\nEvaluación extremo a extremo validada correctamente.')


ROBUSTEZ Y CIERRE DE LA EVALUACIÓN
[1/5] Validando el registro de entrada...



COMPROBACIONES DE ROBUSTEZ


,Comprobación,Resultado
0,La ausencia de registro devuelve HTTP 400,True
1,El registro incompleto devuelve HTTP 400,True
2,Las dos respuestas proporcionan un mensaje con...,True
3,Los errores no exponen credenciales,True
4,El servicio permanece operativo después de los...,True
5,Todas las comprobaciones funcionales extremo a...,True



RESUMEN FINAL DE LA EVALUACIÓN


,Elemento,Valor
0,Código HTTP de la ejecución completa,200
1,Predicciones validadas,4
2,Factores explicativos,12
3,Comprobaciones funcionales superadas,15
4,Comprobaciones de robustez superadas,6
5,Tiempo total extremo a extremo (s),206.21
6,Resultado global,VALIDADO



Evaluación extremo a extremo validada correctamente.


### Resultados

Las solicitudes sin registro y con un registro incompleto son rechazadas correctamente mediante HTTP 400 antes de alcanzar las etapas costosas del flujo.

Los mensajes de error mantienen una estructura controlada y no exponen credenciales ni información sensible de configuración. Después de ambas solicitudes incorrectas, el endpoint de estado confirma que el servicio continúa operativo.

Las seis comprobaciones de robustez y las 15 comprobaciones funcionales de la evaluación resultan correctas.

La evaluación extremo a extremo queda por tanto validada globalmente, manteniendo cuatro predicciones, 12 factores explicativos y una respuesta estructurada reproducible desde la API.

## Síntesis de la sección 9 (memoria)

La evaluación extremo a extremo valida la solución desde el punto de entrada Flask hasta la respuesta final de la aplicación, sin invocar directamente las funciones internas durante la prueba principal.

El endpoint `/api/v1/report` recibe un registro con 813 variables y recorre validación, predicción, cuatro explicaciones SHAP, recuperación documental, preparación del contexto, generación estructurada, guardrails y construcción de la respuesta. La ejecución finaliza con HTTP 200 y mantiene las cuatro predicciones y los 12 factores esperados.

Las probabilidades, umbrales, clasificaciones, familias y modelos coinciden exactamente con la referencia previamente validada. El indicador de planificación suicida conserva sus factores explicativos aunque no supere el umbral, y la respuesta mantiene tanto el contrato del informe como la explicación ampliada de los cuatro indicadores.

La ejecución completa requiere aproximadamente tres minutos. SHAP concentra la mayor parte del tiempo, mientras que la predicción requiere varios segundos, RAG menos de un segundo y la generación mediante Mistral del orden de varios segundos.

Las 15 comprobaciones funcionales y las seis pruebas de robustez resultan correctas. Las entradas ausentes o incompletas se rechazan mediante HTTP 400 antes de alcanzar las operaciones costosas y el servicio permanece operativo después de los errores.

**Tabla candidata:** resumen de la ejecución extremo a extremo y comparación de las cuatro predicciones con la referencia.

**Figura candidata:** esquema extremo a extremo de la arquitectura o captura final del informe, evitando duplicar figuras ya utilizadas.

**Destino:** MEMORIA + ANEXO. La memoria recogerá la validación global, la reproducibilidad predictiva y el coste aproximado del flujo; las 21 comprobaciones completas y la trazabilidad detallada se reservarán para el anexo.

# 10. Conclusiones y preparación del cierre

La última sección consolida el resultado técnico del Notebook 05 después de completar satisfactoriamente su ejecución secuencial.

El cierre del notebook se mantiene separado de la congelación del entorno `TFM_Agentes`. En esta fase no se modifican `TFM_Agentes_environment.yml`, `TFM_Agentes_REPRODUCCION.md`, archivos lock ni hashes del entorno. Esa operación se realizará posteriormente, una vez cerrada la revisión documental del notebook.

La solución mantiene la separación entre la capa predictiva cerrada en `TFM_ML` y la capa de agentes y productivización ejecutada desde `TFM_Agentes`. La primera conserva modelos, preprocesamiento, inferencia y SHAP; la segunda incorpora contratos, bridge, tools, MCP, RAG, LangGraph, generación estructurada, guardrails, Flask, Plotly y las interfaces de utilización.

El objetivo de esta sección es consolidar las comprobaciones realizadas, inventariar las evidencias producidas y confirmar que el notebook reúne las condiciones de coherencia y reproducibilidad necesarias para su cierre definitivo.

## 10.1. Conclusiones técnicas

El Notebook 05 transforma la solución predictiva transferida desde el Notebook 04 en un prototipo funcional sin modificar modelos, variables, umbrales o decisiones predictivas previamente cerradas.

La inferencia se mantiene encapsulada en `TFM_ML` y se consume desde `TFM_Agentes` mediante un bridge JSON de solo lectura. La tool predictiva local constituye la vía principal de utilización y MCP permanece como alternativa interoperable validada.

La recuperación documental utiliza un corpus controlado e independiente de la predicción. LangGraph organiza un único flujo trazable y evita introducir una arquitectura multiagente artificial. Esta infraestructura de ejecución y control del agente —agent harness— integra estado, tools, routing, contratos Pydantic, bridge, RAG, guardrails y trazabilidad sin intervenir en la selección o modificación de los modelos predictivos.

La capa generativa recibe únicamente contexto previamente estructurado. Mistral se limita a generar resumen, interpretación y traducciones, mientras probabilidades, umbrales, clasificaciones, factores SHAP, orientaciones, limitaciones, fuentes y advertencias permanecen bajo control determinista. Los guardrails validan posteriormente el informe antes de aceptarlo.

Flask y Plotly completan la productivización mediante API, interfaz visual, control de acceso académico, generación de tres casos demostrativos e historial de informes. La evaluación extremo a extremo confirma finalmente que esta capa de aplicación conserva la solución previamente validada.

In [44]:
# =============================================================================
# CONCLUSIONES TÉCNICAS
# =============================================================================
print('\nCONCLUSIONES TÉCNICAS')

resumen_solucion_final = pd.DataFrame({
    'elemento': [
        'Variables objetivo',
        'Variables originales de entrada',
        'Predicciones validadas extremo a extremo',
        'Factores explicativos extremo a extremo',
        'Endpoints Flask',
        'Modelo generativo',
        'Versión del prompt',
        'Evaluación extremo a extremo',
        'Acceso autorizado'
    ],
    'valor': [
        len(targets),
        len(vars_predictoras_modelado_dl),
        len(predicciones_e2e),
        len(factores_e2e),
        len(endpoints_flask),
        modelo_mistral,
        version_prompt_informe,
        'VALIDADO' if comprobaciones_e2e['resultado'].all()
        and comprobaciones_robustez_e2e['resultado'].all() else 'REVISAR',
        'VALIDADO' if comprobaciones_acceso_autorizado['resultado'].all()
        else 'REVISAR'
    ]
})

tabla_resumen = resumen_solucion_final.rename(columns=renombrado_comun)

print('\nRESUMEN DE LA SOLUCIÓN FINAL')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_solucion_final, '72_resumen_solucion_final')

print('\nConclusiones técnicas consolidadas correctamente.')


CONCLUSIONES TÉCNICAS

RESUMEN DE LA SOLUCIÓN FINAL


,Elemento,Valor
0,Variables objetivo,4
1,Variables originales de entrada,813
2,Predicciones validadas extremo a extremo,4
3,Factores explicativos extremo a extremo,12
4,Endpoints Flask,10
5,Modelo generativo,mistral-small-latest
6,Versión del prompt,v5
7,Evaluación extremo a extremo,VALIDADO
8,Acceso autorizado,VALIDADO



Conclusiones técnicas consolidadas correctamente.


### Resultados

La solución final integra cuatro variables objetivo y mantiene una entrada general de 813 variables originales.

La evaluación extremo a extremo valida las cuatro predicciones y 12 factores explicativos mediante una aplicación Flask que dispone de 10 endpoints.

La capa generativa utiliza `mistral-small-latest` con el prompt `v5` y permanece restringida a las tareas definidas por el contrato generativo.

Tanto la evaluación extremo a extremo como el escenario de acceso autorizado quedan en estado `VALIDADO`.

El Notebook 05 cumple así su objetivo de transformar la solución predictiva cerrada en un prototipo funcional, explicable, controlado y utilizable desde una aplicación.

## 10.2. Consolidación de validaciones

Las comprobaciones distribuidas a lo largo del notebook se consolidan por bloques funcionales para disponer de una visión única del estado final de la solución después de la ejecución secuencial completa.

Este resumen no sustituye las tablas detalladas generadas en cada apartado. Su objetivo es confirmar de forma consolidada que los principales contratos y capacidades desarrollados permanecen validados antes de cerrar definitivamente el notebook.

In [45]:
# =============================================================================
# CONSOLIDACIÓN DE VALIDACIONES
# =============================================================================
print('\nCONSOLIDACIÓN DE VALIDACIONES')

bloques_validacion = [
    ('0. Preparación', pd.concat(
        [comprobaciones_entorno, comprobaciones_configuracion], ignore_index=True
    )),
    ('1. Transferencia', comprobaciones_recuperacion_transferencia),
    ('2. Contratos', comprobaciones_contratos),
    ('3. Bridge predictivo', comprobaciones_bridge_final),
    ('4. Tools y MCP', comprobaciones_decision_tools),
    ('5. RAG', comprobaciones_evaluacion_rag),
    ('6. LangGraph', comprobaciones_rutas_flujo),
    ('7. Generación y guardrails', pd.concat(
        [guardrails_informe, comprobaciones_finales_generacion], ignore_index=True
    )),
    ('8. Productivización', pd.concat(
        [comprobaciones_flask, comprobaciones_acceso_autorizado], ignore_index=True
    )),
    ('9. Evaluación extremo a extremo', pd.concat(
        [comprobaciones_e2e, comprobaciones_robustez_e2e], ignore_index=True
    ))
]

resumen_validaciones = pd.DataFrame([
    {
        'bloque': nombre,
        'comprobaciones': len(tabla),
        'superadas': int(tabla['resultado'].sum()),
        'estado': 'OK' if tabla['resultado'].all() else 'REVISAR'
    }
    for nombre, tabla in bloques_validacion
])

tabla_resumen = resumen_validaciones.rename(columns=renombrado_comun)

print('\nRESUMEN DE VALIDACIONES')
display(tabla_resumen)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
if not resumen_validaciones['estado'].eq('OK').all():
    raise ValueError('Existen bloques funcionales pendientes de revisión.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(resumen_validaciones, '73_resumen_validaciones_notebook05')

print('\nValidaciones funcionales consolidadas correctamente.')


CONSOLIDACIÓN DE VALIDACIONES

RESUMEN DE VALIDACIONES


,Bloque,Comprobaciones,Superadas,Estado
0,0. Preparación,12,12,OK
1,1. Transferencia,18,18,OK
2,2. Contratos,10,10,OK
3,3. Bridge predictivo,14,14,OK
4,4. Tools y MCP,5,5,OK
5,5. RAG,8,8,OK
6,6. LangGraph,8,8,OK
7,7. Generación y guardrails,22,22,OK
8,8. Productivización,39,39,OK
9,9. Evaluación extremo a extremo,21,21,OK



Validaciones funcionales consolidadas correctamente.


### Resultados

Los diez bloques funcionales incluidos en la consolidación presentan estado `OK`.

En conjunto se consolidan **157 comprobaciones**, todas ellas superadas. Estas validaciones abarcan preparación, transferencia, contratos, bridge predictivo, tools y MCP, RAG, LangGraph, generación y guardrails, productivización y evaluación extremo a extremo.

No se detecta ningún bloque parcialmente validado ni comprobaciones fallidas en el cierre funcional.

La consolidación confirma que las diferentes capas de la solución permanecen coherentes después del `Run All` completo.

## 10.3. Inventario y transferencia de evidencias

Se construye un inventario de las salidas presentes en las carpetas asociadas al Notebook 05 y de los módulos de productivización utilizados por la solución.

El inventario tiene finalidad documental y permite revisar la numeración, detectar duplicidades y localizar posibles archivos obsoletos. No se eliminan archivos automáticamente en esta fase.

También se define la transferencia editorial de las evidencias principales hacia memoria y anexos. La memoria conservará la arquitectura, los resultados esenciales y la demostración funcional; los contratos completos, prompts, pruebas exhaustivas, endpoints, logs y respuestas extensas se reservarán principalmente para anexos.

In [46]:
# =============================================================================
# INVENTARIO Y TRANSFERENCIA DE EVIDENCIAS
# =============================================================================
print('\nINVENTARIO Y TRANSFERENCIA DE EVIDENCIAS')

carpetas_inventario = {
    'Tabla': ruta_tablas,
    'Figura': ruta_figuras,
    'Informe': ruta_informes,
    'Log': ruta_logs
}

archivos_control_cierre = {
    '74_inventario_archivos_notebook05.csv',
    '75_transferencia_memoria_anexos.csv',
    '76_comprobaciones_finales_notebook05.csv'
}

inventario_archivos_05 = pd.DataFrame([
    {
        'tipo': tipo,
        'archivo': ruta.name,
        'ruta': str(ruta.relative_to(ruta_proyecto)),
        'tamano_kb': round(ruta.stat().st_size / 1024, 2)
    }
    for tipo, carpeta in carpetas_inventario.items() for ruta in sorted(carpeta.glob('*'))
    if ruta.is_file() and ruta.name not in archivos_control_cierre
])

inventario_codigo_05 = pd.DataFrame([
    {
        'tipo': 'Código',
        'archivo': ruta.name,
        'ruta': str(ruta.relative_to(ruta_proyecto)),
        'tamano_kb': round(ruta.stat().st_size / 1024, 2)
    }
    for ruta in [ruta_servicio_predictivo, ruta_servidor_mcp] if ruta.is_file()
])

inventario_archivos_05 = pd.concat(
    [inventario_archivos_05, inventario_codigo_05], ignore_index=True
)

resumen_inventario = inventario_archivos_05.groupby('tipo', as_index=False).agg(
    archivos=('archivo', 'count'), tamano_kb=('tamano_kb', 'sum')
)

tabla_resumen = resumen_inventario.rename(columns=renombrado_comun).round({'Tamaño (KB)': 2})

print('\nRESUMEN DEL INVENTARIO')
display(tabla_resumen)

transferencia_evidencias = pd.DataFrame({
    'evidencia': [
        'Arquitectura y flujo LangGraph',
        'Recuperación documental RAG',
        'Generación estructurada y guardrails',
        'Interfaz Flask y acceso autorizado',
        'Evaluación extremo a extremo',
        'Contratos, prompts, logs y pruebas detalladas'
    ],
    'destino': [
        'MEMORIA',
        'MEMORIA + ANEXO',
        'MEMORIA + ANEXO',
        'MEMORIA + ANEXO',
        'MEMORIA + ANEXO',
        'ANEXO'
    ]
})

tabla_transferencia = transferencia_evidencias.rename(columns=renombrado_comun)

print('\nTRANSFERENCIA DE EVIDENCIAS')
display(tabla_transferencia)

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(inventario_archivos_05, '74_inventario_archivos_notebook05')
_ = guardar_csv(transferencia_evidencias, '75_transferencia_memoria_anexos')

print('\nInventario y transferencia de evidencias preparados correctamente.')


INVENTARIO Y TRANSFERENCIA DE EVIDENCIAS

RESUMEN DEL INVENTARIO


,Tipo,Archivos,Tamaño (KB)
0,Código,2,23.04
1,Figura,1,9.24
2,Informe,5,109.03
3,Tabla,73,205.61



TRANSFERENCIA DE EVIDENCIAS


,Evidencia,Destino
0,Arquitectura y flujo LangGraph,MEMORIA
1,Recuperación documental RAG,MEMORIA + ANEXO
2,Generación estructurada y guardrails,MEMORIA + ANEXO
3,Interfaz Flask y acceso autorizado,MEMORIA + ANEXO
4,Evaluación extremo a extremo,MEMORIA + ANEXO
5,"Contratos, prompts, logs y pruebas detalladas",ANEXO



Inventario y transferencia de evidencias preparados correctamente.


### Resultados

El inventario final identifica 81 archivos asociados a las evidencias del Notebook 05: 2 archivos de código, 1 figura, 5 informes y 73 tablas.

El volumen conjunto es reducido, del orden de unos cientos de kilobytes, al no persistirse nuevamente los modelos predictivos ni los registros completos utilizados para cada informe.

La transferencia editorial asigna a la memoria la arquitectura y el flujo LangGraph, mientras que RAG, generación y guardrails, interfaz Flask, acceso autorizado y evaluación extremo a extremo se distribuyen entre memoria y anexos según su nivel de detalle.

Los contratos completos, prompts, logs y pruebas exhaustivas se reservan para anexos, evitando sobrecargar la memoria principal con información de implementación.

## 10.4. Reproducibilidad y comprobaciones finales

La última comprobación verifica las condiciones de reproducibilidad y coherencia del Notebook 05 después de completar satisfactoriamente su ejecución secuencial, sin modificar ni congelar todavía el entorno `TFM_Agentes`.

Se comprueban el entorno activo, la coincidencia de versiones, la disponibilidad de los archivos de reproducción y del smoke test, la transferencia completa desde el Notebook 04, la separación respecto a `TFM_ML`, la protección del archivo `.env`, la existencia de los módulos de productivización y el estado de todas las validaciones funcionales.

También se exige la presencia de los tres CSV independientes destinados a la demostración de acceso autorizado y del inventario final generado por el notebook.

La congelación definitiva de `TFM_Agentes` se realizará fuera del notebook una vez concluida su revisión documental.

In [47]:
# =============================================================================
# REPRODUCIBILIDAD Y COMPROBACIONES FINALES
# =============================================================================
print('\nREPRODUCIBILIDAD Y COMPROBACIONES FINALES')

csv_demo_disponibles = [
    ruta_tablas / f'63_registro_entrada_demo_{numero:02d}.csv'
    for numero in range(1, 4)
]

comprobaciones_finales_notebook05 = pd.DataFrame({
    'comprobacion': [
        'El kernel pertenece a TFM_Agentes',
        'Las versiones fijadas permanecen validadas',
        'Los archivos del entorno están disponibles',
        'La transferencia desde el Notebook 04 está disponible',
        'TFM_Agentes mantiene aisladas las dependencias predictivas',
        'El archivo .env mantiene permisos 600',
        'El archivo .env está ignorado y no versionado',
        'Los módulos predictivo y MCP están disponibles',
        'Los tres CSV independientes de demostración están disponibles',
        'Todos los bloques funcionales permanecen validados',
        'El inventario contiene los archivos generados por el notebook'
    ],
    'resultado': [
        nombre_entorno == 'TFM_Agentes',
        tabla_versiones['estado'].eq('OK').all(),
        tabla_archivos_entorno['disponible'].all(),
        tabla_archivos_transferencia['disponible'].all(),
        all(dependencias_aisladas.values()),
        permisos_env == '0o600',
        env_ignorado_git and env_no_versionado,
        ruta_servicio_predictivo.is_file() and ruta_servidor_mcp.is_file(),
        len(csv_demo_disponibles) == 3 and all(ruta.is_file() for ruta in csv_demo_disponibles),
        resumen_validaciones['estado'].eq('OK').all(),
        not inventario_archivos_05.empty
    ]
})

tabla_comprobaciones = comprobaciones_finales_notebook05.rename(columns=renombrado_comun)

print('\nCOMPROBACIONES FINALES DEL NOTEBOOK')
display(tabla_comprobaciones)

# -----------------------------------------------------------------------------
# COMPROBACIÓN
# -----------------------------------------------------------------------------
estado_notebook05 = (
    'VALIDACIÓN INTERNA SUPERADA'
    if comprobaciones_finales_notebook05['resultado'].all() else 'REVISAR'
)

print(f'\nEstado del Notebook 05: {estado_notebook05}')

if not comprobaciones_finales_notebook05['resultado'].all():
    raise ValueError('Las comprobaciones finales del Notebook 05 no son correctas.')

# -----------------------------------------------------------------------------
# GUARDADO
# -----------------------------------------------------------------------------
_ = guardar_csv(comprobaciones_finales_notebook05, '76_comprobaciones_finales_notebook05')

print('\nEl entorno TFM_Agentes no se modifica ni se congela desde este notebook.')


REPRODUCIBILIDAD Y COMPROBACIONES FINALES



COMPROBACIONES FINALES DEL NOTEBOOK


,Comprobación,Resultado
0,El kernel pertenece a TFM_Agentes,True
1,Las versiones fijadas permanecen validadas,True
2,Los archivos del entorno están disponibles,True
3,La transferencia desde el Notebook 04 está dis...,True
4,TFM_Agentes mantiene aisladas las dependencias...,True
5,El archivo .env mantiene permisos 600,True
6,El archivo .env está ignorado y no versionado,True
7,Los módulos predictivo y MCP están disponibles,True
8,Los tres CSV independientes de demostración es...,True
9,Todos los bloques funcionales permanecen valid...,True



Estado del Notebook 05: VALIDACIÓN INTERNA SUPERADA

El entorno TFM_Agentes no se modifica ni se congela desde este notebook.


### Resultados

Las once comprobaciones finales del notebook resultan correctas.

Se confirma que el kernel pertenece a `TFM_Agentes`, las versiones fijadas permanecen validadas, los archivos de reproducción están disponibles y la transferencia procedente del Notebook 04 continúa accesible.

También se verifica el aislamiento de las dependencias predictivas en `TFM_ML`, la protección del archivo `.env`, la disponibilidad de los módulos predictivo y MCP y la existencia de los tres CSV independientes de demostración.

Todos los bloques funcionales permanecen validados y el inventario contiene las evidencias generadas durante la ejecución.

El estado final del notebook queda establecido como **`VALIDACIÓN INTERNA SUPERADA`**. Esta validación cierra funcionalmente el Notebook 05, aunque la congelación definitiva de `TFM_Agentes` se realizará posteriormente fuera del notebook.

## Síntesis de la sección 10 (memoria)

El Notebook 05 completa la transformación de la solución predictiva desarrollada en los notebooks anteriores en un prototipo funcional de extremo a extremo. La arquitectura conserva estrictamente la separación entre `TFM_ML`, responsable de modelos, preprocesamiento, inferencia y SHAP, y `TFM_Agentes`, responsable de validación, acceso a capacidades, recuperación documental, flujo, generación y productivización.

La capacidad predictiva se consume mediante una tool local como mecanismo principal y mediante MCP como alternativa interoperable validada. LangGraph coordina el flujo dentro de la infraestructura de ejecución y control del agente (agent harness), que integra estado, tools, routing, contratos Pydantic, bridge, RAG, guardrails y trazabilidad sin modificar modelos, variables, umbrales o decisiones predictivas.

La capa RAG recupera documentación controlada procedente del codebook NSDUH 2024 y de las limitaciones metodológicas del proyecto. Mistral recibe únicamente contexto minimizado y genera resumen, interpretación y traducciones; el resto del informe se reconstruye de forma determinista y se valida mediante guardrails.

La aplicación Flask integra las cuatro predicciones, 12 factores principales, explicabilidad ampliada, Plotly, acceso autorizado y un historial persistente. Tres registros independientes de demostración producen correctamente tres informes, que pueden consultarse posteriormente sin repetir las operaciones predictivas o generativas y sin almacenar nuevamente las 813 variables originales.

La evaluación extremo a extremo reproduce exactamente las cuatro predicciones de referencia y supera 15 comprobaciones funcionales y seis pruebas de robustez. La ejecución completa requiere aproximadamente tres minutos, concentrándose el coste principalmente en las cuatro explicaciones SHAP.

La consolidación final reúne 157 comprobaciones superadas en los diez bloques evaluados y once comprobaciones adicionales de cierre. El estado interno del Notebook 05 queda establecido como **`VALIDACIÓN INTERNA SUPERADA`**.

Con este resultado queda funcionalmente cerrada la implementación técnica del Notebook 05: desde el registro original NSDUH hasta una predicción multisalida explicable y su presentación mediante un informe preventivo controlado.

**Tabla candidata:** resumen global de la arquitectura y validaciones del sistema.

**Figura candidata:** arquitectura extremo a extremo definitiva del TFM, desde el registro NSDUH hasta el informe preventivo.

**Destino:** MEMORIA. Las tablas de comprobaciones, contratos, logs, prompts, inventarios y respuestas completas se trasladarán a los anexos.